# Emission-aware PtX scheduling and sold product CI tracking

**Publication support notebook for:**  
*Tracking Product Carbon Intensity in Power-to-X Plants: An Optimization Framework for Emission-Aware Hydrogen, Ammonia, and Methanol Production*

This notebook is a cleaned, paper-aligned extraction from the local working notebook. It keeps the core computational workflow only:

1. **Tier 1** generation mix forecasting using the final ENTSO-E.
2. **Tier 2** emission signal construction: AEF, UpRamp MEF, and MSDR Soft MEF.
3. **Optimization** exogenous input preparation using GFS-based local renewable profiles.
4. **Rolling horizon optimization** using one core block for each model class: M0, M1, and M2.
5. **Representative/stress week selection** used for diagnostic interpretation.
6. **Ex-post sold product CI accounting** for H₂, NH₃, and MeOH.



## Notes

- the ENTSO-E API token was replaced by a placeholder.
- Outputs were cleared to keep the GitHub notebook lightweight.
- Some cells still use Colab-style paths such as `/content/...`; update these paths if running locally.
- Gurobi is required for the optimization cells.
- csv files will be given upon request



# 1. Tier 1 — ENTSO-E generation mix forecasting

The prep cell builds the hourly master feature tables, performs the chronological split, applies scaling, and creates the 24→24 windows used by the technology specific ANN models.


## 1.1 Prep block — fetch, process, split, scale, and create windows



In [ ]:
# =========================
# prep cell – ENTSO-E Fetch + Process + Split + Scale + Sliding Windows (2022–2024)
# =========================

import os, time, traceback, warnings
import numpy as np
import pandas as pd
from functools import reduce
from entsoe import EntsoePandasClient
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings("ignore", category=DeprecationWarning, module="jupyter_client.session")

# --------- CONFIG ----------
API_KEY       = "PUT_YOUR_ENTSOE_API_KEY_HERE"
AREA          = "ES"                   # Spain
TZ_ENTSOE     = "Europe/Brussels"

YEARS_TRAIN   = [2022, 2023]
YEAR_TEST     = 2024

DATA_PROC_DIR = "/content/data_entsoe_es_processed"

# path to MIBGAS day-ahead gas price Excel
MIBGAS_GAS_PATH = "/content/migbas/da_gas_price_migbasPVB_EURperMWh.xlsx"

VAL_DAYS      = 60   # last N days of 2023 as validation
TRAIN_WIN_H   = 24   # sliding window input hours
LABEL_WIN_H   = 24   # sliding window horizon hours

os.makedirs(DATA_PROC_DIR, exist_ok=True)

print("[INFO] Working dir:", os.getcwd())
print("[INFO] PROC  dir  :", DATA_PROC_DIR)

client = EntsoePandasClient(api_key=API_KEY)

# --------- COMMON HELPERS ----------

def _to_utc_hourly(df_like, tz_src=TZ_ENTSOE) -> pd.DataFrame:
    """
    Accepts Series or DataFrame from entsoe-py.
    Converts to hourly UTC DataFrame.
    DOES NOT drop all-NaN rows (we want to see missing hours).
    Resampling to 1H also fixes 15-minute data.
    """
    if df_like is None or len(df_like) == 0:
        return pd.DataFrame()

    # Series → DataFrame
    if isinstance(df_like, pd.Series):
        df = df_like.to_frame(name=df_like.name if df_like.name else "value")
    else:
        df = df_like.copy()

    # Ensure datetime index
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index, errors="coerce")

    # Localize to ENTSO-E market time, then convert to UTC
    if df.index.tz is None:
        df.index = df.index.tz_localize(tz_src, nonexistent="shift_forward", ambiguous="NaT")
    else:
        df = df.tz_convert(tz_src)
    df = df.tz_convert("UTC")

    # Cast numeric
    for c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # Resample to hourly mean in UTC; keep NaN rows
    df = df.resample("1h").mean()

    return df


def add_calendar_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add calendar features based on datetime_utc index (UTC)."""
    df = df.copy()
    idx = df.index

    df["hour"] = idx.hour
    df["dow"] = idx.dayofweek
    df["month"] = idx.month

    doy = idx.dayofyear
    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24.0)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24.0)
    df["doy_sin"]  = np.sin(2 * np.pi * doy / 365.25)
    df["doy_cos"]  = np.cos(2 * np.pi * doy / 365.25)

    df["is_weekend"] = df["dow"].isin([5, 6]).astype(int)

    # I did not add spain holidays
    df["is_holiday"] = 0

    return df


TECH_MAP = {
    "Biomass":                    "gen_biomass_mw",
    "Fossil Gas":                 "gen_fossil_gas_mw",
    "Fossil Hard coal":           "gen_fossil_hard_coal_mw",
    "Fossil Oil":                 "gen_fossil_oil_mw",
    "Hydro Pumped Storage":       "gen_hydro_pumped_storage_mw",
    "Hydro Run-of-river and poundage": "gen_hydro_run_of_river_mw",
    "Hydro Water Reservoir":      "gen_hydro_water_reservoir_mw",
    "Nuclear":                    "gen_nuclear_mw",
    "Solar":                      "gen_solar_mw",
    "Wind Onshore":               "gen_wind_onshore_mw",
    "Waste":                      "gen_waste_mw",
    "Other":                      "gen_other_mw",
    "Other renewable":            "gen_other_renewable_mw",
}

def _year_local_bounds(year: int):
    """
    Choose start/end in Europe/Brussels so that AFTER tz_convert to UTC and
    entsoe's internal reindexing, we cover exactly the UTC range
    [year-01-01 00:00, year-12-31 23:00].
    """
    start = pd.Timestamp(f"{year}-01-01 01:00", tz=TZ_ENTSOE)
    end   = pd.Timestamp(f"{year+1}-01-01 01:00", tz=TZ_ENTSOE)
    return start, end

def _iter_year_chunks(year: int, chunk_days: int = 30):
    """
    Iterate [start, end) windows in TZ_ENTSOE that cover the whole year,
    split into chunks of <= chunk_days.
    """
    start_year, end_year = _year_local_bounds(year)
    cur = start_year
    while cur < end_year:
        nxt = min(cur + pd.Timedelta(days=chunk_days), end_year)
        yield cur, nxt
        cur = nxt


# --------- CSV CACHE WRAPPER (per-year, per-signal) ----------

def cached_fetch_hourly(csv_path: str, fetch_fn, *args, **kwargs) -> pd.DataFrame:
    """
    If csv_path exists, load and return it (expects 'datetime_utc' column).
    Otherwise call fetch_fn(*args, **kwargs), save to csv_path (with datetime_utc),
    and return the DataFrame with datetime index.
    """
    if os.path.exists(csv_path):
        print(f"[CACHE] Using existing file (no API call): {csv_path}")
        df = pd.read_csv(csv_path, parse_dates=["datetime_utc"])
        df = df.set_index("datetime_utc").sort_index()
        return df

    print(f"[CACHE] No file at {csv_path} – calling ENTSO-E API once and saving.")
    df = fetch_fn(*args, **kwargs)
    if df is None or df.empty:
        print(f"[CACHE] WARNING: fetched empty df for {csv_path}")
        return df

    df = df.sort_index()
    df.index.name = "datetime_utc"
    df.reset_index().to_csv(csv_path, index=False)
    print(f"[CACHE] Saved fetched data -> {csv_path}")
    return df


# ---------LOAD MIBGAS DAY-AHEAD GAS PRICE & MAKE HOURLY ----------

def load_da_gas_price_mibgas(path: str = MIBGAS_GAS_PATH) -> pd.DataFrame:
    """
    Load daily MIBGAS PVB day-ahead gas price and expand to hourly UTC.

    Expected columns in Excel:
      - 'delivery_day'
      - 'da_gas_price_migbasPVB_EURperMWh'
    """
    if not os.path.exists(path):
        print(f"[GAS_DA] File not found at {path}. Skipping gas DA feature.")
        return pd.DataFrame()

    df = pd.read_excel(path)
    if "delivery_day" not in df.columns or "da_gas_price_migbasPVB_EURperMWh" not in df.columns:
        raise ValueError("[GAS_DA] Expected columns 'delivery_day' and "
                         "'da_gas_price_migbasPVB_EURperMWh' not found in MIBGAS file.")

    df["delivery_day"] = pd.to_datetime(df["delivery_day"])
    df = df.sort_values("delivery_day")

    # Daily series indexed by delivery_day
    daily = df.set_index("delivery_day")[["da_gas_price_migbasPVB_EURperMWh"]].copy()

    # Expand each day to 24 hourly rows with same price
    n_days = len(daily)
    if n_days == 0:
        print("[GAS_DA] Empty daily gas price data.")
        return pd.DataFrame()

    hourly_index = pd.date_range(
        start=daily.index.min(),
        end=daily.index.max() + pd.Timedelta(hours=23),
        freq="1H",
        tz="UTC",
    )
    hourly = daily.loc[daily.index.repeat(24)].copy()
    hourly.index = hourly_index
    hourly.index.name = "datetime_utc"

    print(f"[GAS_DA] Loaded MIBGAS gas DA prices: daily_rows={n_days}, hourly_rows={len(hourly)}, "
          f"range={hourly.index.min()} -> {hourly.index.max()}")

    # Save hourly gas DA feature as CSV for inspection
    gas_csv_path = os.path.join(DATA_PROC_DIR, "da_gas_price_migbasPVB_EURperMWh_hourly.csv")
    hourly.reset_index().to_csv(gas_csv_path, index=False)
    print(f"[GAS_DA] Saved hourly gas DA price -> {gas_csv_path}")

    return hourly


# --------- FETCHERS ----------
# the entso-e platform updated early 2026, so you could easy download the csv files directly, and this code might have issues in fetching correct timeframes, please be aware
def fetch_gen_actual(country_code: str, year: int) -> pd.DataFrame:
    """
    Actual Generation per Production Type → hourly UTC, only selected techs.
    Fetched in 30-day chunks.
    """
    dfs = []

    for start, end in _iter_year_chunks(year, chunk_days=30):
        try:
            raw_part = client.query_generation(country_code, start=start, end=end, psr_type=None)
        except Exception as e:
            print(f"[GEN_ACT] Error {country_code} {year} chunk {start} -> {end}: {e}")
            continue

        if raw_part is None or len(raw_part) == 0:
            print(f"[GEN_ACT] EMPTY chunk {country_code} {year} {start} -> {end}")
            continue

        df = raw_part.copy()

        # Flatten multiindex and keep "Actual Aggregated" if present
        if isinstance(df.columns, pd.MultiIndex):
            agg_level = None
            for lvl in range(df.columns.nlevels):
                vals = set(map(str, df.columns.get_level_values(lvl)))
                if "Actual Aggregated" in vals or "Actual Consumption" in vals:
                    agg_level = lvl
                    break

            if agg_level is not None:
                cols_keep = [c for c in df.columns if str(c[agg_level]) == "Actual Aggregated"]
                df = df[cols_keep]
                tech_levels = [i for i in range(df.columns.nlevels) if i != agg_level]
                df = df.T.groupby(level=tech_levels[-1]).sum(min_count=1).T
            else:
                df = df.T.groupby(level=-1).sum(min_count=1).T

        # Keep only technologies we care about, rename to gen_* cols
        df.columns = [str(c).strip() for c in df.columns]
        keep_cols = {k: v for k, v in TECH_MAP.items() if k in df.columns}
        if not keep_cols:
            print(f"[GEN_ACT] No mapped techs found in chunk {start} -> {end}")
            time.sleep(5.0)
            continue

        df = df[list(keep_cols.keys())]
        df = df.rename(columns=keep_cols)

        df_part = _to_utc_hourly(df)
        dfs.append(df_part)
        time.sleep(5.0)

    if not dfs:
        print(f"[GEN_ACT] EMPTY for {country_code} {year} (no valid chunks)")
        return pd.DataFrame()

    df_all = pd.concat(dfs, axis=0).sort_index()
    df_all = df_all[~df_all.index.duplicated(keep="first")]

    # Clip strictly to UTC year bounds
    utc_start = pd.Timestamp(f"{year}-01-01 00:00", tz="UTC")
    utc_end   = pd.Timestamp(f"{year+1}-01-01 00:00", tz="UTC")
    df_all = df_all.loc[(df_all.index >= utc_start) & (df_all.index < utc_end)]

    print(f"[GEN_ACT] {year}: shape={df_all.shape}, cols={list(df_all.columns)}, "
          f"range={df_all.index.min()} -> {df_all.index.max()}")
    return df_all


def fetch_load_da(country_code: str, year: int) -> pd.DataFrame:
    """Day Ahead Total Load Forecast (UTC), fetched in 30-day chunks."""
    dfs = []

    for start, end in _iter_year_chunks(year, chunk_days=30):
        try:
            s_part = client.query_load_forecast(country_code, start=start, end=end)
        except Exception as e:
            print(f"[LOAD_DA] Error {country_code} {year} chunk {start} -> {end}: {e}")
            continue

        if s_part is None or len(s_part) == 0:
            print(f"[LOAD_DA] EMPTY chunk {start} -> {end}")
            time.sleep(5.0)
            continue

        df_part = _to_utc_hourly(s_part)
        dfs.append(df_part)
        time.sleep(5.0)

    if not dfs:
        print(f"[LOAD_DA] EMPTY for {country_code} {year} (no chunks)")
        return pd.DataFrame()

    df = pd.concat(dfs, axis=0).sort_index()
    df = df[~df.index.duplicated(keep="first")]

    utc_start = pd.Timestamp(f"{year}-01-01 00:00", tz="UTC")
    utc_end   = pd.Timestamp(f"{year+1}-01-01 00:00", tz="UTC")
    df = df.loc[(df.index >= utc_start) & (df.index < utc_end)]

    if df.shape[1] > 1:
        df = df.iloc[:, :1]
    df.columns = ["da_total_load_mw"]

    print(f"[LOAD_DA] {year}: shape={df.shape}, range={df.index.min()} -> {df.index.max()}")
    return df


def fetch_gen_da_total(country_code: str, year: int) -> pd.DataFrame:
    """Day-ahead total generation forecast (ENTSOE returns total, not per tech)."""
    dfs = []

    for start, end in _iter_year_chunks(year, chunk_days=30):
        try:
            raw_part = client.query_generation_forecast(country_code, start=start, end=end)
        except Exception as e:
            print(f"[GEN_DA] Error {country_code} {year} chunk {start} -> {end}: {e}")
            continue

        if raw_part is None or len(raw_part) == 0:
            print(f"[GEN_DA] EMPTY chunk {start} -> {end}")
            time.sleep(5.0)
            continue

        if isinstance(raw_part, pd.Series):
            df_tmp = raw_part.to_frame("da_gen_total_mw")
        else:
            df_tmp = raw_part.sum(axis=1).to_frame("da_gen_total_mw")

        df_part = _to_utc_hourly(df_tmp)
        dfs.append(df_part)
        time.sleep(5.0)

    if not dfs:
        print(f"[GEN_DA] EMPTY for {country_code} {year} (no chunks)")
        return pd.DataFrame()

    df = pd.concat(dfs, axis=0).sort_index()
    df = df[~df.index.duplicated(keep="first")]

    utc_start = pd.Timestamp(f"{year}-01-01 00:00", tz="UTC")
    utc_end   = pd.Timestamp(f"{year+1}-01-01 00:00", tz="UTC")
    df = df.loc[(df.index >= utc_start) & (df.index < utc_end)]

    print(f"[GEN_DA] {year}: shape={df.shape}, range={df.index.min()} -> {df.index.max()}")
    return df


def fetch_wind_solar_da(country_code: str, year: int) -> pd.DataFrame:
    """Day-ahead forecast for wind + solar (total) in UTC, 30-day chunks."""
    dfs = []

    for start, end in _iter_year_chunks(year, chunk_days=30):
        try:
            raw_part = client.query_wind_and_solar_forecast(country_code, start=start, end=end)
        except Exception as e:
            print(f"[WS_DA] Error {country_code} {year} chunk {start} -> {end}: {e}")
            continue

        if raw_part is None or len(raw_part) == 0:
            print(f"[WS_DA] EMPTY chunk {start} -> {end}")
            time.sleep(5.0)
            continue

        df_part = _to_utc_hourly(raw_part)
        dfs.append(df_part)
        time.sleep(5.0)

    if not dfs:
        print(f"[WS_DA] EMPTY for {country_code} {year} (no chunks)")
        return pd.DataFrame()

    df = pd.concat(dfs, axis=0).sort_index()
    df = df[~df.index.duplicated(keep="first")]

    utc_start = pd.Timestamp(f"{year}-01-01 00:00", tz="UTC")
    utc_end   = pd.Timestamp(f"{year+1}-01-01 00:00", tz="UTC")
    df = df.loc[(df.index >= utc_start) & (df.index < utc_end)]

    cols = list(df.columns)
    wind_cols  = [c for c in cols if "Wind"  in str(c)]
    solar_cols = [c for c in cols if "Solar" in str(c)]

    out = pd.DataFrame(index=df.index)
    if wind_cols:
        out["da_wind_total_mw"] = df[wind_cols].sum(axis=1)
    if solar_cols:
        out["da_solar_total_mw"] = df[solar_cols].sum(axis=1)

    print(f"[WS_DA] {year}: shape={out.shape}, cols={list(out.columns)}, "
          f"range={out.index.min()} -> {out.index.max()}")
    return out


def fetch_exchanges_da(country_from: str, country_to: str, year: int) -> pd.DataFrame:
    """
    Day-ahead scheduled commercial exchanges between ES and neighbor, in UTC.
    Positive series is export from 'country_from' to 'country_to'.
    """
    dfs = []

    for start, end in _iter_year_chunks(year, chunk_days=30):
        try:
            s_part = client.query_scheduled_exchanges(country_from, country_to, start=start, end=end)
        except Exception as e:
            print(f"[XCH_DA] Error {country_from}-{country_to} {year} chunk {start} -> {end}: {e}")
            continue

        if s_part is None or len(s_part) == 0:
            print(f"[XCH_DA] EMPTY chunk {country_from}-{country_to} {start} -> {end}")
            time.sleep(5.0)
            continue

        df_part = _to_utc_hourly(s_part)
        if df_part.shape[1] > 1:
            df_part = df_part.iloc[:, :1]
        df_part.columns = ["flow_mw"]

        dfs.append(df_part)
        time.sleep(5.0)

    if not dfs:
        print(f"[XCH_DA] EMPTY for {country_from}-{country_to} {year} (no chunks)")
        return pd.DataFrame()

    df = pd.concat(dfs, axis=0).sort_index()
    df = df[~df.index.duplicated(keep="first")]

    utc_start = pd.Timestamp(f"{year}-01-01 00:00", tz="UTC")
    utc_end   = pd.Timestamp(f"{year+1}-01-01 00:00", tz="UTC")
    df = df.loc[(df.index >= utc_start) & (df.index < utc_end)]

    flow = df["flow_mw"]
    imports = (-flow).clip(lower=0)
    exports = ( flow).clip(lower=0)

    if country_to == "FR":
        out = pd.DataFrame({
            "da_import_fr_mw": imports,
            "da_export_fr_mw": exports,
        }, index=df.index)
    elif country_to == "PT":
        out = pd.DataFrame({
            "da_import_pt_mw": imports,
            "da_export_pt_mw": exports,
        }, index=df.index)
    else:
        out = pd.DataFrame({
            f"da_import_{country_to.lower()}_mw": imports,
            f"da_export_{country_to.lower()}_mw": exports,
        }, index=df.index)

    print(f"[XCH_DA] {country_from}-{country_to} {year}: shape={out.shape}, "
          f"range={out.index.min()} -> {out.index.max()}")
    return out


# ENTSO-E Day-Ahead Electricity Price
def fetch_elec_price_da(country_code: str, year: int) -> pd.DataFrame:
    """
    Day-ahead electricity price (EUR/MWh) from ENTSO-E, hourly UTC.
    Uses 'Without Sequence - Day-ahead' prices.
    """
    dfs = []
    for start, end in _iter_year_chunks(year, chunk_days=30):
        try:
            s_part = client.query_day_ahead_prices(country_code, start=start, end=end)
        except Exception as e:
            print(f"[PRICE_DA] Error {country_code} {year} chunk {start} -> {end}: {e}")
            continue

        if s_part is None or len(s_part) == 0:
            print(f"[PRICE_DA] EMPTY chunk {start} -> {end}")
            time.sleep(5.0)
            continue

        df_part = _to_utc_hourly(s_part)
        dfs.append(df_part)
        time.sleep(5.0)

    if not dfs:
        print(f"[PRICE_DA] EMPTY for {country_code} {year} (no valid chunks)")
        return pd.DataFrame()

    df = pd.concat(dfs, axis=0).sort_index()
    df = df[~df.index.duplicated(keep="first")]

    utc_start = pd.Timestamp(f"{year}-01-01 00:00", tz="UTC")
    utc_end   = pd.Timestamp(f"{year+1}-01-01 00:00", tz="UTC")
    df = df.loc[(df.index >= utc_start) & (df.index < utc_end)]

    if df.shape[1] > 1:
        df = df.iloc[:, :1]
    df.columns = ["da_elec_price_EURperMWh"]

    print(f"[PRICE_DA] {year}: shape={df.shape}, range={df.index.min()} -> {df.index.max()}")
    return df


def merge_year_and_report_missing(dfs_dict, year: int, outdir_processed: str) -> pd.DataFrame:
    """
    Merge all pieces for a given year on UTC hourly index, do DA interpolation,
    build hydro_dispatchable, coal+oil aggregate, DA-derived features, calendar features.
    NO lag/ramp. NO row drops for missing gen_*.
    """
    # Align on datetime_utc index
    for name, df in dfs_dict.items():
        if df is None or df.empty:
            print(f"[MERGE] Warning: {name} empty for {year}")
            continue
        if "datetime_utc" in df.columns:
            df = df.set_index("datetime_utc")
        df = df.sort_index()
        dfs_dict[name] = df

    frames = [df for df in dfs_dict.values() if df is not None and not df.empty]
    if not frames:
        raise RuntimeError(f"[MERGE] No frames for year {year}")

    master = reduce(lambda left, right: left.join(right, how="outer"), frames)
    master.index.name = "datetime_utc"
    master = master.sort_index()

    print(f"\n[MERGE] Year {year}: merged shape BEFORE interpolation = {master.shape}")
    print(f"[MERGE] Time range: {master.index.min()} -> {master.index.max()}")

    # Missingness BEFORE interpolation
    miss_mask = master.isna()

    long_missing = miss_mask.stack().reset_index()
    long_missing.columns = ["datetime_utc", "column", "is_missing"]
    long_missing = long_missing[long_missing["is_missing"]]

    miss_detail_path = os.path.join(outdir_processed, f"missing_hours_ES_{year}.csv")
    long_missing.to_csv(miss_detail_path, index=False)
    print(f"[MISS] Saved detailed missing hours -> {miss_detail_path} (rows={len(long_missing)})")

    miss_summary = miss_mask.sum().reset_index()
    miss_summary.columns = ["column", "n_missing"]
    miss_summary["year"] = year
    miss_summary_path = os.path.join(outdir_processed, f"missing_summary_ES_{year}.csv")
    miss_summary.to_csv(miss_summary_path, index=False)
    print(f"[MISS] Saved missing summary -> {miss_summary_path}")

    # Interpolate DA features (forecasts only) – first pass
    da_cols = [c for c in master.columns if c.startswith("da_")]
    print(f"[INTERP] Interpolating DA columns (year-level): {da_cols}")
    master[da_cols] = master[da_cols].interpolate(method="time", limit_direction="both")

    # ----- Hydro dispatchable aggregate: reservoir + pumped -----
    has_res = "gen_hydro_water_reservoir_mw" in master.columns
    has_pst = "gen_hydro_pumped_storage_mw" in master.columns
    if has_res or has_pst:
        res_series = master["gen_hydro_water_reservoir_mw"].fillna(0.0) if has_res else 0.0
        pst_series = master["gen_hydro_pumped_storage_mw"].fillna(0.0) if has_pst else 0.0
        master["gen_hydro_dispatchable_mw"] = res_series + pst_series
        print(f"[HYDRO] Added gen_hydro_dispatchable_mw for year {year}")

    # ----- Fossil coal + oil aggregate (NaNs treated as 0 for the sum) -----
    has_coal = "gen_fossil_hard_coal_mw" in master.columns
    has_oil  = "gen_fossil_oil_mw" in master.columns
    if has_coal or has_oil:
        coal_series = master["gen_fossil_hard_coal_mw"].fillna(0.0) if has_coal else 0.0
        oil_series  = master["gen_fossil_oil_mw"].fillna(0.0) if has_oil else 0.0
        master["gen_fossil_coal_oil_mw"] = coal_series + oil_series
        print(f"[FOSSIL] Added gen_fossil_coal_oil_mw for year {year}")

    # Derived DA features (now that DA + hydro are there)
    eps = 1e-6
    if "da_total_load_mw" in master.columns:
        master["da_residual_load_mw"] = (
            master["da_total_load_mw"]
            - master.get("da_wind_total_mw", 0.0)
            - master.get("da_solar_total_mw", 0.0)
        )

        master["da_res_share_wind"] = (
            master.get("da_wind_total_mw", 0.0) / (master["da_total_load_mw"] + eps)
        )
        master["da_res_share_solar"] = (
            master.get("da_solar_total_mw", 0.0) / (master["da_total_load_mw"] + eps)
        )

    master["da_net_imports_mw"] = (
        master.get("da_import_fr_mw", 0.0)
        + master.get("da_import_pt_mw", 0.0)
        - master.get("da_export_fr_mw", 0.0)
        - master.get("da_export_pt_mw", 0.0)
    )
    if "da_total_load_mw" in master.columns:
        master["da_import_share"] = master["da_net_imports_mw"] / (master["da_total_load_mw"] + eps)

    # Calendar features (UTC)
    master = master.sort_index()
    master = add_calendar_features(master)

    # Save raw per-year master (with NaNs still present in non-DA columns)
    master_path = os.path.join(outdir_processed, f"master_ES_{year}.csv")
    master.reset_index().to_csv(master_path, index=False)
    print(f"[MASTER] Saved master ES {year} -> {master_path} (rows={len(master)}, cols={len(master.columns)})")

    return master


# --------- PRECOMPUTE GAS DA HOURLY SERIES ----------
gas_da_hourly = load_da_gas_price_mibgas()

# --------- BUILD MASTERS FOR TRAIN YEARS (2022–2023) ----------
masters_train = []

for yr in YEARS_TRAIN:
    try:
        print(f"\n========== YEAR {yr} ==========")

        gen_actual_path   = os.path.join(DATA_PROC_DIR, f"gen_actual_{AREA}_{yr}.csv")
        load_da_path      = os.path.join(DATA_PROC_DIR, f"da_total_load_{AREA}_{yr}.csv")
        gen_da_total_path = os.path.join(DATA_PROC_DIR, f"da_gen_total_{AREA}_{yr}.csv")
        ws_da_path        = os.path.join(DATA_PROC_DIR, f"da_wind_solar_{AREA}_{yr}.csv")
        exch_es_fr_path   = os.path.join(DATA_PROC_DIR, f"da_exchanges_ES_FR_{yr}.csv")
        exch_es_pt_path   = os.path.join(DATA_PROC_DIR, f"da_exchanges_ES_PT_{yr}.csv")
        price_da_path     = os.path.join(DATA_PROC_DIR, f"da_elec_price_{AREA}_{yr}.csv")

        gen_actual    = cached_fetch_hourly(gen_actual_path,   fetch_gen_actual, AREA, yr)
        load_da       = cached_fetch_hourly(load_da_path,      fetch_load_da, AREA, yr)
        gen_da_total  = cached_fetch_hourly(gen_da_total_path, fetch_gen_da_total, AREA, yr)
        ws_da         = cached_fetch_hourly(ws_da_path,        fetch_wind_solar_da, AREA, yr)
        exch_es_fr    = cached_fetch_hourly(exch_es_fr_path,   fetch_exchanges_da, "ES", "FR", yr)
        exch_es_pt    = cached_fetch_hourly(exch_es_pt_path,   fetch_exchanges_da, "ES", "PT", yr)
        price_da      = cached_fetch_hourly(price_da_path,     fetch_elec_price_da, AREA, yr)

        # Subset gas DA hourly to this year (if available)
        if gas_da_hourly is not None and not gas_da_hourly.empty:
            mask_gas = (
                (gas_da_hourly.index >= pd.Timestamp(f"{yr}-01-01 00:00", tz="UTC")) &
                (gas_da_hourly.index <  pd.Timestamp(f"{yr+1}-01-01 00:00", tz="UTC"))
            )
            gas_da_year = gas_da_hourly.loc[mask_gas].copy()
        else:
            gas_da_year = pd.DataFrame()
            print(f"[GAS_DA] No gas DA data available for year {yr}")

        dfs_year = {
            "gen_actual": gen_actual,
            "load_da": load_da,
            "gen_da_total": gen_da_total,
            "ws_da": ws_da,
            "xch_fr": exch_es_fr,
            "xch_pt": exch_es_pt,
            "price_da": price_da,
            "gas_da": gas_da_year,
        }

        master_y = merge_year_and_report_missing(dfs_year, yr, DATA_PROC_DIR)
        masters_train.append(master_y)

    except Exception as e:
        print(f"[ERROR] Year {yr}: {e}")
        traceback.print_exc()
    time.sleep(1.0)

if not masters_train:
    raise RuntimeError("[TRAIN] No master data built for TRAIN years.")

master_train = pd.concat(masters_train, axis=0).sort_index()
print("\n[TRAIN] CONCAT 2022–2023:", master_train.shape,
      "| range:", master_train.index.min(), "->", master_train.index.max())

train_master_path = os.path.join(DATA_PROC_DIR, "master_ES_train_2022_2023_raw.csv")
master_train.reset_index().to_csv(train_master_path, index=False)
print(f"[TRAIN] Saved raw concatenated TRAIN master -> {train_master_path}")

# --------- BUILD MASTER FOR TEST YEAR (2024) ----------
print(f"\n========== YEAR {YEAR_TEST} (TEST) ==========")
try:
    gen_actual_path   = os.path.join(DATA_PROC_DIR, f"gen_actual_{AREA}_{YEAR_TEST}.csv")
    load_da_path      = os.path.join(DATA_PROC_DIR, f"da_total_load_{AREA}_{YEAR_TEST}.csv")
    gen_da_total_path = os.path.join(DATA_PROC_DIR, f"da_gen_total_{AREA}_{YEAR_TEST}.csv")
    ws_da_path        = os.path.join(DATA_PROC_DIR, f"da_wind_solar_{AREA}_{YEAR_TEST}.csv")
    exch_es_fr_path   = os.path.join(DATA_PROC_DIR, f"da_exchanges_ES_FR_{YEAR_TEST}.csv")
    exch_es_pt_path   = os.path.join(DATA_PROC_DIR, f"da_exchanges_ES_PT_{YEAR_TEST}.csv")
    price_da_path     = os.path.join(DATA_PROC_DIR, f"da_elec_price_{AREA}_{YEAR_TEST}.csv")

    gen_actual    = cached_fetch_hourly(gen_actual_path,   fetch_gen_actual, AREA, YEAR_TEST)
    load_da       = cached_fetch_hourly(load_da_path,      fetch_load_da, AREA, YEAR_TEST)
    gen_da_total  = cached_fetch_hourly(gen_da_total_path, fetch_gen_da_total, AREA, YEAR_TEST)
    ws_da         = cached_fetch_hourly(ws_da_path,        fetch_wind_solar_da, AREA, YEAR_TEST)
    exch_es_fr    = cached_fetch_hourly(exch_es_fr_path,   fetch_exchanges_da, "ES", "FR", YEAR_TEST)
    exch_es_pt    = cached_fetch_hourly(exch_es_pt_path,   fetch_exchanges_da, "ES", "PT", YEAR_TEST)
    price_da      = cached_fetch_hourly(price_da_path,     fetch_elec_price_da, AREA, YEAR_TEST)

    if gas_da_hourly is not None and not gas_da_hourly.empty:
        mask_gas = (
            (gas_da_hourly.index >= pd.Timestamp(f"{YEAR_TEST}-01-01 00:00", tz="UTC")) &
            (gas_da_hourly.index <  pd.Timestamp(f"{YEAR_TEST+1}-01-01 00:00", tz="UTC"))
        )
        gas_da_year = gas_da_hourly.loc[mask_gas].copy()
    else:
        gas_da_year = pd.DataFrame()
        print(f"[GAS_DA] No gas DA data available for year {YEAR_TEST}")

    dfs_year = {
        "gen_actual": gen_actual,
        "load_da": load_da,
        "gen_da_total": gen_da_total,
        "ws_da": ws_da,
        "xch_fr": exch_es_fr,
        "xch_pt": exch_es_pt,
        "price_da": price_da,
        "gas_da": gas_da_year,
    }

    master_test = merge_year_and_report_missing(dfs_year, YEAR_TEST, DATA_PROC_DIR)
    master_test = master_test.sort_index()

except Exception as e:
    print(f"[ERROR] Year {YEAR_TEST}: {e}")
    traceback.print_exc()
    raise

test_master_path = os.path.join(DATA_PROC_DIR, "master_ES_test_2024_raw.csv")
master_test.reset_index().to_csv(test_master_path, index=False)
print(f"[TEST] Saved raw TEST master -> {test_master_path}")
print("[TEST] master_test:", master_test.shape,
      "| range:", master_test.index.min(), "->", master_test.index.max())

# --------- GLOBAL LINEAR INTERPOLATION (ALL NUMERIC COLS, TRAIN & TEST) ----------
def interpolate_all_numeric(df: pd.DataFrame, name: str) -> pd.DataFrame:
    df = df.sort_index()
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    print(f"[INTERP-ALL] {name}: numeric cols={len(num_cols)}")
    df[num_cols] = df[num_cols].interpolate(method="time", limit_direction="both")
    return df

master_train = interpolate_all_numeric(master_train, "TRAIN(2022–2023)")
master_test  = interpolate_all_numeric(master_test,  "TEST(2024)")

train_interp_path = os.path.join(DATA_PROC_DIR, "master_ES_train_2022_2023_interp.csv")
test_interp_path  = os.path.join(DATA_PROC_DIR, "master_ES_test_2024_interp.csv")
master_train.reset_index().to_csv(train_interp_path, index=False)
master_test.reset_index().to_csv(test_interp_path, index=False)
print(f"[SAVE] TRAIN interpolated -> {train_interp_path}")
print(f"[SAVE] TEST  interpolated -> {test_interp_path}")

# --------- FEATURE SELECTION (GEN, DA, CALENDAR) ----------
gen_cols = [c for c in master_train.columns if c.startswith("gen_")]
da_cols  = [c for c in master_train.columns if c.startswith("da_")]
cal_cols = ["hour", "dow", "month", "hour_sin", "hour_cos",
            "doy_sin", "doy_cos", "is_weekend", "is_holiday"]
cal_cols = [c for c in cal_cols if c in master_train.columns]

# --- AGGREGATE CLEANUP FOR TARGET TECHS ---
# If we have dispatchable hydro, drop component techs from the prediction list
if "gen_hydro_dispatchable_mw" in gen_cols:
    for c in ["gen_hydro_water_reservoir_mw", "gen_hydro_pumped_storage_mw"]:
        if c in gen_cols:
            gen_cols.remove(c)

# Ensure coal+oil aggregate is included and donors removed
if "gen_fossil_coal_oil_mw" in master_train.columns:
    if "gen_fossil_coal_oil_mw" not in gen_cols:
        gen_cols.append("gen_fossil_coal_oil_mw")
    for c in ["gen_fossil_hard_coal_mw", "gen_fossil_oil_mw"]:
        if c in gen_cols:
            gen_cols.remove(c)

print("\n[FEATURES] gen_cols:", gen_cols)
print("[FEATURES] da_cols :", da_cols)
print("[FEATURES] cal_cols:", cal_cols)

# Save feature lists
feat_list_path = os.path.join(DATA_PROC_DIR, "feature_lists_entsoe.csv")
feat_df_list = (
    pd.concat([
        pd.DataFrame({"type": "gen", "name": gen_cols}),
        pd.DataFrame({"type": "da",  "name": da_cols}),
        pd.DataFrame({"type": "cal", "name": cal_cols}),
    ], axis=0, ignore_index=True)
)
feat_df_list.to_csv(feat_list_path, index=False)
print(f"[FEATURES] Saved feature lists -> {feat_list_path}")

# --------- TRAIN / VAL SPLIT (TIME-BASED INSIDE 2022–2023) ----------
train_end = master_train.index.max()
val_start = train_end - pd.Timedelta(days=VAL_DAYS) + pd.Timedelta(hours=1)

train_df = master_train.loc[master_train.index < val_start].copy()
val_df   = master_train.loc[master_train.index >= val_start].copy()

print("\n[SPLIT] TRAIN_df:", train_df.shape,
      "| range:", train_df.index.min(), "->", train_df.index.max())
print("[SPLIT] VAL_df  :", val_df.shape,
      "| range:", val_df.index.min(), "->", val_df.index.max())

# --------- SCALE (MINMAX) – FIT ON TRAIN ONLY ----------
# For MLPs, we keep:
#   - history features: gen_cols + cal_cols (known from past)
#   - DA features: da_cols (future horizon)
hist_feature_cols = gen_cols + cal_cols
da_feature_cols   = da_cols

all_feature_cols  = hist_feature_cols + da_feature_cols

# ensure no duplicates in all_feature_cols
all_feature_cols = list(dict.fromkeys(all_feature_cols))

# consistency check – must exist in all dfs
for c in all_feature_cols:
    if c not in train_df.columns or c not in val_df.columns or c not in master_test.columns:
        raise ValueError(f"[FEATURES] Column {c} not present in train/val/test dfs.")

# We also keep all gen_cols in y, but per-tech training will pick one target later.
# For scaling we treat all feature columns with MinMax, and also all gen_* as potential targets for later.
X_train_raw = train_df[all_feature_cols].values
X_val_raw   = val_df[all_feature_cols].values
X_test_raw  = master_test[all_feature_cols].values

y_train_raw = train_df[gen_cols].values
y_val_raw   = val_df[gen_cols].values
y_test_raw  = master_test[gen_cols].values

scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_train_scaled = scaler_X.fit_transform(X_train_raw)
X_val_scaled   = scaler_X.transform(X_val_raw)
X_test_scaled  = scaler_X.transform(X_test_raw)

y_train_scaled = scaler_y.fit_transform(y_train_raw)
y_val_scaled   = scaler_y.transform(y_val_raw)
y_test_scaled  = scaler_y.transform(y_test_raw)

train_scaled = pd.DataFrame(X_train_scaled, index=train_df.index, columns=all_feature_cols)
val_scaled   = pd.DataFrame(X_val_scaled,   index=val_df.index,   columns=all_feature_cols)
test_scaled  = pd.DataFrame(X_test_scaled,  index=master_test.index, columns=all_feature_cols)

# add scaled gen_* columns (targets) back
for i, col in enumerate(gen_cols):
    train_scaled[col] = y_train_scaled[:, i]
    val_scaled[col]   = y_val_scaled[:, i]
    test_scaled[col]  = y_test_scaled[:, i]

print("\n[SCALE] train_scaled:", train_scaled.shape,
      "| range:", train_scaled.index.min(), "->", train_scaled.index.max())
print("[SCALE] val_scaled  :", val_scaled.shape,
      "| range:", val_scaled.index.min(),   "->", val_scaled.index.max())
print("[SCALE] test_scaled :", test_scaled.shape,
      "| range:", test_scaled.index.min(),  "->", test_scaled.index.max())

train_scaled_path = os.path.join(DATA_PROC_DIR, "entsoe_train_scaled_hourly.csv")
val_scaled_path   = os.path.join(DATA_PROC_DIR, "entsoe_val_scaled_hourly.csv")
test_scaled_path  = os.path.join(DATA_PROC_DIR, "entsoe_test_scaled_hourly.csv")

train_scaled.reset_index().to_csv(train_scaled_path, index=False)
val_scaled.reset_index().to_csv(val_scaled_path,   index=False)
test_scaled.reset_index().to_csv(test_scaled_path, index=False)

print(f"[SAVE] train_scaled_hourly -> {train_scaled_path}")
print(f"[SAVE] val_scaled_hourly   -> {val_scaled_path}")
print(f"[SAVE] test_scaled_hourly  -> {test_scaled_path}")

# --------- SLIDING WINDOWS 24→24 (CarbonCast-style, with DA horizon)
def build_sliding_windows_with_da(df_scaled: pd.DataFrame,
                                  hist_cols,
                                  da_cols,
                                  gen_cols_for_y,
                                  input_hours=24,
                                  label_hours=24):
    """
    Building CarbonCast-style sliding windows:

    For each index i:
      H = df[hist_cols][i : i+input_hours]                  (past 24h)
      F = df[da_cols][i+input_hours : i+input_hours+label_hours] (DA horizon 24h)
      X_window = concat(H_t, F_t) along feature axis       -> shape (24, F_hist+F_da)
      y_window = df[gen_cols_for_y][i+input_hours : i+input_hours+label_hours]

    Returned X is flattened for MLP: (N_samples, 24 * (F_hist+F_da))
    Returned y is shape: (N_samples, 24, n_gen_cols_for_y)  – you will pick per-tech later.
    """
    df_scaled = df_scaled.sort_index()
    H_data = df_scaled[hist_cols].values
    F_data = df_scaled[da_cols].values
    Y_data = df_scaled[gen_cols_for_y].values

    n = len(df_scaled)
    max_i = n - (input_hours + label_hours) + 1
    if max_i <= 0:
        raise RuntimeError("[SLIDING] Not enough data points for given input/label hours.")

    X_list, Y_list = [], []

    for i in range(max_i):
        h_block = H_data[i : i + input_hours, :]  # (24, F_hist)
        f_block = F_data[i + input_hours : i + input_hours + label_hours, :]  # (24, F_da)

        if h_block.shape[0] != input_hours or f_block.shape[0] != label_hours:
            continue  # safety

        # concatenate along feature axis: shape (24, F_hist + F_da)
        x_block = np.concatenate([h_block, f_block], axis=1)
        y_block = Y_data[i + input_hours : i + input_hours + label_hours, :]  # (24, n_gen_cols_for_y)

        X_list.append(x_block.reshape(-1))  # flatten to (24*(F_hist+F_da),)
        Y_list.append(y_block)              # keep (24, n_gen_cols_for_y)

    X = np.stack(X_list)
    Y = np.stack(Y_list)

    return X, Y

print("\n[SLIDING] Building sliding windows 24→24 for TRAIN and VAL...")

X_train, Y_train = build_sliding_windows_with_da(
    train_scaled, hist_feature_cols, da_feature_cols, gen_cols,
    input_hours=TRAIN_WIN_H, label_hours=LABEL_WIN_H
)

X_val, Y_val = build_sliding_windows_with_da(
    val_scaled, hist_feature_cols, da_feature_cols, gen_cols,
    input_hours=TRAIN_WIN_H, label_hours=LABEL_WIN_H
)

print("[SLIDING] X_train:", X_train.shape, "| Y_train:", Y_train.shape)
print("[SLIDING] X_val  :", X_val.shape,   "| Y_val  :", Y_val.shape)

# Save sliding-window pairs as CSV (flattened X, flattened Y per sample)
# Y has shape (N, 24, n_gen_cols)
n_train, _, n_gen_targets = Y_train.shape
n_val,   _, _             = Y_val.shape

Y_train_flat = Y_train.reshape(n_train, LABEL_WIN_H * n_gen_targets)
Y_val_flat   = Y_val.reshape(n_val,   LABEL_WIN_H * n_gen_targets)

X_train_cols = [f"x_{i}" for i in range(X_train.shape[1])]
X_val_cols   = [f"x_{i}" for i in range(X_val.shape[1])]

Y_train_cols = [f"y_{tech}_{h}"
                for tech in gen_cols
                for h in range(LABEL_WIN_H)]
Y_val_cols   = [f"y_{tech}_{h}"
                for tech in gen_cols
                for h in range(LABEL_WIN_H)]

X_train_df = pd.DataFrame(X_train, columns=X_train_cols)
X_val_df   = pd.DataFrame(X_val,   columns=X_val_cols)
Y_train_df = pd.DataFrame(Y_train_flat, columns=Y_train_cols)
Y_val_df   = pd.DataFrame(Y_val_flat,   columns=Y_val_cols)

X_train_path = os.path.join(DATA_PROC_DIR, "entsoe_train_X_sliding_24_24.csv")
Y_train_path = os.path.join(DATA_PROC_DIR, "entsoe_train_Y_sliding_24_24.csv")
X_val_path   = os.path.join(DATA_PROC_DIR, "entsoe_val_X_sliding_24_24.csv")
Y_val_path   = os.path.join(DATA_PROC_DIR, "entsoe_val_Y_sliding_24_24.csv")

X_train_df.to_csv(X_train_path, index=False)
Y_train_df.to_csv(Y_train_path, index=False)
X_val_df.to_csv(X_val_path,     index=False)
Y_val_df.to_csv(Y_val_path,     index=False)

print(f"[SAVE] X_train_sliding -> {X_train_path}")
print(f"[SAVE] Y_train_sliding -> {Y_train_path}")
print(f"[SAVE] X_val_sliding   -> {X_val_path}")
print(f"[SAVE] Y_val_sliding   -> {Y_val_path}")

# --------- INITIAL HISTORY FOR FORECASTING (END OF VAL) ----------
# Last 24 hours of VAL (scaled) – full row with all features + gen_*.
history_init = val_scaled.iloc[-TRAIN_WIN_H:].copy()
history_init_path = os.path.join(DATA_PROC_DIR, "entsoe_history_init_val_last24.csv")
history_init.reset_index().to_csv(history_init_path, index=False)
print(f"[SAVE] history_init (last 24h of VAL) -> {history_init_path}")

print("\n[DONE] Prep cell finished. You can now build per-tech MLPs using:")
print("- Sliding window TRAIN/VAL CSVs for supervised learning")
print("- hourly scaled TRAIN/VAL/TEST for day-ahead walk-forward forecasting")
print("- gen_cols / da_cols / cal_cols from feature_lists_entsoe.csv")


## 1.2 Technology specific ANN training and 2024 testing

Each technology is kept as a separate code block because the final manuscript/SI used technology specific ANN configurations rather than one shared architecture.  
Excluded here: individual hard coal, fossil oil, hydro pumped storage, and hydro water reservoir cells, because the final target groups use the aggregated **coal-oil** and **hydro dispatchable** categories.


### 1.2.1 Fossil coal-oil aggregated technology



In [ ]:
# =========================
# CELL – Per-Tech MLP for Fossil Coal+Oil (CarbonCast-style)
# =========================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import timedelta

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, InputLayer
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import mean_squared_error, mean_absolute_error

# --------- CONFIG ----------
DATA_PROC_DIR = "/content/data_entsoe_es_processed"
TECH_NAME     = "gen_fossil_coal_oil_mw"   # aggregated target tech
TRAIN_WIN_H   = 24
LABEL_WIN_H   = 24

# Per-tech subfolder
TECH_DIR = os.path.join(DATA_PROC_DIR, TECH_NAME)
os.makedirs(TECH_DIR, exist_ok=True)

# ---------------------------
# 1) Load feature lists
# ---------------------------
feat_list_path = os.path.join(DATA_PROC_DIR, "feature_lists_entsoe.csv")
feat_df_list   = pd.read_csv(feat_list_path)

gen_cols = feat_df_list[feat_df_list["type"] == "gen"]["name"].tolist()
da_cols  = feat_df_list[feat_df_list["type"] == "da"]["name"].tolist()
cal_cols = feat_df_list[feat_df_list["type"] == "cal"]["name"].tolist()

if TECH_NAME not in gen_cols:
    raise ValueError(f"{TECH_NAME} not found in gen_cols")

tech_idx = gen_cols.index(TECH_NAME)
print(f"[TECH] Using tech={TECH_NAME}, index={tech_idx}")
print("[TECH] gen_cols:", gen_cols)
print("[TECH] da_cols :", da_cols)
print("[TECH] cal_cols:", cal_cols)

# Hist features = all gen + calendar; DA features separate
hist_feature_cols = gen_cols + cal_cols
da_feature_cols   = da_cols
all_feature_cols  = list(dict.fromkeys(hist_feature_cols + da_feature_cols))

# ---------------------------
# 2) Load sliding-window train/val
# ---------------------------
X_train_flat = pd.read_csv(os.path.join(DATA_PROC_DIR, "entsoe_train_X_sliding_24_24.csv")).values
Y_train_flat = pd.read_csv(os.path.join(DATA_PROC_DIR, "entsoe_train_Y_sliding_24_24.csv")).values
X_val_flat   = pd.read_csv(os.path.join(DATA_PROC_DIR, "entsoe_val_X_sliding_24_24.csv")).values
Y_val_flat   = pd.read_csv(os.path.join(DATA_PROC_DIR, "entsoe_val_Y_sliding_24_24.csv")).values

n_train = X_train_flat.shape[0]
n_val   = X_val_flat.shape[0]

F_total        = X_train_flat.shape[1] // TRAIN_WIN_H
n_gen_targets  = len(gen_cols)

print(f"[SHAPE] X_train_flat: {X_train_flat.shape} -> (N, 24, {F_total})")
print(f"[SHAPE] Y_train_flat: {Y_train_flat.shape} -> (N, 24, {n_gen_targets})")

X_train_seq = X_train_flat.reshape(n_train, TRAIN_WIN_H, F_total)
X_val_seq   = X_val_flat.reshape(n_val,   TRAIN_WIN_H, F_total)

Y_train_all = Y_train_flat.reshape(n_train, LABEL_WIN_H, n_gen_targets)
Y_val_all   = Y_val_flat.reshape(n_val,   LABEL_WIN_H, n_gen_targets)

# Extract target only
Y_train = Y_train_all[:, :, tech_idx]
Y_val   = Y_val_all[:, :, tech_idx]

print(f"[SHAPE] X_train_seq: {X_train_seq.shape}, Y_train: {Y_train.shape}")
print(f"[SHAPE] X_val_seq  : {X_val_seq.shape}, Y_val:   {Y_val.shape}")

# ---------------------------
# 3) MLP model
# ---------------------------
EPOCHS      = 100
BATCH_SIZE  = 10
ACTIVATION  = "relu"
LOSS_FUNC   = "mse"
LEARNING_RT = 0.0011
HIDDEN      = [50, 40]

model = Sequential()
model.add(InputLayer(input_shape=(TRAIN_WIN_H, F_total)))
model.add(Flatten())
model.add(Dense(HIDDEN[0], activation=ACTIVATION))
model.add(Dense(HIDDEN[1], activation=ACTIVATION))
model.add(Dense(LABEL_WIN_H))

opt = tf.keras.optimizers.Adam(learning_rate=LEARNING_RT)
model.compile(loss=LOSS_FUNC, optimizer=opt, metrics=["mae"])

model.summary()

es = EarlyStopping(
    monitor="val_loss",
    patience=10,
    mode="min",
    verbose=1,
    restore_best_weights=True,
)

# ---------------------------
# 4) Train
# ---------------------------
history = model.fit(
    X_train_seq, Y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val_seq, Y_val),
    callbacks=[es],
    verbose=2
)

print("[TRAIN] Best weights restored.")

model_path = os.path.join(TECH_DIR, "best_mlp_coal_oil.keras")
model.save(model_path)
print(f"[TRAIN] Saved model -> {model_path}")

# ---------------------------
# 5) Load hourly scaled val+test and strict day-ahead forecast
# ---------------------------
val_scaled  = pd.read_csv(os.path.join(DATA_PROC_DIR, "entsoe_val_scaled_hourly.csv"),
                          parse_dates=["datetime_utc"]).set_index("datetime_utc").sort_index()
test_scaled = pd.read_csv(os.path.join(DATA_PROC_DIR, "entsoe_test_scaled_hourly.csv"),
                          parse_dates=["datetime_utc"]).set_index("datetime_utc").sort_index()

full_df = pd.concat([val_scaled, test_scaled], axis=0)

test_days = sorted(test_scaled.index.normalize().unique())

print(f"[TEST] Days in 2024: {len(test_days)}")

all_pred_scaled = []
all_true_scaled = []
all_time_index  = []

for day in test_days:
    horizon_start = pd.Timestamp(day)
    horizon_end   = horizon_start + timedelta(hours=23)

    hist_start = horizon_start - timedelta(hours=24)
    hist_end   = horizon_start - timedelta(hours=1)

    H = full_df.loc[hist_start:hist_end][hist_feature_cols]
    F = full_df.loc[horizon_start:horizon_end][da_feature_cols]

    if len(H) != 24 or len(F) != 24:
        continue

    X_day = np.concatenate([H.values, F.values], axis=1).reshape(1, 24, F_total)

    y_pred_scaled = model.predict(X_day, verbose=0)[0]
    y_true_scaled = full_df.loc[horizon_start:horizon_end][TECH_NAME].values

    all_pred_scaled.append(y_pred_scaled)
    all_true_scaled.append(y_true_scaled)
    all_time_index.append(full_df.loc[horizon_start:horizon_end].index.values)

y_pred_scaled = np.concatenate(all_pred_scaled)
y_true_scaled = np.concatenate(all_true_scaled)
time_index    = np.concatenate(all_time_index)

print(f"[TEST] Total hourly predictions: {len(y_true_scaled)}")

# ---------------------------
# 6) Inverse scaling
# ---------------------------
try:
    scaler_y
except:
    raise RuntimeError("scaler_y missing. Run prep cell first.")

n_techs = len(gen_cols)
k       = tech_idx
T       = len(y_true_scaled)

y_true_scaled_full = np.zeros((T, n_techs))
y_pred_scaled_full = np.zeros((T, n_techs))
y_true_scaled_full[:, k] = y_true_scaled
y_pred_scaled_full[:, k] = y_pred_scaled

y_true = scaler_y.inverse_transform(y_true_scaled_full)[:, k]
y_pred = scaler_y.inverse_transform(y_pred_scaled_full)[:, k]

# ---------------------------
# 7) Hourly metrics
# ---------------------------
rmse_h = np.sqrt(mean_squared_error(y_true, y_pred))
mae_h  = mean_absolute_error(y_true, y_pred)
eps = 1e-6
mape_h = np.mean(np.abs((y_true - y_pred) /
                        np.clip(np.abs(y_true), eps, None))) * 100
nmae_h = mae_h / (np.mean(y_true) + eps)

print("\n[METRICS – HOURLY 2024, FOSSIL COAL+OIL]")
print(f"RMSE      = {rmse_h:.4f} MW")
print(f"MAE       = {mae_h:.4f} MW")
print(f"MAPE      = {mape_h:.2f} %")
print(f"nMAE_mean = {nmae_h:.4f}")

# ---------------------------
# 8) Daily metrics
# ---------------------------
hourly_df = pd.DataFrame({
    "datetime_utc": time_index,
    "y_true_mw": y_true,
    "y_pred_mw": y_pred
}).set_index("datetime_utc").sort_index()

hourly_df["abs_error"] = np.abs(hourly_df["y_true_mw"] - hourly_df["y_pred_mw"])
hourly_df["nabs_error_mean"] = hourly_df["abs_error"] / (np.mean(y_true) + eps)

daily_df = hourly_df.groupby(hourly_df.index.normalize()).agg({
    "y_true_mw": "mean",
    "abs_error": "mean",
    "y_pred_mw": (lambda s: np.sqrt(np.mean((s - hourly_df.loc[s.index, "y_true_mw"])**2)))
})

# ---------------------------
# 9) Save CSVs
# ---------------------------
hourly_df.reset_index().to_csv(os.path.join(TECH_DIR, "coal_oil_hourly_predictions_2024.csv"), index=False)
daily_df.to_csv(os.path.join(TECH_DIR, "coal_oil_daily_metrics_2024.csv"))
pd.DataFrame([{
    "RMSE_MW": rmse_h,
    "MAE_MW": mae_h,
    "MAPE_percent": mape_h,
    "nMAE_mean": nmae_h
}]).to_csv(os.path.join(TECH_DIR, "coal_oil_overall_metrics_2024.csv"), index=False)

# ---------------------------
# 10) Plots
# ---------------------------

# Actual vs predicted
plt.figure(figsize=(14,4))
plt.plot(hourly_df.index, hourly_df["y_true_mw"], label="Actual", linewidth=1)
plt.plot(hourly_df.index, hourly_df["y_pred_mw"], label="Predicted", linewidth=0.8, alpha=0.7)
plt.title("Fossil Coal+Oil – Hourly Actual vs Predicted (2024)")
plt.xlabel("Time (UTC)")
plt.ylabel("Power [MW]")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(TECH_DIR, "coal_oil_hourly_actual_vs_pred_2024.png"), dpi=150)
plt.show()

# Normalized abs error histogram
plt.figure(figsize=(6,4))
plt.hist(hourly_df["nabs_error_mean"], bins=50)
plt.title("Fossil Coal+Oil – Distribution of Hourly Normalized Abs Error (2024)")
plt.xlabel("Normalized Abs Error")
plt.ylabel("Frequency")
plt.tight_layout()
plt.savefig(os.path.join(TECH_DIR, "coal_oil_hourly_norm_abs_error_hist_2024.png"), dpi=150)
plt.show()

print("\n[DONE] Per-tech MLP for fossil coal+oil finished.")


### 1.2.2 Biomass


In [ ]:
# =========================
# CELL – Per-Tech MLP for Biomass (CarbonCast-style)
# =========================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import timedelta

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, InputLayer
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import mean_squared_error, mean_absolute_error

# --------- CONFIG ----------
DATA_PROC_DIR = "/content/data_entsoe_es_processed"
TECH_NAME     = "gen_biomass_mw"      # target tech
TRAIN_WIN_H   = 24
LABEL_WIN_H   = 24

# Per-tech subfolder
TECH_DIR = os.path.join(DATA_PROC_DIR, TECH_NAME)
os.makedirs(TECH_DIR, exist_ok=True)

# ---------------------------
# 1) Load feature lists to get gen/da/cal columns and biomass index
# ---------------------------
feat_list_path = os.path.join(DATA_PROC_DIR, "feature_lists_entsoe.csv")
feat_df_list   = pd.read_csv(feat_list_path)

gen_cols = feat_df_list[feat_df_list["type"] == "gen"]["name"].tolist()
da_cols  = feat_df_list[feat_df_list["type"] == "da"]["name"].tolist()
cal_cols = feat_df_list[feat_df_list["type"] == "cal"]["name"].tolist()

if TECH_NAME not in gen_cols:
    raise ValueError(f"{TECH_NAME} not found in gen_cols: {gen_cols}")

tech_idx = gen_cols.index(TECH_NAME)
print(f"[TECH] Using tech={TECH_NAME}, index={tech_idx}")
print("[TECH] gen_cols:", gen_cols)
print("[TECH] da_cols :", da_cols)
print("[TECH] cal_cols:", cal_cols)

# Hist features = all gen + calendar; DA features as separate block
hist_feature_cols = gen_cols + cal_cols
da_feature_cols   = da_cols
all_feature_cols  = list(dict.fromkeys(hist_feature_cols + da_feature_cols))

# ---------------------------
# 2) Load sliding-window TRAIN / VAL data and reshape
# ---------------------------
X_train_path = os.path.join(DATA_PROC_DIR, "entsoe_train_X_sliding_24_24.csv")
Y_train_path = os.path.join(DATA_PROC_DIR, "entsoe_train_Y_sliding_24_24.csv")
X_val_path   = os.path.join(DATA_PROC_DIR, "entsoe_val_X_sliding_24_24.csv")
Y_val_path   = os.path.join(DATA_PROC_DIR, "entsoe_val_Y_sliding_24_24.csv")

X_train_flat = pd.read_csv(X_train_path).values  # (N_train, 24*(F_hist+F_da))
Y_train_flat = pd.read_csv(Y_train_path).values  # (N_train, 24 * n_gen_cols)
X_val_flat   = pd.read_csv(X_val_path).values
Y_val_flat   = pd.read_csv(Y_val_path).values

n_train = X_train_flat.shape[0]
n_val   = X_val_flat.shape[0]

# infer feature dimension per timestep and number of gen targets
F_total        = X_train_flat.shape[1] // TRAIN_WIN_H
n_gen_targets  = len(gen_cols)

if Y_train_flat.shape[1] != LABEL_WIN_H * n_gen_targets:
    raise RuntimeError("Y_train_flat shape is inconsistent with LABEL_WIN_H and n_gen_targets")

print(f"[SHAPE] X_train_flat: {X_train_flat.shape} -> (N, {TRAIN_WIN_H}, {F_total})")
print(f"[SHAPE] Y_train_flat: {Y_train_flat.shape} -> (N, {LABEL_WIN_H}, {n_gen_targets})")

# reshape to (N, 24, F_total) and (N, 24, n_gen_targets)
X_train_seq = X_train_flat.reshape(n_train, TRAIN_WIN_H, F_total)
X_val_seq   = X_val_flat.reshape(n_val,   TRAIN_WIN_H, F_total)

Y_train_all = Y_train_flat.reshape(n_train, LABEL_WIN_H, n_gen_targets)
Y_val_all   = Y_val_flat.reshape(n_val,   LABEL_WIN_H, n_gen_targets)

# extract biomass only: shape (N, 24)
Y_train = Y_train_all[:, :, tech_idx]
Y_val   = Y_val_all[:, :, tech_idx]

print(f"[SHAPE] X_train_seq: {X_train_seq.shape}, Y_train: {Y_train.shape}")
print(f"[SHAPE] X_val_seq  : {X_val_seq.shape}, Y_val  : {Y_val.shape}")

# ---------------------------
# 3) Build CarbonCast-style MLP for this tech
# ---------------------------
EPOCHS      = 100
BATCH_SIZE  = 10
ACTIVATION  = "relu"
LOSS_FUNC   = "mse"
LEARNING_RT = 0.001
HIDDEN      = [50, 34]

model = Sequential()
model.add(InputLayer(input_shape=(TRAIN_WIN_H, F_total)))
model.add(Flatten())
model.add(Dense(HIDDEN[0], activation=ACTIVATION))
model.add(Dense(HIDDEN[1], activation=ACTIVATION))
model.add(Dense(LABEL_WIN_H))  # 24 outputs (24h horizon)

opt = tf.keras.optimizers.Adam(learning_rate=LEARNING_RT)
model.compile(loss=LOSS_FUNC, optimizer=opt, metrics=["mae"])

model_summary = []
model.summary(print_fn=lambda x: model_summary.append(x))
print("\n".join(model_summary))

# EarlyStopping only – restore best weights in memory
es = EarlyStopping(
    monitor="val_loss",
    mode="min",
    patience=10,
    verbose=1,
    restore_best_weights=True,
)

# ---------------------------
# 4) Train model
# ---------------------------
history = model.fit(
    X_train_seq, Y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val_seq, Y_val),
    callbacks=[es],
    verbose=2
)

print("[TRAIN] Training finished. Best weights are already loaded in `model`.")
best_model = model  # use this directly

# Save model in native Keras format in tech subfolder (optional, no reload here)
model_path = os.path.join(TECH_DIR, "best_mlp_biomass.keras")
best_model.save(model_path)
print(f"[TRAIN] Saved model to {model_path}")

# ---------------------------
# 5) Load hourly scaled VAL + TEST and build strict day-ahead forecasts for 2024
# ---------------------------
train_scaled_path = os.path.join(DATA_PROC_DIR, "entsoe_train_scaled_hourly.csv")
val_scaled_path   = os.path.join(DATA_PROC_DIR, "entsoe_val_scaled_hourly.csv")
test_scaled_path  = os.path.join(DATA_PROC_DIR, "entsoe_test_scaled_hourly.csv")

val_scaled  = pd.read_csv(val_scaled_path, parse_dates=["datetime_utc"]).set_index("datetime_utc").sort_index()
test_scaled = pd.read_csv(test_scaled_path, parse_dates=["datetime_utc"]).set_index("datetime_utc").sort_index()

print("[TEST] val_scaled :", val_scaled.shape,  "|", val_scaled.index.min(), "->", val_scaled.index.max())
print("[TEST] test_scaled:", test_scaled.shape, "|", test_scaled.index.min(),  "->", test_scaled.index.max())

# Combine val tail + test for history context
full_df = pd.concat([val_scaled, test_scaled], axis=0).sort_index()

# List of test days (UTC midnights) with full 24h coverage
test_days = sorted(test_scaled.index.normalize().unique())
print(f"[TEST] Number of test days in 2024: {len(test_days)}")

all_pred_scaled = []
all_true_scaled = []
all_time_index  = []

for day in test_days:
    horizon_start = pd.Timestamp(day)                      # e.g. 2024-01-01 00:00
    horizon_end   = horizon_start + timedelta(hours=23)    # inclusive

    # History: previous 24 hours
    hist_start = horizon_start - timedelta(hours=24)
    hist_end   = horizon_start - timedelta(hours=1)

    hist_slice    = full_df.loc[hist_start:hist_end]
    horizon_slice = full_df.loc[horizon_start:horizon_end]

    if len(hist_slice) != TRAIN_WIN_H or len(horizon_slice) != LABEL_WIN_H:
        print(f"[WARN] Skipping day {day.date()} due to incomplete history/horizon "
              f"({len(hist_slice)} hist hrs, {len(horizon_slice)} horizon hrs).")
        continue

    H = hist_slice[hist_feature_cols].values    # (24, F_hist)
    F = horizon_slice[da_feature_cols].values   # (24, F_da)

    if H.shape[0] != TRAIN_WIN_H or F.shape[0] != LABEL_WIN_H:
        print(f"[WARN] Shape mismatch on day {day.date()}, skipping.")
        continue

    X_day = np.concatenate([H, F], axis=1)  # (24, F_total) – must match training
    X_day = X_day.reshape(1, TRAIN_WIN_H, F_total)

    y_pred_day_scaled = best_model.predict(X_day, verbose=0)[0]  # (24,)
    y_true_day_scaled = horizon_slice[TECH_NAME].values          # (24,)

    all_pred_scaled.append(y_pred_day_scaled)
    all_true_scaled.append(y_true_day_scaled)
    all_time_index.append(horizon_slice.index.values)

# Stack to hourly arrays
if not all_pred_scaled:
    raise RuntimeError("[TEST] No valid test days were forecasted.")

y_pred_scaled = np.concatenate(all_pred_scaled, axis=0)
y_true_scaled = np.concatenate(all_true_scaled, axis=0)
time_index    = np.concatenate(all_time_index, axis=0)

print(f"[TEST] Aggregated test hours: {len(y_true_scaled)}")

# ---------------------------
# 6) Inverse scaling for biomass (using scaler_y from prep cell)
# ---------------------------
try:
    scaler_y  # just to check it exists
except NameError:
    raise RuntimeError("scaler_y is not available in this runtime. "
                       "You must run the prep cell (with scaler_y) before this.")

n_techs = len(gen_cols)
k       = tech_idx

# build placeholder arrays to inverse_transform
T = len(y_true_scaled)
y_true_scaled_full = np.zeros((T, n_techs))
y_pred_scaled_full = np.zeros((T, n_techs))

y_true_scaled_full[:, k] = y_true_scaled
y_pred_scaled_full[:, k] = y_pred_scaled

y_true_unscaled_full = scaler_y.inverse_transform(y_true_scaled_full)
y_pred_unscaled_full = scaler_y.inverse_transform(y_pred_scaled_full)

y_true = y_true_unscaled_full[:, k]
y_pred = y_pred_unscaled_full[:, k]

# ---------------------------
# 7) Compute hourly metrics
# ---------------------------
rmse_hourly = np.sqrt(mean_squared_error(y_true, y_pred))
mae_hourly  = mean_absolute_error(y_true, y_pred)

# MAPE (%)
eps = 1e-6
mape_hourly = np.mean(np.abs((y_true - y_pred) / np.clip(np.abs(y_true), eps, None))) * 100.0

mean_true = np.mean(y_true)
nmae_mean_hourly = mae_hourly / (mean_true + eps)

print("\n[METRICS – HOURLY 2024, BIOMASS]")
print(f"RMSE        = {rmse_hourly:.4f} MW")
print(f"MAE         = {mae_hourly:.4f} MW")
print(f"MAPE        = {mape_hourly:.2f} %")
print(f"nMAE_mean   = {nmae_mean_hourly:.4f}")

# ---------------------------
# 8) Daily metrics
# ---------------------------
# Build DataFrame with hourly results
hourly_df = pd.DataFrame({
    "datetime_utc": pd.to_datetime(time_index),
    "y_true_mw": y_true,
    "y_pred_mw": y_pred
}).set_index("datetime_utc").sort_index()

hourly_df["abs_error"] = np.abs(hourly_df["y_true_mw"] - hourly_df["y_pred_mw"])
hourly_df["sq_error"]  = (hourly_df["y_true_mw"] - hourly_df["y_pred_mw"])**2
hourly_df["ape"]       = np.where(
    np.abs(hourly_df["y_true_mw"]) > eps,
    np.abs(hourly_df["y_true_mw"] - hourly_df["y_pred_mw"]) / np.abs(hourly_df["y_true_mw"]),
    np.nan
)
# per-hour normalized abs error using global mean_true (nMAE_mean-style)
hourly_df["nabs_error_mean"] = hourly_df["abs_error"] / (mean_true + eps)

# daily aggregates
daily_groups = hourly_df.groupby(hourly_df.index.normalize())
daily_metrics = []

for day, group in daily_groups:
    if len(group) == 0:
        continue

    y_t = group["y_true_mw"].values
    y_p = group["y_pred_mw"].values

    rmse_d = np.sqrt(np.mean((y_t - y_p)**2))
    mae_d  = np.mean(np.abs(y_t - y_p))
    denom   = np.clip(np.abs(y_t), eps, None)
    mape_d  = np.mean(np.abs((y_t - y_p) / denom)) * 100.0
    mean_d  = np.mean(y_t)
    nmae_d  = mae_d / (mean_d + eps)

    daily_metrics.append({
        "date": day.date(),
        "RMSE_MW": rmse_d,
        "MAE_MW": mae_d,
        "MAPE_percent": mape_d,
        "nMAE_mean": nmae_d,
        "mean_true_MW": mean_d
    })

daily_df = pd.DataFrame(daily_metrics).sort_values("date")

print("\n[METRICS – DAILY 2024, BIOMASS]")
print(daily_df.describe()[["RMSE_MW", "MAE_MW", "MAPE_percent", "nMAE_mean"]])

# ---------------------------
# 9) Save CSVs (into tech subfolder)
# ---------------------------
hourly_csv_path = os.path.join(TECH_DIR, "biomass_hourly_predictions_2024.csv")
daily_csv_path  = os.path.join(TECH_DIR, "biomass_daily_metrics_2024.csv")
summary_csv_path = os.path.join(TECH_DIR, "biomass_overall_metrics_2024.csv")

hourly_df.reset_index().to_csv(hourly_csv_path, index=False)
daily_df.to_csv(daily_csv_path, index=False)

# small overall summary row
summary_df = pd.DataFrame([{
    "RMSE_MW": rmse_hourly,
    "MAE_MW": mae_hourly,
    "MAPE_percent": mape_hourly,
    "nMAE_mean": nmae_mean_hourly,
    "mean_true_MW": mean_true
}])
summary_df.to_csv(summary_csv_path, index=False)

print(f"\n[SAVE] Hourly predictions/errors -> {hourly_csv_path}")
print(f"[SAVE] Daily metrics             -> {daily_csv_path}")
print(f"[SAVE] Overall metrics           -> {summary_csv_path}")

# ---------------------------
# 10) Plots (save + show)
# ---------------------------

# (a) Hourly actual vs predicted for 2024
plt.figure(figsize=(14, 4))
plt.plot(hourly_df.index, hourly_df["y_true_mw"], label="Actual", linewidth=1)
plt.plot(hourly_df.index, hourly_df["y_pred_mw"], label="Predicted", linewidth=0.8, alpha=0.7)
plt.title("Biomass Generation – Hourly Actual vs Predicted (2024)")
plt.xlabel("Time (UTC)")
plt.ylabel("Power [MW]")
plt.legend()
plt.tight_layout()
plot1_path = os.path.join(TECH_DIR, "biomass_hourly_actual_vs_pred_2024.png")
plt.savefig(plot1_path, dpi=150)
plt.show()
plt.close()
print(f"[PLOT] Saved hourly actual vs predicted -> {plot1_path}")

# (b) Histogram of hourly normalized abs error (nMAE_mean-style)
plt.figure(figsize=(6, 4))
plt.hist(hourly_df["nabs_error_mean"].values, bins=50)
plt.title("Biomass – Distribution of Hourly Normalized Abs Error (2024)")
plt.xlabel("Normalized Abs Error (|e| / mean_true)")
plt.ylabel("Frequency [hours]")
plt.tight_layout()
plot2_path = os.path.join(TECH_DIR, "biomass_hourly_norm_abs_error_hist_2024.png")
plt.savefig(plot2_path, dpi=150)
plt.show()
plt.close()
print(f"[PLOT] Saved hourly normalized abs error histogram -> {plot2_path}")

print("\n[DONE] Per-tech MLP for biomass finished.")


### 1.2.3 Wind onshore


In [ ]:
# =========================
# CELL – Per-Tech MLP for Wind Onshore (CarbonCast-style)
# =========================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import timedelta

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, InputLayer
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import mean_squared_error, mean_absolute_error

# --------- CONFIG ----------
DATA_PROC_DIR = "/content/data_entsoe_es_processed"
TECH_NAME     = "gen_wind_onshore_mw"      # target tech
TRAIN_WIN_H   = 24
LABEL_WIN_H   = 24

# Per-tech subfolder
TECH_DIR = os.path.join(DATA_PROC_DIR, TECH_NAME)
os.makedirs(TECH_DIR, exist_ok=True)

# ---------------------------
# 1) Load feature lists to get gen/da/cal columns and tech index
# ---------------------------
feat_list_path = os.path.join(DATA_PROC_DIR, "feature_lists_entsoe.csv")
feat_df_list   = pd.read_csv(feat_list_path)

gen_cols = feat_df_list[feat_df_list["type"] == "gen"]["name"].tolist()
da_cols  = feat_df_list[feat_df_list["type"] == "da"]["name"].tolist()
cal_cols = feat_df_list[feat_df_list["type"] == "cal"]["name"].tolist()

if TECH_NAME not in gen_cols:
    raise ValueError(f"{TECH_NAME} not found in gen_cols: {gen_cols}")

tech_idx = gen_cols.index(TECH_NAME)
print(f"[TECH] Using tech={TECH_NAME}, index={tech_idx}")
print("[TECH] gen_cols:", gen_cols)
print("[TECH] da_cols :", da_cols)
print("[TECH] cal_cols:", cal_cols)

# Hist features = all gen + calendar; DA features as separate block
hist_feature_cols = gen_cols + cal_cols
da_feature_cols   = da_cols
all_feature_cols  = list(dict.fromkeys(hist_feature_cols + da_feature_cols))

# ---------------------------
# 2) Load sliding-window TRAIN / VAL data and reshape
# ---------------------------
X_train_path = os.path.join(DATA_PROC_DIR, "entsoe_train_X_sliding_24_24.csv")
Y_train_path = os.path.join(DATA_PROC_DIR, "entsoe_train_Y_sliding_24_24.csv")
X_val_path   = os.path.join(DATA_PROC_DIR, "entsoe_val_X_sliding_24_24.csv")
Y_val_path   = os.path.join(DATA_PROC_DIR, "entsoe_val_Y_sliding_24_24.csv")

X_train_flat = pd.read_csv(X_train_path).values  # (N_train, 24*(F_hist+F_da))
Y_train_flat = pd.read_csv(Y_train_path).values  # (N_train, 24 * n_gen_cols)
X_val_flat   = pd.read_csv(X_val_path).values
Y_val_flat   = pd.read_csv(Y_val_path).values

n_train = X_train_flat.shape[0]
n_val   = X_val_flat.shape[0]

# infer feature dimension per timestep and number of gen targets
F_total        = X_train_flat.shape[1] // TRAIN_WIN_H
n_gen_targets  = len(gen_cols)

if Y_train_flat.shape[1] != LABEL_WIN_H * n_gen_targets:
    raise RuntimeError("Y_train_flat shape is inconsistent with LABEL_WIN_H and n_gen_targets")

print(f"[SHAPE] X_train_flat: {X_train_flat.shape} -> (N, {TRAIN_WIN_H}, {F_total})")
print(f"[SHAPE] Y_train_flat: {Y_train_flat.shape} -> (N, {LABEL_WIN_H}, {n_gen_targets})")

# reshape to (N, 24, F_total) and (N, 24, n_gen_targets)
X_train_seq = X_train_flat.reshape(n_train, TRAIN_WIN_H, F_total)
X_val_seq   = X_val_flat.reshape(n_val,   TRAIN_WIN_H, F_total)

Y_train_all = Y_train_flat.reshape(n_train, LABEL_WIN_H, n_gen_targets)
Y_val_all   = Y_val_flat.reshape(n_val,   LABEL_WIN_H, n_gen_targets)

# extract wind onshore only: shape (N, 24)
Y_train = Y_train_all[:, :, tech_idx]
Y_val   = Y_val_all[:, :, tech_idx]

print(f"[SHAPE] X_train_seq: {X_train_seq.shape}, Y_train: {Y_train.shape}")
print(f"[SHAPE] X_val_seq  : {X_val_seq.shape}, Y_val  : {Y_val.shape}")

# ---------------------------
# 3) Build CarbonCast-style MLP for this tech
# ---------------------------
EPOCHS      = 100
BATCH_SIZE  = 10
ACTIVATION  = "relu"
LOSS_FUNC   = "mse"
LEARNING_RT = 0.001
HIDDEN      = [50, 34]

model = Sequential()
model.add(InputLayer(input_shape=(TRAIN_WIN_H, F_total)))
model.add(Flatten())
model.add(Dense(HIDDEN[0], activation=ACTIVATION))
model.add(Dense(HIDDEN[1], activation=ACTIVATION))
model.add(Dense(LABEL_WIN_H))  # 24 outputs (24h horizon)

opt = tf.keras.optimizers.Adam(learning_rate=LEARNING_RT)
model.compile(loss=LOSS_FUNC, optimizer=opt, metrics=["mae"])

model_summary = []
model.summary(print_fn=lambda x: model_summary.append(x))
print("\n".join(model_summary))

# EarlyStopping only – restore best weights in memory
es = EarlyStopping(
    monitor="val_loss",
    mode="min",
    patience=10,
    verbose=1,
    restore_best_weights=True,
)

# ---------------------------
# 4) Train model
# ---------------------------
history = model.fit(
    X_train_seq, Y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val_seq, Y_val),
    callbacks=[es],
    verbose=2
)

print("[TRAIN] Training finished. Best weights are already loaded in `model`.")
best_model = model

# Save model in native Keras format in tech subfolder (optional, no reload here)
model_path = os.path.join(TECH_DIR, "best_mlp_wind_onshore.keras")
best_model.save(model_path)
print(f"[TRAIN] Saved model to {model_path}")

# ---------------------------
# 5) Load hourly scaled VAL + TEST and build strict day-ahead forecasts for 2024
# ---------------------------
train_scaled_path = os.path.join(DATA_PROC_DIR, "entsoe_train_scaled_hourly.csv")
val_scaled_path   = os.path.join(DATA_PROC_DIR, "entsoe_val_scaled_hourly.csv")
test_scaled_path  = os.path.join(DATA_PROC_DIR, "entsoe_test_scaled_hourly.csv")

val_scaled  = pd.read_csv(val_scaled_path, parse_dates=["datetime_utc"]).set_index("datetime_utc").sort_index()
test_scaled = pd.read_csv(test_scaled_path, parse_dates=["datetime_utc"]).set_index("datetime_utc").sort_index()

print("[TEST] val_scaled :", val_scaled.shape,  "|", val_scaled.index.min(), "->", val_scaled.index.max())
print("[TEST] test_scaled:", test_scaled.shape, "|", test_scaled.index.min(),  "->", test_scaled.index.max())

# Combine val tail + test for history context
full_df = pd.concat([val_scaled, test_scaled], axis=0).sort_index()

# List of test days (UTC midnights) with full 24h coverage
test_days = sorted(test_scaled.index.normalize().unique())
print(f"[TEST] Number of test days in 2024: {len(test_days)}")

all_pred_scaled = []
all_true_scaled = []
all_time_index  = []

for day in test_days:
    horizon_start = pd.Timestamp(day)                      # e.g. 2024-01-01 00:00
    horizon_end   = horizon_start + timedelta(hours=23)    # inclusive

    # History: previous 24 hours
    hist_start = horizon_start - timedelta(hours=24)
    hist_end   = horizon_start - timedelta(hours=1)

    hist_slice    = full_df.loc[hist_start:hist_end]
    horizon_slice = full_df.loc[horizon_start:horizon_end]

    if len(hist_slice) != TRAIN_WIN_H or len(horizon_slice) != LABEL_WIN_H:
        print(f"[WARN] Skipping day {day.date()} due to incomplete history/horizon "
              f"({len(hist_slice)} hist hrs, {len(horizon_slice)} horizon hrs).")
        continue

    H = hist_slice[hist_feature_cols].values    # (24, F_hist)
    F = horizon_slice[da_feature_cols].values   # (24, F_da)

    if H.shape[0] != TRAIN_WIN_H or F.shape[0] != LABEL_WIN_H:
        print(f"[WARN] Shape mismatch on day {day.date()}, skipping.")
        continue

    X_day = np.concatenate([H, F], axis=1)  # (24, F_total) – must match training
    X_day = X_day.reshape(1, TRAIN_WIN_H, F_total)

    y_pred_day_scaled = best_model.predict(X_day, verbose=0)[0]  # (24,)
    y_true_day_scaled = horizon_slice[TECH_NAME].values          # (24,)

    all_pred_scaled.append(y_pred_day_scaled)
    all_true_scaled.append(y_true_day_scaled)
    all_time_index.append(horizon_slice.index.values)

# Stack to hourly arrays
if not all_pred_scaled:
    raise RuntimeError("[TEST] No valid test days were forecasted.")

y_pred_scaled = np.concatenate(all_pred_scaled, axis=0)
y_true_scaled = np.concatenate(all_true_scaled, axis=0)
time_index    = np.concatenate(all_time_index, axis=0)

print(f"[TEST] Aggregated test hours: {len(y_true_scaled)}")

# ---------------------------
# 6) Inverse scaling for wind onshore (using scaler_y from prep cell)
# ---------------------------
try:
    scaler_y  # just to check it exists
except NameError:
    raise RuntimeError("scaler_y is not available in this runtime. "
                       "You must run the prep cell (with scaler_y) before this.")

n_techs = len(gen_cols)
k       = tech_idx

# build placeholder arrays to inverse_transform
T = len(y_true_scaled)
y_true_scaled_full = np.zeros((T, n_techs))
y_pred_scaled_full = np.zeros((T, n_techs))

y_true_scaled_full[:, k] = y_true_scaled
y_pred_scaled_full[:, k] = y_pred_scaled

y_true_unscaled_full = scaler_y.inverse_transform(y_true_scaled_full)
y_pred_unscaled_full = scaler_y.inverse_transform(y_pred_scaled_full)

y_true = y_true_unscaled_full[:, k]
y_pred = y_pred_unscaled_full[:, k]

# ---------------------------
# 7) Compute hourly metrics
# ---------------------------
rmse_hourly = np.sqrt(mean_squared_error(y_true, y_pred))
mae_hourly  = mean_absolute_error(y_true, y_pred)

# MAPE (%)
eps = 1e-6
mape_hourly = np.mean(np.abs((y_true - y_pred) / np.clip(np.abs(y_true), eps, None))) * 100.0

mean_true = np.mean(y_true)
nmae_mean_hourly = mae_hourly / (mean_true + eps)

print("\n[METRICS – HOURLY 2024, WIND_ONSHORE]")
print(f"RMSE        = {rmse_hourly:.4f} MW")
print(f"MAE         = {mae_hourly:.4f} MW")
print(f"MAPE        = {mape_hourly:.2f} %")
print(f"nMAE_mean   = {nmae_mean_hourly:.4f}")

# ---------------------------
# 8) Daily metrics
# ---------------------------
# Build DataFrame with hourly results
hourly_df = pd.DataFrame({
    "datetime_utc": pd.to_datetime(time_index),
    "y_true_mw": y_true,
    "y_pred_mw": y_pred
}).set_index("datetime_utc").sort_index()

hourly_df["abs_error"] = np.abs(hourly_df["y_true_mw"] - hourly_df["y_pred_mw"])
hourly_df["sq_error"]  = (hourly_df["y_true_mw"] - hourly_df["y_pred_mw"])**2
hourly_df["ape"]       = np.where(
    np.abs(hourly_df["y_true_mw"]) > eps,
    np.abs(hourly_df["y_true_mw"] - hourly_df["y_pred_mw"]) / np.abs(hourly_df["y_true_mw"]),
    np.nan
)
# per-hour normalized abs error using global mean_true (nMAE_mean-style)
hourly_df["nabs_error_mean"] = hourly_df["abs_error"] / (mean_true + eps)

# daily aggregates
daily_groups = hourly_df.groupby(hourly_df.index.normalize())
daily_metrics = []

for day, group in daily_groups:
    if len(group) == 0:
        continue

    y_t = group["y_true_mw"].values
    y_p = group["y_pred_mw"].values

    rmse_d = np.sqrt(np.mean((y_t - y_p)**2))
    mae_d  = np.mean(np.abs(y_t - y_p))
    denom   = np.clip(np.abs(y_t), eps, None)
    mape_d  = np.mean(np.abs((y_t - y_p) / denom)) * 100.0
    mean_d  = np.mean(y_t)
    nmae_d  = mae_d / (mean_d + eps)

    daily_metrics.append({
        "date": day.date(),
        "RMSE_MW": rmse_d,
        "MAE_MW": mae_d,
        "MAPE_percent": mape_d,
        "nMAE_mean": nmae_d,
        "mean_true_MW": mean_d
    })

daily_df = pd.DataFrame(daily_metrics).sort_values("date")

print("\n[METRICS – DAILY 2024, WIND_ONSHORE]")
print(daily_df.describe()[["RMSE_MW", "MAE_MW", "MAPE_percent", "nMAE_mean"]])

# ---------------------------
# 9) Save CSVs (into tech subfolder)
# ---------------------------
hourly_csv_path  = os.path.join(TECH_DIR, "wind_onshore_hourly_predictions_2024.csv")
daily_csv_path   = os.path.join(TECH_DIR, "wind_onshore_daily_metrics_2024.csv")
summary_csv_path = os.path.join(TECH_DIR, "wind_onshore_overall_metrics_2024.csv")

hourly_df.reset_index().to_csv(hourly_csv_path, index=False)
daily_df.to_csv(daily_csv_path, index=False)

# small overall summary row
summary_df = pd.DataFrame([{
    "RMSE_MW": rmse_hourly,
    "MAE_MW": mae_hourly,
    "MAPE_percent": mape_hourly,
    "nMAE_mean": nmae_mean_hourly,
    "mean_true_MW": mean_true
}])
summary_df.to_csv(summary_csv_path, index=False)

print(f"\n[SAVE] Hourly predictions/errors -> {hourly_csv_path}")
print(f"[SAVE] Daily metrics             -> {daily_csv_path}")
print(f"[SAVE] Overall metrics           -> {summary_csv_path}")

# ---------------------------
# 10) Plots (save + show)
# ---------------------------

# (a) Hourly actual vs predicted for 2024
plt.figure(figsize=(14, 4))
plt.plot(hourly_df.index, hourly_df["y_true_mw"], label="Actual", linewidth=1)
plt.plot(hourly_df.index, hourly_df["y_pred_mw"], label="Predicted", linewidth=0.8, alpha=0.7)
plt.title("Wind Onshore Generation – Hourly Actual vs Predicted (2024)")
plt.xlabel("Time (UTC)")
plt.ylabel("Power [MW]")
plt.legend()
plt.tight_layout()
plot1_path = os.path.join(TECH_DIR, "wind_onshore_hourly_actual_vs_pred_2024.png")
plt.savefig(plot1_path, dpi=150)
plt.show()
plt.close()
print(f"[PLOT] Saved hourly actual vs predicted -> {plot1_path}")

# (b) Histogram of hourly normalized abs error (nMAE_mean-style)
plt.figure(figsize=(6, 4))
plt.hist(hourly_df["nabs_error_mean"].values, bins=50)
plt.title("Wind Onshore – Distribution of Hourly Normalized Abs Error (2024)")
plt.xlabel("Normalized Abs Error (|e| / mean_true)")
plt.ylabel("Frequency [hours]")
plt.tight_layout()
plot2_path = os.path.join(TECH_DIR, "wind_onshore_hourly_norm_abs_error_hist_2024.png")
plt.savefig(plot2_path, dpi=150)
plt.show()
plt.close()
print(f"[PLOT] Saved hourly normalized abs error histogram -> {plot2_path}")

print("\n[DONE] Per-tech MLP for wind onshore finished.")


### 1.2.4 Fossil gas


In [ ]:
# =========================
# CELL – Per-Tech MLP for Fossil Gas (CarbonCast-style)
# =========================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import timedelta

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, InputLayer
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import mean_squared_error, mean_absolute_error

# --------- CONFIG ----------
DATA_PROC_DIR = "/content/data_entsoe_es_processed"
TECH_NAME     = "gen_fossil_gas_mw"      # target tech
TRAIN_WIN_H   = 24
LABEL_WIN_H   = 24

# Per-tech subfolder
TECH_DIR = os.path.join(DATA_PROC_DIR, TECH_NAME)
os.makedirs(TECH_DIR, exist_ok=True)

# ---------------------------
# 1) Load feature lists to get gen/da/cal columns and tech index
# ---------------------------
feat_list_path = os.path.join(DATA_PROC_DIR, "feature_lists_entsoe.csv")
feat_df_list   = pd.read_csv(feat_list_path)

gen_cols = feat_df_list[feat_df_list["type"] == "gen"]["name"].tolist()
da_cols  = feat_df_list[feat_df_list["type"] == "da"]["name"].tolist()
cal_cols = feat_df_list[feat_df_list["type"] == "cal"]["name"].tolist()

if TECH_NAME not in gen_cols:
    raise ValueError(f"{TECH_NAME} not found in gen_cols.")

tech_idx = gen_cols.index(TECH_NAME)
print(f"[TECH] Using tech={TECH_NAME}, index={tech_idx}")

# Hist = all gen + calendar
hist_feature_cols = gen_cols + cal_cols
da_feature_cols   = da_cols
all_feature_cols  = list(dict.fromkeys(hist_feature_cols + da_feature_cols))

# ---------------------------
# 2) Load sliding-window TRAIN/VAL datasets
# ---------------------------
X_train = pd.read_csv(os.path.join(DATA_PROC_DIR, "entsoe_train_X_sliding_24_24.csv")).values
Y_train = pd.read_csv(os.path.join(DATA_PROC_DIR, "entsoe_train_Y_sliding_24_24.csv")).values
X_val   = pd.read_csv(os.path.join(DATA_PROC_DIR, "entsoe_val_X_sliding_24_24.csv")).values
Y_val   = pd.read_csv(os.path.join(DATA_PROC_DIR, "entsoe_val_Y_sliding_24_24.csv")).values

n_train = X_train.shape[0]
n_val   = X_val.shape[0]
F_total = X_train.shape[1] // TRAIN_WIN_H
n_gen_targets = len(gen_cols)

# reshape
X_train_seq = X_train.reshape(n_train, TRAIN_WIN_H, F_total)
X_val_seq   = X_val.reshape(n_val,   TRAIN_WIN_H, F_total)

Y_train_all = Y_train.reshape(n_train, LABEL_WIN_H, n_gen_targets)
Y_val_all   = Y_val.reshape(n_val,   LABEL_WIN_H, n_gen_targets)

# select fossil gas only
Y_train = Y_train_all[:, :, tech_idx]
Y_val   = Y_val_all[:, :, tech_idx]

print(f"[SHAPE] X_train_seq: {X_train_seq.shape}, Y_train: {Y_train.shape}")
print(f"[SHAPE] X_val_seq  : {X_val_seq.shape}, Y_val:   {Y_val.shape}")

# ---------------------------
# 3) Build CarbonCast-style MLP
# ---------------------------
EPOCHS      = 100
BATCH_SIZE  = 10
ACTIVATION  = "relu"
LOSS_FUNC   = "mse"
LEARNING_RT = 0.001
HIDDEN      = [50, 34]

model = Sequential([
    InputLayer(input_shape=(TRAIN_WIN_H, F_total)),
    Flatten(),
    Dense(HIDDEN[0], activation=ACTIVATION),
    Dense(HIDDEN[1], activation=ACTIVATION),
    Dense(LABEL_WIN_H)  # 24h output
])

model.compile(
    loss=LOSS_FUNC,
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RT),
    metrics=["mae"]
)

print(model.summary())

es = EarlyStopping(
    monitor="val_loss",
    mode="min",
    patience=10,
    verbose=1,
    restore_best_weights=True
)

# ---------------------------
# 4) Train
# ---------------------------
history = model.fit(
    X_train_seq, Y_train,
    validation_data=(X_val_seq, Y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[es],
    verbose=2
)

print("[TRAIN] Training finished. Best weights restored.")
best_model = model

model_path = os.path.join(TECH_DIR, "best_mlp_fossil_gas.keras")
best_model.save(model_path)
print(f"[TRAIN] Saved model to {model_path}")

# ---------------------------
# 5) Load scaled VAL + TEST for strict 24→24 forecasting
# ---------------------------
val_scaled  = pd.read_csv(os.path.join(DATA_PROC_DIR, "entsoe_val_scaled_hourly.csv"),
                          parse_dates=["datetime_utc"]).set_index("datetime_utc")
test_scaled = pd.read_csv(os.path.join(DATA_PROC_DIR, "entsoe_test_scaled_hourly.csv"),
                          parse_dates=["datetime_utc"]).set_index("datetime_utc")

full_df = pd.concat([val_scaled, test_scaled]).sort_index()

test_days = sorted(test_scaled.index.normalize().unique())
print(f"[TEST] Days in 2024: {len(test_days)}")

all_pred_scaled = []
all_true_scaled = []
all_times = []

for day in test_days:
    h0 = pd.Timestamp(day)
    h1 = h0 + timedelta(hours=23)

    hist = full_df.loc[h0 - timedelta(hours=24):h0 - timedelta(hours=1)]
    horiz = full_df.loc[h0:h1]

    if len(hist) != 24 or len(horiz) != 24:
        continue

    H = hist[hist_feature_cols].values
    F = horiz[da_feature_cols].values
    X_day = np.concatenate([H, F], axis=1).reshape(1, 24, F_total)

    y_pred_s = best_model.predict(X_day, verbose=0)[0]
    y_true_s = horiz[TECH_NAME].values

    all_pred_scaled.append(y_pred_s)
    all_true_scaled.append(y_true_s)
    all_times.append(horiz.index.values)

y_pred_scaled = np.concatenate(all_pred_scaled)
y_true_scaled = np.concatenate(all_true_scaled)
time_index    = np.concatenate(all_times)

print(f"[TEST] Total hourly predictions: {len(y_true_scaled)}")

# ---------------------------
# 6) Inverse scaling
# ---------------------------
try:
    scaler_y
except NameError:
    raise RuntimeError("Run prep cell first to load scaler_y.")

T = len(y_true_scaled)
n_techs = len(gen_cols)
k       = tech_idx

Y_true_full = np.zeros((T, n_techs))
Y_pred_full = np.zeros((T, n_techs))

Y_true_full[:, k] = y_true_scaled
Y_pred_full[:, k] = y_pred_scaled

y_true = scaler_y.inverse_transform(Y_true_full)[:, k]
y_pred = scaler_y.inverse_transform(Y_pred_full)[:, k]

# ---------------------------
# 7) Hourly metrics
# ---------------------------
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mae  = mean_absolute_error(y_true, y_pred)
eps  = 1e-6
mape = np.mean(np.abs((y_true - y_pred) / np.clip(np.abs(y_true), eps, None))) * 100
mean_true = np.mean(y_true)
nmae = mae / (mean_true + eps)

print("\n[METRICS – HOURLY 2024, FOSSIL GAS]")
print(f"RMSE      = {rmse:.3f} MW")
print(f"MAE       = {mae:.3f} MW")
print(f"MAPE      = {mape:.2f} %")
print(f"nMAE_mean = {nmae:.4f}")

# ---------------------------
# 8) Daily metrics
# ---------------------------
hourly_df = pd.DataFrame({
    "datetime_utc": time_index,
    "y_true": y_true,
    "y_pred": y_pred
}).set_index("datetime_utc").sort_index()

hourly_df["abs_err"] = np.abs(hourly_df["y_true"] - hourly_df["y_pred"])
hourly_df["nabs_err_mean"] = hourly_df["abs_err"] / (mean_true + eps)

daily_df = hourly_df.groupby(hourly_df.index.normalize()).apply(
    lambda g: pd.Series({
        "RMSE_MW": np.sqrt(np.mean((g.y_true - g.y_pred)**2)),
        "MAE_MW": np.mean(np.abs(g.y_true - g.y_pred)),
        "MAPE_percent": np.mean(
            np.abs((g.y_true - g.y_pred) /
                   np.clip(np.abs(g.y_true), eps, None))
        ) * 100,
        "nMAE_mean": np.mean(np.abs(g.y_true - g.y_pred)) / (np.mean(g.y_true) + eps),
        "mean_true_MW": np.mean(g.y_true)
    })
)

print("\n[METRICS – DAILY 2024, FOSSIL GAS]")
print(daily_df.describe()[["RMSE_MW", "MAE_MW", "MAPE_percent", "nMAE_mean"]])

# ---------------------------
# 9) Save results
# ---------------------------
hourly_df.to_csv(os.path.join(TECH_DIR, "fossil_gas_hourly_predictions_2024.csv"))
daily_df.to_csv(os.path.join(TECH_DIR, "fossil_gas_daily_metrics_2024.csv"))

pd.DataFrame([{
    "RMSE_MW": rmse,
    "MAE_MW": mae,
    "MAPE_percent": mape,
    "nMAE_mean": nmae,
    "mean_true_MW": mean_true
}]).to_csv(os.path.join(TECH_DIR, "fossil_gas_overall_metrics_2024.csv"), index=False)

# ---------------------------
# 10) Plots
# ---------------------------
plt.figure(figsize=(14,4))
plt.plot(hourly_df.index, hourly_df.y_true, label="Actual", linewidth=1)
plt.plot(hourly_df.index, hourly_df.y_pred, label="Predicted", linewidth=0.8)
plt.title("Fossil Gas – Hourly Actual vs Predicted (2024)")
plt.xlabel("Time (UTC)")
plt.ylabel("Power [MW]")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(TECH_DIR, "fossil_gas_hourly_actual_vs_pred_2024.png"), dpi=150)
plt.show()

plt.figure(figsize=(6,4))
plt.hist(hourly_df["nabs_err_mean"], bins=50)
plt.title("Fossil Gas – Distribution of Hourly Normalized Abs Error (2024)")
plt.xlabel("Normalized Abs Error (|e| / mean_true)")
plt.ylabel("Frequency")
plt.tight_layout()
plt.savefig(os.path.join(TECH_DIR, "fossil_gas_hourly_norm_abs_error_hist_2024.png"), dpi=150)
plt.show()

print("\n[DONE] Per-tech MLP for fossil gas finished.")


### 1.2.5 Nuclear


In [ ]:
# =========================
# CELL – Per-Tech MLP for Nuclear (CarbonCast-style)
# =========================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import timedelta

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, InputLayer
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import mean_squared_error, mean_absolute_error

# --------- CONFIG ----------
DATA_PROC_DIR = "/content/data_entsoe_es_processed"
TECH_NAME     = "gen_nuclear_mw"      # target tech
TRAIN_WIN_H   = 24
LABEL_WIN_H   = 24

# Per-tech subfolder
TECH_DIR = os.path.join(DATA_PROC_DIR, TECH_NAME)
os.makedirs(TECH_DIR, exist_ok=True)

# ---------------------------
# 1) Load feature lists to get gen/da/cal columns and tech index
# ---------------------------
feat_list_path = os.path.join(DATA_PROC_DIR, "feature_lists_entsoe.csv")
feat_df_list   = pd.read_csv(feat_list_path)

gen_cols = feat_df_list[feat_df_list["type"] == "gen"]["name"].tolist()
da_cols  = feat_df_list[feat_df_list["type"] == "da"]["name"].tolist()
cal_cols = feat_df_list[feat_df_list["type"] == "cal"]["name"].tolist()

if TECH_NAME not in gen_cols:
    raise ValueError(f"{TECH_NAME} not found in gen_cols: {gen_cols}")

tech_idx = gen_cols.index(TECH_NAME)
print(f"[TECH] Using tech={TECH_NAME}, index={tech_idx}")
print("[TECH] gen_cols:", gen_cols)
print("[TECH] da_cols :", da_cols)
print("[TECH] cal_cols:", cal_cols)

# Hist features = all gen + calendar; DA features as separate block
hist_feature_cols = gen_cols + cal_cols
da_feature_cols   = da_cols
all_feature_cols  = list(dict.fromkeys(hist_feature_cols + da_feature_cols))

# ---------------------------
# 2) Load sliding-window TRAIN / VAL data and reshape
# ---------------------------
X_train_path = os.path.join(DATA_PROC_DIR, "entsoe_train_X_sliding_24_24.csv")
Y_train_path = os.path.join(DATA_PROC_DIR, "entsoe_train_Y_sliding_24_24.csv")
X_val_path   = os.path.join(DATA_PROC_DIR, "entsoe_val_X_sliding_24_24.csv")
Y_val_path   = os.path.join(DATA_PROC_DIR, "entsoe_val_Y_sliding_24_24.csv")

X_train_flat = pd.read_csv(X_train_path).values
Y_train_flat = pd.read_csv(Y_train_path).values
X_val_flat   = pd.read_csv(X_val_path).values
Y_val_flat   = pd.read_csv(Y_val_path).values

n_train = X_train_flat.shape[0]
n_val   = X_val_flat.shape[0]

F_total        = X_train_flat.shape[1] // TRAIN_WIN_H
n_gen_targets  = len(gen_cols)

print(f"[SHAPE] X_train_flat: {X_train_flat.shape} -> (N, {TRAIN_WIN_H}, {F_total})")
print(f"[SHAPE] Y_train_flat: {Y_train_flat.shape} -> (N, {LABEL_WIN_H}, {n_gen_targets})")

X_train_seq = X_train_flat.reshape(n_train, TRAIN_WIN_H, F_total)
X_val_seq   = X_val_flat.reshape(n_val,   TRAIN_WIN_H, F_total)

Y_train_all = Y_train_flat.reshape(n_train, LABEL_WIN_H, n_gen_targets)
Y_val_all   = Y_val_flat.reshape(n_val,   LABEL_WIN_H, n_gen_targets)

Y_train = Y_train_all[:, :, tech_idx]
Y_val   = Y_val_all[:, :, tech_idx]

print(f"[SHAPE] X_train_seq: {X_train_seq.shape}, Y_train: {Y_train.shape}")
print(f"[SHAPE] X_val_seq  : {X_val_seq.shape}, Y_val  : {Y_val.shape}")

# ---------------------------
# 3) Build CarbonCast-style MLP for this tech
# ---------------------------
EPOCHS      = 100
BATCH_SIZE  = 10
ACTIVATION  = "relu"
LOSS_FUNC   = "mse"
LEARNING_RT = 0.01
HIDDEN      = [40, 34]

model = Sequential()
model.add(InputLayer(input_shape=(TRAIN_WIN_H, F_total)))
model.add(Flatten())
model.add(Dense(HIDDEN[0], activation=ACTIVATION))
model.add(Dense(HIDDEN[1], activation=ACTIVATION))
model.add(Dense(LABEL_WIN_H))  # 24 outputs

opt = tf.keras.optimizers.Adam(learning_rate=LEARNING_RT)
model.compile(loss=LOSS_FUNC, optimizer=opt, metrics=["mae"])

model.summary()

# EarlyStopping
es = EarlyStopping(
    monitor="val_loss",
    mode="min",
    patience=10,
    verbose=1,
    restore_best_weights=True,
)

# ---------------------------
# 4) Train model
# ---------------------------
history = model.fit(
    X_train_seq, Y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val_seq, Y_val),
    callbacks=[es],
    verbose=2
)

print("[TRAIN] Training finished. Best weights restored.")
best_model = model

# Save
model_path = os.path.join(TECH_DIR, "best_mlp_nuclear.keras")
best_model.save(model_path)
print(f"[TRAIN] Saved model to {model_path}")

# ---------------------------
# 5) Strict day-ahead prediction on 2024
# ---------------------------
train_scaled_path = os.path.join(DATA_PROC_DIR, "entsoe_train_scaled_hourly.csv")
val_scaled_path   = os.path.join(DATA_PROC_DIR, "entsoe_val_scaled_hourly.csv")
test_scaled_path  = os.path.join(DATA_PROC_DIR, "entsoe_test_scaled_hourly.csv")

val_scaled  = pd.read_csv(val_scaled_path, parse_dates=["datetime_utc"]).set_index("datetime_utc").sort_index()
test_scaled = pd.read_csv(test_scaled_path, parse_dates=["datetime_utc"]).set_index("datetime_utc").sort_index()

full_df = pd.concat([val_scaled, test_scaled], axis=0).sort_index()
test_days = sorted(test_scaled.index.normalize().unique())
print(f"[TEST] Days in 2024: {len(test_days)}")

all_pred_scaled = []
all_true_scaled = []
all_time_index  = []

for day in test_days:
    horizon_start = pd.Timestamp(day)
    horizon_end   = horizon_start + timedelta(hours=23)

    hist_slice    = full_df.loc[horizon_start - timedelta(hours=24):horizon_start - timedelta(hours=1)]
    horizon_slice = full_df.loc[horizon_start:horizon_end]

    if len(hist_slice) != 24 or len(horizon_slice) != 24:
        print(f"[WARN] Skipping {day}")
        continue

    H = hist_slice[hist_feature_cols].values
    F = horizon_slice[da_feature_cols].values

    X_day = np.concatenate([H, F], axis=1).reshape(1, 24, F_total)

    y_pred_day_scaled = best_model.predict(X_day, verbose=0)[0]
    y_true_day_scaled = horizon_slice[TECH_NAME].values

    all_pred_scaled.append(y_pred_day_scaled)
    all_true_scaled.append(y_true_day_scaled)
    all_time_index.append(horizon_slice.index.values)

y_pred_scaled = np.concatenate(all_pred_scaled)
y_true_scaled = np.concatenate(all_true_scaled)
time_index    = np.concatenate(all_time_index)

print(f"[TEST] Total hourly predictions: {len(y_true_scaled)}")

# ---------------------------
# 6) Inverse scaling
# ---------------------------
n_techs = len(gen_cols)
k       = tech_idx
T       = len(y_true_scaled)

y_true_scaled_full = np.zeros((T, n_techs))
y_pred_scaled_full = np.zeros((T, n_techs))

y_true_scaled_full[:, k] = y_true_scaled
y_pred_scaled_full[:, k] = y_pred_scaled

y_true_unscaled_full = scaler_y.inverse_transform(y_true_scaled_full)
y_pred_unscaled_full = scaler_y.inverse_transform(y_pred_scaled_full)

y_true = y_true_unscaled_full[:, k]
y_pred = y_pred_unscaled_full[:, k]

# ---------------------------
# 7) Hourly metrics
# ---------------------------
rmse_hourly = np.sqrt(mean_squared_error(y_true, y_pred))
mae_hourly  = mean_absolute_error(y_true, y_pred)

eps = 1e-6
mape_hourly = np.mean(np.abs((y_true - y_pred) /
                             np.clip(np.abs(y_true), eps, None))) * 100

mean_true = np.mean(y_true)
nmae_mean_hourly = mae_hourly / (mean_true + eps)

print("\n[METRICS – HOURLY 2024, NUCLEAR]")
print(f"RMSE      = {rmse_hourly:.4f} MW")
print(f"MAE       = {mae_hourly:.4f} MW")
print(f"MAPE      = {mape_hourly:.2f} %")
print(f"nMAE_mean = {nmae_mean_hourly:.4f}")

# ---------------------------
# 8) Daily metrics
# ---------------------------
hourly_df = pd.DataFrame({
    "datetime_utc": pd.to_datetime(time_index),
    "y_true_mw": y_true,
    "y_pred_mw": y_pred
}).set_index("datetime_utc").sort_index()

hourly_df["abs_error"] = np.abs(hourly_df["y_true_mw"] - hourly_df["y_pred_mw"])
hourly_df["sq_error"]  = (hourly_df["y_true_mw"] - hourly_df["y_pred_mw"])**2
hourly_df["nabs_error_mean"] = hourly_df["abs_error"] / (mean_true + eps)

daily_df = hourly_df.groupby(hourly_df.index.normalize()).agg({
    "sq_error": lambda x: np.sqrt(np.mean(x)),
    "abs_error": "mean",
    "y_true_mw": "mean"
})

daily_df.rename(columns={
    "sq_error": "RMSE_MW",
    "abs_error": "MAE_MW",
    "y_true_mw": "mean_true_MW"
}, inplace=True)

daily_df["MAPE_percent"] = (daily_df["MAE_MW"] / np.clip(daily_df["mean_true_MW"], eps, None)) * 100
daily_df["nMAE_mean"]    = daily_df["MAE_MW"] / (daily_df["mean_true_MW"] + eps)

print("\n[METRICS – DAILY 2024, NUCLEAR]")
print(daily_df.describe()[["RMSE_MW", "MAE_MW", "MAPE_percent", "nMAE_mean"]])

# ---------------------------
# 9) Save CSVs
# ---------------------------
hourly_csv_path  = os.path.join(TECH_DIR, "nuclear_hourly_predictions_2024.csv")
daily_csv_path   = os.path.join(TECH_DIR, "nuclear_daily_metrics_2024.csv")
summary_csv_path = os.path.join(TECH_DIR, "nuclear_overall_metrics_2024.csv")

hourly_df.reset_index().to_csv(hourly_csv_path, index=False)
daily_df.to_csv(daily_csv_path, index=False)

pd.DataFrame([{
    "RMSE_MW": rmse_hourly,
    "MAE_MW": mae_hourly,
    "MAPE_percent": mape_hourly,
    "nMAE_mean": nmae_mean_hourly,
    "mean_true_MW": mean_true
}]).to_csv(summary_csv_path, index=False)

print(f"[SAVE] Hourly  -> {hourly_csv_path}")
print(f"[SAVE] Daily   -> {daily_csv_path}")
print(f"[SAVE] Summary -> {summary_csv_path}")

# ---------------------------
# 10) Plots
# ---------------------------
plt.figure(figsize=(14, 4))
plt.plot(hourly_df.index, hourly_df["y_true_mw"], label="Actual", linewidth=1)
plt.plot(hourly_df.index, hourly_df["y_pred_mw"], label="Predicted", linewidth=0.8, alpha=0.7)
plt.title("Nuclear – Hourly Actual vs Predicted (2024)")
plt.xlabel("Time (UTC)")
plt.ylabel("Power [MW]")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(TECH_DIR, "nuclear_hourly_actual_vs_pred_2024.png"), dpi=150)
plt.show()

plt.figure(figsize=(6, 4))
plt.hist(hourly_df["nabs_error_mean"], bins=50)
plt.title("Nuclear – Distribution of Hourly Normalized Abs Error (2024)")
plt.xlabel("Normalized Abs Error (|e| / mean_true)")
plt.ylabel("Frequency")
plt.tight_layout()
plt.savefig(os.path.join(TECH_DIR, "nuclear_hourly_norm_abs_error_hist_2024.png"), dpi=150)
plt.show()

print("\n[DONE] Per-tech MLP for nuclear finished.")


### 1.2.6 Solar


In [ ]:
# =========================
# CELL – Per-Tech MLP for Solar (CarbonCast-style)
# =========================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import timedelta

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, InputLayer
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import mean_squared_error, mean_absolute_error

# --------- CONFIG ----------
DATA_PROC_DIR = "/content/data_entsoe_es_processed"
TECH_NAME     = "gen_solar_mw"   # target tech
TRAIN_WIN_H   = 24
LABEL_WIN_H   = 24

# Per-tech subfolder
TECH_DIR = os.path.join(DATA_PROC_DIR, TECH_NAME)
os.makedirs(TECH_DIR, exist_ok=True)

# ---------------------------
# 1) Load feature lists to get gen/da/cal columns and tech index
# ---------------------------
feat_list_path = os.path.join(DATA_PROC_DIR, "feature_lists_entsoe.csv")
feat_df_list   = pd.read_csv(feat_list_path)

gen_cols = feat_df_list[feat_df_list["type"] == "gen"]["name"].tolist()
da_cols  = feat_df_list[feat_df_list["type"] == "da"]["name"].tolist()
cal_cols = feat_df_list[feat_df_list["type"] == "cal"]["name"].tolist()

if TECH_NAME not in gen_cols:
    raise ValueError(f"{TECH_NAME} not found in gen_cols: {gen_cols}")

tech_idx = gen_cols.index(TECH_NAME)
print(f"[TECH] Using tech={TECH_NAME}, index={tech_idx}")
print("[TECH] gen_cols:", gen_cols)
print("[TECH] da_cols :", da_cols)
print("[TECH] cal_cols:", cal_cols)

# Hist features = all gen + calendar; DA features as separate block
hist_feature_cols = gen_cols + cal_cols
da_feature_cols   = da_cols
all_feature_cols  = list(dict.fromkeys(hist_feature_cols + da_feature_cols))

# ---------------------------
# 2) Load sliding-window TRAIN / VAL data and reshape
# ---------------------------
X_train_path = os.path.join(DATA_PROC_DIR, "entsoe_train_X_sliding_24_24.csv")
Y_train_path = os.path.join(DATA_PROC_DIR, "entsoe_train_Y_sliding_24_24.csv")
X_val_path   = os.path.join(DATA_PROC_DIR, "entsoe_val_X_sliding_24_24.csv")
Y_val_path   = os.path.join(DATA_PROC_DIR, "entsoe_val_Y_sliding_24_24.csv")

X_train_flat = pd.read_csv(X_train_path).values  # (N_train, 24*(F_hist+F_da))
Y_train_flat = pd.read_csv(Y_train_path).values  # (N_train, 24 * n_gen_cols)
X_val_flat   = pd.read_csv(X_val_path).values
Y_val_flat   = pd.read_csv(Y_val_path).values

n_train = X_train_flat.shape[0]
n_val   = X_val_flat.shape[0]

# infer feature dimension per timestep and number of gen targets
F_total       = X_train_flat.shape[1] // TRAIN_WIN_H
n_gen_targets = len(gen_cols)

if Y_train_flat.shape[1] != LABEL_WIN_H * n_gen_targets:
    raise RuntimeError("Y_train_flat shape is inconsistent with LABEL_WIN_H and n_gen_targets")

print(f"[SHAPE] X_train_flat: {X_train_flat.shape} -> (N, {TRAIN_WIN_H}, {F_total})")
print(f"[SHAPE] Y_train_flat: {Y_train_flat.shape} -> (N, {LABEL_WIN_H}, {n_gen_targets})")

# reshape to (N, 24, F_total) and (N, 24, n_gen_targets)
X_train_seq = X_train_flat.reshape(n_train, TRAIN_WIN_H, F_total)
X_val_seq   = X_val_flat.reshape(n_val,   TRAIN_WIN_H, F_total)

Y_train_all = Y_train_flat.reshape(n_train, LABEL_WIN_H, n_gen_targets)
Y_val_all   = Y_val_flat.reshape(n_val,   LABEL_WIN_H, n_gen_targets)

# extract solar only: shape (N, 24)
Y_train = Y_train_all[:, :, tech_idx]
Y_val   = Y_val_all[:, :, tech_idx]

print(f"[SHAPE] X_train_seq: {X_train_seq.shape}, Y_train: {Y_train.shape}")
print(f"[SHAPE] X_val_seq  : {X_val_seq.shape}, Y_val  : {Y_val.shape}")

# ---------------------------
# 3) Build CarbonCast-style MLP for this tech
# ---------------------------
EPOCHS      = 100
BATCH_SIZE  = 10
ACTIVATION  = "relu"
LOSS_FUNC   = "mse"
LEARNING_RT = 0.001
HIDDEN      = [50, 34]

model = Sequential()
model.add(InputLayer(input_shape=(TRAIN_WIN_H, F_total)))
model.add(Flatten())
model.add(Dense(HIDDEN[0], activation=ACTIVATION))
model.add(Dense(HIDDEN[1], activation=ACTIVATION))
model.add(Dense(LABEL_WIN_H))  # 24 outputs (24h horizon)

opt = tf.keras.optimizers.Adam(learning_rate=LEARNING_RT)
model.compile(loss=LOSS_FUNC, optimizer=opt, metrics=["mae"])

model_summary = []
model.summary(print_fn=lambda x: model_summary.append(x))
print("\n".join(model_summary))

# EarlyStopping – restore best weights in memory
es = EarlyStopping(
    monitor="val_loss",
    mode="min",
    patience=10,
    verbose=1,
    restore_best_weights=True,
)

# ---------------------------
# 4) Train model
# ---------------------------
history = model.fit(
    X_train_seq, Y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val_seq, Y_val),
    callbacks=[es],
    verbose=2
)

print("[TRAIN] Training finished. Best weights restored.")
best_model = model

# Save model in native Keras format in tech subfolder
model_path = os.path.join(TECH_DIR, "best_mlp_solar.keras")
best_model.save(model_path)
print(f"[TRAIN] Saved model to {model_path}")

# ---------------------------
# 5) Load hourly scaled VAL + TEST and build strict day-ahead forecasts for 2024
# ---------------------------
train_scaled_path = os.path.join(DATA_PROC_DIR, "entsoe_train_scaled_hourly.csv")
val_scaled_path   = os.path.join(DATA_PROC_DIR, "entsoe_val_scaled_hourly.csv")
test_scaled_path  = os.path.join(DATA_PROC_DIR, "entsoe_test_scaled_hourly.csv")

val_scaled  = pd.read_csv(val_scaled_path, parse_dates=["datetime_utc"]).set_index("datetime_utc").sort_index()
test_scaled = pd.read_csv(test_scaled_path, parse_dates=["datetime_utc"]).set_index("datetime_utc").sort_index()

print("[TEST] val_scaled :", val_scaled.shape,  "|", val_scaled.index.min(), "->", val_scaled.index.max())
print("[TEST] test_scaled:", test_scaled.shape, "|", test_scaled.index.min(),  "->", test_scaled.index.max())

# Combine val tail + test for history context
full_df = pd.concat([val_scaled, test_scaled], axis=0).sort_index()

# List of test days (UTC midnights) with full 24h coverage
test_days = sorted(test_scaled.index.normalize().unique())
print(f"[TEST] Days in 2024: {len(test_days)}")

all_pred_scaled = []
all_true_scaled = []
all_time_index  = []

for day in test_days:
    horizon_start = pd.Timestamp(day)                   # e.g. 2024-01-01 00:00
    horizon_end   = horizon_start + timedelta(hours=23) # inclusive

    # History: previous 24 hours
    hist_start = horizon_start - timedelta(hours=24)
    hist_end   = horizon_start - timedelta(hours=1)

    hist_slice    = full_df.loc[hist_start:hist_end]
    horizon_slice = full_df.loc[horizon_start:horizon_end]

    if len(hist_slice) != TRAIN_WIN_H or len(horizon_slice) != LABEL_WIN_H:
        print(f"[WARN] Skipping day {day.date()} due to incomplete history/horizon "
              f"({len(hist_slice)} hist hrs, {len(horizon_slice)} horizon hrs).")
        continue

    H = hist_slice[hist_feature_cols].values    # (24, F_hist)
    F = horizon_slice[da_feature_cols].values   # (24, F_da)

    if H.shape[0] != TRAIN_WIN_H or F.shape[0] != LABEL_WIN_H:
        print(f"[WARN] Shape mismatch on day {day.date()}, skipping.")
        continue

    X_day = np.concatenate([H, F], axis=1)  # (24, F_total)
    X_day = X_day.reshape(1, TRAIN_WIN_H, F_total)

    y_pred_day_scaled = best_model.predict(X_day, verbose=0)[0]  # (24,)
    y_true_day_scaled = horizon_slice[TECH_NAME].values          # (24,)

    all_pred_scaled.append(y_pred_day_scaled)
    all_true_scaled.append(y_true_day_scaled)
    all_time_index.append(horizon_slice.index.values)

# Stack to hourly arrays
if not all_pred_scaled:
    raise RuntimeError("[TEST] No valid test days were forecasted.")

y_pred_scaled = np.concatenate(all_pred_scaled, axis=0)
y_true_scaled = np.concatenate(all_true_scaled, axis=0)
time_index    = np.concatenate(all_time_index, axis=0)

print(f"[TEST] Total hourly predictions: {len(y_true_scaled)}")

# ---------------------------
# 6) Inverse scaling for solar (using scaler_y from prep cell)
# ---------------------------
try:
    scaler_y  # just to check it exists
except NameError:
    raise RuntimeError("scaler_y is not available in this runtime. "
                       "You must run the prep cell (with scaler_y) before this.")

n_techs = len(gen_cols)
k       = tech_idx

T = len(y_true_scaled)
y_true_scaled_full = np.zeros((T, n_techs))
y_pred_scaled_full = np.zeros((T, n_techs))

y_true_scaled_full[:, k] = y_true_scaled
y_pred_scaled_full[:, k] = y_pred_scaled

y_true_unscaled_full = scaler_y.inverse_transform(y_true_scaled_full)
y_pred_unscaled_full = scaler_y.inverse_transform(y_pred_scaled_full)

y_true = y_true_unscaled_full[:, k]
y_pred = y_pred_unscaled_full[:, k]
y_pred = np.maximum(y_pred, 0.0)
# ---------------------------
# 7) Compute hourly metrics
# ---------------------------
rmse_hourly = np.sqrt(mean_squared_error(y_true, y_pred))
mae_hourly  = mean_absolute_error(y_true, y_pred)

eps = 1e-6
mape_hourly = np.mean(np.abs((y_true - y_pred) /
                             np.clip(np.abs(y_true), eps, None))) * 100.0

mean_true = np.mean(y_true)
nmae_mean_hourly = mae_hourly / (mean_true + eps)

print("\n[METRICS – HOURLY 2024, SOLAR]")
print(f"RMSE      = {rmse_hourly:.4f} MW")
print(f"MAE       = {mae_hourly:.4f} MW")
print(f"MAPE      = {mape_hourly:.2f} %")
print(f"nMAE_mean = {nmae_mean_hourly:.4f}")

# ---------------------------
# 8) Daily metrics
# ---------------------------
hourly_df = pd.DataFrame({
    "datetime_utc": pd.to_datetime(time_index),
    "y_true_mw": y_true,
    "y_pred_mw": y_pred
}).set_index("datetime_utc").sort_index()

hourly_df["abs_error"] = np.abs(hourly_df["y_true_mw"] - hourly_df["y_pred_mw"])
hourly_df["sq_error"]  = (hourly_df["y_true_mw"] - hourly_df["y_pred_mw"])**2
hourly_df["ape"]       = np.where(
    np.abs(hourly_df["y_true_mw"]) > eps,
    np.abs(hourly_df["y_true_mw"] - hourly_df["y_pred_mw"]) /
    np.abs(hourly_df["y_true_mw"]),
    np.nan
)
hourly_df["nabs_error_mean"] = hourly_df["abs_error"] / (mean_true + eps)

daily_groups = hourly_df.groupby(hourly_df.index.normalize())
daily_metrics = []

for day, group in daily_groups:
    if len(group) == 0:
        continue

    y_t = group["y_true_mw"].values
    y_p = group["y_pred_mw"].values

    rmse_d = np.sqrt(np.mean((y_t - y_p)**2))
    mae_d  = np.mean(np.abs(y_t - y_p))
    denom  = np.clip(np.abs(y_t), eps, None)
    mape_d = np.mean(np.abs((y_t - y_p) / denom)) * 100.0
    mean_d = np.mean(y_t)
    nmae_d = mae_d / (mean_d + eps)

    daily_metrics.append({
        "date": day.date(),
        "RMSE_MW": rmse_d,
        "MAE_MW": mae_d,
        "MAPE_percent": mape_d,
        "nMAE_mean": nmae_d,
        "mean_true_MW": mean_d
    })

daily_df = pd.DataFrame(daily_metrics).sort_values("date")

print("\n[METRICS – DAILY 2024, SOLAR]")
print(daily_df.describe()[["RMSE_MW", "MAE_MW", "MAPE_percent", "nMAE_mean"]])

# ---------------------------
# 9) Save CSVs
# ---------------------------
hourly_csv_path  = os.path.join(TECH_DIR, "solar_hourly_predictions_2024.csv")
daily_csv_path   = os.path.join(TECH_DIR, "solar_daily_metrics_2024.csv")
summary_csv_path = os.path.join(TECH_DIR, "solar_overall_metrics_2024.csv")

hourly_df.reset_index().to_csv(hourly_csv_path, index=False)
daily_df.to_csv(daily_csv_path, index=False)

summary_df = pd.DataFrame([{
    "RMSE_MW": rmse_hourly,
    "MAE_MW": mae_hourly,
    "MAPE_percent": mape_hourly,
    "nMAE_mean": nmae_mean_hourly,
    "mean_true_MW": mean_true
}])
summary_df.to_csv(summary_csv_path, index=False)

print(f"\n[SAVE] Hourly  -> {hourly_csv_path}")
print(f"[SAVE] Daily   -> {daily_csv_path}")
print(f"[SAVE] Summary -> {summary_csv_path}")

# ---------------------------
# 10) Plots
# ---------------------------

# (a) Hourly actual vs predicted
plt.figure(figsize=(14, 4))
plt.plot(hourly_df.index, hourly_df["y_true_mw"], label="Actual", linewidth=1)
plt.plot(hourly_df.index, hourly_df["y_pred_mw"], label="Predicted",
         linewidth=0.8, alpha=0.7)
plt.title("Solar – Hourly Actual vs Predicted (2024)")
plt.xlabel("Time (UTC)")
plt.ylabel("Power [MW]")
plt.legend()
plt.tight_layout()
plot1_path = os.path.join(TECH_DIR, "solar_hourly_actual_vs_pred_2024.png")
plt.savefig(plot1_path, dpi=150)
plt.show()
plt.close()
print(f"[PLOT] Saved hourly actual vs predicted -> {plot1_path}")

# (b) Histogram of hourly normalized abs error
plt.figure(figsize=(6, 4))
plt.hist(hourly_df["nabs_error_mean"].values, bins=50)
plt.title("Solar – Distribution of Hourly Normalized Abs Error (2024)")
plt.xlabel("Normalized Abs Error (|e| / mean_true)")
plt.ylabel("Frequency")
plt.tight_layout()
plot2_path = os.path.join(TECH_DIR, "solar_hourly_norm_abs_error_hist_2024.png")
plt.savefig(plot2_path, dpi=150)
plt.show()
plt.close()
print(f"[PLOT] Saved hourly normalized abs error histogram -> {plot2_path}")

print("\n[DONE] Per-tech MLP for solar finished.")


### 1.2.7 Waste


In [ ]:
# =========================
# CELL – Per-Tech MLP for Waste (CarbonCast-style)
# =========================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import timedelta

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, InputLayer
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import mean_squared_error, mean_absolute_error

# --------- CONFIG ----------
DATA_PROC_DIR = "/content/data_entsoe_es_processed"
TECH_NAME     = "gen_waste_mw"      # target tech
TRAIN_WIN_H   = 24
LABEL_WIN_H   = 24

# Per-tech subfolder
TECH_DIR = os.path.join(DATA_PROC_DIR, TECH_NAME)
os.makedirs(TECH_DIR, exist_ok=True)

# ---------------------------
# 1) Load feature lists to get gen/da/cal columns and tech index
# ---------------------------
feat_list_path = os.path.join(DATA_PROC_DIR, "feature_lists_entsoe.csv")
feat_df_list   = pd.read_csv(feat_list_path)

gen_cols = feat_df_list[feat_df_list["type"] == "gen"]["name"].tolist()
da_cols  = feat_df_list[feat_df_list["type"] == "da"]["name"].tolist()
cal_cols = feat_df_list[feat_df_list["type"] == "cal"]["name"].tolist()

if TECH_NAME not in gen_cols:
    raise ValueError(f"{TECH_NAME} not found in gen_cols: {gen_cols}")

tech_idx = gen_cols.index(TECH_NAME)
print(f"[TECH] Using tech={TECH_NAME}, index={tech_idx}")
print("[TECH] gen_cols:", gen_cols)
print("[TECH] da_cols :", da_cols)
print("[TECH] cal_cols:", cal_cols)

# Hist features = all gen + calendar; DA features as separate block
hist_feature_cols = gen_cols + cal_cols
da_feature_cols   = da_cols
all_feature_cols  = list(dict.fromkeys(hist_feature_cols + da_feature_cols))

# ---------------------------
# 2) Load sliding-window TRAIN / VAL data and reshape
# ---------------------------
X_train_path = os.path.join(DATA_PROC_DIR, "entsoe_train_X_sliding_24_24.csv")
Y_train_path = os.path.join(DATA_PROC_DIR, "entsoe_train_Y_sliding_24_24.csv")
X_val_path   = os.path.join(DATA_PROC_DIR, "entsoe_val_X_sliding_24_24.csv")
Y_val_path   = os.path.join(DATA_PROC_DIR, "entsoe_val_Y_sliding_24_24.csv")

X_train_flat = pd.read_csv(X_train_path).values  # (N_train, 24*(F_hist+F_da))
Y_train_flat = pd.read_csv(Y_train_path).values  # (N_train, 24 * n_gen_cols)
X_val_flat   = pd.read_csv(X_val_path).values
Y_val_flat   = pd.read_csv(Y_val_path).values

n_train = X_train_flat.shape[0]
n_val   = X_val_flat.shape[0]

# infer feature dimension per timestep and number of gen targets
F_total        = X_train_flat.shape[1] // TRAIN_WIN_H
n_gen_targets  = len(gen_cols)

if Y_train_flat.shape[1] != LABEL_WIN_H * n_gen_targets:
    raise RuntimeError("Y_train_flat shape is inconsistent with LABEL_WIN_H and n_gen_targets")

print(f"[SHAPE] X_train_flat: {X_train_flat.shape} -> (N, {TRAIN_WIN_H}, {F_total})")
print(f"[SHAPE] Y_train_flat: {Y_train_flat.shape} -> (N, {LABEL_WIN_H}, {n_gen_targets})")

# reshape to (N, 24, F_total) and (N, 24, n_gen_targets)
X_train_seq = X_train_flat.reshape(n_train, TRAIN_WIN_H, F_total)
X_val_seq   = X_val_flat.reshape(n_val,   TRAIN_WIN_H, F_total)

Y_train_all = Y_train_flat.reshape(n_train, LABEL_WIN_H, n_gen_targets)
Y_val_all   = Y_val_flat.reshape(n_val,   LABEL_WIN_H, n_gen_targets)

# extract waste only: shape (N, 24)
Y_train = Y_train_all[:, :, tech_idx]
Y_val   = Y_val_all[:, :, tech_idx]

print(f"[SHAPE] X_train_seq: {X_train_seq.shape}, Y_train: {Y_train.shape}")
print(f"[SHAPE] X_val_seq  : {X_val_seq.shape}, Y_val  : {Y_val.shape}")

# ---------------------------
# 3) Build CarbonCast-style MLP for this tech
# ---------------------------
EPOCHS      = 100
BATCH_SIZE  = 10
ACTIVATION  = "relu"
LOSS_FUNC   = "mse"
LEARNING_RT = 0.001
HIDDEN      = [50, 34]

model = Sequential()
model.add(InputLayer(input_shape=(TRAIN_WIN_H, F_total)))
model.add(Flatten())
model.add(Dense(HIDDEN[0], activation=ACTIVATION))
model.add(Dense(HIDDEN[1], activation=ACTIVATION))
model.add(Dense(LABEL_WIN_H))  # 24 outputs (24h horizon)

opt = tf.keras.optimizers.Adam(learning_rate=LEARNING_RT)
model.compile(loss=LOSS_FUNC, optimizer=opt, metrics=["mae"])

model_summary = []
model.summary(print_fn=lambda x: model_summary.append(x))
print("\n".join(model_summary))

# EarlyStopping only – restore best weights in memory
es = EarlyStopping(
    monitor="val_loss",
    mode="min",
    patience=10,
    verbose=1,
    restore_best_weights=True,
)

# ---------------------------
# 4) Train model
# ---------------------------
history = model.fit(
    X_train_seq, Y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val_seq, Y_val),
    callbacks=[es],
    verbose=2
)

print("[TRAIN] Training finished. Best weights restored.")
best_model = model

# Save model in native Keras format in tech subfolder
model_path = os.path.join(TECH_DIR, "best_mlp_waste.keras")
best_model.save(model_path)
print(f"[TRAIN] Saved model -> {model_path}")

# ---------------------------
# 5) Load hourly scaled VAL + TEST and build strict day-ahead forecasts for 2024
# ---------------------------
val_scaled_path   = os.path.join(DATA_PROC_DIR, "entsoe_val_scaled_hourly.csv")
test_scaled_path  = os.path.join(DATA_PROC_DIR, "entsoe_test_scaled_hourly.csv")

val_scaled  = pd.read_csv(val_scaled_path,  parse_dates=["datetime_utc"]).set_index("datetime_utc").sort_index()
test_scaled = pd.read_csv(test_scaled_path, parse_dates=["datetime_utc"]).set_index("datetime_utc").sort_index()

print("[TEST] val_scaled :", val_scaled.shape,  "|", val_scaled.index.min(), "->", val_scaled.index.max())
print("[TEST] test_scaled:", test_scaled.shape, "|", test_scaled.index.min(),  "->", test_scaled.index.max())

# history context = val tail + test
full_df = pd.concat([val_scaled, test_scaled], axis=0).sort_index()

test_days = sorted(test_scaled.index.normalize().unique())
print(f"[TEST] Days in 2024: {len(test_days)}")

all_pred_scaled = []
all_true_scaled = []
all_time_index  = []

for day in test_days:
    horizon_start = pd.Timestamp(day)
    horizon_end   = horizon_start + timedelta(hours=23)

    hist_start = horizon_start - timedelta(hours=24)
    hist_end   = horizon_start - timedelta(hours=1)

    hist_slice    = full_df.loc[hist_start:hist_end]
    horizon_slice = full_df.loc[horizon_start:horizon_end]

    if len(hist_slice) != TRAIN_WIN_H or len(horizon_slice) != LABEL_WIN_H:
        print(f"[WARN] Skipping {day.date()} – hist={len(hist_slice)}, hor={len(horizon_slice)}")
        continue

    H = hist_slice[hist_feature_cols].values
    F = horizon_slice[da_feature_cols].values

    if H.shape[0] != TRAIN_WIN_H or F.shape[0] != LABEL_WIN_H:
        print(f"[WARN] Shape mismatch on {day.date()}, skipping.")
        continue

    X_day = np.concatenate([H, F], axis=1).reshape(1, TRAIN_WIN_H, F_total)

    y_pred_day_scaled = best_model.predict(X_day, verbose=0)[0]   # (24,)
    y_true_day_scaled = horizon_slice[TECH_NAME].values           # (24,)

    all_pred_scaled.append(y_pred_day_scaled)
    all_true_scaled.append(y_true_day_scaled)
    all_time_index.append(horizon_slice.index.values)

if not all_pred_scaled:
    raise RuntimeError("[TEST] No valid test days were forecasted for waste.")

y_pred_scaled = np.concatenate(all_pred_scaled, axis=0)
y_true_scaled = np.concatenate(all_true_scaled, axis=0)
time_index    = np.concatenate(all_time_index,  axis=0)

print(f"[TEST] Total hourly predictions: {len(y_true_scaled)}")

# ---------------------------
# 6) Inverse scaling for waste (using scaler_y from prep cell)
# ---------------------------
try:
    scaler_y
except NameError:
    raise RuntimeError("scaler_y not found – run the prep cell in the same session first.")

n_techs = len(gen_cols)
k       = tech_idx

T = len(y_true_scaled)
y_true_scaled_full = np.zeros((T, n_techs))
y_pred_scaled_full = np.zeros((T, n_techs))

y_true_scaled_full[:, k] = y_true_scaled
y_pred_scaled_full[:, k] = y_pred_scaled

y_true_unscaled_full = scaler_y.inverse_transform(y_true_scaled_full)
y_pred_unscaled_full = scaler_y.inverse_transform(y_pred_scaled_full)

y_true = y_true_unscaled_full[:, k]
y_pred = y_pred_unscaled_full[:, k]

# ---------------------------
# 7) Hourly metrics
# ---------------------------
rmse_hourly = np.sqrt(mean_squared_error(y_true, y_pred))
mae_hourly  = mean_absolute_error(y_true, y_pred)

eps = 1e-6
mape_hourly = np.mean(
    np.abs((y_true - y_pred) / np.clip(np.abs(y_true), eps, None))
) * 100.0

mean_true = np.mean(y_true)
nmae_mean_hourly = mae_hourly / (mean_true + eps)

print("\n[METRICS – HOURLY 2024, WASTE]")
print(f"RMSE      = {rmse_hourly:.4f} MW")
print(f"MAE       = {mae_hourly:.4f} MW")
print(f"MAPE      = {mape_hourly:.2f} %")
print(f"nMAE_mean = {nmae_mean_hourly:.4f}")

# ---------------------------
# 8) Daily metrics
# ---------------------------
hourly_df = pd.DataFrame({
    "datetime_utc": pd.to_datetime(time_index),
    "y_true_mw": y_true,
    "y_pred_mw": y_pred
}).set_index("datetime_utc").sort_index()

hourly_df["abs_error"] = np.abs(hourly_df["y_true_mw"] - hourly_df["y_pred_mw"])
hourly_df["sq_error"]  = (hourly_df["y_true_mw"] - hourly_df["y_pred_mw"])**2
hourly_df["ape"]       = np.where(
    np.abs(hourly_df["y_true_mw"]) > eps,
    np.abs(hourly_df["y_true_mw"] - hourly_df["y_pred_mw"]) / np.abs(hourly_df["y_true_mw"]),
    np.nan
)
hourly_df["nabs_error_mean"] = hourly_df["abs_error"] / (mean_true + eps)

daily_groups = hourly_df.groupby(hourly_df.index.normalize())
daily_metrics = []

for day, group in daily_groups:
    if len(group) == 0:
        continue

    y_t = group["y_true_mw"].values
    y_p = group["y_pred_mw"].values

    rmse_d = np.sqrt(np.mean((y_t - y_p)**2))
    mae_d  = np.mean(np.abs(y_t - y_p))
    denom  = np.clip(np.abs(y_t), eps, None)
    mape_d = np.mean(np.abs((y_t - y_p) / denom)) * 100.0
    mean_d = np.mean(y_t)
    nmae_d = mae_d / (mean_d + eps)

    daily_metrics.append({
        "date": day.date(),
        "RMSE_MW": rmse_d,
        "MAE_MW": mae_d,
        "MAPE_percent": mape_d,
        "nMAE_mean": nmae_d,
        "mean_true_MW": mean_d
    })

daily_df = pd.DataFrame(daily_metrics).sort_values("date")

print("\n[METRICS – DAILY 2024, WASTE]")
print(daily_df.describe()[["RMSE_MW", "MAE_MW", "MAPE_percent", "nMAE_mean"]])

# ---------------------------
# 9) Save CSVs
# ---------------------------
hourly_csv_path  = os.path.join(TECH_DIR, "waste_hourly_predictions_2024.csv")
daily_csv_path   = os.path.join(TECH_DIR, "waste_daily_metrics_2024.csv")
summary_csv_path = os.path.join(TECH_DIR, "waste_overall_metrics_2024.csv")

hourly_df.reset_index().to_csv(hourly_csv_path, index=False)
daily_df.to_csv(daily_csv_path, index=False)

summary_df = pd.DataFrame([{
    "RMSE_MW": rmse_hourly,
    "MAE_MW": mae_hourly,
    "MAPE_percent": mape_hourly,
    "nMAE_mean": nmae_mean_hourly,
    "mean_true_MW": mean_true
}])
summary_df.to_csv(summary_csv_path, index=False)

print(f"\n[SAVE] Hourly  -> {hourly_csv_path}")
print(f"[SAVE] Daily   -> {daily_csv_path}")
print(f"[SAVE] Summary -> {summary_csv_path}")

# ---------------------------
# 10) Plots
# ---------------------------

# (a) Hourly actual vs predicted
plt.figure(figsize=(14, 4))
plt.plot(hourly_df.index, hourly_df["y_true_mw"], label="Actual", linewidth=1)
plt.plot(hourly_df.index, hourly_df["y_pred_mw"], label="Predicted", linewidth=0.8, alpha=0.7)
plt.title("Waste – Hourly Actual vs Predicted (2024)")
plt.xlabel("Time (UTC)")
plt.ylabel("Power [MW]")
plt.legend()
plt.tight_layout()
plot1_path = os.path.join(TECH_DIR, "waste_hourly_actual_vs_pred_2024.png")
plt.savefig(plot1_path, dpi=150)
plt.show()
plt.close()
print(f"[PLOT] Saved hourly actual vs predicted -> {plot1_path}")

# (b) Histogram of hourly normalized abs error
plt.figure(figsize=(6, 4))
plt.hist(hourly_df["nabs_error_mean"].values, bins=50)
plt.title("Waste – Distribution of Hourly Normalized Abs Error (2024)")
plt.xlabel("Normalized Abs Error (|e| / mean_true)")
plt.ylabel("Frequency [hours]")
plt.tight_layout()
plot2_path = os.path.join(TECH_DIR, "waste_hourly_norm_abs_error_hist_2024.png")
plt.savefig(plot2_path, dpi=150)
plt.show()
plt.close()
print(f"[PLOT] Saved hourly normalized abs error histogram -> {plot2_path}")

print("\n[DONE] Per-tech MLP for waste finished.")


### 1.2.8 Other renewables


In [ ]:
# =========================
# CELL – Per-Tech MLP for Other Renewables (CarbonCast-style)
# =========================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import timedelta

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, InputLayer
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import mean_squared_error, mean_absolute_error

# --------- CONFIG ----------
DATA_PROC_DIR = "/content/data_entsoe_es_processed"
TECH_NAME     = "gen_other_renewable_mw"      # target tech
TRAIN_WIN_H   = 24
LABEL_WIN_H   = 24

# Per-tech subfolder
TECH_DIR = os.path.join(DATA_PROC_DIR, TECH_NAME)
os.makedirs(TECH_DIR, exist_ok=True)

# ---------------------------
# 1) Load feature lists (gen/da/cal) and find index
# ---------------------------
feat_list_path = os.path.join(DATA_PROC_DIR, "feature_lists_entsoe.csv")
feat_df_list   = pd.read_csv(feat_list_path)

gen_cols = feat_df_list[feat_df_list["type"] == "gen"]["name"].tolist()
da_cols  = feat_df_list[feat_df_list["type"] == "da"]["name"].tolist()
cal_cols = feat_df_list[feat_df_list["type"] == "cal"]["name"].tolist()

if TECH_NAME not in gen_cols:
    raise ValueError(f"{TECH_NAME} not found in gen_cols: {gen_cols}")

tech_idx = gen_cols.index(TECH_NAME)
print(f"[TECH] Using tech={TECH_NAME}, index={tech_idx}")
print("[TECH] gen_cols:", gen_cols)
print("[TECH] da_cols :", da_cols)
print("[TECH] cal_cols:", cal_cols)

hist_feature_cols = gen_cols + cal_cols
da_feature_cols   = da_cols
all_feature_cols  = list(dict.fromkeys(hist_feature_cols + da_feature_cols))

# ---------------------------
# 2) Load sliding-window TRAIN / VAL data and reshape
# ---------------------------
X_train_path = os.path.join(DATA_PROC_DIR, "entsoe_train_X_sliding_24_24.csv")
Y_train_path = os.path.join(DATA_PROC_DIR, "entsoe_train_Y_sliding_24_24.csv")
X_val_path   = os.path.join(DATA_PROC_DIR, "entsoe_val_X_sliding_24_24.csv")
Y_val_path   = os.path.join(DATA_PROC_DIR, "entsoe_val_Y_sliding_24_24.csv")

X_train_flat = pd.read_csv(X_train_path).values
Y_train_flat = pd.read_csv(Y_train_path).values
X_val_flat   = pd.read_csv(X_val_path).values
Y_val_flat   = pd.read_csv(Y_val_path).values

n_train = X_train_flat.shape[0]
n_val   = X_val_flat.shape[0]

F_total       = X_train_flat.shape[1] // TRAIN_WIN_H
n_gen_targets = len(gen_cols)

if Y_train_flat.shape[1] != LABEL_WIN_H * n_gen_targets:
    raise RuntimeError("Y_train_flat shape is inconsistent with LABEL_WIN_H and n_gen_targets")

print(f"[SHAPE] X_train_flat: {X_train_flat.shape} -> (N, 24, {F_total})")
print(f"[SHAPE] Y_train_flat: {Y_train_flat.shape} -> (N, 24, {n_gen_targets})")

X_train_seq = X_train_flat.reshape(n_train, TRAIN_WIN_H, F_total)
X_val_seq   = X_val_flat.reshape(n_val,   TRAIN_WIN_H, F_total)

Y_train_all = Y_train_flat.reshape(n_train, LABEL_WIN_H, n_gen_targets)
Y_val_all   = Y_val_flat.reshape(n_val,   LABEL_WIN_H, n_gen_targets)

# Extract target tech (other renewables)
Y_train = Y_train_all[:, :, tech_idx]
Y_val   = Y_val_all[:, :, tech_idx]

print(f"[SHAPE] X_train_seq: {X_train_seq.shape}, Y_train: {Y_train.shape}")
print(f"[SHAPE] X_val_seq  : {X_val_seq.shape}, Y_val  : {Y_val.shape}")

# ---------------------------
# 3) Build CarbonCast-style MLP for this tech
# ---------------------------
EPOCHS      = 100
BATCH_SIZE  = 10
ACTIVATION  = "relu"
LOSS_FUNC   = "mse"
LEARNING_RT = 0.01
HIDDEN      = [52, 34]

model = Sequential([
    InputLayer(input_shape=(TRAIN_WIN_H, F_total)),
    Flatten(),
    Dense(HIDDEN[0], activation=ACTIVATION),
    Dense(HIDDEN[1], activation=ACTIVATION),
    Dense(LABEL_WIN_H)
])

opt = tf.keras.optimizers.Adam(learning_rate=LEARNING_RT)
model.compile(loss=LOSS_FUNC, optimizer=opt, metrics=["mae"])

model.summary()

es = EarlyStopping(
    monitor="val_loss",
    mode="min",
    patience=10,
    verbose=1,
    restore_best_weights=True,
)

# ---------------------------
# 4) Train model
# ---------------------------
history = model.fit(
    X_train_seq, Y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val_seq, Y_val),
    callbacks=[es],
    verbose=2
)

print("[TRAIN] Training finished. Best weights restored.")
best_model = model

# Save model
model_path = os.path.join(TECH_DIR, "best_mlp_other_renewables.keras")
best_model.save(model_path)
print(f"[TRAIN] Saved model -> {model_path}")

# ---------------------------
# 5) Strict day-ahead test forecasts (CarbonCast)
# ---------------------------
val_scaled  = pd.read_csv(os.path.join(DATA_PROC_DIR, "entsoe_val_scaled_hourly.csv"),
                           parse_dates=["datetime_utc"]).set_index("datetime_utc")
test_scaled = pd.read_csv(os.path.join(DATA_PROC_DIR, "entsoe_test_scaled_hourly.csv"),
                           parse_dates=["datetime_utc"]).set_index("datetime_utc")

full_df = pd.concat([val_scaled, test_scaled]).sort_index()

test_days = sorted(test_scaled.index.normalize().unique())
print(f"[TEST] Days in 2024: {len(test_days)}")

all_pred_scaled = []
all_true_scaled = []
all_time_index  = []

for day in test_days:
    s0 = pd.Timestamp(day)
    s1 = s0 + timedelta(hours=23)

    hist_start = s0 - timedelta(hours=24)
    hist_end   = s0 - timedelta(hours=1)

    hist = full_df.loc[hist_start:hist_end]
    fut  = full_df.loc[s0:s1]

    if len(hist) != 24 or len(fut) != 24:
        continue

    H = hist[hist_feature_cols].values
    F = fut[da_feature_cols].values
    X = np.concatenate([H, F], axis=1).reshape(1, 24, F_total)

    yp = best_model.predict(X, verbose=0)[0]
    yt = fut[TECH_NAME].values

    all_pred_scaled.append(yp)
    all_true_scaled.append(yt)
    all_time_index.append(fut.index.values)

y_pred_scaled = np.concatenate(all_pred_scaled)
y_true_scaled = np.concatenate(all_true_scaled)
time_index    = np.concatenate(all_time_index)

print("[TEST] Total hourly predictions:", len(y_true_scaled))

# ---------------------------
# 6) Inverse scaling
# ---------------------------
n_techs = len(gen_cols)
k = tech_idx

T = len(y_true_scaled)
yt_full = np.zeros((T, n_techs))
yp_full = np.zeros((T, n_techs))
yt_full[:, k] = y_true_scaled
yp_full[:, k] = y_pred_scaled

y_true = scaler_y.inverse_transform(yt_full)[:, k]
y_pred = scaler_y.inverse_transform(yp_full)[:, k]

# ---------------------------
# 7) Hourly metrics
# ---------------------------
rmse_hourly = np.sqrt(mean_squared_error(y_true, y_pred))
mae_hourly  = mean_absolute_error(y_true, y_pred)
eps = 1e-6
mape_hourly = np.mean(np.abs((y_true - y_pred) / np.clip(np.abs(y_true), eps, None))) * 100
mean_true   = np.mean(y_true)
nmae_mean_hourly = mae_hourly / (mean_true + eps)

print("\n[METRICS – HOURLY 2024, OTHER RENEWABLES]")
print(f"RMSE      = {rmse_hourly:.3f} MW")
print(f"MAE       = {mae_hourly:.3f} MW")
print(f"MAPE      = {mape_hourly:.3f} %")
print(f"nMAE_mean = {nmae_mean_hourly:.4f}")

# ---------------------------
# 8) Daily metrics
# ---------------------------
hourly_df = pd.DataFrame({
    "datetime_utc": time_index,
    "y_true_mw": y_true,
    "y_pred_mw": y_pred
}).set_index("datetime_utc")

daily_df = []
for d, g in hourly_df.groupby(hourly_df.index.normalize()):
    y_t = g["y_true_mw"].values
    y_p = g["y_pred_mw"].values

    rmse_d = np.sqrt(np.mean((y_t - y_p)**2))
    mae_d  = np.mean(np.abs(y_t - y_p))
    mape_d = np.mean(np.abs((y_t - y_p) / np.clip(np.abs(y_t), eps, None))) * 100
    mean_d = np.mean(y_t)
    nmae_d = mae_d / (mean_d + eps)

    daily_df.append({
        "date": d.date(),
        "RMSE_MW": rmse_d,
        "MAE_MW": mae_d,
        "MAPE_percent": mape_d,
        "nMAE_mean": nmae_d,
        "mean_true_MW": mean_d
    })

daily_df = pd.DataFrame(daily_df).sort_values("date")

print("\n[METRICS – DAILY SUMMARY]")
print(daily_df.describe()[["RMSE_MW","MAE_MW","MAPE_percent","nMAE_mean"]])

# ---------------------------
# 9) Save CSVs
# ---------------------------
hourly_path  = os.path.join(TECH_DIR, "other_renewables_hourly_predictions_2024.csv")
daily_path   = os.path.join(TECH_DIR, "other_renewables_daily_metrics_2024.csv")
summary_path = os.path.join(TECH_DIR, "other_renewables_overall_metrics_2024.csv")

hourly_df.reset_index().to_csv(hourly_path, index=False)
daily_df.to_csv(daily_path, index=False)

summary_df = pd.DataFrame([{
    "RMSE_MW": rmse_hourly,
    "MAE_MW": mae_hourly,
    "MAPE_percent": mape_hourly,
    "nMAE_mean": nmae_mean_hourly,
    "mean_true_MW": mean_true
}])
summary_df.to_csv(summary_path, index=False)

print(f"[SAVE] Hourly  -> {hourly_path}")
print(f"[SAVE] Daily   -> {daily_path}")
print(f"[SAVE] Summary -> {summary_path}")

# ---------------------------
# 10) Plots
# ---------------------------
plt.figure(figsize=(14,4))
plt.plot(hourly_df.index, hourly_df["y_true_mw"], label="Actual", linewidth=1)
plt.plot(hourly_df.index, hourly_df["y_pred_mw"], label="Predicted", linewidth=0.8, alpha=0.7)
plt.title("Other Renewables – Hourly Actual vs Predicted (2024)")
plt.xlabel("Time (UTC)")
plt.ylabel("Power [MW]")
plt.legend()
plt.tight_layout()
plot1 = os.path.join(TECH_DIR, "other_renewables_hourly_actual_vs_pred_2024.png")
plt.savefig(plot1, dpi=150)
plt.show()

plt.figure(figsize=(6,4))
plt.hist(np.abs(hourly_df["y_true_mw"] - hourly_df["y_pred_mw"]) / (mean_true + eps), bins=50)
plt.title("Other Renewables – Distribution of Hourly Normalized Abs Error (2024)")
plt.xlabel("Normalized Abs Error")
plt.ylabel("Frequency [hours]")
plt.tight_layout()
plot2 = os.path.join(TECH_DIR, "other_renewables_norm_abs_error_hist_2024.png")
plt.savefig(plot2, dpi=150)
plt.show()

print(f"[PLOT] Saved -> {plot1}")
print(f"[PLOT] Saved -> {plot2}")

print("\n[DONE] Per-tech MLP for other renewables finished.")


### 1.2.9 Hydro run-of-river


In [ ]:
# =========================
# CELL – Per-Tech MLP for Hydro Run-of-River (CarbonCast-style)
# =========================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import timedelta

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, InputLayer
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import mean_squared_error, mean_absolute_error

# --------- CONFIG ----------
DATA_PROC_DIR = "/content/data_entsoe_es_processed"
TECH_NAME     = "gen_hydro_run_of_river_mw"   # target tech
TRAIN_WIN_H   = 24
LABEL_WIN_H   = 24

# Per-tech subfolder
TECH_DIR = os.path.join(DATA_PROC_DIR, TECH_NAME)
os.makedirs(TECH_DIR, exist_ok=True)

# ---------------------------
# 1) Load feature lists & tech index
# ---------------------------
feat_list_path = os.path.join(DATA_PROC_DIR, "feature_lists_entsoe.csv")
feat_df_list   = pd.read_csv(feat_list_path)

gen_cols = feat_df_list[feat_df_list["type"] == "gen"]["name"].tolist()
da_cols  = feat_df_list[feat_df_list["type"] == "da"]["name"].tolist()
cal_cols = feat_df_list[feat_df_list["type"] == "cal"]["name"].tolist()

if TECH_NAME not in gen_cols:
    raise ValueError(f"{TECH_NAME} not found in gen_cols: {gen_cols}")

tech_idx = gen_cols.index(TECH_NAME)
print(f"[TECH] Using tech={TECH_NAME}, index={tech_idx}")
print("[TECH] gen_cols:", gen_cols)
print("[TECH] da_cols :", da_cols)
print("[TECH] cal_cols:", cal_cols)

hist_feature_cols = gen_cols + cal_cols
da_feature_cols   = da_cols
all_feature_cols  = list(dict.fromkeys(hist_feature_cols + da_feature_cols))

# ---------------------------
# 2) Load sliding-window TRAIN / VAL data
# ---------------------------
X_train_path = os.path.join(DATA_PROC_DIR, "entsoe_train_X_sliding_24_24.csv")
Y_train_path = os.path.join(DATA_PROC_DIR, "entsoe_train_Y_sliding_24_24.csv")
X_val_path   = os.path.join(DATA_PROC_DIR, "entsoe_val_X_sliding_24_24.csv")
Y_val_path   = os.path.join(DATA_PROC_DIR, "entsoe_val_Y_sliding_24_24.csv")

X_train_flat = pd.read_csv(X_train_path).values
Y_train_flat = pd.read_csv(Y_train_path).values
X_val_flat   = pd.read_csv(X_val_path).values
Y_val_flat   = pd.read_csv(Y_val_path).values

n_train = X_train_flat.shape[0]
n_val   = X_val_flat.shape[0]

F_total        = X_train_flat.shape[1] // TRAIN_WIN_H
n_gen_targets  = len(gen_cols)

print(f"[SHAPE] X_train_flat: {X_train_flat.shape}")
print(f"[SHAPE] Y_train_flat: {Y_train_flat.shape}")

X_train_seq = X_train_flat.reshape(n_train, TRAIN_WIN_H, F_total)
X_val_seq   = X_val_flat.reshape(n_val,   TRAIN_WIN_H, F_total)

Y_train_all = Y_train_flat.reshape(n_train, LABEL_WIN_H, n_gen_targets)
Y_val_all   = Y_val_flat.reshape(n_val,   LABEL_WIN_H, n_gen_targets)

Y_train = Y_train_all[:, :, tech_idx]
Y_val   = Y_val_all[:, :, tech_idx]

print(f"[SHAPE] X_train_seq: {X_train_seq.shape}, Y_train: {Y_train.shape}")
print(f"[SHAPE] X_val_seq  : {X_val_seq.shape}, Y_val:  {Y_val.shape}")

# ---------------------------
# 3) Build CarbonCast-style MLP
# ---------------------------
EPOCHS      = 100
BATCH_SIZE  = 10
ACTIVATION  = "relu"
LOSS_FUNC   = "mse"
LEARNING_RT = 0.001
HIDDEN      = [50, 34]

model = Sequential([
    InputLayer(input_shape=(TRAIN_WIN_H, F_total)),
    Flatten(),
    Dense(HIDDEN[0], activation=ACTIVATION),
    Dense(HIDDEN[1], activation=ACTIVATION),
    Dense(LABEL_WIN_H),
])

opt = tf.keras.optimizers.Adam(learning_rate=LEARNING_RT)
model.compile(loss=LOSS_FUNC, optimizer=opt, metrics=["mae"])

model.summary()

es = EarlyStopping(monitor="val_loss", mode="min", patience=10,
                   verbose=1, restore_best_weights=True)

# ---------------------------
# 4) Train
# ---------------------------
history = model.fit(
    X_train_seq, Y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val_seq, Y_val),
    callbacks=[es],
    verbose=2
)

print("[TRAIN] Training finished. Best weights restored.")
best_model = model

# Save model
model_path = os.path.join(TECH_DIR, "best_mlp_hydro_run_of_river.keras")
best_model.save(model_path)
print(f"[TRAIN] Saved model -> {model_path}")

# ---------------------------
# 5) Strict day-ahead forecasting for 2024
# ---------------------------
val_scaled_path  = os.path.join(DATA_PROC_DIR, "entsoe_val_scaled_hourly.csv")
test_scaled_path = os.path.join(DATA_PROC_DIR, "entsoe_test_scaled_hourly.csv")

val_scaled  = pd.read_csv(val_scaled_path,  parse_dates=["datetime_utc"]).set_index("datetime_utc")
test_scaled = pd.read_csv(test_scaled_path, parse_dates=["datetime_utc"]).set_index("datetime_utc")

print("[TEST] val_scaled:", val_scaled.shape,
      "|", val_scaled.index.min(), "->", val_scaled.index.max())
print("[TEST] test_scaled:", test_scaled.shape,
      "|", test_scaled.index.min(), "->", test_scaled.index.max())

full_df = pd.concat([val_scaled, test_scaled]).sort_index()
test_days = sorted(test_scaled.index.normalize().unique())
print(f"[TEST] Days in 2024: {len(test_days)}")

all_pred_scaled = []
all_true_scaled = []
all_time_index  = []

for day in test_days:
    horizon_start = pd.Timestamp(day)
    horizon_end   = horizon_start + timedelta(hours=23)

    hist_start = horizon_start - timedelta(hours=24)
    hist_end   = horizon_start - timedelta(hours=1)

    H_slice = full_df.loc[hist_start:hist_end]
    F_slice = full_df.loc[horizon_start:horizon_end]

    if len(H_slice) != 24 or len(F_slice) != 24:
        print(f"[WARN] Skipping {day.date()} (incomplete window)")
        continue

    H = H_slice[hist_feature_cols].values
    F = F_slice[da_feature_cols].values

    X_day = np.concatenate([H, F], axis=1).reshape(1, 24, F_total)

    y_pred_scaled = best_model.predict(X_day, verbose=0)[0]
    y_true_scaled = F_slice[TECH_NAME].values

    all_pred_scaled.append(y_pred_scaled)
    all_true_scaled.append(y_true_scaled)
    all_time_index.append(F_slice.index.values)

# Stack predictions
y_pred_scaled = np.concatenate(all_pred_scaled)
y_true_scaled = np.concatenate(all_true_scaled)
time_index    = np.concatenate(all_time_index)

print(f"[TEST] Total hourly predictions: {len(y_true_scaled)}")

# ---------------------------
# 6) Inverse scaling (requires scaler_y)
# ---------------------------
try:
    scaler_y
except NameError:
    raise RuntimeError("scaler_y missing — run prep cell first.")

n_techs = len(gen_cols)
k = tech_idx
T = len(y_true_scaled)

yt_full  = np.zeros((T, n_techs))
yp_full  = np.zeros((T, n_techs))

yt_full[:, k] = y_true_scaled
yp_full[:, k] = y_pred_scaled

y_true = scaler_y.inverse_transform(yt_full)[:, k]
y_pred = scaler_y.inverse_transform(yp_full)[:, k]

# ---------------------------
# 7) Hourly metrics
# ---------------------------
rmse_hourly = np.sqrt(mean_squared_error(y_true, y_pred))
mae_hourly  = mean_absolute_error(y_true, y_pred)

eps = 1e-6
mape_hourly = np.mean(np.abs((y_true - y_pred) /
                             np.clip(np.abs(y_true), eps, None))) * 100

mean_true = np.mean(y_true)
nmae_mean_hourly = mae_hourly / (mean_true + eps)

print("\n[METRICS – HOURLY 2024, HYDRO_RUN_OF_RIVER]")
print(f"RMSE      = {rmse_hourly:.4f} MW")
print(f"MAE       = {mae_hourly:.4f} MW")
print(f"MAPE      = {mape_hourly:.2f} %")
print(f"nMAE_mean = {nmae_mean_hourly:.4f}")

# ---------------------------
# 8) Daily metrics
# ---------------------------
hourly_df = pd.DataFrame({
    "datetime_utc": time_index,
    "y_true_mw": y_true,
    "y_pred_mw": y_pred
}).set_index("datetime_utc").sort_index()

hourly_df["abs_error"] = np.abs(hourly_df["y_true_mw"] - hourly_df["y_pred_mw"])
hourly_df["sq_error"]  = (hourly_df["y_true_mw"] - hourly_df["y_pred_mw"])**2
hourly_df["nabs_error_mean"] = hourly_df["abs_error"] / (mean_true + eps)

daily_df = hourly_df.groupby(hourly_df.index.normalize()).agg({
    "sq_error":  lambda x: np.sqrt(np.mean(x)),
    "abs_error": "mean",
    "y_true_mw": "mean"
}).rename(columns={
    "sq_error": "RMSE_MW",
    "abs_error": "MAE_MW",
    "y_true_mw": "mean_true_MW"
})
daily_df["MAPE_percent"] = (
    daily_df["MAE_MW"] /
    np.clip(np.abs(daily_df["mean_true_MW"]), eps, None)
) * 100
daily_df["nMAE_mean"] = daily_df["MAE_MW"] / (daily_df["mean_true_MW"] + eps)

print("\n[METRICS – DAILY 2024, HYDRO_RUN_OF_RIVER]")
print(daily_df.describe()[["RMSE_MW","MAE_MW","MAPE_percent","nMAE_mean"]])

# ---------------------------
# 9) Save CSVs
# ---------------------------
hourly_df.reset_index().to_csv(
    os.path.join(TECH_DIR, "hydro_run_of_river_hourly_predictions_2024.csv"),
    index=False
)
daily_df.reset_index().to_csv(
    os.path.join(TECH_DIR, "hydro_run_of_river_daily_metrics_2024.csv"),
    index=False
)

summary_df = pd.DataFrame([{
    "RMSE_MW": rmse_hourly,
    "MAE_MW": mae_hourly,
    "MAPE_percent": mape_hourly,
    "nMAE_mean": nmae_mean_hourly,
    "mean_true_MW": mean_true
}])
summary_df.to_csv(
    os.path.join(TECH_DIR, "hydro_run_of_river_overall_metrics_2024.csv"),
    index=False
)

# ---------------------------
# 10) Plots
# ---------------------------
plt.figure(figsize=(14,4))
plt.plot(hourly_df.index, hourly_df["y_true_mw"], label="Actual", linewidth=1)
plt.plot(hourly_df.index, hourly_df["y_pred_mw"], label="Predicted", linewidth=0.7)
plt.title("Hydro Run-of-River – Hourly Actual vs Predicted (2024)")
plt.xlabel("Time (UTC)")
plt.ylabel("Power [MW]")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(TECH_DIR, "hydro_run_of_river_hourly_actual_vs_pred_2024.png"), dpi=150)
plt.show()

plt.figure(figsize=(6,4))
plt.hist(hourly_df["nabs_error_mean"], bins=50)
plt.title("Hydro Run-of-River – Normalized Abs Error Distribution (2024)")
plt.xlabel("|Error| / mean_true")
plt.ylabel("Frequency (hours)")
plt.tight_layout()
plt.savefig(os.path.join(TECH_DIR, "hydro_run_of_river_hourly_norm_abs_error_hist_2024.png"), dpi=150)
plt.show()

print("\n[DONE] Per-tech MLP for hydro run-of-river finished.")


### 1.2.10 Other generation


In [ ]:
# =========================
# CELL – Per-Tech MLP for Other (CarbonCast-style)
# =========================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import timedelta

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, InputLayer
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import mean_squared_error, mean_absolute_error

# --------- CONFIG ----------
DATA_PROC_DIR = "/content/data_entsoe_es_processed"
TECH_NAME     = "gen_other_mw"      # target tech
TRAIN_WIN_H   = 24
LABEL_WIN_H   = 24

# Per-tech subfolder
TECH_DIR = os.path.join(DATA_PROC_DIR, TECH_NAME)
os.makedirs(TECH_DIR, exist_ok=True)

# ---------------------------
# 1) Load feature lists to get gen/da/cal columns and tech index
# ---------------------------
feat_list_path = os.path.join(DATA_PROC_DIR, "feature_lists_entsoe.csv")
feat_df_list   = pd.read_csv(feat_list_path)

gen_cols = feat_df_list[feat_df_list["type"] == "gen"]["name"].tolist()
da_cols  = feat_df_list[feat_df_list["type"] == "da"]["name"].tolist()
cal_cols = feat_df_list[feat_df_list["type"] == "cal"]["name"].tolist()

if TECH_NAME not in gen_cols:
    raise ValueError(f"{TECH_NAME} not found in gen_cols: {gen_cols}")

tech_idx = gen_cols.index(TECH_NAME)
print(f"[TECH] Using tech={TECH_NAME}, index={tech_idx}")
print("[TECH] gen_cols:", gen_cols)
print("[TECH] da_cols :", da_cols)
print("[TECH] cal_cols:", cal_cols)

# Hist features = all gen + calendar; DA features as separate block
hist_feature_cols = gen_cols + cal_cols
da_feature_cols   = da_cols
all_feature_cols  = list(dict.fromkeys(hist_feature_cols + da_feature_cols))

# ---------------------------
# 2) Load sliding-window TRAIN / VAL data and reshape
# ---------------------------
X_train_path = os.path.join(DATA_PROC_DIR, "entsoe_train_X_sliding_24_24.csv")
Y_train_path = os.path.join(DATA_PROC_DIR, "entsoe_train_Y_sliding_24_24.csv")
X_val_path   = os.path.join(DATA_PROC_DIR, "entsoe_val_X_sliding_24_24.csv")
Y_val_path   = os.path.join(DATA_PROC_DIR, "entsoe_val_Y_sliding_24_24.csv")

X_train_flat = pd.read_csv(X_train_path).values  # (N_train, 24*(F_hist+F_da))
Y_train_flat = pd.read_csv(Y_train_path).values  # (N_train, 24 * n_gen_cols)
X_val_flat   = pd.read_csv(X_val_path).values
Y_val_flat   = pd.read_csv(Y_val_path).values

n_train = X_train_flat.shape[0]
n_val   = X_val_flat.shape[0]

# infer feature dimension per timestep and number of gen targets
F_total        = X_train_flat.shape[1] // TRAIN_WIN_H
n_gen_targets  = len(gen_cols)

if Y_train_flat.shape[1] != LABEL_WIN_H * n_gen_targets:
    raise RuntimeError("Y_train_flat shape is inconsistent with LABEL_WIN_H and n_gen_targets")

print(f"[SHAPE] X_train_flat: {X_train_flat.shape} -> (N, {TRAIN_WIN_H}, {F_total})")
print(f"[SHAPE] Y_train_flat: {Y_train_flat.shape} -> (N, {LABEL_WIN_H}, {n_gen_targets})")

# reshape to (N, 24, F_total) and (N, 24, n_gen_targets)
X_train_seq = X_train_flat.reshape(n_train, TRAIN_WIN_H, F_total)
X_val_seq   = X_val_flat.reshape(n_val,   TRAIN_WIN_H, F_total)

Y_train_all = Y_train_flat.reshape(n_train, LABEL_WIN_H, n_gen_targets)
Y_val_all   = Y_val_flat.reshape(n_val,   LABEL_WIN_H, n_gen_targets)

# extract "other" only: shape (N, 24)
Y_train = Y_train_all[:, :, tech_idx]
Y_val   = Y_val_all[:, :, tech_idx]

print(f"[SHAPE] X_train_seq: {X_train_seq.shape}, Y_train: {Y_train.shape}")
print(f"[SHAPE] X_val_seq  : {X_val_seq.shape}, Y_val  : {Y_val.shape}")

# ---------------------------
# 3) Build CarbonCast-style MLP for this tech
# ---------------------------
EPOCHS      = 100
BATCH_SIZE  = 10
ACTIVATION  = "relu"
LOSS_FUNC   = "mse"
LEARNING_RT = 0.01
HIDDEN      = [48, 34]

model = Sequential()
model.add(InputLayer(input_shape=(TRAIN_WIN_H, F_total)))
model.add(Flatten())
model.add(Dense(HIDDEN[0], activation=ACTIVATION))
model.add(Dense(HIDDEN[1], activation=ACTIVATION))
model.add(Dense(LABEL_WIN_H))  # 24 outputs (24h horizon)

opt = tf.keras.optimizers.Adam(learning_rate=LEARNING_RT)
model.compile(loss=LOSS_FUNC, optimizer=opt, metrics=["mae"])

model_summary = []
model.summary(print_fn=lambda x: model_summary.append(x))
print("\n".join(model_summary))

# EarlyStopping only – restore best weights in memory
es = EarlyStopping(
    monitor="val_loss",
    mode="min",
    patience=10,
    verbose=1,
    restore_best_weights=True,
)

# ---------------------------
# 4) Train model
# ---------------------------
history = model.fit(
    X_train_seq, Y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val_seq, Y_val),
    callbacks=[es],
    verbose=2
)

print("[TRAIN] Training finished. Best weights are already loaded in `model`.")
best_model = model  # use this directly

# Save model in native Keras format in tech subfolder
model_path = os.path.join(TECH_DIR, "best_mlp_other.keras")
best_model.save(model_path)
print(f"[TRAIN] Saved model to {model_path}")

# ---------------------------
# 5) Load hourly scaled VAL + TEST and build strict day-ahead forecasts for 2024
# ---------------------------
val_scaled_path   = os.path.join(DATA_PROC_DIR, "entsoe_val_scaled_hourly.csv")
test_scaled_path  = os.path.join(DATA_PROC_DIR, "entsoe_test_scaled_hourly.csv")

val_scaled  = pd.read_csv(val_scaled_path,  parse_dates=["datetime_utc"]).set_index("datetime_utc").sort_index()
test_scaled = pd.read_csv(test_scaled_path, parse_dates=["datetime_utc"]).set_index("datetime_utc").sort_index()

print("[TEST] val_scaled :", val_scaled.shape,  "|", val_scaled.index.min(), "->", val_scaled.index.max())
print("[TEST] test_scaled:", test_scaled.shape, "|", test_scaled.index.min(),  "->", test_scaled.index.max())

# Combine val tail + test for history context
full_df = pd.concat([val_scaled, test_scaled], axis=0).sort_index()

# List of test days (UTC midnights) with full 24h coverage
test_days = sorted(test_scaled.index.normalize().unique())
print(f"[TEST] Number of test days in 2024: {len(test_days)}")

all_pred_scaled = []
all_true_scaled = []
all_time_index  = []

for day in test_days:
    horizon_start = pd.Timestamp(day)                      # e.g. 2024-01-01 00:00
    horizon_end   = horizon_start + timedelta(hours=23)    # inclusive

    # History: previous 24 hours
    hist_start = horizon_start - timedelta(hours=24)
    hist_end   = horizon_start - timedelta(hours=1)

    hist_slice    = full_df.loc[hist_start:hist_end]
    horizon_slice = full_df.loc[horizon_start:horizon_end]

    if len(hist_slice) != TRAIN_WIN_H or len(horizon_slice) != LABEL_WIN_H:
        print(f"[WARN] Skipping day {day.date()} due to incomplete history/horizon "
              f"({len(hist_slice)} hist hrs, {len(horizon_slice)} horizon hrs).")
        continue

    H = hist_slice[hist_feature_cols].values    # (24, F_hist)
    F = horizon_slice[da_feature_cols].values   # (24, F_da)

    if H.shape[0] != TRAIN_WIN_H or F.shape[0] != LABEL_WIN_H:
        print(f"[WARN] Shape mismatch on day {day.date()}, skipping.")
        continue

    X_day = np.concatenate([H, F], axis=1)  # (24, F_total) – must match training
    X_day = X_day.reshape(1, TRAIN_WIN_H, F_total)

    y_pred_day_scaled = best_model.predict(X_day, verbose=0)[0]  # (24,)
    y_true_day_scaled = horizon_slice[TECH_NAME].values          # (24,)

    all_pred_scaled.append(y_pred_day_scaled)
    all_true_scaled.append(y_true_day_scaled)
    all_time_index.append(horizon_slice.index.values)

# Stack to hourly arrays
if not all_pred_scaled:
    raise RuntimeError("[TEST] No valid test days were forecasted.")

y_pred_scaled = np.concatenate(all_pred_scaled, axis=0)
y_true_scaled = np.concatenate(all_true_scaled, axis=0)
time_index    = np.concatenate(all_time_index, axis=0)

print(f"[TEST] Aggregated test hours: {len(y_true_scaled)}")

# ---------------------------
# 6) Inverse scaling for OTHER (using scaler_y from prep cell)
# ---------------------------
try:
    scaler_y  # just to check it exists
except NameError:
    raise RuntimeError("scaler_y is not available in this runtime. "
                       "You must run the prep cell (with scaler_y) before this.")

n_techs = len(gen_cols)
k       = tech_idx

# build placeholder arrays to inverse_transform
T = len(y_true_scaled)
y_true_scaled_full = np.zeros((T, n_techs))
y_pred_scaled_full = np.zeros((T, n_techs))

y_true_scaled_full[:, k] = y_true_scaled
y_pred_scaled_full[:, k] = y_pred_scaled

y_true_unscaled_full = scaler_y.inverse_transform(y_true_scaled_full)
y_pred_unscaled_full = scaler_y.inverse_transform(y_pred_scaled_full)

y_true = y_true_unscaled_full[:, k]
y_pred = y_pred_unscaled_full[:, k]

# ---------------------------
# 7) Compute hourly metrics
# ---------------------------
rmse_hourly = np.sqrt(mean_squared_error(y_true, y_pred))
mae_hourly  = mean_absolute_error(y_true, y_pred)

# MAPE (%)
eps = 1e-6
mape_hourly = np.mean(np.abs((y_true - y_pred) / np.clip(np.abs(y_true), eps, None))) * 100.0

mean_true = np.mean(y_true)
nmae_mean_hourly = mae_hourly / (mean_true + eps)

print("\n[METRICS – HOURLY 2024, OTHER]")
print(f"RMSE        = {rmse_hourly:.4f} MW")
print(f"MAE         = {mae_hourly:.4f} MW")
print(f"MAPE        = {mape_hourly:.2f} %")
print(f"nMAE_mean   = {nmae_mean_hourly:.4f}")

# ---------------------------
# 8) Daily metrics
# ---------------------------
# Build DataFrame with hourly results
hourly_df = pd.DataFrame({
    "datetime_utc": pd.to_datetime(time_index),
    "y_true_mw": y_true,
    "y_pred_mw": y_pred
}).set_index("datetime_utc").sort_index()

hourly_df["abs_error"] = np.abs(hourly_df["y_true_mw"] - hourly_df["y_pred_mw"])
hourly_df["sq_error"]  = (hourly_df["y_true_mw"] - hourly_df["y_pred_mw"])**2
hourly_df["ape"]       = np.where(
    np.abs(hourly_df["y_true_mw"]) > eps,
    np.abs(hourly_df["y_true_mw"] - hourly_df["y_pred_mw"]) / np.abs(hourly_df["y_true_mw"]),
    np.nan
)
# per-hour normalized abs error using global mean_true (nMAE_mean-style)
hourly_df["nabs_error_mean"] = hourly_df["abs_error"] / (mean_true + eps)

# daily aggregates
daily_groups = hourly_df.groupby(hourly_df.index.normalize())
daily_metrics = []

for day, group in daily_groups:
    if len(group) == 0:
        continue

    y_t = group["y_true_mw"].values
    y_p = group["y_pred_mw"].values

    rmse_d = np.sqrt(np.mean((y_t - y_p)**2))
    mae_d  = np.mean(np.abs(y_t - y_p))
    denom   = np.clip(np.abs(y_t), eps, None)
    mape_d  = np.mean(np.abs((y_t - y_p) / denom)) * 100.0
    mean_d  = np.mean(y_t)
    nmae_d  = mae_d / (mean_d + eps)

    daily_metrics.append({
        "date": day.date(),
        "RMSE_MW": rmse_d,
        "MAE_MW": mae_d,
        "MAPE_percent": mape_d,
        "nMAE_mean": nmae_d,
        "mean_true_MW": mean_d
    })

daily_df = pd.DataFrame(daily_metrics).sort_values("date")

print("\n[METRICS – DAILY 2024, OTHER]")
print(daily_df.describe()[["RMSE_MW", "MAE_MW", "MAPE_percent", "nMAE_mean"]])

# ---------------------------
# 9) Save CSVs (into tech subfolder)
# ---------------------------
hourly_csv_path  = os.path.join(TECH_DIR, "other_hourly_predictions_2024.csv")
daily_csv_path   = os.path.join(TECH_DIR, "other_daily_metrics_2024.csv")
summary_csv_path = os.path.join(TECH_DIR, "other_overall_metrics_2024.csv")

hourly_df.reset_index().to_csv(hourly_csv_path, index=False)
daily_df.to_csv(daily_csv_path, index=False)

# small overall summary row
summary_df = pd.DataFrame([{
    "RMSE_MW": rmse_hourly,
    "MAE_MW": mae_hourly,
    "MAPE_percent": mape_hourly,
    "nMAE_mean": nmae_mean_hourly,
    "mean_true_MW": mean_true
}])
summary_df.to_csv(summary_csv_path, index=False)

print(f"\n[SAVE] Hourly predictions/errors -> {hourly_csv_path}")
print(f"[SAVE] Daily metrics             -> {daily_csv_path}")
print(f"[SAVE] Overall metrics           -> {summary_csv_path}")

# ---------------------------
# 10) Plots (save + show)
# ---------------------------

# (a) Hourly actual vs predicted for 2024
plt.figure(figsize=(14, 4))
plt.plot(hourly_df.index, hourly_df["y_true_mw"], label="Actual", linewidth=1)
plt.plot(hourly_df.index, hourly_df["y_pred_mw"], label="Predicted", linewidth=0.8, alpha=0.7)
plt.title("Other Generation – Hourly Actual vs Predicted (2024)")
plt.xlabel("Time (UTC)")
plt.ylabel("Power [MW]")
plt.legend()
plt.tight_layout()
plot1_path = os.path.join(TECH_DIR, "other_hourly_actual_vs_pred_2024.png")
plt.savefig(plot1_path, dpi=150)
plt.show()
plt.close()
print(f"[PLOT] Saved hourly actual vs predicted -> {plot1_path}")

# (b) Histogram of hourly normalized abs error (nMAE_mean-style)
plt.figure(figsize=(6, 4))
plt.hist(hourly_df["nabs_error_mean"].values, bins=50)
plt.title("Other – Distribution of Hourly Normalized Abs Error (2024)")
plt.xlabel("Normalized Abs Error (|e| / mean_true)")
plt.ylabel("Frequency [hours]")
plt.tight_layout()
plot2_path = os.path.join(TECH_DIR, "other_hourly_norm_abs_error_hist_2024.png")
plt.savefig(plot2_path, dpi=150)
plt.show()
plt.close()
print(f"[PLOT] Saved hourly normalized abs error histogram -> {plot2_path}")

print("\n[DONE] Per-tech MLP for other finished.")


### 1.2.11 Hydro dispatchable


In [ ]:
# =========================
# CELL – Per-Tech MLP for Hydro Dispatchable (CarbonCast-style)
# =========================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import timedelta

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, InputLayer
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import mean_squared_error, mean_absolute_error

# --------- CONFIG ----------
DATA_PROC_DIR = "/content/data_entsoe_es_processed"
TECH_NAME     = "gen_hydro_dispatchable_mw"   # target tech
TRAIN_WIN_H   = 24
LABEL_WIN_H   = 24

# Per-tech subfolder
TECH_DIR = os.path.join(DATA_PROC_DIR, TECH_NAME)
os.makedirs(TECH_DIR, exist_ok=True)

# ---------------------------
# 1) Load feature lists to get gen/da/cal columns and tech index
# ---------------------------
feat_list_path = os.path.join(DATA_PROC_DIR, "feature_lists_entsoe.csv")
feat_df_list   = pd.read_csv(feat_list_path)

gen_cols = feat_df_list[feat_df_list["type"] == "gen"]["name"].tolist()
da_cols  = feat_df_list[feat_df_list["type"] == "da"]["name"].tolist()
cal_cols = feat_df_list[feat_df_list["type"] == "cal"]["name"].tolist()

if TECH_NAME not in gen_cols:
    raise ValueError(f"{TECH_NAME} not found in gen_cols: {gen_cols}")

tech_idx = gen_cols.index(TECH_NAME)
print(f"[TECH] Using tech={TECH_NAME}, index={tech_idx}")
print("[TECH] gen_cols:", gen_cols)
print("[TECH] da_cols :", da_cols)
print("[TECH] cal_cols:", cal_cols)

# Hist features = all gen + calendar; DA features as separate block
hist_feature_cols = gen_cols + cal_cols
da_feature_cols   = da_cols
all_feature_cols  = list(dict.fromkeys(hist_feature_cols + da_feature_cols))


# ---------------------------
# 2) Load sliding-window TRAIN / VAL data and reshape
# ---------------------------
X_train_path = os.path.join(DATA_PROC_DIR, "entsoe_train_X_sliding_24_24.csv")
Y_train_path = os.path.join(DATA_PROC_DIR, "entsoe_train_Y_sliding_24_24.csv")
X_val_path   = os.path.join(DATA_PROC_DIR, "entsoe_val_X_sliding_24_24.csv")
Y_val_path   = os.path.join(DATA_PROC_DIR, "entsoe_val_Y_sliding_24_24.csv")

X_train_flat = pd.read_csv(X_train_path).values  # (N_train, 24*(F_hist+F_da))
Y_train_flat = pd.read_csv(Y_train_path).values  # (N_train, 24 * n_gen_cols)
X_val_flat   = pd.read_csv(X_val_path).values
Y_val_flat   = pd.read_csv(Y_val_path).values

n_train = X_train_flat.shape[0]
n_val   = X_val_flat.shape[0]

# infer feature dimension per timestep and number of gen targets
F_total        = X_train_flat.shape[1] // TRAIN_WIN_H
n_gen_targets  = len(gen_cols)

if Y_train_flat.shape[1] != LABEL_WIN_H * n_gen_targets:
    raise RuntimeError("Y_train_flat shape is inconsistent with LABEL_WIN_H and n_gen_targets")

print(f"[SHAPE] X_train_flat: {X_train_flat.shape} -> (N, {TRAIN_WIN_H}, {F_total})")
print(f"[SHAPE] Y_train_flat: {Y_train_flat.shape} -> (N, {LABEL_WIN_H}, {n_gen_targets})")

# reshape to (N, 24, F_total) and (N, 24, n_gen_targets)
X_train_seq = X_train_flat.reshape(n_train, TRAIN_WIN_H, F_total)
X_val_seq   = X_val_flat.reshape(n_val,   TRAIN_WIN_H, F_total)



Y_train_all = Y_train_flat.reshape(n_train, LABEL_WIN_H, n_gen_targets)
Y_val_all   = Y_val_flat.reshape(n_val,   LABEL_WIN_H, n_gen_targets)

# extract hydro dispatchable only: shape (N, 24)
Y_train = Y_train_all[:, :, tech_idx]
Y_val   = Y_val_all[:, :, tech_idx]

print(f"[SHAPE] X_train_seq: {X_train_seq.shape}, Y_train: {Y_train.shape}")
print(f"[SHAPE] X_val_seq  : {X_val_seq.shape}, Y_val  : {Y_val.shape}")

# ---------------------------
# 3) Build CarbonCast-style MLP for this tech
# ---------------------------
EPOCHS      = 100
BATCH_SIZE  = 10
ACTIVATION  = "relu"
LOSS_FUNC   = "mse"
LEARNING_RT = 0.001
HIDDEN      = [51, 36]

model = Sequential()
model.add(InputLayer(input_shape=(TRAIN_WIN_H, F_total)))
model.add(Flatten())
model.add(Dense(HIDDEN[0], activation=ACTIVATION))
model.add(Dense(HIDDEN[1], activation=ACTIVATION))
model.add(Dense(LABEL_WIN_H))  # 24 outputs (24h horizon)

opt = tf.keras.optimizers.Adam(learning_rate=LEARNING_RT)
model.compile(loss=LOSS_FUNC, optimizer=opt, metrics=["mae"])

model_summary = []
model.summary(print_fn=lambda x: model_summary.append(x))
print("\n".join(model_summary))

# EarlyStopping only – restore best weights in memory
es = EarlyStopping(
    monitor="val_loss",
    mode="min",
    patience=10,
    verbose=1,
    restore_best_weights=True,
)

# ---------------------------
# 4) Train model
# ---------------------------
history = model.fit(
    X_train_seq, Y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val_seq, Y_val),
    callbacks=[es],
    verbose=2
)

print("[TRAIN] Training finished. Best weights restored.")
best_model = model  # use this directly

# Save model in native Keras format in tech subfolder
model_path = os.path.join(TECH_DIR, "best_mlp_hydro_dispatchable.keras")
best_model.save(model_path)
print(f"[TRAIN] Saved model -> {model_path}")

# ---------------------------
# 5) Load hourly scaled VAL + TEST and build strict day-ahead forecasts for 2024
# ---------------------------
train_scaled_path = os.path.join(DATA_PROC_DIR, "entsoe_train_scaled_hourly.csv")
val_scaled_path   = os.path.join(DATA_PROC_DIR, "entsoe_val_scaled_hourly.csv")
test_scaled_path  = os.path.join(DATA_PROC_DIR, "entsoe_test_scaled_hourly.csv")

val_scaled  = pd.read_csv(val_scaled_path, parse_dates=["datetime_utc"]).set_index("datetime_utc").sort_index()
test_scaled = pd.read_csv(test_scaled_path, parse_dates=["datetime_utc"]).set_index("datetime_utc").sort_index()

print("[TEST] val_scaled :", val_scaled.shape,  "|", val_scaled.index.min(), "->", val_scaled.index.max())
print("[TEST] test_scaled:", test_scaled.shape, "|", test_scaled.index.min(),  "->", test_scaled.index.max())

# Combine val tail + test for history context
full_df = pd.concat([val_scaled, test_scaled], axis=0).sort_index()

# List of test days (UTC midnights) with full 24h coverage
test_days = sorted(test_scaled.index.normalize().unique())
print(f"[TEST] Days in 2024: {len(test_days)}")

all_pred_scaled = []
all_true_scaled = []
all_time_index  = []

for day in test_days:
    horizon_start = pd.Timestamp(day)                      # e.g. 2024-01-01 00:00
    horizon_end   = horizon_start + timedelta(hours=23)    # inclusive

    # History: previous 24 hours
    hist_start = horizon_start - timedelta(hours=24)
    hist_end   = horizon_start - timedelta(hours=1)

    hist_slice    = full_df.loc[hist_start:hist_end]
    horizon_slice = full_df.loc[horizon_start:horizon_end]

    if len(hist_slice) != TRAIN_WIN_H or len(horizon_slice) != LABEL_WIN_H:
        print(f"[WARN] Skipping day {day.date()} due to incomplete history/horizon "
              f"({len(hist_slice)} hist hrs, {len(horizon_slice)} horizon hrs).")
        continue

    H = hist_slice[hist_feature_cols].values    # (24, F_hist)
    F = horizon_slice[da_feature_cols].values   # (24, F_da)

    if H.shape[0] != TRAIN_WIN_H or F.shape[0] != LABEL_WIN_H:
        print(f"[WARN] Shape mismatch on day {day.date()}, skipping.")
        continue

    X_day = np.concatenate([H, F], axis=1)  # (24, F_total) – must match training





    X_day = X_day.reshape(1, TRAIN_WIN_H, F_total)

    y_pred_day_scaled = best_model.predict(X_day, verbose=0)[0]  # (24,)
    y_true_day_scaled = horizon_slice[TECH_NAME].values          # (24,)

    all_pred_scaled.append(y_pred_day_scaled)
    all_true_scaled.append(y_true_day_scaled)
    all_time_index.append(horizon_slice.index.values)

# Stack to hourly arrays
if not all_pred_scaled:
    raise RuntimeError("[TEST] No valid test days were forecasted.")

y_pred_scaled = np.concatenate(all_pred_scaled, axis=0)
y_true_scaled = np.concatenate(all_true_scaled, axis=0)
time_index    = np.concatenate(all_time_index, axis=0)

print(f"[TEST] Total hourly predictions: {len(y_true_scaled)}")

# ---------------------------
# 6) Inverse scaling for hydro dispatchable (using scaler_y from prep cell)
# ---------------------------
try:
    scaler_y  # just to check it exists
except NameError:
    raise RuntimeError("scaler_y is not available in this runtime. "
                       "You must run the prep cell (with scaler_y) before this.")

n_techs = len(gen_cols)
k       = tech_idx

# build placeholder arrays to inverse_transform
T = len(y_true_scaled)
y_true_scaled_full = np.zeros((T, n_techs))
y_pred_scaled_full = np.zeros((T, n_techs))

y_true_scaled_full[:, k] = y_true_scaled
y_pred_scaled_full[:, k] = y_pred_scaled

y_true_unscaled_full = scaler_y.inverse_transform(y_true_scaled_full)
y_pred_unscaled_full = scaler_y.inverse_transform(y_pred_scaled_full)

y_true = y_true_unscaled_full[:, k]
y_pred = y_pred_unscaled_full[:, k]

# ---------------------------
# 7) Compute hourly metrics
# ---------------------------
rmse_hourly = np.sqrt(mean_squared_error(y_true, y_pred))
mae_hourly  = mean_absolute_error(y_true, y_pred)

# MAPE (%)
eps = 1e-6
mape_hourly = np.mean(np.abs((y_true - y_pred) / np.clip(np.abs(y_true), eps, None))) * 100.0

mean_true = np.mean(y_true)
nmae_mean_hourly = mae_hourly / (mean_true + eps)

print("\n[METRICS – HOURLY 2024, HYDRO_DISPATCHABLE]")
print(f"RMSE      = {rmse_hourly:.4f} MW")
print(f"MAE       = {mae_hourly:.4f} MW")
print(f"MAPE      = {mape_hourly:.2f} %")
print(f"nMAE_mean = {nmae_mean_hourly:.4f}")

# ---------------------------
# 8) Daily metrics
# ---------------------------
# Build DataFrame with hourly results
hourly_df = pd.DataFrame({
    "datetime_utc": pd.to_datetime(time_index),
    "y_true_mw": y_true,
    "y_pred_mw": y_pred
}).set_index("datetime_utc").sort_index()

hourly_df["abs_error"] = np.abs(hourly_df["y_true_mw"] - hourly_df["y_pred_mw"])
hourly_df["sq_error"]  = (hourly_df["y_true_mw"] - hourly_df["y_pred_mw"])**2
hourly_df["ape"]       = np.where(
    np.abs(hourly_df["y_true_mw"]) > eps,
    np.abs(hourly_df["y_true_mw"] - hourly_df["y_pred_mw"]) / np.abs(hourly_df["y_true_mw"]),
    np.nan
)
# per-hour normalized abs error using global mean_true (nMAE_mean-style)
hourly_df["nabs_error_mean"] = hourly_df["abs_error"] / (mean_true + eps)

# daily aggregates
daily_groups = hourly_df.groupby(hourly_df.index.normalize())
daily_metrics = []

for day, group in daily_groups:
    if len(group) == 0:
        continue

    y_t = group["y_true_mw"].values
    y_p = group["y_pred_mw"].values

    rmse_d = np.sqrt(np.mean((y_t - y_p)**2))
    mae_d  = np.mean(np.abs(y_t - y_p))
    denom   = np.clip(np.abs(y_t), eps, None)
    mape_d  = np.mean(np.abs((y_t - y_p) / denom)) * 100.0
    mean_d  = np.mean(y_t)
    nmae_d  = mae_d / (mean_d + eps)

    daily_metrics.append({
        "date": day.date(),
        "RMSE_MW": rmse_d,
        "MAE_MW": mae_d,
        "MAPE_percent": mape_d,
        "nMAE_mean": nmae_d,
        "mean_true_MW": mean_d
    })

daily_df = pd.DataFrame(daily_metrics).sort_values("date")

print("\n[METRICS – DAILY 2024, HYDRO_DISPATCHABLE]")
print(daily_df.describe()[["RMSE_MW", "MAE_MW", "MAPE_percent", "nMAE_mean"]])

# ---------------------------
# 9) Save CSVs (into tech subfolder)
# ---------------------------
hourly_csv_path  = os.path.join(TECH_DIR, "hydro_dispatchable_hourly_predictions_2024.csv")
daily_csv_path   = os.path.join(TECH_DIR, "hydro_dispatchable_daily_metrics_2024.csv")
summary_csv_path = os.path.join(TECH_DIR, "hydro_dispatchable_overall_metrics_2024.csv")

hourly_df.reset_index().to_csv(hourly_csv_path, index=False)
daily_df.to_csv(daily_csv_path, index=False)

# small overall summary row
summary_df = pd.DataFrame([{
    "RMSE_MW": rmse_hourly,
    "MAE_MW": mae_hourly,
    "MAPE_percent": mape_hourly,
    "nMAE_mean": nmae_mean_hourly,
    "mean_true_MW": mean_true
}])
summary_df.to_csv(summary_csv_path, index=False)

print(f"\n[SAVE] Hourly  -> {hourly_csv_path}")
print(f"[SAVE] Daily   -> {daily_csv_path}")
print(f"[SAVE] Summary -> {summary_csv_path}")

# ---------------------------
# 10) Plots (save + show)
# ---------------------------

# (a) Hourly actual vs predicted for 2024
plt.figure(figsize=(14, 4))
plt.plot(hourly_df.index, hourly_df["y_true_mw"], label="Actual", linewidth=1)
plt.plot(hourly_df.index, hourly_df["y_pred_mw"], label="Predicted", linewidth=0.8, alpha=0.7)
plt.title("Hydro Dispatchable Generation – Hourly Actual vs Predicted (2024)")
plt.xlabel("Time (UTC)")
plt.ylabel("Power [MW]")
plt.legend()
plt.tight_layout()
plot1_path = os.path.join(TECH_DIR, "hydro_dispatchable_hourly_actual_vs_pred_2024.png")
plt.savefig(plot1_path, dpi=150)
plt.show()
plt.close()
print(f"[PLOT] Saved hourly actual vs predicted -> {plot1_path}")

# (b) Histogram of hourly normalized abs error (nMAE_mean-style)
plt.figure(figsize=(6, 4))
plt.hist(hourly_df["nabs_error_mean"].values, bins=50)
plt.title("Hydro Dispatchable – Distribution of Hourly Normalized Abs Error (2024)")
plt.xlabel("Normalized Abs Error (|e| / mean_true)")
plt.ylabel("Frequency [hours]")
plt.tight_layout()
plot2_path = os.path.join(TECH_DIR, "hydro_dispatchable_hourly_norm_abs_error_hist_2024.png")
plt.savefig(plot2_path, dpi=150)
plt.show()
plt.close()
print(f"[PLOT] Saved hourly normalized abs error histogram -> {plot2_path}")

print("\n[DONE] Per-tech MLP for hydro dispatchable finished.")


# 2. Tier 2 — dynamic day ahead electricity emission signals

This section keeps the final emission-signal construction blocks used in the paper:

- **AEF**
- **UpRamp MEF**
- **MSDR Soft MEF**


## 2.1 AEF from actual generation


In [ ]:
# ============================================
# TIER 2.1.A – Build hourly AEF from actual gen (2022–23 train + 2024 test)
# ============================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_PROC_DIR = "/content/data_entsoe_es_processed"
EMBER_CO2_DIR = "/content/ember_co2"
TIER2_DIR     = "/content/data_tier2_mef"
TIER2_AEF_DIR = os.path.join(TIER2_DIR, "tier2_1_aef")

os.makedirs(TIER2_DIR, exist_ok=True)
os.makedirs(TIER2_AEF_DIR, exist_ok=True)

TRAIN_MASTER_PATH = os.path.join(DATA_PROC_DIR, "master_ES_train_2022_2023_interp.csv")
TEST_MASTER_PATH  = os.path.join(DATA_PROC_DIR, "master_ES_test_2024_interp.csv")

print("[AEF-1A] Loading master train/test...")
train_master = pd.read_csv(TRAIN_MASTER_PATH, parse_dates=["datetime_utc"]).set_index("datetime_utc").sort_index()
test_master  = pd.read_csv(TEST_MASTER_PATH,  parse_dates=["datetime_utc"]).set_index("datetime_utc").sort_index()

print("[AEF-1A] TRAIN master:", train_master.shape,
      "| range:", train_master.index.min(), "->", train_master.index.max())
print("[AEF-1A] TEST  master:", test_master.shape,
      "| range:", test_master.index.min(), "->", test_master.index.max())


# ---------- EF map loader (kgCO2/kWh per Tier1 tech) ----------

def load_ef_map(ember_dir: str) -> dict:
    csvs = [
        os.path.join(ember_dir, f)
        for f in os.listdir(ember_dir)
        if f.startswith("gv2-tier2-ember-per-tech-emission-") and f.endswith(".csv")
    ]
    if not csvs:
        raise RuntimeError("[AEF-1A] No Ember EF files found")

    frames = []
    for path in sorted(csvs):
        df = pd.read_csv(path)
        if "Tier1 Tech" not in df.columns or "EF (kgCo2/MWh)" not in df.columns:
            raise ValueError(f"[AEF-1A] Unexpected columns in {path}: {df.columns}")
        frames.append(df[["Tier1 Tech", "EF (kgCo2/MWh)"]].copy())

    ef_all = pd.concat(frames, axis=0)

    ef_all = ef_all.groupby("Tier1 Tech", as_index=False)["EF (kgCo2/MWh)"].mean()
    ef_all["EF_kg_per_kwh"] = ef_all["EF (kgCo2/MWh)"] / 1000.0

    ef_map = dict(zip(ef_all["Tier1 Tech"], ef_all["EF_kg_per_kwh"]))
    print("\n[AEF-1A] EF map (kgCO2/kWh):")
    for k, v in ef_map.items():
        print("  ", k, "->", f"{v:.4f}")
    return ef_map

ef_map = load_ef_map(EMBER_CO2_DIR)


# ---------- Helper: compute CO2 and AEF from actual gen ----------

def add_co2_and_aef_from_gen(df: pd.DataFrame,
                             ef_map: dict,
                             name: str) -> pd.DataFrame:
    df = df.copy()
    eps = 1e-6

    # Determine which gen_* columns are in both df and EF map
    gen_cols_from_ef = [tech for tech in ef_map.keys() if tech in df.columns]
    if not gen_cols_from_ef:
        raise RuntimeError(f"[AEF-1A] No overlapping gen_* columns + EF map for {name}")

    print(f"\n[AEF-1A] {name}: using gen techs:", gen_cols_from_ef)

    # Compute per-tech CO2 (kg/h)
    co2_cols = []
    for tech in gen_cols_from_ef:
        ef_kg_per_kwh = ef_map[tech]
        out_col = f"co2_{tech}_kgph"
        df[out_col] = df[tech] * 1000.0 * ef_kg_per_kwh
        co2_cols.append(out_col)

    df["co2_total_kgph"] = df[co2_cols].sum(axis=1)

    # Total generation in kWh from the same techs
    df["gen_total_kwh"] = df[gen_cols_from_ef].sum(axis=1) * 1000.0

    # AEF as kgCO2 per kWh consumed/produced
    df["aef_true_kg_per_kwh"] = df["co2_total_kgph"] / (df["gen_total_kwh"] + eps)

    # Basic debug stats
    print(f"[AEF-1A] {name} co2_total_kgph stats:")
    print(df["co2_total_kgph"].describe())
    print(f"\n[AEF-1A] {name} AEF_true stats (kgCO2/kWh):")
    print(df["aef_true_kg_per_kwh"].describe())

    return df

train_with_aef = add_co2_and_aef_from_gen(train_master, ef_map, "TRAIN(2022–2023)")
test_with_aef  = add_co2_and_aef_from_gen(test_master,  ef_map, "TEST(2024)")


# ---------- Build AEF-ML training tables (features + target) ----------

aef_feature_cols = [
    "da_total_load_mw",
    "da_residual_load_mw",
    "da_net_imports_mw",
    "da_import_share",
    "da_elec_price_EURperMWh",
    "da_gas_price_migbasPVB_EURperMWh",
    "da_res_share_wind",
    "da_res_share_solar",
    "hour",
    "dow",
    "month",
    "is_weekend",
    "is_holiday",
]

missing_train = [c for c in aef_feature_cols if c not in train_with_aef.columns]
missing_test  = [c for c in aef_feature_cols if c not in test_with_aef.columns]
if missing_train or missing_test:
    raise ValueError(f"[AEF-1A] Missing features – train: {missing_train}, test: {missing_test}")

aef_target_col = "aef_true_kg_per_kwh"

aef_train_tbl = train_with_aef[aef_feature_cols + [aef_target_col]].copy()
aef_test_tbl  = test_with_aef[aef_feature_cols + [aef_target_col]].copy()

aef_train_path = os.path.join(TIER2_AEF_DIR, "tier2_1_aef_train_2022_2023.csv")
aef_test_path  = os.path.join(TIER2_AEF_DIR, "tier2_1_aef_true_2024.csv")

aef_train_tbl.reset_index().to_csv(aef_train_path, index=False)
aef_test_tbl.reset_index().to_csv(aef_test_path, index=False)

print(f"\n[AEF-1A] Saved AEF train table -> {aef_train_path} (rows={len(aef_train_tbl)})")
print(f"[AEF-1A] Saved AEF TRUE 2024 table -> {aef_test_path} (rows={len(aef_test_tbl)})")


# ---------- Quick sanity plots ----------

try:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    train_with_aef["aef_true_kg_per_kwh"].hist(bins=40, ax=axes[0], alpha=0.7)
    axes[0].set_title("AEF_true distribution – 2022–2023")
    axes[0].set_xlabel("kgCO2/kWh")
    axes[0].set_ylabel("Hours")

    test_with_aef["aef_true_kg_per_kwh"].hist(bins=40, ax=axes[1], alpha=0.7)
    axes[1].set_title("AEF_true distribution – 2024")
    axes[1].set_xlabel("kgCO2/kWh")
    axes[1].set_ylabel("Hours")

    plt.tight_layout()
    hist_path = os.path.join(TIER2_AEF_DIR, "tier2_1_aef_true_hist_2022_2024.png")
    plt.savefig(hist_path, dpi=150)
    plt.close(fig)
    print("[AEF-1A] Saved AEF histograms ->", hist_path)
except Exception as e:
    print("[AEF-1A] Skipped AEF plots due to error:", e)

print("\n[AEF-1A] Done. Next cell will build AEF_2024 from Tier-1 forecasts.")


## 2.2 AEF from Tier-1 forecasted generation


In [ ]:
# =========================
# Tier 2.1.B – AEF 2024 from Tier-1 forecasts (fixed tz alignment)
# =========================

import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

TIER2_DIR      = "/content/data_tier2_mef"
TIER2_AEF_DIR  = os.path.join(TIER2_DIR, "tier2_1_aef")
DATA_PROC_DIR  = "/content/data_entsoe_es_processed"
EMBER_CO2_DIR  = "/content/ember_co2"

TRUE_2024_CSV  = os.path.join(TIER2_AEF_DIR, "tier2_1_aef_true_2024.csv")
OUT_FORECAST_CSV = os.path.join(TIER2_AEF_DIR, "tier2_1_aef_forecast_from_tier1_2024_full.csv")
OUT_METRICS_CSV  = os.path.join(TIER2_AEF_DIR, "tier2_1_aef_eval_tier1_vs_true_2024_metrics.csv")
OUT_SERIES_CSV   = os.path.join(TIER2_AEF_DIR, "tier2_1_aef_eval_tier1_vs_true_2024_series.csv")

print("[AEF-1B] Loading TRUE AEF 2024 table...")
aef_true_2024 = (
    pd.read_csv(TRUE_2024_CSV, parse_dates=["datetime_utc"])
      .set_index("datetime_utc")
      .sort_index()
)

# ensure UTC tz-aware index
if aef_true_2024.index.tz is None:
    aef_true_2024.index = aef_true_2024.index.tz_localize("UTC")
else:
    aef_true_2024 = aef_true_2024.tz_convert("UTC")

print("[AEF-1B] TRUE AEF 2024:", aef_true_2024.shape,
      "| range:", aef_true_2024.index.min(), "->", aef_true_2024.index.max())

# --------- EF map from EMBER files (2022–2024) ----------

ef_files = [
    os.path.join(EMBER_CO2_DIR, f)
    for f in os.listdir(EMBER_CO2_DIR)
    if f.startswith("gv2-tier2-ember-per-tech-emission-") and f.endswith(".csv")
]
if not ef_files:
    raise RuntimeError("[AEF-1B] No EF files found in EMBER_CO2_DIR.")

ef_frames = []
for path in sorted(ef_files):
    df_ef = pd.read_csv(path)
    if "Tier1 Tech" not in df_ef.columns or "EF (kgCo2/MWh)" not in df_ef.columns:
        raise ValueError(f"[EF] Unexpected columns in {path}: {df_ef.columns}")
    df_ef = df_ef[["Tier1 Tech", "EF (kgCo2/MWh)"]].copy()
    df_ef["EF_kg_per_kwh"] = df_ef["EF (kgCo2/MWh)"] / 1000.0
    ef_frames.append(df_ef)

ef_df = (pd.concat(ef_frames, axis=0)
           .drop_duplicates(subset=["Tier1 Tech"])
           .set_index("Tier1 Tech")
           .sort_index())
ef_map = ef_df["EF_kg_per_kwh"].to_dict()

print("\n[AEF-1B] EF map (kgCO2/kWh):")
for k, v in ef_map.items():
    print("  ", k, "->", f"{v:.4f}")

tech_cols = sorted(ef_map.keys())

# --------- Load Tier-1 per-tech predictions and align tz ----------

print("\n[AEF-1B] Loading Tier-1 per-tech hourly predictions for 2024...")

pred_frames = []

TIER1_FLAT_DIR = os.path.join(DATA_PROC_DIR, "tier1forecasts")

for tech in tech_cols:

    matches = []

    tech_folder = os.path.join(DATA_PROC_DIR, tech)
    pattern_sub = os.path.join(tech_folder, "*hourly_predictions_2024*.csv")
    matches = glob.glob(pattern_sub)


    if not matches and os.path.isdir(TIER1_FLAT_DIR):
        pattern_flat = os.path.join(
            TIER1_FLAT_DIR,
            f"*{tech}*hourly_predictions_2024*.csv"
        )
        matches = glob.glob(pattern_flat)

    if not matches:
        print(f"[AEF-1B] WARNING: no prediction file for {tech} in subfolder or tier1forecasts – skipping.")
        continue

    pred_path = matches[0]

    df_pred = (
        pd.read_csv(pred_path, parse_dates=["datetime_utc"])
          .set_index("datetime_utc")
          .sort_index()
    )


    if df_pred.index.tz is None:
        df_pred.index = df_pred.index.tz_localize("UTC")
    else:
        df_pred = df_pred.tz_convert("UTC")

    # fossil gas special case, if u run code after tier 1 wihtour edit in the csv file or master file
    #if tech == "gen_fossil_gas_mw":
     #   pred_col = "y_pred"
    #else:
    pred_col = "y_pred_mw"

    if pred_col not in df_pred.columns:
        raise ValueError(
            f"[AEF-1B] Column '{pred_col}' not found in {pred_path} "
            f"(cols={df_pred.columns.tolist()})"
        )

    df_pred = df_pred[[pred_col]].rename(columns={pred_col: tech})
    pred_frames.append(df_pred)

    print(f"[AEF-1B] {tech}: loaded {df_pred.shape[0]} rows from {os.path.basename(pred_path)}")

if not pred_frames:
    raise RuntimeError("[AEF-1B] No Tier-1 prediction series loaded.")

pred_all = pd.concat(pred_frames, axis=1).sort_index()

# align prediction horizon to TRUE AEF horizon
pred_all = pred_all.reindex(aef_true_2024.index)

print("\n[AEF-1B] Combined Tier-1 prediction matrix:", pred_all.shape)
print("[AEF-1B] 2024_Tier1 – raw prediction ranges (MW):")
stats = pred_all.agg(['min', 'max', 'mean']).T
print(stats)


# sanity check: ensure we did not kill everything by tz mismatch
if pred_all.isna().all().all():
    raise RuntimeError(
        "[AEF-1B] All Tier-1 predictions are NaN after reindexing. "
        "Check datetime_utc formats / timezones in prediction CSVs."
    )

# --------- Helper: compute AEF from per-tech MW predictions ----------

def compute_aef_from_pred(df_pred: pd.DataFrame, ef_map: dict, label: str):
    df = df_pred.copy()
    eps = 1e-6

    co2_cols = []
    for tech, ef in ef_map.items():
        if tech not in df.columns:
            continue
        out_col = f"co2_{tech}_kgph_pred"
        df[out_col] = df[tech] * 1000.0 * ef   # MW → kW → kg
        co2_cols.append(out_col)

    if not co2_cols:
        df["co2_total_kgph_pred"] = 0.0
    else:
        df["co2_total_kgph_pred"] = df[co2_cols].sum(axis=1)

    df["gen_total_mw_pred"]  = df[[c for c in df.columns if c.startswith("gen_")]].sum(axis=1)
    df["gen_total_kwh_pred"] = df["gen_total_mw_pred"] * 1000.0
    df["aef_tier1_forecast_kg_per_kwh"] = df["co2_total_kgph_pred"] / (df["gen_total_kwh_pred"] + eps)

    print(f"\n[AEF-1B] {label} co2_total_kgph_pred stats:")
    print(df["co2_total_kgph_pred"].describe())
    print(f"\n[AEF-1B] {label} AEF_tier1_forecast stats (kgCO2/kWh):")
    print(df["aef_tier1_forecast_kg_per_kwh"].describe())

    return df

# --------- Compute Tier-1 based AEF forecast and compare to TRUE 2024 ----------

pred_with_aef = compute_aef_from_pred(pred_all, ef_map, "2024_Tier1")

out = pd.concat(
    [
        aef_true_2024[["aef_true_kg_per_kwh"]],
        pred_with_aef[["aef_tier1_forecast_kg_per_kwh"]],
    ],
    axis=1,
)
out = out.dropna(subset=["aef_true_kg_per_kwh", "aef_tier1_forecast_kg_per_kwh"]).copy()

# metrics
y_true = out["aef_true_kg_per_kwh"].values
y_pred = out["aef_tier1_forecast_kg_per_kwh"].values

mae  = np.mean(np.abs(y_pred - y_true))
rmse = np.sqrt(np.mean((y_pred - y_true) ** 2))
mape = np.mean(np.abs((y_pred - y_true) / np.maximum(np.abs(y_true), 1e-6))) * 100.0

metrics_df = pd.DataFrame(
    {
        "MAE_kg_per_kwh":  [mae],
        "RMSE_kg_per_kwh": [rmse],
        "MAPE_percent":    [mape],
        "n_hours":         [len(out)],
    }
)

print("\n[AEF-1B] AEF 2024 – Tier1 vs TRUE metrics:")
print(metrics_df)

# save outputs
pred_with_aef.reset_index().to_csv(OUT_FORECAST_CSV, index=False)
metrics_df.to_csv(OUT_METRICS_CSV, index=False)
out.reset_index().to_csv(OUT_SERIES_CSV, index=False)

print(f"[AEF-1B] Saved Tier-1-based AEF forecast table -> {OUT_FORECAST_CSV}")
print(f"[AEF-1B] Saved metrics -> {OUT_METRICS_CSV}")
print(f"[AEF-1B] Saved TRUE vs Tier1 AEF series -> {OUT_SERIES_CSV}")

# quick scatter + time series for sanity
try:
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.scatter(
        out["aef_true_kg_per_kwh"],
        out["aef_tier1_forecast_kg_per_kwh"],
        alpha=0.3,
        s=8,
        label="hours",
    )
    lims = [
        min(out["aef_true_kg_per_kwh"].min(), out["aef_tier1_forecast_kg_per_kwh"].min()),
        max(out["aef_true_kg_per_kwh"].max(), out["aef_tier1_forecast_kg_per_kwh"].max()),
    ]
    ax.plot(lims, lims, "k--", linewidth=1)
    ax.set_xlabel("AEF TRUE [kgCO2/kWh]")
    ax.set_ylabel("AEF Tier1 forecast [kgCO2/kWh]")
    ax.set_title("AEF 2024 – TRUE vs Tier1 forecast")
    plt.tight_layout()
    scatter_path = os.path.join(TIER2_AEF_DIR, "tier2_1_aef_true_vs_tier1_scatter_2024.png")
    plt.savefig(scatter_path, dpi=150)
    plt.close(fig)
    print(f"[AEF-1B] Saved scatter plot -> {scatter_path}")

    fig, ax = plt.subplots(figsize=(10, 4))
    out["aef_true_kg_per_kwh"].plot(ax=ax, label="TRUE", linewidth=1)
    out["aef_tier1_forecast_kg_per_kwh"].plot(ax=ax, label="Tier1", linewidth=1, alpha=0.8)
    ax.set_ylabel("kgCO2/kWh")
    ax.set_title("AEF 2024 – TRUE vs Tier1 forecast (time series)")
    ax.legend()
    plt.tight_layout()
    ts_path = os.path.join(TIER2_AEF_DIR, "tier2_1_aef_true_vs_tier1_timeseries_2024.png")
    plt.savefig(ts_path, dpi=150)
    plt.close(fig)
    print(f"[AEF-1B] Saved time-series plot -> {ts_path}")
except Exception as e:
    print("[AEF-1B] Skipped plots due to error:", e)

print("\n[AEF-1B] Done. Next step will be Tier 2.1.C – ML AEF model for 2022–23 and comparison on 2024.")


## 2.3 UpRamp MEF proxy


In [ ]:
# ============================================================
# MEF-2.2 – Ramp-based MEF_up proxy with k-hour ramps + shrinkage-to-AEF
#   - TRAIN (2022–2023 actual genmix): compute stack_up^(k), MEF_up^(k), tau=Q_q(stack_up>0)
#   - TEST  (2024): compute TRUE MEF_up^(k) from y_true_mw, PRED MEF_up^(k) from y_pred_mw
#   - Stabilize via shrinkage: MEF*(t)=w(t)*MEF_up(t)+(1-w(t))*AEF(t), w=stack/(stack+tau)
#   - Compare (TRUE vs PRED) for 4 configs: k∈{1,3}, q∈{0.10,0.20}
#   - Save: detailed CSV diagnostics + metrics + plots (PNG+SVG) + weight diagnostics + a ZIP bundle
#
# Folder expectations under /content:
#   /content/Genmix_input/
#        master_ES_train_2022_2023_interp_modified.csv   (datetime_utc + gen_* columns)
#        <tech>_hourly_predictions_2024*.csv             (datetime_utc, y_true_mw, y_pred_mw)
#   /content/ember emission/
#        gv2-tier2-ember-per-tech-emission-2022.csv
#        gv2-tier2-ember-per-tech-emission-2023.csv
#        gv2-tier2-ember-per-tech-emission-2024.csv      (Tier1 Tech, EF (kgCo2/MWh))
#   /content/AEF/
#        tier2_1_aef_eval_tier1_vs_true_2024_series.csv  (datetime_utc, aef_true_kg_per_kwh, aef_tier1_forecast_kg_per_kwh)
# ============================================================

import os, glob, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -----------------------------
# Plot style (your settings)
# -----------------------------
def apply_plot_settings(ax, title, xlabel, ylabel):
    ax.set_title(title, fontsize=14, fontweight='bold', pad=15)
    ax.set_xlabel(xlabel, fontsize=12, labelpad=10)
    ax.set_ylabel(ylabel, fontsize=12, labelpad=10)
    ax.tick_params(axis='both', which='major', labelsize=10)
    for spine in ax.spines.values():
        spine.set_color('#dadedf')
        spine.set_linewidth(2)
    ax.grid(False)

def savefig_png_svg(fig, out_noext, dpi=150):
    png_path = out_noext + ".png"
    svg_path = out_noext + ".svg"
    fig.savefig(png_path, dpi=dpi, bbox_inches="tight")
    fig.savefig(svg_path, bbox_inches="tight")
    plt.close(fig)
    return png_path, svg_path

# -----------------------------
# CONFIG
# -----------------------------
BASE_DIR   = "/content"
GENMIX_DIR = os.path.join(BASE_DIR, "Genmix_input")
EMBER_DIR  = os.path.join(BASE_DIR, "ember emission")
AEF_DIR    = os.path.join(BASE_DIR, "AEF")

OUT_DIR    = os.path.join(BASE_DIR, "tier2_2_mef_vnew2")
os.makedirs(OUT_DIR, exist_ok=True)

TRAIN_GENMIX_FILE = os.path.join(GENMIX_DIR, "master_ES_train_2022_2023_interp_modified with only true train genmix.csv")

EMBER_2022 = os.path.join(EMBER_DIR, "gv2-tier2-ember-per-tech-emission-2022.csv")
EMBER_2023 = os.path.join(EMBER_DIR, "gv2-tier2-ember-per-tech-emission-2023.csv")
EMBER_2024 = os.path.join(EMBER_DIR, "gv2-tier2-ember-per-tech-emission-2024.csv")

AEF_SERIES_FILE = os.path.join(AEF_DIR, "tier2_1_aef_eval_tier1_vs_true_2024_series.csv")

# Sweeps
K_LIST = [1, 3]
Q_LIST = [0.10, 0.20]

# Small eps
EPS_STACK = 1e-9

# -----------------------------
# Helper: UTC index alignment
# -----------------------------
def ensure_utc_index(df: pd.DataFrame, dt_col: str = "datetime_utc") -> pd.DataFrame:
    df = df.copy()
    if dt_col in df.columns:
        df[dt_col] = pd.to_datetime(df[dt_col], utc=True)
        df = df.set_index(dt_col)
    else:
        df.index = pd.to_datetime(df.index, utc=True)
    df = df.sort_index()
    if df.index.tz is None:
        df.index = df.index.tz_localize("UTC")
    else:
        df.index = df.index.tz_convert("UTC")
    return df

# -----------------------------
# Helper: Load Ember EF map (kgCO2/kWh)
# -----------------------------
def load_ember_ef_map(paths, label):
    frames = []
    for p in paths:
        if not os.path.exists(p):
            raise FileNotFoundError(f"[VNEW2] Missing Ember EF file ({label}): {p}")
        df = pd.read_csv(p)
        required = ["Tier1 Tech", "EF (kgCo2/MWh)"]
        miss = [c for c in required if c not in df.columns]
        if miss:
            raise ValueError(f"[VNEW2] Ember EF file missing columns {miss}: {p}")
        df = df[["Tier1 Tech", "EF (kgCo2/MWh)"]].copy()
        df["EF_kg_per_kwh"] = df["EF (kgCo2/MWh)"] / 1000.0
        frames.append(df[["Tier1 Tech", "EF_kg_per_kwh"]])
    ef_all = pd.concat(frames, axis=0)
    ef_avg = ef_all.groupby("Tier1 Tech", as_index=False)["EF_kg_per_kwh"].mean()
    ef_map = dict(zip(ef_avg["Tier1 Tech"], ef_avg["EF_kg_per_kwh"]))
    return ef_map, ef_avg

# -----------------------------
# Helper: Compute k-hour ramp MEF_up proxy + stack_up
#   ΔP_i^(k)(t) = P_i(t) - P_i(t-k)
#   ΔP_i^up     = max(ΔP_i^(k), 0)
#   stack_up    = Σ ΔP_i^up
#   CO2_up      = Σ ΔP_i^up * EF_i
#   MEF_up      = CO2_up / stack_up  if stack_up>0 else NaN
# -----------------------------
def compute_mef_up_proxy_k(gen_df: pd.DataFrame, ef_map: dict, tech_cols: list, k: int, label: str) -> pd.DataFrame:
    df = gen_df[tech_cols].copy().sort_index()
    for c in tech_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    dP = df.diff(k)
    dP_up = dP.clip(lower=0.0)

    stack_up = dP_up.sum(axis=1)

    ef_vec = np.array([ef_map[t] for t in tech_cols], dtype=float)
    co2_up = (dP_up.values * ef_vec).sum(axis=1)

    mef_up = np.full(shape=(len(stack_up),), fill_value=np.nan, dtype=float)
    mask = stack_up.values > EPS_STACK
    mef_up[mask] = co2_up[mask] / stack_up.values[mask]

    out = pd.DataFrame(index=df.index)
    out[f"{label}_stack_up_mw_k{k}"] = stack_up
    out[f"{label}_co2_up_weighted_k{k}"] = co2_up
    out[f"{label}_mef_up_kg_per_kwh_k{k}"] = mef_up
    return out

# -----------------------------
# Helper: Compute tau from TRAIN stack_up distribution
# -----------------------------
def compute_tau_from_train(stack_up: pd.Series, q: float) -> float:
    s = stack_up.dropna()
    s = s[s > EPS_STACK]
    if len(s) == 0:
        return np.nan
    return float(s.quantile(q))

# -----------------------------
# Helper: Shrinkage-to-AEF stabilization
#   w(t) = stack_up(t) / (stack_up(t) + tau)
#   MEF*(t) = w(t)*MEF_up(t) + (1-w(t))*AEF(t)
# -----------------------------
def stabilize_mef(mef_up: pd.Series, stack_up: pd.Series, aef: pd.Series, tau: float) -> pd.Series:
    if (tau is None) or (not np.isfinite(tau)) or (tau <= 0):
        return mef_up.copy()
    w = stack_up / (stack_up + tau)
    mef_star = w * mef_up + (1.0 - w) * aef
    return mef_star

# -----------------------------
# Weight diagnostics (hourly + summary)
# -----------------------------
def compute_weight_diagnostics_tables(
    index_utc,
    tau_mw: float,
    stack_up: pd.Series,
    mef_raw: pd.Series,
    aef: pd.Series,
    mef_final: pd.Series,
    tag: str,
):
    df = pd.DataFrame(index=index_utc)

    stack_up = pd.to_numeric(stack_up, errors="coerce")
    mef_raw  = pd.to_numeric(mef_raw,  errors="coerce")
    aef      = pd.to_numeric(aef,      errors="coerce")
    mef_final= pd.to_numeric(mef_final,errors="coerce")

    # w(t)=stack/(stack+tau)
    if (tau_mw is None) or (not np.isfinite(tau_mw)) or (tau_mw <= 0):
        w = pd.Series(np.nan, index=index_utc)
    else:
        w = stack_up / (stack_up + float(tau_mw))
        w = w.clip(lower=0.0, upper=1.0)

    mef_component = w * mef_raw
    aef_component = (1.0 - w) * aef
    recon_err = (mef_component + aef_component) - mef_final

    df["stack_up_mw"] = stack_up
    df["tau_mw"] = float(tau_mw) if np.isfinite(tau_mw) else np.nan
    df["w_mef"] = w
    df["mef_raw_kg_per_kwh"] = mef_raw
    df["aef_kg_per_kwh"] = aef
    df["mef_final_kg_per_kwh"] = mef_final
    df["mef_component"] = mef_component
    df["aef_component"] = aef_component
    df["reconstruction_error"] = recon_err

    w_valid = w.dropna()
    stack_valid = stack_up.dropna()

    thresholds = [0.2, 0.5, 0.8, 0.9]
    summary = {
        "tag": tag,
        "tau_mw": float(tau_mw) if np.isfinite(tau_mw) else np.nan,
        "n_hours_total": int(len(df)),
        "n_hours_w_valid": int(len(w_valid)),
        "w_mean": float(w_valid.mean()) if len(w_valid) else np.nan,
        "w_median": float(w_valid.median()) if len(w_valid) else np.nan,
        "w_q10": float(w_valid.quantile(0.10)) if len(w_valid) else np.nan,
        "w_q90": float(w_valid.quantile(0.90)) if len(w_valid) else np.nan,
        "stack_mean_mw": float(stack_valid.mean()) if len(stack_valid) else np.nan,
        "stack_median_mw": float(stack_valid.median()) if len(stack_valid) else np.nan,
        "stack_q10_mw": float(stack_valid.quantile(0.10)) if len(stack_valid) else np.nan,
        "stack_q90_mw": float(stack_valid.quantile(0.90)) if len(stack_valid) else np.nan,
        "recon_err_abs_mean": float(np.nanmean(np.abs(recon_err.values))),
        "recon_err_abs_max": float(np.nanmax(np.abs(recon_err.values))),
    }
    for th in thresholds:
        summary[f"frac_w_ge_{th}"] = float((w_valid >= th).mean()) if len(w_valid) else np.nan

    summary_df = pd.DataFrame([summary])
    return df, summary_df

# ============================================================
# 0) Load AEF series (2024)
# ============================================================
if not os.path.exists(AEF_SERIES_FILE):
    raise FileNotFoundError(f"[VNEW2] Missing AEF series file: {AEF_SERIES_FILE}")

aef_2024 = pd.read_csv(AEF_SERIES_FILE)
needed_aef = ["datetime_utc", "aef_true_kg_per_kwh", "aef_tier1_forecast_kg_per_kwh"]
miss = [c for c in needed_aef if c not in aef_2024.columns]
if miss:
    raise ValueError(f"[VNEW2] AEF series missing columns {miss}. Found: {aef_2024.columns.tolist()}")

aef_2024 = ensure_utc_index(aef_2024, "datetime_utc")
aef_2024 = aef_2024[["aef_true_kg_per_kwh", "aef_tier1_forecast_kg_per_kwh"]].copy()

# ============================================================
# 1) Load EF maps
# ============================================================
ef_train_map, ef_train_df = load_ember_ef_map([EMBER_2022, EMBER_2023], label="2022-2023")
ef_2024_map,  ef_2024_df  = load_ember_ef_map([EMBER_2024], label="2024")

# ============================================================
# 2) Load TRAIN genmix (actual, 2022–2023)
# ============================================================
if not os.path.exists(TRAIN_GENMIX_FILE):
    raise FileNotFoundError(f"[VNEW2] Missing TRAIN genmix file: {TRAIN_GENMIX_FILE}")

train_gen = pd.read_csv(TRAIN_GENMIX_FILE)
train_gen = ensure_utc_index(train_gen, "datetime_utc")

train_techs = sorted([t for t in ef_train_map.keys() if t in train_gen.columns])
if not train_techs:
    raise RuntimeError("[VNEW2] No overlapping gen tech columns between TRAIN genmix file and TRAIN EF map.")

# ============================================================
# 3) Load TEST 2024 per-tech prediction files
# ============================================================
if not os.path.isdir(GENMIX_DIR):
    raise FileNotFoundError(f"[VNEW2] Missing folder: {GENMIX_DIR}")

techs_2024 = sorted(list(ef_2024_map.keys()))

true_frames = []
pred_frames = []
loaded_techs = []

for tech in techs_2024:
    pattern = os.path.join(GENMIX_DIR, f"{tech}*hourly_predictions_2024*.csv")
    matches = glob.glob(pattern)
    if not matches:
        print(f"[VNEW2] WARNING: missing 2024 genmix file for {tech} (pattern: {pattern})")
        continue

    path = matches[0]
    df = pd.read_csv(path)
    needed = ["datetime_utc", "y_true_mw", "y_pred_mw"]
    miss = [c for c in needed if c not in df.columns]
    if miss:
        raise ValueError(f"[VNEW2] {os.path.basename(path)} missing {miss}. Found: {df.columns.tolist()}")

    df = ensure_utc_index(df, "datetime_utc")
    df["y_true_mw"] = pd.to_numeric(df["y_true_mw"], errors="coerce")
    df["y_pred_mw"] = pd.to_numeric(df["y_pred_mw"], errors="coerce")

    true_frames.append(df[["y_true_mw"]].rename(columns={"y_true_mw": tech}))
    pred_frames.append(df[["y_pred_mw"]].rename(columns={"y_pred_mw": tech}))
    loaded_techs.append(tech)

if not true_frames or not pred_frames:
    raise RuntimeError("[VNEW2] No 2024 per-tech prediction files loaded. Check file names in Genmix_input.")

gen_true_2024 = pd.concat(true_frames, axis=1).sort_index()
gen_pred_2024 = pd.concat(pred_frames, axis=1).sort_index()

common_idx_2024 = gen_true_2024.index.intersection(gen_pred_2024.index)
gen_true_2024 = gen_true_2024.reindex(common_idx_2024)
gen_pred_2024 = gen_pred_2024.reindex(common_idx_2024)

aef_2024 = aef_2024.reindex(common_idx_2024)

test_techs = sorted([t for t in gen_true_2024.columns if t in ef_2024_map])
if not test_techs:
    raise RuntimeError("[VNEW2] No overlapping gen tech columns between 2024 prediction files and 2024 EF map.")

# ============================================================
# 4) Run the grid of (k,q)
# ============================================================
all_outputs = []
summary_rows = []

# A) TRAIN precompute
train_mef_by_k = {}
for k in K_LIST:
    train_mef_by_k[k] = compute_mef_up_proxy_k(
        gen_df=train_gen,
        ef_map=ef_train_map,
        tech_cols=train_techs,
        k=k,
        label="train_true"
    )

# B) Tau table
tau_table_rows = []
for k in K_LIST:
    stack_series = train_mef_by_k[k][f"train_true_stack_up_mw_k{k}"]
    for q in Q_LIST:
        tau = compute_tau_from_train(stack_series, q)
        tau_table_rows.append({
            "k": k, "q": q, "tau_mw": tau,
            "train_stack_up_pos_count": int((stack_series > EPS_STACK).sum())
        })

tau_table = pd.DataFrame(tau_table_rows)
tau_csv = os.path.join(OUT_DIR, "vnew2_train_tau_values_by_k_q.csv")
tau_table.to_csv(tau_csv, index=False)
all_outputs.append(tau_csv)

# TRAIN stack hist plots
for k in K_LIST:
    stack_series = train_mef_by_k[k][f"train_true_stack_up_mw_k{k}"].dropna()
    pos = stack_series[stack_series > EPS_STACK]
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(pos.values, bins=60, alpha=0.8)
    for q in Q_LIST:
        tau = float(tau_table[(tau_table["k"] == k) & (tau_table["q"] == q)]["tau_mw"].iloc[0])
        ax.axvline(tau, linewidth=2, linestyle="--", label=f"q={int(q*100)}% → tau={tau:.2f} MW")
    apply_plot_settings(
        ax,
        title=f"TRAIN (2022–2023) stack_up distribution (k={k}h) with tau quantiles",
        xlabel="stack_up (MW)",
        ylabel="count (hours)"
    )
    ax.legend(fontsize=10)
    out_noext = os.path.join(OUT_DIR, f"vnew2_train_stackup_hist_k{k}")
    png, svg = savefig_png_svg(fig, out_noext)
    all_outputs += [png, svg]

# TRAIN raw stats
train_stats_rows = []
for k in K_LIST:
    mef_series = train_mef_by_k[k][f"train_true_mef_up_kg_per_kwh_k{k}"]
    stack_series = train_mef_by_k[k][f"train_true_stack_up_mw_k{k}"]
    train_stats_rows.append({
        "k": k,
        "n_hours": int(len(mef_series)),
        "n_nan_mef": int(mef_series.isna().sum()),
        "nan_frac_mef": float(mef_series.isna().mean()),
        "n_pos_stack": int((stack_series > EPS_STACK).sum()),
        "mef_mean": float(mef_series.dropna().mean()),
        "mef_std": float(mef_series.dropna().std()),
        "mef_p05": float(mef_series.dropna().quantile(0.05)) if mef_series.notna().any() else np.nan,
        "mef_p50": float(mef_series.dropna().quantile(0.50)) if mef_series.notna().any() else np.nan,
        "mef_p95": float(mef_series.dropna().quantile(0.95)) if mef_series.notna().any() else np.nan,
    })
train_stats = pd.DataFrame(train_stats_rows)
train_stats_csv = os.path.join(OUT_DIR, "vnew2_train_mef_stack_stats_by_k.csv")
train_stats.to_csv(train_stats_csv, index=False)
all_outputs.append(train_stats_csv)

# ============================================================
# 5) Per config outputs + NEW weight diagnostics CSVs
# ============================================================
def compute_metrics_like_aef(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mae  = float(np.mean(np.abs(y_pred - y_true)))
    rmse = float(np.sqrt(np.mean((y_pred - y_true) ** 2)))
    mape = float(np.mean(np.abs((y_pred - y_true) / np.maximum(np.abs(y_true), 1e-6))) * 100.0)
    return mae, rmse, mape

def pick_week_window(index_utc):
    start = index_utc.min()
    end = start + pd.Timedelta(days=7)
    return start, end

#collect all weight-diag summaries in one file too
weight_summary_all = []

for k in K_LIST:
    out_true = compute_mef_up_proxy_k(gen_true_2024, ef_2024_map, test_techs, k=k, label="true")
    out_pred = compute_mef_up_proxy_k(gen_pred_2024, ef_2024_map, test_techs, k=k, label="pred")

    mef_true_raw = out_true[f"true_mef_up_kg_per_kwh_k{k}"]
    mef_pred_raw = out_pred[f"pred_mef_up_kg_per_kwh_k{k}"]
    stack_true   = out_true[f"true_stack_up_mw_k{k}"]
    stack_pred   = out_pred[f"pred_stack_up_mw_k{k}"]

    for q in Q_LIST:
        tag = f"k{k}_q{int(q*100)}"
        cfg_dir = os.path.join(OUT_DIR, f"vnew2_{tag}")
        os.makedirs(cfg_dir, exist_ok=True)

        tau = float(tau_table[(tau_table["k"] == k) & (tau_table["q"] == q)]["tau_mw"].iloc[0])

        aef_true_2024 = aef_2024["aef_true_kg_per_kwh"]
        aef_pred_2024 = aef_2024["aef_tier1_forecast_kg_per_kwh"]

        mef_true_star = stabilize_mef(mef_true_raw, stack_true, aef_true_2024, tau=tau)
        mef_pred_star = stabilize_mef(mef_pred_raw, stack_pred, aef_pred_2024, tau=tau)

        mef_true_final = mef_true_star.fillna(aef_true_2024)
        mef_pred_final = mef_pred_star.fillna(aef_pred_2024)

        # Build detailed output table
        df_out = pd.DataFrame(index=common_idx_2024)
        for tech in test_techs:
            df_out[f"{tech}_true_mw"] = gen_true_2024[tech]
            df_out[f"{tech}_pred_mw"] = gen_pred_2024[tech]

        df_out[f"stack_up_true_mw_k{k}"] = stack_true
        df_out[f"stack_up_pred_mw_k{k}"] = stack_pred
        df_out[f"mef_up_true_raw_kg_per_kwh_k{k}"] = mef_true_raw
        df_out[f"mef_up_pred_raw_kg_per_kwh_k{k}"] = mef_pred_raw

        df_out["aef_true_kg_per_kwh"] = aef_true_2024
        df_out["aef_tier1_forecast_kg_per_kwh"] = aef_pred_2024

        df_out[f"mef_true_star_shrink_kg_per_kwh_{tag}"] = mef_true_star
        df_out[f"mef_pred_star_shrink_kg_per_kwh_{tag}"] = mef_pred_star
        df_out[f"mef_true_final_filled_kg_per_kwh_{tag}"] = mef_true_final
        df_out[f"mef_pred_final_filled_kg_per_kwh_{tag}"] = mef_pred_final

        out_csv = os.path.join(cfg_dir, f"vnew2_mef_proxy_fulltable_2024_{tag}.csv")
        df_out.reset_index().rename(columns={"index": "datetime_utc"}).to_csv(out_csv, index=False)
        all_outputs.append(out_csv)

        # weight diagnostics (hourly + summary) for TRUE and PRED ----
        # TRUE weight diag
        wt_hourly, wt_summary = compute_weight_diagnostics_tables(
            index_utc=common_idx_2024,
            tau_mw=tau,
            stack_up=stack_true,
            mef_raw=mef_true_raw,
            aef=aef_true_2024,
            mef_final=mef_true_final,
            tag=f"{tag}_TRUE"
        )
        wt_hourly_csv = os.path.join(cfg_dir, f"vnew2_weight_diagnostics_hourly_2024_{tag}_TRUE.csv")
        wt_summary_csv = os.path.join(cfg_dir, f"vnew2_weight_diagnostics_summary_2024_{tag}_TRUE.csv")
        wt_hourly.reset_index().rename(columns={"index":"datetime_utc"}).to_csv(wt_hourly_csv, index=False)
        wt_summary.to_csv(wt_summary_csv, index=False)
        all_outputs += [wt_hourly_csv, wt_summary_csv]
        weight_summary_all.append(wt_summary.iloc[0].to_dict())

        # PRED weight diag
        wp_hourly, wp_summary = compute_weight_diagnostics_tables(
            index_utc=common_idx_2024,
            tau_mw=tau,
            stack_up=stack_pred,
            mef_raw=mef_pred_raw,
            aef=aef_pred_2024,
            mef_final=mef_pred_final,
            tag=f"{tag}_PRED"
        )
        wp_hourly_csv = os.path.join(cfg_dir, f"vnew2_weight_diagnostics_hourly_2024_{tag}_PRED.csv")
        wp_summary_csv = os.path.join(cfg_dir, f"vnew2_weight_diagnostics_summary_2024_{tag}_PRED.csv")
        wp_hourly.reset_index().rename(columns={"index":"datetime_utc"}).to_csv(wp_hourly_csv, index=False)
        wp_summary.to_csv(wp_summary_csv, index=False)
        all_outputs += [wp_hourly_csv, wp_summary_csv]
        weight_summary_all.append(wp_summary.iloc[0].to_dict())

        # Existing NaN diagnostics (kept)
        diag_rows = []
        def add_diag(name, s):
            s = pd.Series(s)
            diag_rows.append({
                "series": name,
                "n_hours": int(len(s)),
                "n_nan": int(s.isna().sum()),
                "nan_fraction": float(s.isna().mean()),
            })

        add_diag(f"mef_true_raw_{tag}", mef_true_raw)
        add_diag(f"mef_pred_raw_{tag}", mef_pred_raw)
        add_diag(f"mef_true_star_{tag}", mef_true_star)
        add_diag(f"mef_pred_star_{tag}", mef_pred_star)
        add_diag(f"mef_true_final_{tag}", mef_true_final)
        add_diag(f"mef_pred_final_{tag}", mef_pred_final)
        add_diag(f"stack_true_{tag}", stack_true)
        add_diag(f"stack_pred_{tag}", stack_pred)

        diag_df = pd.DataFrame(diag_rows)
        diag_csv = os.path.join(cfg_dir, f"vnew2_nan_and_weight_diagnostics_2024_{tag}.csv")
        diag_df.to_csv(diag_csv, index=False)
        all_outputs.append(diag_csv)

        # Metrics
        eval_df = pd.DataFrame({
            "true_final": mef_true_final,
            "pred_final": mef_pred_final
        }).dropna()

        mae, rmse, mape = compute_metrics_like_aef(eval_df["true_final"].values, eval_df["pred_final"].values)
        metrics_df = pd.DataFrame({
            "k": [k],
            "q": [q],
            "tau_mw": [tau],
            "MAE_kg_per_kwh": [mae],
            "RMSE_kg_per_kwh": [rmse],
            "MAPE_percent": [mape],
            "n_hours": [len(eval_df)],
        })
        metrics_csv = os.path.join(cfg_dir, f"vnew2_metrics_pred_vs_true_2024_{tag}.csv")
        metrics_df.to_csv(metrics_csv, index=False)
        all_outputs.append(metrics_csv)
        summary_rows.append(metrics_df.iloc[0].to_dict())

        # Distribution plot
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.hist(mef_true_final.values, bins=60, alpha=0.65, label="TRUE (final)")
        ax.hist(mef_pred_final.values, bins=60, alpha=0.65, label="PRED (final)")
        apply_plot_settings(
            ax,
            title=f"MEF* distribution 2024 (final) – {tag} (tau={tau:.2f} MW)",
            xlabel="kgCO2/kWh",
            ylabel="count (hours)"
        )
        ax.legend(fontsize=10)
        out_noext = os.path.join(cfg_dir, f"vnew2_dist_mef_final_2024_{tag}")
        png, svg = savefig_png_svg(fig, out_noext)
        all_outputs += [png, svg]

        # Parity plot
        y_true = eval_df["true_final"].values
        y_pred = eval_df["pred_final"].values
        fig, ax = plt.subplots(figsize=(6, 6))
        ax.scatter(y_true, y_pred, s=10, alpha=0.35, label="hours")
        lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
        ax.plot(lims, lims, "k--", linewidth=1.25, label="Ideal")
        apply_plot_settings(
            ax,
            title=f"MEF* parity 2024 – TRUE vs PRED ({tag})",
            xlabel="TRUE MEF* (final) [kgCO2/kWh]",
            ylabel="PRED MEF* (final) [kgCO2/kWh]"
        )
        ax.legend(fontsize=10)
        out_noext = os.path.join(cfg_dir, f"vnew2_parity_mef_final_2024_{tag}")
        png, svg = savefig_png_svg(fig, out_noext)
        all_outputs += [png, svg]

        # Full-year time series
        fig, ax = plt.subplots(figsize=(14, 4))
        pd.Series(mef_true_final, index=common_idx_2024).plot(ax=ax, linewidth=1.0, label=f"MEF* TRUE (final) {tag}")
        pd.Series(mef_pred_final, index=common_idx_2024).plot(ax=ax, linewidth=1.0, alpha=0.85, label=f"MEF* PRED (final) {tag}")
        aef_true_2024.plot(ax=ax, linewidth=0.9, alpha=0.55, label="AEF TRUE")
        aef_pred_2024.plot(ax=ax, linewidth=0.9, alpha=0.55, label="AEF Tier1 forecast")
        apply_plot_settings(
            ax,
            title=f"Full-year 2024 time series – MEF* + AEF ({tag})",
            xlabel="Time (UTC)",
            ylabel="kgCO2/kWh"
        )
        ax.legend(fontsize=9, ncol=2)
        out_noext = os.path.join(cfg_dir, f"vnew2_timeseries_full_year_2024_{tag}")
        png, svg = savefig_png_svg(fig, out_noext)
        all_outputs += [png, svg]

        # One-week plots
        week_start, week_end = pick_week_window(common_idx_2024)
        week_mask = (common_idx_2024 >= week_start) & (common_idx_2024 < week_end)

        week_true_mef = pd.Series(mef_true_final.values, index=common_idx_2024).loc[week_mask]
        week_pred_mef = pd.Series(mef_pred_final.values, index=common_idx_2024).loc[week_mask]
        week_aef_true = aef_true_2024.loc[week_mask]
        week_aef_pred = aef_pred_2024.loc[week_mask]

        fig, ax = plt.subplots(figsize=(14, 4))
        week_true_mef.plot(ax=ax, linewidth=1.2, label=f"MEF* TRUE (final) {tag}")
        week_pred_mef.plot(ax=ax, linewidth=1.2, alpha=0.9, label=f"MEF* PRED (final) {tag}")
        week_aef_true.plot(ax=ax, linewidth=1.0, alpha=0.55, label="AEF TRUE")
        week_aef_pred.plot(ax=ax, linewidth=1.0, alpha=0.55, label="AEF Tier1 forecast")
        apply_plot_settings(
            ax,
            title=f"One-week 2024 – MEF* + AEF ({tag}) [{week_start.date()} to {(week_end-pd.Timedelta(hours=1)).date()}]",
            xlabel="Time (UTC)",
            ylabel="kgCO2/kWh"
        )
        ax.legend(fontsize=9, ncol=2)
        out_noext = os.path.join(cfg_dir, f"vnew2_timeseries_one_week_overlay_2024_{tag}")
        png, svg = savefig_png_svg(fig, out_noext)
        all_outputs += [png, svg]

        fig, ax = plt.subplots(figsize=(14, 3.8))
        week_true_mef.plot(ax=ax, linewidth=1.2, label=f"MEF* TRUE (final) {tag}")
        week_aef_true.plot(ax=ax, linewidth=1.0, alpha=0.55, label="AEF TRUE")
        apply_plot_settings(
            ax,
            title=f"One-week 2024 – TRUE signals ({tag})",
            xlabel="Time (UTC)",
            ylabel="kgCO2/kWh"
        )
        ax.legend(fontsize=10)
        out_noext = os.path.join(cfg_dir, f"vnew2_timeseries_one_week_TRUE_2024_{tag}")
        png, svg = savefig_png_svg(fig, out_noext)
        all_outputs += [png, svg]

        fig, ax = plt.subplots(figsize=(14, 3.8))
        week_pred_mef.plot(ax=ax, linewidth=1.2, label=f"MEF* PRED (final) {tag}")
        week_aef_pred.plot(ax=ax, linewidth=1.0, alpha=0.55, label="AEF Tier1 forecast")
        apply_plot_settings(
            ax,
            title=f"One-week 2024 – PRED signals ({tag})",
            xlabel="Time (UTC)",
            ylabel="kgCO2/kWh"
        )
        ax.legend(fontsize=10)
        out_noext = os.path.join(cfg_dir, f"vnew2_timeseries_one_week_PRED_2024_{tag}")
        png, svg = savefig_png_svg(fig, out_noext)
        all_outputs += [png, svg]

        # Grid MEF file
        grid_mef = pd.DataFrame(index=common_idx_2024)
        grid_mef["datetime_utc"] = common_idx_2024
        grid_mef["grid_mef_pred_kg_per_kwh"] = mef_pred_final.values
        grid_mef["grid_mef_true_kg_per_kwh"] = mef_true_final.values
        grid_mef["aef_tier1_forecast_kg_per_kwh"] = aef_pred_2024.values
        grid_mef["aef_true_kg_per_kwh"] = aef_true_2024.values
        grid_mef["stack_up_pred_mw"] = stack_pred.values
        grid_mef["stack_up_true_mw"] = stack_true.values
        grid_mef["tau_mw_used"] = tau
        grid_mef["k_hours"] = k
        grid_mef["q_quantile"] = q

        grid_csv = os.path.join(cfg_dir, f"grid_mef_hourly_2024_{tag}.csv")
        grid_mef.to_csv(grid_csv, index=False)
        all_outputs.append(grid_csv)

# ============================================================
# 6) Global summary table + global weight summary + ZIP
# ============================================================
summary_df = pd.DataFrame(summary_rows).sort_values(["k", "q"])
summary_csv = os.path.join(OUT_DIR, "vnew2_summary_metrics_all_configs.csv")
summary_df.to_csv(summary_csv, index=False)
all_outputs.append(summary_csv)

# weight summaries across all configs
weight_summary_all_df = pd.DataFrame(weight_summary_all)
weight_summary_all_csv = os.path.join(OUT_DIR, "vnew2_weight_diagnostics_summary_all_configs.csv")
weight_summary_all_df.to_csv(weight_summary_all_csv, index=False)
all_outputs.append(weight_summary_all_csv)

print("\n[VNEW2] Summary metrics across configs:")
print(summary_df)

print("\n[VNEW2] Weight diagnostic summaries across configs (TRUE+PRED):")
print(weight_summary_all_df)

zip_path = os.path.join(BASE_DIR, "tier2_2_mef_vnew2_outputs.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for root, dirs, files in os.walk(OUT_DIR):
        for f in files:
            full = os.path.join(root, f)
            rel = os.path.relpath(full, OUT_DIR)
            z.write(full, arcname=os.path.join("tier2_2_mef_vnew2", rel))

print(f"\n[VNEW2] DONE. ZIP created at: {zip_path}")
print("Open the zip and you will see one subfolder per config: vnew2_k1_q10, vnew2_k1_q20, vnew2_k3_q10, vnew2_k3_q20")

## 2.4 MSDR Soft MEF proxy


In [ ]:
# ============================================================
# MEF 2.3
# ------------------------------------------------------------
#
# A) MSDR outputs:
#    1) Smoothed regime probabilities exported per hour (and plotted)
#    2) Regime parameters exported (const, slope, sigma2)
#    3) Model-summary text dumps (NOT for every hour; selectable subset)
#
# B) DLR baseline block (rolling OLS on the same 168h window):
#    - slope/intercept/R2 per target hour
#    - timeseries plots for DLR slope
#    - overlay plot: DLR slope vs MSDR soft slope (same units)
#
# C) Regime interpretation helpers:
#    - fossil-share series from genmix (heuristic fossil tech detection)
#    - plots: fossil share vs MEF (daily) and fossil share vs regime probabilities (daily)
#    - correlation table between fossil share and regime probs / MEF
#
# D) No scaling:
#    - Slopes remain in physical units: kgCO2/MWh -> final kgCO2/kWh
#
# E) Stability
#    - robust target hour selection
#    - window conditioning checks (dg variance)
#    - corrected capping logic
#    - NaN/negative are explicitly reported (exact hours) and filled with AEF
#    - robust transition matrix reconstruction
#
# ============================================================

import os, glob, zipfile, warnings, re, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import statsmodels.api as sm
from statsmodels.tsa.regime_switching.markov_regression import MarkovRegression

warnings.filterwarnings("ignore")

# ============================================================
# CONFIG
# ============================================================

# ---- I/O ----
INPUT_DIR = "/content/mef2_3_input"
OUT_DIR   = "/content/mef2_3_v6_modified_outputs"

OUT_CSV_DIR  = os.path.join(OUT_DIR, "csv")
OUT_PLOT_DIR = os.path.join(OUT_DIR, "plots")
OUT_META_DIR = os.path.join(OUT_DIR, "meta")
OUT_TEXT_DIR = os.path.join(OUT_DIR, "model_summaries")
for d in [OUT_CSV_DIR, OUT_PLOT_DIR, OUT_META_DIR, OUT_TEXT_DIR]:
    os.makedirs(d, exist_ok=True)

APPLY_YEAR = 2024

ROLL_HOURS = 336
# ---- regimes (paper selects between {2,3}; keep 2 while debugging fast) ----
K_LIST = [2]

# ============================================================
# Target-hour loop control (debug-fast)
# ============================================================
TARGET_HOURS_MODE = "full_year"  # full_year | first_n | random_n | explicit_list | range_slice
TARGET_FIRST_N     = 500
TARGET_RANDOM_N    = 200
TARGET_RANDOM_SEED = 42
TARGET_EXPLICIT_LIST = []
TARGET_RANGE_START = "2024-01-01 00:00:00+00:00"
TARGET_RANGE_END   = "2024-01-10 23:00:00+00:00"

# ============================================================
# Stability guards for ill-conditioned windows
# ============================================================
DG_STD_MIN = 1e-6
DG_EPS = 1e-6
DG_MIN_FRAC_BIG = 0.10  # require at least 10% of points with |dG| > DG_EPS

# ============================================================
# Capping (fixed logic)
# ============================================================
CAP_Q_LOW  = 0.001
CAP_Q_HIGH = 0.999
CAP_ABS_FACTOR = 2.0  # abs bound = ± factor * EF_MAX_2024

# ============================================================
# Smoothed vs filtered probabilities
# ============================================================
# Filtered:  P(S_t=k | y_1..y_t)
# Smoothed:  P(S_t=k | y_1..y_T)  (uses future data within the window)
PROB_MODE = "smoothed"  # use smoothed for outputs / soft+hard MEF selection

# ============================================================
# Model summary
# ============================================================
# Dumping summary for all hours is huge; use a controlled policy:
SUMMARY_DUMP_MODE = "some"  # none | some | all
SUMMARY_DUMP_MAX  = 20      # max windows to dump in "some" mode
SUMMARY_DUMP_EVERY = 50     # dump every Nth successful window in "some" mode

# ============================================================
# Plot style helper
# ============================================================
def apply_plot_settings(ax, title, xlabel, ylabel):
    ax.set_title(title, fontsize=14, fontweight='bold', pad=15)
    ax.set_xlabel(xlabel, fontsize=12, labelpad=10)
    ax.set_ylabel(ylabel, fontsize=12, labelpad=10)
    ax.tick_params(axis='both', which='major', labelsize=10)
    for spine in ax.spines.values():
        spine.set_color('#dadedf')
        spine.set_linewidth(2)
    ax.grid(False)

def save_fig(fig, filename_base):
    png_path = os.path.join(OUT_PLOT_DIR, f"{filename_base}.png")
    svg_path = os.path.join(OUT_PLOT_DIR, f"{filename_base}.svg")
    fig.savefig(png_path, dpi=150, bbox_inches="tight")
    fig.savefig(svg_path, bbox_inches="tight")
    plt.close(fig)

# ============================================================
# Utilities
# ============================================================
def _find_one(patterns, must_exist=True):
    matches = []
    for p in patterns:
        matches.extend(glob.glob(p))
    matches = sorted(list(set(matches)))
    if not matches and must_exist:
        raise FileNotFoundError(
            "No files matched patterns:\n" + "\n".join(patterns) +
            f"\n\nCheck INPUT_DIR={INPUT_DIR} and file names."
        )
    return matches[0] if matches else None

def read_any_table(path):
    if path.lower().endswith(".csv"):
        return pd.read_csv(path)
    if path.lower().endswith((".xlsx", ".xls")):
        return pd.read_excel(path)
    raise ValueError(f"Unsupported file type: {path}")

def to_utc_index_robust(df, col="datetime_utc"):
    df = df.copy()
    if col not in df.columns:
        raise ValueError(f"Missing datetime column '{col}'. Found: {list(df.columns)}")

    s = df[col].astype(str).str.strip()
    dt1 = pd.to_datetime(s, utc=True, errors="coerce")
    nat_rate = dt1.isna().mean()

    if nat_rate > 0.01:
        dt2 = pd.to_datetime(s, utc=True, errors="coerce", dayfirst=True)
        dt = dt2 if dt2.isna().mean() < nat_rate else dt1
    else:
        dt = dt1

    df[col] = dt
    df = df.dropna(subset=[col]).sort_values(col).set_index(col)

    if df.index.tz is None:
        df.index = df.index.tz_localize("UTC")
    else:
        df.index = df.index.tz_convert("UTC")

    df.index = df.index.floor("H")
    if df.index.has_duplicates:
        df = df.groupby(df.index).mean(numeric_only=True)

    return df.sort_index()

def complete_hourly_index(df, name_for_report):
    if df.empty:
        return df, []
    full_idx = pd.date_range(df.index.min(), df.index.max(), freq="H", tz="UTC")
    missing = full_idx.difference(df.index)
    pd.DataFrame({"datetime_utc_missing": missing}).to_csv(
        os.path.join(OUT_CSV_DIR, f"{name_for_report}_missing_hours.csv"),
        index=False
    )
    return df, list(missing)

def clip_negative_generation(df, gen_cols, label):
    neg_rows = []
    df2 = df.copy()
    for c in gen_cols:
        neg_mask = df2[c] < 0
        if neg_mask.any():
            ts_list = df2.index[neg_mask].tolist()
            neg_rows.append({
                "stream": label,
                "column": c,
                "n_negative": int(neg_mask.sum()),
                "first_negative_utc": str(ts_list[0]),
                "last_negative_utc": str(ts_list[-1]),
            })
            df2.loc[neg_mask, c] = 0.0
            pd.DataFrame({"datetime_utc": ts_list}).to_csv(
                os.path.join(OUT_CSV_DIR, f"negatives_{label}_{c}.csv"),
                index=False
            )
    pd.DataFrame(neg_rows).to_csv(
        os.path.join(OUT_CSV_DIR, f"negatives_summary_{label}.csv"),
        index=False
    )
    return df2

def load_ef(path, year_label):
    d = read_any_table(path).copy()
    cols_lower = {c.lower(): c for c in d.columns}
    tech_col = cols_lower.get("tier1 tech", None)

    ef_col = None
    for k, orig in cols_lower.items():
        if "ef" in k and ("kg" in k) and ("mwh" in k):
            ef_col = orig
            break

    if tech_col is None or ef_col is None:
        raise ValueError(
            f"EF file {year_label} missing expected columns. "
            f"Need 'Tier1 Tech' and EF column like 'EF (kgCo2/Mwh)'. Found: {list(d.columns)}"
        )

    d = d[[tech_col, ef_col]].rename(columns={tech_col: "tech", ef_col: "ef_kg_per_mwh"})
    d["tech"] = d["tech"].astype(str).str.strip()
    d["ef_kg_per_mwh"] = pd.to_numeric(d["ef_kg_per_mwh"], errors="coerce")
    d = d.dropna(subset=["tech", "ef_kg_per_mwh"])
    return d

def compute_emissions_one_row(gen_row, gen_cols, ef_map):
    s = 0.0
    missing = 0
    for c in gen_cols:
        ef = ef_map.get(c, np.nan)
        g  = gen_row.get(c, np.nan)
        if pd.isna(ef) or pd.isna(g):
            if pd.isna(ef):
                missing += 1
            continue
        s += float(g) * float(ef)
    return s, missing

# ============================================================
# Fossil-share helper (regime interpretation aid)
# ============================================================
def detect_fossil_cols(gen_cols):
    """
    Heuristic fossil tech detection from gen_ column names.
    Adjust if you want explicit mapping later.
    """
    fossil_keys = [
        "coal", "hard_coal", "oil",
        "gas", "fossil"
    ]
    exclude_keys = [
        "biomass", "waste", "renew", "solar", "wind", "hydro", "nuclear",
        "other"
    ]
    fossil = []
    for c in gen_cols:
        lc = c.lower()
        if any(k in lc for k in exclude_keys):
            continue
        if any(k in lc for k in fossil_keys):
            fossil.append(c)
    return sorted(list(set(fossil)))

def fossil_share_from_genmix(df_2024, gen_cols, label):
    fossil_cols = detect_fossil_cols(gen_cols)
    meta = {
        "label": label,
        "n_gen_cols": len(gen_cols),
        "fossil_cols_used": fossil_cols,
        "n_fossil_cols_used": len(fossil_cols),
    }
    with open(os.path.join(OUT_META_DIR, f"fossil_cols_{label}.json"), "w") as f:
        json.dump(meta, f, indent=2)

    if len(fossil_cols) == 0:
        # produce empty/NaN series for transparency
        s = pd.Series(index=df_2024.index, data=np.nan, name="fossil_share")
        s.to_frame().reset_index().rename(columns={"index":"datetime_utc"}).to_csv(
            os.path.join(OUT_CSV_DIR, f"fossil_share_{label}_hourly.csv"), index=False
        )
        return s, fossil_cols

    total = df_2024[gen_cols].sum(axis=1)
    fossil = df_2024[fossil_cols].sum(axis=1)
    share = (fossil / total).replace([np.inf, -np.inf], np.nan)
    share.name = "fossil_share"

    share.to_frame().reset_index().rename(columns={"index":"datetime_utc"}).to_csv(
        os.path.join(OUT_CSV_DIR, f"fossil_share_{label}_hourly.csv"), index=False
    )
    return share, fossil_cols

# ============================================================
# MSDR fitting helpers
# ============================================================
def safe_fit_msdr(dE, dG, K):
    endog = np.asarray(dE, dtype=float)
    exog  = np.asarray(dG, dtype=float).reshape(-1, 1)

    good = np.isfinite(endog) & np.isfinite(exog[:, 0])
    endog = endog[good]
    exog  = exog[good, :]

    if len(endog) < max(50, 10*K):
        return None, "too_few_obs"

    dg = exog[:, 0]
    if np.nanstd(dg) < DG_STD_MIN:
        return None, "dg_std_too_small"

    frac_big = float(np.mean(np.abs(dg) > DG_EPS))
    if frac_big < DG_MIN_FRAC_BIG:
        return None, "dg_near_zero_too_often"

    mod = MarkovRegression(
        endog=endog,
        exog=exog,
        k_regimes=K,
        trend="c",
        switching_exog=True,
        switching_trend=True,
        switching_variance=True
    )

    try:
        res = mod.fit(disp=False, maxiter=200)
        return res, "ok"
    except Exception:
        return None, "fit_failed"

def extract_slopes(res, K):
    params = np.asarray(res.params, dtype=float)
    names  = list(res.model.param_names)
    slopes = [np.nan]*K
    for nm, val in zip(names, params):
        n = nm.lower().replace(" ", "")
        if n.startswith("x1[") and "[" in nm and "]" in nm:
            try:
                k = int(nm.split("[")[-1].split("]")[0])
                if 0 <= k < K:
                    slopes[k] = float(val)
            except:
                pass
    slopes = np.asarray(slopes, dtype=float)
    if not np.isfinite(slopes).all():
        raise RuntimeError(f"Could not extract {K} slopes. got={slopes}")
    return slopes

def extract_intercepts_sigmas(res, K):
    params = np.asarray(res.params, dtype=float)
    names  = list(res.model.param_names)
    consts = [np.nan]*K
    sig2   = [np.nan]*K
    for nm, val in zip(names, params):
        n = nm.lower().replace(" ", "")
        if n.startswith("const["):
            try:
                k = int(nm.split("[")[-1].split("]")[0])
                if 0 <= k < K:
                    consts[k] = float(val)
            except:
                pass
        if "sigma2[" in n:
            try:
                k = int(nm.split("[")[-1].split("]")[0])
                if 0 <= k < K:
                    sig2[k] = float(val)
            except:
                pass
    return np.asarray(consts, dtype=float), np.asarray(sig2, dtype=float)

def get_regime_probs(res, K, mode="smoothed"):
    if mode == "smoothed":
        pm = getattr(res, "smoothed_marginal_probabilities", None)
        if pm is None:
            pm = res.filtered_marginal_probabilities
    else:
        pm = res.filtered_marginal_probabilities

    if hasattr(pm, "iloc"):
        p_last = np.asarray(pm.iloc[-1].values, dtype=float)
    else:
        pm = np.asarray(pm, dtype=float)
        if pm.ndim != 2:
            return None
        if pm.shape[1] == K:
            p_last = pm[-1, :]
        elif pm.shape[0] == K:
            p_last = pm[:, -1]
        else:
            return None

    p_last = np.where(np.isfinite(p_last), p_last, 0.0)
    s = float(np.sum(p_last))
    if s <= 0:
        p_last = np.ones(K, dtype=float) / K
    else:
        p_last = p_last / s
    return p_last

def reconstruct_transition_matrix(res, K, eps=1e-12):
    names = list(res.model.param_names)
    params = np.asarray(res.params, dtype=float)
    pat = re.compile(r"p\[(\d+)->(\d+)\]")

    row_dict = {i: {} for i in range(K)}
    for nm, val in zip(names, params):
        m = pat.search(nm.replace(" ", ""))
        if m:
            i = int(m.group(1)); j = int(m.group(2))
            if 0 <= i < K and 0 <= j < K:
                row_dict[i][j] = float(val)

    P = np.full((K, K), np.nan, dtype=float)
    for i in range(K):
        for j, v in row_dict[i].items():
            P[i, j] = v

        missing_js = [j for j in range(K) if not np.isfinite(P[i, j])]
        if len(missing_js) == 1:
            jmiss = missing_js[0]
            known_sum = np.nansum(P[i, :])
            P[i, jmiss] = 1.0 - known_sum

        if not np.isfinite(P[i, :]).all():
            P[i, :] = 1.0 / K

        P[i, :] = np.clip(P[i, :], eps, 1.0)
        P[i, :] = P[i, :] / np.sum(P[i, :])

    return P

def hqic_from(llf, nobs, k_params):
    if nobs <= 1 or not np.isfinite(llf) or not np.isfinite(k_params):
        return np.nan
    return float(-2.0*llf + 2.0*k_params*np.log(np.log(nobs)))

def summarize_series_stats(x, label_prefix):
    x = pd.Series(x).dropna()
    if x.empty:
        return {f"{label_prefix}_n": 0}
    return {
        f"{label_prefix}_n": int(x.shape[0]),
        f"{label_prefix}_min": float(x.min()),
        f"{label_prefix}_p01": float(x.quantile(0.01)),
        f"{label_prefix}_median": float(x.median()),
        f"{label_prefix}_p99": float(x.quantile(0.99)),
        f"{label_prefix}_max": float(x.max()),
        f"{label_prefix}_mean": float(x.mean()),
        f"{label_prefix}_std": float(x.std(ddof=0)),
    }

# ============================================================
# DLR baseline (rolling OLS) on same window
# ============================================================
def rolling_dlr(dfw):
    """
    OLS: dE = a + b*dG
    returns a,b,r2 or None
    """
    y = np.asarray(dfw["dE"].values, dtype=float)
    x = np.asarray(dfw["dG"].values, dtype=float)
    good = np.isfinite(y) & np.isfinite(x)
    y = y[good]; x = x[good]
    if len(y) < 50:
        return None

    # conditioning guard consistent with MSDR guard
    if np.nanstd(x) < DG_STD_MIN:
        return None
    if float(np.mean(np.abs(x) > DG_EPS)) < DG_MIN_FRAC_BIG:
        return None

    X = np.column_stack([np.ones_like(x), x])
    try:
        beta, *_ = np.linalg.lstsq(X, y, rcond=None)
        yhat = X @ beta
        ss_res = float(np.sum((y - yhat)**2))
        ss_tot = float(np.sum((y - np.mean(y))**2))
        r2 = 1.0 - ss_res/ss_tot if ss_tot > 0 else np.nan
        a = float(beta[0]); b = float(beta[1])
        return a, b, r2
    except Exception:
        return None

# ============================================================
# 1) FIND INPUT FILES
# ============================================================
gen_2023_tail_path = _find_one([
    os.path.join(INPUT_DIR, "genmix_actual_last month of 2023_tier2.3*.csv"),
    os.path.join(INPUT_DIR, "*last*month*2023*tier2.3*.csv"),
])

gen_actual_2024_path = _find_one([
    os.path.join(INPUT_DIR, "genmix_actual_2024_tier2.3*.csv"),
    os.path.join(INPUT_DIR, "*genmix*actual*2024*tier2.3*.csv"),
])

gen_forecast_2024_path = _find_one([
    os.path.join(INPUT_DIR, "genmix_tier1forecast_2024_tier2.3*.csv"),
    os.path.join(INPUT_DIR, "*genmix*tier1*forecast*2024*tier2.3*.csv"),
])

ef_2023_path = _find_one([os.path.join(INPUT_DIR, "*ember*emission*2023*.*")])
ef_2024_path = _find_one([os.path.join(INPUT_DIR, "*ember*emission*2024*.*")])

aef_2024_path = _find_one([
    os.path.join(INPUT_DIR, "tier2_1_aef_eval_tier1_vs_true_2024_series*.csv"),
    os.path.join(INPUT_DIR, "*aef*2024*series*.csv"),
])

print("[MEF2.3_v6_modified] Using files:")
print("  gen_2023_tail:", gen_2023_tail_path)
print("  gen_actual_2024:", gen_actual_2024_path)
print("  gen_forecast_2024:", gen_forecast_2024_path)
print("  ef_2023:", ef_2023_path)
print("  ef_2024:", ef_2024_path)
print("  aef_2024:", aef_2024_path)

# ============================================================
# 2) LOAD EF TABLES
# ============================================================
ef_2023 = load_ef(ef_2023_path, "2023")
ef_2024 = load_ef(ef_2024_path, "2024")
efmap_2023 = dict(zip(ef_2023["tech"], ef_2023["ef_kg_per_mwh"]))
efmap_2024 = dict(zip(ef_2024["tech"], ef_2024["ef_kg_per_mwh"]))

ef_2023.to_csv(os.path.join(OUT_CSV_DIR, "ember_ef_2023_loaded.csv"), index=False)
ef_2024.to_csv(os.path.join(OUT_CSV_DIR, "ember_ef_2024_loaded.csv"), index=False)

EF_MAX = float(pd.Series(list(efmap_2024.values())).max())
pd.DataFrame([{"EF_MAX_kg_per_mwh_2024": EF_MAX}]).to_csv(
    os.path.join(OUT_CSV_DIR, "ef_max_2024_for_capping.csv"), index=False
)

# ============================================================
# 3) LOAD AEF SERIES (kg/kWh)
# ============================================================
df_aef = to_utc_index_robust(pd.read_csv(aef_2024_path), col="datetime_utc")
need_aef_cols = ["aef_true_kg_per_kwh", "aef_tier1_forecast_kg_per_kwh"]
missing = [c for c in need_aef_cols if c not in df_aef.columns]
if missing:
    raise ValueError(f"AEF file missing columns: {missing}. Found: {list(df_aef.columns)}")

df_aef = df_aef[need_aef_cols].copy()
df_aef.reset_index().rename(columns={"index":"datetime_utc"}).to_csv(
    os.path.join(OUT_CSV_DIR, "aef_2024_loaded.csv"), index=False
)

# ============================================================
# 4) LOAD GENMIX, FIX TIME, CHECK MISSING HOURS, CLIP NEGATIVES
# ============================================================
df_2023_tail = to_utc_index_robust(pd.read_csv(gen_2023_tail_path), col="datetime_utc")
df_act_2024  = to_utc_index_robust(pd.read_csv(gen_actual_2024_path), col="datetime_utc")
df_for_2024  = to_utc_index_robust(pd.read_csv(gen_forecast_2024_path), col="datetime_utc")

gen_cols_2023 = [c for c in df_2023_tail.columns if c.startswith("gen_")]
gen_cols_act  = [c for c in df_act_2024.columns  if c.startswith("gen_")]
gen_cols_for  = [c for c in df_for_2024.columns  if c.startswith("gen_")]

if not gen_cols_2023 or not gen_cols_act or not gen_cols_for:
    raise ValueError("Could not find gen_* columns in one of the genmix files.")

pd.DataFrame({"gen_cols_2023_tail": gen_cols_2023}).to_csv(os.path.join(OUT_CSV_DIR, "gen_cols_2023_tail.csv"), index=False)
pd.DataFrame({"gen_cols_actual_2024": gen_cols_act}).to_csv(os.path.join(OUT_CSV_DIR, "gen_cols_actual_2024.csv"), index=False)
pd.DataFrame({"gen_cols_forecast_2024": gen_cols_for}).to_csv(os.path.join(OUT_CSV_DIR, "gen_cols_forecast_2024.csv"), index=False)

df_2023_tail, _ = complete_hourly_index(df_2023_tail, "genmix_2023_tail")
df_act_2024,  _ = complete_hourly_index(df_act_2024,  "genmix_actual_2024")
df_for_2024,  _ = complete_hourly_index(df_for_2024,  "genmix_forecast_2024")

df_2023_tail = clip_negative_generation(df_2023_tail, gen_cols_2023, "actual_2023_tail")
df_act_2024  = clip_negative_generation(df_act_2024,  gen_cols_act,  "actual_2024")
df_for_2024  = clip_negative_generation(df_for_2024,  gen_cols_for,  "forecast_2024")

# save cleaned 2024 genmix for fossil-share analysis
df_act_2024.reset_index().rename(columns={"index":"datetime_utc"}).to_csv(os.path.join(OUT_CSV_DIR, "genmix_actual_2024_clean.csv"), index=False)
df_for_2024.reset_index().rename(columns={"index":"datetime_utc"}).to_csv(os.path.join(OUT_CSV_DIR, "genmix_forecast_2024_clean.csv"), index=False)

# fossil share series (2024 only)
fossil_share_actual, fossil_cols_actual = fossil_share_from_genmix(df_act_2024, gen_cols_act, "actual_2024")
fossil_share_forec,  fossil_cols_forec  = fossil_share_from_genmix(df_for_2024, gen_cols_for, "forecast_2024")

# ============================================================
# 5) BUILD CONTIGUOUS SERIES FOR ROLLING WINDOWS (seed 2023 tail)
# ============================================================
seed_start = df_2023_tail.index.max() - pd.Timedelta(hours=ROLL_HOURS-1)
df_seed = df_2023_tail.loc[df_2023_tail.index >= seed_start].copy()

df_stream_actual = pd.concat([df_seed, df_act_2024], axis=0).sort_index()
df_stream_forec  = pd.concat([df_seed, df_for_2024], axis=0).sort_index()

# ============================================================
# 6) COMPUTE (E_t, G_t, ΔE_t, ΔG_t)
# ============================================================
def build_E_G_deltas(df_stream, gen_cols, label):
    G = df_stream[gen_cols].sum(axis=1)

    E_list = []
    miss_rows = []
    for ts, row in df_stream.iterrows():
        ef_map = efmap_2023 if int(ts.year) == 2023 else efmap_2024
        e_val, nmiss = compute_emissions_one_row(row, gen_cols, ef_map)
        E_list.append(e_val)
        miss_rows.append({"datetime_utc": str(ts), "year": int(ts.year), "n_missing_ef": int(nmiss)})

    E = pd.Series(E_list, index=df_stream.index, name="E_kgco2_per_h")
    G = pd.Series(G.values, index=df_stream.index, name="G_mw")

    dE = E.diff()
    dG = G.diff()

    out = pd.DataFrame({"E_kgco2_per_h": E, "G_mw": G, "dE": dE, "dG": dG}).dropna()

    pd.DataFrame(miss_rows).to_csv(os.path.join(OUT_CSV_DIR, f"ef_coverage_over_time_{label}.csv"), index=False)
    out.reset_index().rename(columns={"index":"datetime_utc"}).to_csv(os.path.join(OUT_CSV_DIR, f"E_G_deltas_{label}.csv"), index=False)

    return out

series_actual = build_E_G_deltas(df_stream_actual, gen_cols_act, "actual_stream")
series_forec  = build_E_G_deltas(df_stream_forec,  gen_cols_for, "forecast_stream")

# ============================================================
# 7) TARGET HOURS (selectable loop)
# ============================================================
full_hours_2024 = pd.date_range(
    pd.Timestamp(f"{APPLY_YEAR}-01-01 00:00:00", tz="UTC"),
    pd.Timestamp(f"{APPLY_YEAR}-12-31 23:00:00", tz="UTC"),
    freq="H",
    tz="UTC"
)

def make_target_hours():
    if TARGET_HOURS_MODE == "full_year":
        return full_hours_2024
    if TARGET_HOURS_MODE == "first_n":
        return full_hours_2024[:int(TARGET_FIRST_N)]
    if TARGET_HOURS_MODE == "random_n":
        rng = np.random.default_rng(TARGET_RANDOM_SEED)
        idx = rng.choice(len(full_hours_2024), size=int(TARGET_RANDOM_N), replace=False)
        return full_hours_2024[np.sort(idx)]
    if TARGET_HOURS_MODE == "explicit_list":
        if not TARGET_EXPLICIT_LIST:
            raise ValueError("TARGET_EXPLICIT_LIST is empty.")
        return pd.to_datetime(TARGET_EXPLICIT_LIST, utc=True)
    if TARGET_HOURS_MODE == "range_slice":
        a = pd.to_datetime(TARGET_RANGE_START, utc=True)
        b = pd.to_datetime(TARGET_RANGE_END, utc=True)
        return full_hours_2024[(full_hours_2024 >= a) & (full_hours_2024 <= b)]
    raise ValueError(f"Unknown TARGET_HOURS_MODE: {TARGET_HOURS_MODE}")

target_hours = make_target_hours()
pd.DataFrame({"datetime_utc": target_hours.astype(str)}).to_csv(
    os.path.join(OUT_CSV_DIR, "target_hours_used.csv"), index=False
)
print(f"[MEF2.3_v6_modified] Target hours: {len(target_hours)} (mode={TARGET_HOURS_MODE})")

# ============================================================
# 8) ROLLING MSDR + DLR baseline + exports
# ============================================================
def maybe_dump_summary(res, label, K, t_str, dump_counter, is_success=True):
    if SUMMARY_DUMP_MODE == "none":
        return dump_counter
    if SUMMARY_DUMP_MODE == "all":
        do = True
    else:
        # "some": dump every SUMMARY_DUMP_EVERY-th successful window, up to MAX
        do = is_success and (dump_counter < SUMMARY_DUMP_MAX) and ((dump_counter == 0) or (dump_counter % SUMMARY_DUMP_EVERY == 0))
    if not do:
        return dump_counter

    fname = f"summary_{label}_K{K}_{t_str.replace(':','-').replace(' ','_')}.txt"
    fpath = os.path.join(OUT_TEXT_DIR, fname)
    try:
        with open(fpath, "w") as f:
            f.write(res.summary().as_text())
        dump_counter += 1
    except Exception:
        pass
    return dump_counter

def rolling_msdr_and_dlr(series_df, label, K):
    rows_mef = []
    rows_fit = []
    rows_reg = []
    rows_tm  = []
    rows_dlr = []

    dump_counter = 0

    for idx, t in enumerate(target_hours):
        w_end   = t - pd.Timedelta(hours=1)
        w_start = t - pd.Timedelta(hours=ROLL_HOURS)
        t_str = str(t)

        if idx % 50 == 0:
            print(f"  [{label},K={K}] processed {idx}/{len(target_hours)} windows...")

        dfw = series_df.loc[(series_df.index >= w_start) & (series_df.index <= w_end)].copy()

        # ---- DLR baseline (rolling OLS) ----
        dlr_out = rolling_dlr(dfw) if dfw.shape[0] >= 50 else None
        if dlr_out is None:
            rows_dlr.append({
                "datetime_utc": t_str,
                "status": "dlr_failed_or_ill_conditioned",
                "n_obs_window": int(dfw.shape[0]),
                "dlr_intercept_kgco2_per_h": np.nan,
                "dlr_slope_kg_per_mwh": np.nan,
                "dlr_r2": np.nan,
            })
        else:
            a, b, r2 = dlr_out
            rows_dlr.append({
                "datetime_utc": t_str,
                "status": "ok",
                "n_obs_window": int(dfw.shape[0]),
                "dlr_intercept_kgco2_per_h": a,
                "dlr_slope_kg_per_mwh": b,
                "dlr_r2": r2,
            })

        # ---- MSDR ----
        if dfw.shape[0] < 50:
            rows_fit.append({"datetime_utc": t_str, "K": K, "status": "insufficient_window_rows", "n_obs_window": int(dfw.shape[0])})
            continue

        res, status = safe_fit_msdr(dfw["dE"].values, dfw["dG"].values, K=K)
        if res is None:
            rows_fit.append({"datetime_utc": t_str, "K": K, "status": status, "n_obs_window": int(dfw.shape[0])})
            continue

        dump_counter = maybe_dump_summary(res, label, K, t_str, dump_counter, is_success=True)

        # slopes / params
        slopes = extract_slopes(res, K)
        consts, sig2 = extract_intercepts_sigmas(res, K)

        # smoothed probs (paper-style interpretation)
        p_last = get_regime_probs(res, K, mode=PROB_MODE)
        if p_last is None or not np.isfinite(p_last).all():
            p_last = np.ones(K, dtype=float) / K

        k_hat = int(np.argmax(p_last))
        mef_hard = float(slopes[k_hat])
        mef_soft = float(np.dot(p_last, slopes))

        # transition matrix full reconstruction
        P = reconstruct_transition_matrix(res, K)

        # fit diagnostics
        llf = float(res.llf)
        aic = float(res.aic)
        bic = float(res.bic)
        nobs = int(getattr(res, "nobs", dfw.shape[0]))
        k_params = int(len(res.params))
        hqic = float(getattr(res, "hqic", np.nan))
        if not np.isfinite(hqic):
            hqic = hqic_from(llf, nobs, k_params)

        rows_fit.append({
            "datetime_utc": t_str, "K": K, "status": "ok",
            "n_obs_window": nobs, "log_likelihood": llf, "aic": aic, "bic": bic, "hqic": hqic, "params_count": k_params,
            "prob_mode": PROB_MODE
        })

        # main MEF + probabilities
        row_mef = {
            "datetime_utc": t_str,
            "K": K,
            "mef_soft_kg_per_mwh_raw": mef_soft,
            "mef_hard_kg_per_mwh_raw": mef_hard,
            "hard_regime": k_hat,
            **{f"p_{PROB_MODE}_regime_{k}": float(p_last[k]) for k in range(K)},
            **{f"slope_regime_{k}_kg_per_mwh": float(slopes[k]) for k in range(K)},
        }
        rows_mef.append(row_mef)

        # regime parameter rows (Table-3 style)
        for k in range(K):
            rows_reg.append({
                "datetime_utc": t_str, "K": K, "regime": k,
                "const": float(consts[k]),
                "slope_kg_per_mwh": float(slopes[k]),
                "sigma2": float(sig2[k]),
            })

        # transition matrix row (flattened)
        row_tm = {"datetime_utc": t_str, "K": K}
        for i in range(K):
            for j in range(K):
                row_tm[f"P_{i}{j}"] = float(P[i, j])
        rows_tm.append(row_tm)

    df_mef = pd.DataFrame(rows_mef)
    df_fit = pd.DataFrame(rows_fit)
    df_reg = pd.DataFrame(rows_reg)
    df_tm  = pd.DataFrame(rows_tm)
    df_dlr = pd.DataFrame(rows_dlr)

    df_mef.to_csv(os.path.join(OUT_CSV_DIR, f"mef_{label}_K{K}_raw.csv"), index=False)
    df_fit.to_csv(os.path.join(OUT_CSV_DIR, f"fit_diagnostics_{label}_K{K}.csv"), index=False)
    df_reg.to_csv(os.path.join(OUT_CSV_DIR, f"regime_params_{label}_K{K}.csv"), index=False)
    df_tm.to_csv(os.path.join(OUT_CSV_DIR, f"transition_matrix_{label}_K{K}_FULL.csv"), index=False)
    df_dlr.to_csv(os.path.join(OUT_CSV_DIR, f"dlr_baseline_{label}_window{ROLL_HOURS}h.csv"), index=False)

    return df_mef, df_fit, df_reg, df_tm, df_dlr

results = {}
for stream_label, series_df in [("actual", series_actual), ("forecast", series_forec)]:
    for K in K_LIST:
        print(f"[MEF2.3_v6_modified] Rolling MSDR+DLR: stream={stream_label}, K={K}, prob_mode={PROB_MODE}")
        results[(stream_label, K)] = rolling_msdr_and_dlr(series_df, label=stream_label, K=K)

# ============================================================
# 9) POST-PROCESS: CAP + UNIT CONVERT + AEF FILL (NaN/negative)
# ============================================================
def attach_aef_and_postprocess(df_mef_raw, stream_label, K):
    if df_mef_raw.empty:
        return df_mef_raw.copy()

    d = df_mef_raw.copy()
    d["datetime_utc"] = pd.to_datetime(d["datetime_utc"], utc=True)
    d = d.sort_values("datetime_utc").set_index("datetime_utc")

    d = d.join(df_aef, how="left")
    aef_col = "aef_true_kg_per_kwh" if stream_label == "actual" else "aef_tier1_forecast_kg_per_kwh"
    d["aef_used_kg_per_kwh"] = d[aef_col].astype(float)

    soft_raw = d["mef_soft_kg_per_mwh_raw"].astype(float)
    hard_raw = d["mef_hard_kg_per_mwh_raw"].astype(float)

    # quantiles
    q_soft_lo  = float(soft_raw.quantile(CAP_Q_LOW))
    q_soft_hi  = float(soft_raw.quantile(CAP_Q_HIGH))
    q_hard_lo  = float(hard_raw.quantile(CAP_Q_LOW))
    q_hard_hi  = float(hard_raw.quantile(CAP_Q_HIGH))

    # absolute bounds
    abs_lo = -CAP_ABS_FACTOR * EF_MAX
    abs_hi =  CAP_ABS_FACTOR * EF_MAX

    # fixed bounding: intersection (tighten)
    soft_cap_lo = max(q_soft_lo, abs_lo)
    soft_cap_hi = min(q_soft_hi, abs_hi)
    hard_cap_lo = max(q_hard_lo, abs_lo)
    hard_cap_hi = min(q_hard_hi, abs_hi)

    if soft_cap_lo > soft_cap_hi:
        soft_cap_lo, soft_cap_hi = abs_lo, abs_hi
    if hard_cap_lo > hard_cap_hi:
        hard_cap_lo, hard_cap_hi = abs_lo, abs_hi

    d["mef_soft_kg_per_mwh_capped"] = soft_raw.clip(lower=soft_cap_lo, upper=soft_cap_hi)
    d["mef_hard_kg_per_mwh_capped"] = hard_raw.clip(lower=hard_cap_lo, upper=hard_cap_hi)

    d["soft_was_capped"] = (d["mef_soft_kg_per_mwh_capped"] != d["mef_soft_kg_per_mwh_raw"])
    d["hard_was_capped"] = (d["mef_hard_kg_per_mwh_capped"] != d["mef_hard_kg_per_mwh_raw"])

    # convert to kg/kWh
    d["mef_soft_kg_per_kwh_raw"]    = d["mef_soft_kg_per_mwh_raw"] / 1000.0
    d["mef_hard_kg_per_kwh_raw"]    = d["mef_hard_kg_per_mwh_raw"] / 1000.0
    d["mef_soft_kg_per_kwh_capped"] = d["mef_soft_kg_per_mwh_capped"] / 1000.0
    d["mef_hard_kg_per_kwh_capped"] = d["mef_hard_kg_per_mwh_capped"] / 1000.0

    soft_c = d["mef_soft_kg_per_kwh_capped"].astype(float).values
    hard_c = d["mef_hard_kg_per_kwh_capped"].astype(float).values
    aef_u  = d["aef_used_kg_per_kwh"].astype(float).values

    d["soft_is_nan_pre_fill"] = ~np.isfinite(soft_c)
    d["hard_is_nan_pre_fill"] = ~np.isfinite(hard_c)
    d["soft_is_negative_pre_fill"] = np.isfinite(soft_c) & (soft_c < 0)
    d["hard_is_negative_pre_fill"] = np.isfinite(hard_c) & (hard_c < 0)

    # report exact timestamps
    pd.DataFrame({"datetime_utc": d.index[d["soft_is_nan_pre_fill"]]}).to_csv(
        os.path.join(OUT_CSV_DIR, f"{stream_label}_K{K}_soft_nan_hours_pre_fill.csv"), index=False
    )
    pd.DataFrame({"datetime_utc": d.index[d["hard_is_nan_pre_fill"]]}).to_csv(
        os.path.join(OUT_CSV_DIR, f"{stream_label}_K{K}_hard_nan_hours_pre_fill.csv"), index=False
    )
    pd.DataFrame({"datetime_utc": d.index[d["soft_is_negative_pre_fill"]]}).to_csv(
        os.path.join(OUT_CSV_DIR, f"{stream_label}_K{K}_soft_negative_hours_pre_fill.csv"), index=False
    )
    pd.DataFrame({"datetime_utc": d.index[d["hard_is_negative_pre_fill"]]}).to_csv(
        os.path.join(OUT_CSV_DIR, f"{stream_label}_K{K}_hard_negative_hours_pre_fill.csv"), index=False
    )

    # Fill rule: if NaN OR negative -> AEF
    def fill(x, aef):
        x = np.asarray(x, dtype=float)
        y = x.copy()
        bad = (~np.isfinite(y)) | (y < 0)
        y[bad] = aef[bad]
        return y, bad

    soft_filled, soft_bad = fill(soft_c, aef_u)
    hard_filled, hard_bad = fill(hard_c, aef_u)

    d["mef_soft_kg_per_kwh_filled"] = soft_filled
    d["mef_hard_kg_per_kwh_filled"] = hard_filled
    d["soft_was_filled"] = soft_bad
    d["hard_was_filled"] = hard_bad

    d["mef_soft_kg_per_mwh_filled"] = d["mef_soft_kg_per_kwh_filled"] * 1000.0
    d["mef_hard_kg_per_mwh_filled"] = d["mef_hard_kg_per_kwh_filled"] * 1000.0

    diag = [{
        "stream": stream_label, "K": K,
        "prob_mode": PROB_MODE,
        "CAP_Q_LOW": CAP_Q_LOW, "CAP_Q_HIGH": CAP_Q_HIGH,
        "CAP_ABS_FACTOR": CAP_ABS_FACTOR, "EF_MAX_kg_per_mwh": EF_MAX,
        "soft_cap_lo_kg_per_mwh": soft_cap_lo, "soft_cap_hi_kg_per_mwh": soft_cap_hi,
        "hard_cap_lo_kg_per_mwh": hard_cap_lo, "hard_cap_hi_kg_per_mwh": hard_cap_hi,
        "n_rows": int(d.shape[0]),
        "n_soft_capped": int(d["soft_was_capped"].sum()),
        "n_hard_capped": int(d["hard_was_capped"].sum()),
        "n_soft_nan_pre_fill": int(d["soft_is_nan_pre_fill"].sum()),
        "n_hard_nan_pre_fill": int(d["hard_is_nan_pre_fill"].sum()),
        "n_soft_negative_pre_fill": int(d["soft_is_negative_pre_fill"].sum()),
        "n_hard_negative_pre_fill": int(d["hard_is_negative_pre_fill"].sum()),
        "n_soft_filled_total": int(d["soft_was_filled"].sum()),
        "n_hard_filled_total": int(d["hard_was_filled"].sum()),
        "aef_missing_rows": int(pd.isna(d["aef_used_kg_per_kwh"]).sum()),
    }]
    pd.DataFrame(diag).to_csv(os.path.join(OUT_CSV_DIR, f"postprocess_diagnostics_{stream_label}_K{K}.csv"), index=False)

    out = d.reset_index().rename(columns={"index":"datetime_utc"})
    out.to_csv(os.path.join(OUT_CSV_DIR, f"mef_{stream_label}_K{K}_raw_capped_filled.csv"), index=False)

    compact = out[[
        "datetime_utc",
        "mef_soft_kg_per_kwh_filled",
        "mef_hard_kg_per_kwh_filled",
        "aef_used_kg_per_kwh",
        "soft_was_capped","hard_was_capped",
        "soft_is_nan_pre_fill","hard_is_nan_pre_fill",
        "soft_is_negative_pre_fill","hard_is_negative_pre_fill",
        "soft_was_filled","hard_was_filled",
        "hard_regime",
    ]].copy()
    compact.to_csv(os.path.join(OUT_CSV_DIR, f"mef_{stream_label}_K{K}_FINAL_stage2_inputs_kg_per_kwh.csv"), index=False)

    return out

post = {}
dlr_tables = {}
for (stream_label, K), (df_mef_raw, df_fit, df_reg, df_tm, df_dlr) in results.items():
    post[(stream_label, K)] = attach_aef_and_postprocess(df_mef_raw, stream_label, K)
    dlr_tables[(stream_label, K)] = df_dlr

# ============================================================
# 10) DLR post: convert slope to kg/kWh and provide overlay vs MSDR soft
# ============================================================
def dlr_postprocess(df_dlr, stream_label, K):
    d = df_dlr.copy()
    d["datetime_utc"] = pd.to_datetime(d["datetime_utc"], utc=True)
    d = d.sort_values("datetime_utc").set_index("datetime_utc")
    d["dlr_slope_kg_per_kwh"] = d["dlr_slope_kg_per_mwh"] / 1000.0
    d = d.reset_index()
    d.to_csv(os.path.join(OUT_CSV_DIR, f"dlr_baseline_{stream_label}_K{K}_kg_per_kwh.csv"), index=False)
    return d

dlr_post = {}
for (stream_label, K), df_dlr in dlr_tables.items():
    dlr_post[(stream_label, K)] = dlr_postprocess(df_dlr, stream_label, K)

# ============================================================
# 11) STATS (MSDR)
# ============================================================
stats_rows = []
for (stream_label, K), df_out in post.items():
    if df_out.empty:
        continue
    row = {"stream": stream_label, "K": K, "prob_mode": PROB_MODE}
    row.update(summarize_series_stats(df_out["mef_soft_kg_per_kwh_raw"],  "soft_raw"))
    row.update(summarize_series_stats(df_out["mef_hard_kg_per_kwh_raw"],  "hard_raw"))
    row.update(summarize_series_stats(df_out["mef_soft_kg_per_kwh_capped"],  "soft_capped"))
    row.update(summarize_series_stats(df_out["mef_hard_kg_per_kwh_capped"],  "hard_capped"))
    row.update(summarize_series_stats(df_out["mef_soft_kg_per_kwh_filled"], "soft_filled"))
    row.update(summarize_series_stats(df_out["mef_hard_kg_per_kwh_filled"], "hard_filled"))
    row["n_soft_capped"] = int(df_out["soft_was_capped"].sum())
    row["n_hard_capped"] = int(df_out["hard_was_capped"].sum())
    row["n_soft_filled_total"] = int(df_out["soft_was_filled"].sum())
    row["n_hard_filled_total"] = int(df_out["hard_was_filled"].sum())
    row["aef_missing_rows"] = int(pd.isna(df_out["aef_used_kg_per_kwh"]).sum())
    stats_rows.append(row)

pd.DataFrame(stats_rows).to_csv(os.path.join(OUT_CSV_DIR, "msdr_mef_stats_raw_capped_filled_by_stream.csv"), index=False)

# ============================================================
# 12) PLOTS
#   - MEF hourly and daily mean (soft/hard separately)
#   - regime probability plots (hourly and daily)
#   - DLR slope plot + overlay vs MSDR soft
#   - fossil share vs MEF and fossil share vs regime probabilities (daily)
# ============================================================
def _prep_ts(df_out):
    d = df_out.copy()
    d["datetime_utc"] = pd.to_datetime(d["datetime_utc"], utc=True)
    d = d.sort_values("datetime_utc").set_index("datetime_utc")
    return d

def plot_single_ts(index, y, title, fname, ylabel):
    fig, ax = plt.subplots(figsize=(12,4))
    ax.plot(index, y, linewidth=1.2)
    apply_plot_settings(ax, title, "Time (UTC)", ylabel)
    save_fig(fig, fname)

def plot_single_daily(index, y, title, fname, ylabel):
    s = pd.Series(y, index=index).resample("D").mean()
    fig, ax = plt.subplots(figsize=(12,4))
    ax.plot(s.index, s.values, linewidth=1.5)
    apply_plot_settings(ax, title, "Date (UTC)", ylabel)
    save_fig(fig, fname)

def plot_overlay(index, y1, y2, title, fname, ylabel, lab1, lab2):
    fig, ax = plt.subplots(figsize=(12,4))
    ax.plot(index, y1, linewidth=1.2, label=lab1)
    ax.plot(index, y2, linewidth=1.2, label=lab2)
    apply_plot_settings(ax, title, "Time (UTC)", ylabel)
    ax.legend(fontsize=10)
    save_fig(fig, fname)

def plot_overlay_daily(index, y1, y2, title, fname, ylabel, lab1, lab2):
    s1 = pd.Series(y1, index=index).resample("D").mean()
    s2 = pd.Series(y2, index=index).resample("D").mean()
    fig, ax = plt.subplots(figsize=(12,4))
    ax.plot(s1.index, s1.values, linewidth=1.5, label=lab1)
    ax.plot(s2.index, s2.values, linewidth=1.5, label=lab2)
    apply_plot_settings(ax, title, "Date (UTC)", ylabel)
    ax.legend(fontsize=10)
    save_fig(fig, fname)

# --- MEF & probs plots ---
for (stream_label, K), df_out in post.items():
    if df_out.empty:
        continue
    d = _prep_ts(df_out)

    # MEF filled hourly separate
    plot_single_ts(
        d.index, d["mef_soft_kg_per_kwh_filled"].values,
        f"{stream_label.upper()} K={K} — Soft MEF FILLED (hourly)",
        f"{stream_label}_K{K}_soft_mef_filled_hourly",
        "kgCO₂/kWh"
    )
    plot_single_ts(
        d.index, d["mef_hard_kg_per_kwh_filled"].values,
        f"{stream_label.upper()} K={K} — Hard MEF FILLED (hourly)",
        f"{stream_label}_K{K}_hard_mef_filled_hourly",
        "kgCO₂/kWh"
    )

    # MEF filled daily mean separate
    plot_single_daily(
        d.index, d["mef_soft_kg_per_kwh_filled"].values,
        f"{stream_label.upper()} K={K} — Soft MEF FILLED (daily mean)",
        f"{stream_label}_K{K}_soft_mef_filled_daily_mean",
        "kgCO₂/kWh"
    )
    plot_single_daily(
        d.index, d["mef_hard_kg_per_kwh_filled"].values,
        f"{stream_label.upper()} K={K} — Hard MEF FILLED (daily mean)",
        f"{stream_label}_K{K}_hard_mef_filled_daily_mean",
        "kgCO₂/kWh"
    )

    # Probability plots (hourly + daily) per regime
    prob_cols = [c for c in d.columns if c.startswith(f"p_{PROB_MODE}_regime_")]
    for pc in prob_cols:
        kname = pc.split("_")[-1]
        plot_single_ts(
            d.index, d[pc].values,
            f"{stream_label.upper()} K={K} — Regime prob (smoothed) k={kname} (hourly)",
            f"{stream_label}_K{K}_prob_k{kname}_hourly",
            "probability"
        )
        plot_single_daily(
            d.index, d[pc].values,
            f"{stream_label.upper()} K={K} — Regime prob (smoothed) k={kname} (daily mean)",
            f"{stream_label}_K{K}_prob_k{kname}_daily_mean",
            "probability"
        )

# --- DLR plots + overlays vs MSDR soft ---
for (stream_label, K), df_dlr in dlr_post.items():
    if ("actual", K) in post and stream_label == "actual":
        ms = _prep_ts(post[(stream_label, K)])
    elif ("forecast", K) in post and stream_label == "forecast":
        ms = _prep_ts(post[(stream_label, K)])
    else:
        continue

    dd = df_dlr.copy()
    dd["datetime_utc"] = pd.to_datetime(dd["datetime_utc"], utc=True)
    dd = dd.sort_values("datetime_utc").set_index("datetime_utc")

    common = ms.index.intersection(dd.index)
    ms2 = ms.loc[common]
    dd2 = dd.loc[common]

    # DLR slope hourly
    plot_single_ts(
        common, dd2["dlr_slope_kg_per_kwh"].values,
        f"{stream_label.upper()} K={K} — DLR slope (hourly)",
        f"{stream_label}_K{K}_dlr_slope_hourly",
        "kgCO₂/kWh"
    )
    # overlay hourly (DLR vs MSDR soft)
    plot_overlay(
        common,
        dd2["dlr_slope_kg_per_kwh"].values,
        ms2["mef_soft_kg_per_kwh_filled"].values,
        f"{stream_label.upper()} K={K} — Overlay: DLR slope vs MSDR Soft MEF (hourly)",
        f"{stream_label}_K{K}_overlay_dlr_vs_msdr_soft_hourly",
        "kgCO₂/kWh",
        "DLR slope",
        "MSDR soft (filled)"
    )
    # overlay daily
    plot_overlay_daily(
        common,
        dd2["dlr_slope_kg_per_kwh"].values,
        ms2["mef_soft_kg_per_kwh_filled"].values,
        f"{stream_label.upper()} K={K} — Overlay: DLR slope vs MSDR Soft MEF (daily mean)",
        f"{stream_label}_K{K}_overlay_dlr_vs_msdr_soft_daily_mean",
        "kgCO₂/kWh",
        "DLR slope",
        "MSDR soft (filled)"
    )

# --- fossil share vs MEF and vs probs (daily) ---
def fossil_share_plots(stream_label, fossil_share_series, K):
    if (stream_label, K) not in post:
        return
    ms = _prep_ts(post[(stream_label, K)])
    fs = fossil_share_series.copy()
    # align on 2024 target hours only (ms is only target hours)
    common = ms.index.intersection(fs.index)
    if len(common) == 0:
        return

    ms = ms.loc[common]
    fs = fs.loc[common]

    # daily series
    fs_d = fs.resample("D").mean()
    mef_soft_d = pd.Series(ms["mef_soft_kg_per_kwh_filled"].values, index=common).resample("D").mean()
    mef_hard_d = pd.Series(ms["mef_hard_kg_per_kwh_filled"].values, index=common).resample("D").mean()

    # fossil share vs MEF soft daily
    plot_overlay(
        fs_d.index,
        fs_d.values,
        mef_soft_d.values,
        f"{stream_label.upper()} K={K} — Fossil share vs Soft MEF (daily mean)",
        f"fossil_share_vs_mef_{stream_label}_K{K}_soft_daily",
        "value",
        "fossil share",
        "soft MEF (kg/kWh)"
    )
    # fossil share vs MEF hard daily
    plot_overlay(
        fs_d.index,
        fs_d.values,
        mef_hard_d.values,
        f"{stream_label.upper()} K={K} — Fossil share vs Hard MEF (daily mean)",
        f"fossil_share_vs_mef_{stream_label}_K{K}_hard_daily",
        "value",
        "fossil share",
        "hard MEF (kg/kWh)"
    )

    # fossil share vs regime probs daily
    prob_cols = [c for c in ms.columns if c.startswith(f"p_{PROB_MODE}_regime_")]
    corr_rows = []
    for pc in prob_cols:
        pk = pd.Series(ms[pc].values, index=common).resample("D").mean()
        plot_overlay(
            fs_d.index,
            fs_d.values,
            pk.values,
            f"{stream_label.upper()} K={K} — Fossil share vs Prob {pc} (daily mean)",
            f"fossil_share_vs_prob_{stream_label}_K{K}_{pc}_daily",
            "value",
            "fossil share",
            pc
        )
        # correlations (daily)
        tmp = pd.concat([fs_d.rename("fossil_share"), pk.rename(pc), mef_soft_d.rename("mef_soft")], axis=1).dropna()
        if not tmp.empty:
            corr_rows.append({
                "stream": stream_label, "K": K, "prob_col": pc,
                "corr_fossil_prob": float(tmp["fossil_share"].corr(tmp[pc])),
                "corr_fossil_mef_soft": float(tmp["fossil_share"].corr(tmp["mef_soft"])),
            })

    pd.DataFrame(corr_rows).to_csv(os.path.join(OUT_CSV_DIR, f"fossil_share_correlations_{stream_label}_K{K}_daily.csv"), index=False)

# run for each stream/K
for K in K_LIST:
    fossil_share_plots("actual", fossil_share_actual, K)
    fossil_share_plots("forecast", fossil_share_forec, K)

# ============================================================
# 13) ZIP EVERYTHING
# ============================================================
zip_path = "/content/mef2_3_v6_modified_outputs.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(OUT_DIR):
        for fn in files:
            full = os.path.join(root, fn)
            rel  = os.path.relpath(full, OUT_DIR)
            z.write(full, arcname=os.path.join("mef2_3_v6_modified_outputs", rel))

print("\n[MEF2.3_v6_modified] Done.")
print("Outputs folder:", OUT_DIR)
print("ZIP:", zip_path)

# 3. GFS-based local renewable input preparation

This section keeps the GFS organizing cells used before the optimization utility cell. It prepares hourly 2024 weather inputs for PV and wind availability calculations.


## 3.1 T2M hourly reconstruction


In [ ]:
import pandas as pd
import numpy as np
# Clean GFS T2M f3/f6 instantaneous forecasts, merge them into a complete 3-hourly series,
# and generate a smooth hourly T2M timeseries for 2024 via linear interpolation.

# ============================
# CONFIG – file paths
# ============================
PATH_T2M_F3 = "/content/data/t2m_f3_2024.csv"
PATH_T2M_F6 = "/content/data/t2m_f6_2024.csv"

OUT_T2M_3H = "/content/data/t2m_3hour_2024_combined.csv"
OUT_T2M_1H = "/content/data/t2m_hourly_2024_from_f3f6.csv"

# Time horizon for 2024
YEAR_START       = pd.Timestamp("2024-01-01 00:00:00", tz="UTC")
YEAR_END_HOURLY  = pd.Timestamp("2024-12-31 23:00:00", tz="UTC")
YEAR_END_3H      = pd.Timestamp("2024-12-31 21:00:00", tz="UTC")  # last 3h step

# F3 valid times: 03, 09, 15, 21 (4 per day)
F3_START = YEAR_START + pd.Timedelta(hours=3)
F3_END   = pd.Timestamp("2024-12-31 21:00:00", tz="UTC")
EXPECTED_F3_INDEX = pd.date_range(start=F3_START, end=F3_END, freq="6h", tz="UTC")

# F6 valid times: 00, 06, 12, 18 (4 per day)
F6_START = YEAR_START
F6_END   = pd.Timestamp("2024-12-31 18:00:00", tz="UTC")
EXPECTED_F6_INDEX = pd.date_range(start=F6_START, end=F6_END, freq="6h", tz="UTC")


# ============================
# Helper: fix missing by previous day and report
# ============================
def fix_by_previous_day_for_index(df, value_col, name, expected_index):
    df = df.sort_index()
    df_reg = df.reindex(expected_index)
    df_reg.index.name = "datetime_utc"

    missing_before = df_reg[value_col].isna().sum()
    print(f"\n=== {name}: before filling ===")
    print(f"Start (expected): {df_reg.index.min()}")
    print(f"End   (expected): {df_reg.index.max()}")
    print(f"Expected count   : {len(df_reg)}")
    print(f"Missing values   : {missing_before}")

    filled_times = []

    for ts in df_reg.index[df_reg[value_col].isna()]:
        prev = ts - pd.Timedelta(days=1)
        if prev in df_reg.index and pd.notna(df_reg.loc[prev, value_col]):
            df_reg.loc[ts, value_col] = df_reg.loc[prev, value_col]
            filled_times.append(ts)

    missing_after = df_reg[value_col].isna().sum()

    print(f"\n=== {name}: after filling from previous day ===")
    print(f"Filled from previous day: {len(filled_times)}")
    print(f"Remaining missing      : {missing_after}")

    if filled_times:
        print("First filled timestamps (copied from t-24h):")
        print(filled_times[:10])

    if missing_after > 0:
        still_missing = df_reg.index[df_reg[value_col].isna()]
        print("First still-missing timestamps:")
        print(still_missing[:10])

    return df_reg


# ============================
# 1) Read raw T2M f3 and f6 files
# ============================
t3 = pd.read_csv(PATH_T2M_F3)
t6 = pd.read_csv(PATH_T2M_F6)

# Build datetime_utc from "day" + "valid_time"
t3["datetime_utc"] = pd.to_datetime(
    t3["day"].astype(str) + " " + t3["valid_time"].astype(str),
    utc=True
)
t6["datetime_utc"] = pd.to_datetime(
    t6["day"].astype(str) + " " + t6["valid_time"].astype(str),
    utc=True
)

# Keep only needed columns and set index
t3 = t3[["datetime_utc", "t2m"]].set_index("datetime_utc")
t6 = t6[["datetime_utc", "t2m"]].set_index("datetime_utc")

# Enforce expected grids (4 points/day) and fill real gaps
t3_reg = fix_by_previous_day_for_index(
    t3, "t2m", "T2M F3", EXPECTED_F3_INDEX
)
t6_reg = fix_by_previous_day_for_index(
    t6, "t2m", "T2M F6", EXPECTED_F6_INDEX
)

# ============================
# 2) Combine into full 3-hourly instantaneous T2M
# ============================

s3 = t3_reg["t2m"].sort_index()
s6 = t6_reg["t2m"].sort_index()

# Union of the two series: 0,3,6,...,21 every day
s_all = pd.concat([s3, s6]).sort_index()
s_all = s_all.groupby(level=0).mean()  # just in case of duplicates

df3h = s_all.to_frame(name="t2m_3h")
df3h.index.name = "datetime_utc"

# Force full 3-hour grid for 2024
expected_3h = pd.date_range(
    start=YEAR_START,
    end=YEAR_END_3H,
    freq="3h",
    tz="UTC"
)
df3h = df3h.reindex(expected_3h)

missing_3h = df3h["t2m_3h"].isna().sum()
print("\n=== Combined 3-hour T2M grid (full 2024) ===")
print(f"Start: {df3h.index.min()}")
print(f"End  : {df3h.index.max()}")
print(f"Total 3h points : {len(df3h)}")
print(f"Missing 3h points: {missing_3h}")

if missing_3h > 0:
    miss_times_3h = df3h.index[df3h["t2m_3h"].isna()]
    print("First missing 3h timestamps:")
    print(miss_times_3h[:10])

# Save 3-hour T2M file
df3h_reset = df3h.reset_index().rename(columns={"t2m_3h": "t2m"})
df3h_reset.to_csv(OUT_T2M_3H, index=False)
print(f"\nSaved 3-hour combined T2M file to: {OUT_T2M_3H}")

# ============================
# 3) Build hourly T2M by linear interpolation
# ============================

hourly_index = pd.date_range(
    start=YEAR_START,
    end=YEAR_END_HOURLY,
    freq="1h",
    tz="UTC"
)
df1h = pd.DataFrame(index=hourly_index)
df1h["t2m"] = df3h["t2m_3h"]

missing_hours_before = df1h["t2m"].isna().sum()
print("\n=== Hourly T2M before interpolation (full 2024) ===")
print(f"Total hourly timestamps : {len(df1h)}")
print(f"Missing t2m values      : {missing_hours_before}")

if missing_hours_before > 0:
    missing_times = df1h.index[df1h["t2m"].isna()]
    print("First missing hourly timestamps:")
    print(missing_times[:10])

# Linear interpolation (instantaneous field) + edge ffill/bfill
df1h["t2m"] = df1h["t2m"].interpolate(method="linear")
df1h["t2m"] = df1h["t2m"].ffill().bfill()

final_missing = df1h["t2m"].isna().sum()
print(f"\nMissing values AFTER interpolation + ffill/bfill: {final_missing}")

df1h_final = df1h.reset_index().rename(columns={"index": "datetime_utc"})
df1h_final.to_csv(OUT_T2M_1H, index=False)
print(f"Final hourly T2M saved to: {OUT_T2M_1H}")

df1h_final.head()


## 3.2 U100 hourly reconstruction


In [ ]:
import pandas as pd
import numpy as np

# =========================================
# CONFIG – file paths for U100
# =========================================
PATH_U100_F3 = "/content/data/u100_f3_2024.csv"
PATH_U100_F6 = "/content/data/u100_f6_2024.csv"

OUT_U100_3H = "/content/data/u100_3hour_2024_combined.csv"
OUT_U100_1H = "/content/data/u100_hourly_2024_from_f3f6.csv"

# Time horizon for 2024
YEAR_START       = pd.Timestamp("2024-01-01 00:00:00", tz="UTC")
YEAR_END_HOURLY  = pd.Timestamp("2024-12-31 23:00:00", tz="UTC")
YEAR_END_3H      = pd.Timestamp("2024-12-31 21:00:00", tz="UTC")  # last 3h step

# F3 valid times: 03, 09, 15, 21 (4 per day)
F3_START = YEAR_START + pd.Timedelta(hours=3)
F3_END   = pd.Timestamp("2024-12-31 21:00:00", tz="UTC")
EXPECTED_F3_INDEX = pd.date_range(start=F3_START, end=F3_END, freq="6h", tz="UTC")

# F6 valid times: 00, 06, 12, 18 (4 per day)
F6_START = YEAR_START
F6_END   = pd.Timestamp("2024-12-31 18:00:00", tz="UTC")
EXPECTED_F6_INDEX = pd.date_range(start=F6_START, end=F6_END, freq="6h", tz="UTC")


# =========================================
# Helper: fix missing by previous day and report
# =========================================
def fix_by_previous_day_for_index(df, value_col, name, expected_index):
    df = df.sort_index()
    df_reg = df.reindex(expected_index)
    df_reg.index.name = "datetime_utc"

    missing_before = df_reg[value_col].isna().sum()
    print(f"\n=== {name}: before filling ===")
    print(f"Start (expected): {df_reg.index.min()}")
    print(f"End   (expected): {df_reg.index.max()}")
    print(f"Expected count   : {len(df_reg)}")
    print(f"Missing values   : {missing_before}")

    filled_times = []

    for ts in df_reg.index[df_reg[value_col].isna()]:
        prev = ts - pd.Timedelta(days=1)
        if prev in df_reg.index and pd.notna(df_reg.loc[prev, value_col]):
            df_reg.loc[ts, value_col] = df_reg.loc[prev, value_col]
            filled_times.append(ts)

    missing_after = df_reg[value_col].isna().sum()

    print(f"\n=== {name}: after filling from previous day ===")
    print(f"Filled from previous day: {len(filled_times)}")
    print(f"Remaining missing      : {missing_after}")

    if filled_times:
        print("First filled timestamps (copied from t-24h):")
        print(filled_times[:10])

    if missing_after > 0:
        still_missing = df_reg.index[df_reg[value_col].isna()]
        print("First still-missing timestamps:")
        print(still_missing[:10])

    return df_reg


# =========================================
# 1) Read raw U100 f3 and f6 files
# =========================================
u3 = pd.read_csv(PATH_U100_F3)
u6 = pd.read_csv(PATH_U100_F6)

# Build datetime_utc from "day" + "valid_time"
u3["datetime_utc"] = pd.to_datetime(
    u3["day"].astype(str) + " " + u3["valid_time"].astype(str),
    utc=True
)
u6["datetime_utc"] = pd.to_datetime(
    u6["day"].astype(str) + " " + u6["valid_time"].astype(str),
    utc=True
)

# Keep only needed columns and set index
# (assumes the column with wind at 100 m is named 'u100')
u3 = u3[["datetime_utc", "u100"]].set_index("datetime_utc")
u6 = u6[["datetime_utc", "u100"]].set_index("datetime_utc")

# Enforce expected grids (4 points/day) and fill real gaps
u3_reg = fix_by_previous_day_for_index(
    u3, "u100", "U100 F3", EXPECTED_F3_INDEX
)
u6_reg = fix_by_previous_day_for_index(
    u6, "u100", "U100 F6", EXPECTED_F6_INDEX
)

# =========================================
# 2) Combine into full 3-hourly instantaneous U100
# =========================================

s3 = u3_reg["u100"].sort_index()
s6 = u6_reg["u100"].sort_index()

# Union of the two series: 0,3,6,...,21 every day
s_all = pd.concat([s3, s6]).sort_index()
s_all = s_all.groupby(level=0).mean()  # just in case of duplicates

df3h = s_all.to_frame(name="u100_3h")
df3h.index.name = "datetime_utc"

# Force full 3-hour grid for 2024
expected_3h = pd.date_range(
    start=YEAR_START,
    end=YEAR_END_3H,
    freq="3h",
    tz="UTC"
)
df3h = df3h.reindex(expected_3h)

missing_3h = df3h["u100_3h"].isna().sum()
print("\n=== Combined 3-hour U100 grid (full 2024) ===")
print(f"Start: {df3h.index.min()}")
print(f"End  : {df3h.index.max()}")
print(f"Total 3h points : {len(df3h)}")
print(f"Missing 3h points: {missing_3h}")

if missing_3h > 0:
    miss_times_3h = df3h.index[df3h["u100_3h"].isna()]
    print("First missing 3h timestamps:")
    print(miss_times_3h[:10])

# Save 3-hour U100 file
df3h_reset = df3h.reset_index().rename(columns={"u100_3h": "u100"})
df3h_reset.to_csv(OUT_U100_3H, index=False)
print(f"\nSaved 3-hour combined U100 file to: {OUT_U100_3H}")

# =========================================
# 3) Build hourly U100 by linear interpolation
# =========================================

hourly_index = pd.date_range(
    start=YEAR_START,
    end=YEAR_END_HOURLY,
    freq="1h",
    tz="UTC"
)
df1h = pd.DataFrame(index=hourly_index)
df1h["u100"] = df3h["u100_3h"]

missing_hours_before = df1h["u100"].isna().sum()
print("\n=== Hourly U100 before interpolation (full 2024) ===")
print(f"Total hourly timestamps : {len(df1h)}")
print(f"Missing u100 values     : {missing_hours_before}")

if missing_hours_before > 0:
    missing_times = df1h.index[df1h["u100"].isna()]
    print("First missing hourly timestamps:")
    print(missing_times[:10])

# Linear interpolation (instantaneous field) + edge ffill/bfill
df1h["u100"] = df1h["u100"].interpolate(method="linear")
df1h["u100"] = df1h["u100"].ffill().bfill()

final_missing = df1h["u100"].isna().sum()
print(f"\nMissing values AFTER interpolation + ffill/bfill: {final_missing}")

df1h_final = df1h.reset_index().rename(columns={"index": "datetime_utc"})
df1h_final.to_csv(OUT_U100_1H, index=False)
print(f"Final hourly U100 saved to: {OUT_U100_1H}")

df1h_final.head()


## 3.3 V100 hourly reconstruction


In [ ]:
import pandas as pd
import numpy as np

# =========================================
# CONFIG – file paths for V100
# =========================================
PATH_V100_F3 = "/content/data/v100_f3_2024.csv"
PATH_V100_F6 = "/content/data/v100_f6_2024.csv"

OUT_V100_3H = "/content/data/v100_3hour_2024_combined.csv"
OUT_V100_1H = "/content/data/v100_hourly_2024_from_f3f6.csv"

# Time horizon for 2024
YEAR_START       = pd.Timestamp("2024-01-01 00:00:00", tz="UTC")
YEAR_END_HOURLY  = pd.Timestamp("2024-12-31 23:00:00", tz="UTC")
YEAR_END_3H      = pd.Timestamp("2024-12-31 21:00:00", tz="UTC")  # last 3h step

# F3 valid times: 03, 09, 15, 21 (4 per day)
F3_START = YEAR_START + pd.Timedelta(hours=3)
F3_END   = pd.Timestamp("2024-12-31 21:00:00", tz="UTC")
EXPECTED_F3_INDEX = pd.date_range(start=F3_START, end=F3_END, freq="6h", tz="UTC")

# F6 valid times: 00, 06, 12, 18 (4 per day)
F6_START = YEAR_START
F6_END   = pd.Timestamp("2024-12-31 18:00:00", tz="UTC")
EXPECTED_F6_INDEX = pd.date_range(start=F6_START, end=F6_END, freq="6h", tz="UTC")


# =========================================
# Helper: fix missing by previous day and report
# =========================================
def fix_by_previous_day_for_index(df, value_col, name, expected_index):
    df = df.sort_index()
    df_reg = df.reindex(expected_index)
    df_reg.index.name = "datetime_utc"

    missing_before = df_reg[value_col].isna().sum()
    print(f"\n=== {name}: before filling ===")
    print(f"Start (expected): {df_reg.index.min()}")
    print(f"End   (expected): {df_reg.index.max()}")
    print(f"Expected count   : {len(df_reg)}")
    print(f"Missing values   : {missing_before}")

    filled_times = []

    for ts in df_reg.index[df_reg[value_col].isna()]:
        prev = ts - pd.Timedelta(days=1)
        if prev in df_reg.index and pd.notna(df_reg.loc[prev, value_col]):
            df_reg.loc[ts, value_col] = df_reg.loc[prev, value_col]
            filled_times.append(ts)

    missing_after = df_reg[value_col].isna().sum()

    print(f"\n=== {name}: after filling from previous day ===")
    print(f"Filled from previous day: {len(filled_times)}")
    print(f"Remaining missing      : {missing_after}")

    if filled_times:
        print("First filled timestamps (copied from t-24h):")
        print(filled_times[:10])

    if missing_after > 0:
        still_missing = df_reg.index[df_reg[value_col].isna()]
        print("First still-missing timestamps:")
        print(still_missing[:10])

    return df_reg


# =========================================
# 1) Read raw V100 f3 and f6 files
# =========================================
v3 = pd.read_csv(PATH_V100_F3)
v6 = pd.read_csv(PATH_V100_F6)

# Build datetime_utc from "day" + "valid_time"
v3["datetime_utc"] = pd.to_datetime(
    v3["day"].astype(str) + " " + v3["valid_time"].astype(str),
    utc=True
)
v6["datetime_utc"] = pd.to_datetime(
    v6["day"].astype(str) + " " + v6["valid_time"].astype(str),
    utc=True
)

# Keep only needed columns and set index
# (assumes the column with wind at 100 m (v-component) is named 'v100')
v3 = v3[["datetime_utc", "v100"]].set_index("datetime_utc")
v6 = v6[["datetime_utc", "v100"]].set_index("datetime_utc")

# Enforce expected grids (4 points/day) and fill real gaps
v3_reg = fix_by_previous_day_for_index(
    v3, "v100", "V100 F3", EXPECTED_F3_INDEX
)
v6_reg = fix_by_previous_day_for_index(
    v6, "v100", "V100 F6", EXPECTED_F6_INDEX
)

# =========================================
# 2) Combine into full 3-hourly instantaneous V100
# =========================================

s3 = v3_reg["v100"].sort_index()
s6 = v6_reg["v100"].sort_index()

# Union of the two series: 0,3,6,...,21 every day
s_all = pd.concat([s3, s6]).sort_index()
s_all = s_all.groupby(level=0).mean()  # just in case of duplicates

df3h = s_all.to_frame(name="v100_3h")
df3h.index.name = "datetime_utc"

# Force full 3-hour grid for 2024
expected_3h = pd.date_range(
    start=YEAR_START,
    end=YEAR_END_3H,
    freq="3h",
    tz="UTC"
)
df3h = df3h.reindex(expected_3h)

missing_3h = df3h["v100_3h"].isna().sum()
print("\n=== Combined 3-hour V100 grid (full 2024) ===")
print(f"Start: {df3h.index.min()}")
print(f"End  : {df3h.index.max()}")
print(f"Total 3h points : {len(df3h)}")
print(f"Missing 3h points: {missing_3h}")

if missing_3h > 0:
    miss_times_3h = df3h.index[df3h["v100_3h"].isna()]
    print("First missing 3h timestamps:")
    print(miss_times_3h[:10])

# Save 3-hour V100 file
df3h_reset = df3h.reset_index().rename(columns={"v100_3h": "v100"})
df3h_reset.to_csv(OUT_V100_3H, index=False)
print(f"\nSaved 3-hour combined V100 file to: {OUT_V100_3H}")

# =========================================
# 3) Build hourly V100 by linear interpolation
# =========================================

hourly_index = pd.date_range(
    start=YEAR_START,
    end=YEAR_END_HOURLY,
    freq="1h",
    tz="UTC"
)
df1h = pd.DataFrame(index=hourly_index)
df1h["v100"] = df3h["v100_3h"]

missing_hours_before = df1h["v100"].isna().sum()
print("\n=== Hourly V100 before interpolation (full 2024) ===")
print(f"Total hourly timestamps : {len(df1h)}")
print(f"Missing v100 values     : {missing_hours_before}")

if missing_hours_before > 0:
    missing_times = df1h.index[df1h["v100"].isna()]
    print("First missing hourly timestamps:")
    print(missing_times[:10])

# Linear interpolation (instantaneous field) + edge ffill/bfill
df1h["v100"] = df1h["v100"].interpolate(method="linear")
df1h["v100"] = df1h["v100"].ffill().bfill()

final_missing = df1h["v100"].isna().sum()
print(f"\nMissing values AFTER interpolation + ffill/bfill: {final_missing}")

df1h_final = df1h.reset_index().rename(columns={"index": "datetime_utc"})
df1h_final.to_csv(OUT_V100_1H, index=False)
print(f"Final hourly V100 saved to: {OUT_V100_1H}")

df1h_final.head()


## 3.4 DSWRF hourly reconstruction

In [ ]:
import pandas as pd
import numpy as np
# Rebuild complete 2024 DSWRF from GFS f3/f6 averages by correcting gaps, deriving missing 3-hour blocks,
# and converting each 3-hour mean into consistent hourly values using block-wise disaggregation.

# ============================
# CONFIG – file paths
# ============================
PATH_F3 = "/content/data/dswrf_f3avg_2024.csv"
PATH_F6 = "/content/data/dswrf_f6avg_2024.csv"

OUT_3H = "/content/data/dswrf_3hour_reconstructed_2024.csv"
OUT_1H = "/content/data/dswrf_hourly_2024_from_f3f6.csv"

# Time horizon for 2024
YEAR_START       = pd.Timestamp("2024-01-01 00:00:00", tz="UTC")
YEAR_END_HOURLY  = pd.Timestamp("2024-12-31 23:00:00", tz="UTC")
YEAR_END_3H      = pd.Timestamp("2024-12-31 21:00:00", tz="UTC")  # last 3h step

# F3 is valid at 03, 09, 15, 21 (4 per day, 6-hourly with offset 3h)
F3_START = YEAR_START + pd.Timedelta(hours=3)
F3_END   = pd.Timestamp("2024-12-31 21:00:00", tz="UTC")
EXPECTED_F3_INDEX = pd.date_range(start=F3_START, end=F3_END, freq="6h", tz="UTC")

# F6 is valid at 00, 06, 12, 18 (4 per day, 6-hourly starting at midnight)
F6_START = YEAR_START
F6_END   = pd.Timestamp("2024-12-31 18:00:00", tz="UTC")
EXPECTED_F6_INDEX = pd.date_range(start=F6_START, end=F6_END, freq="6h", tz="UTC")


# ============================
# Helper: fix missing by previous day and report
# ============================
def fix_by_previous_day_for_index(df, value_col, name, expected_index):
    df = df.sort_index()
    df_reg = df.reindex(expected_index)
    df_reg.index.name = "datetime_utc"

    missing_before = df_reg[value_col].isna().sum()
    print(f"\n=== {name}: before filling ===")
    print(f"Start (expected): {df_reg.index.min()}")
    print(f"End   (expected): {df_reg.index.max()}")
    print(f"Expected count   : {len(df_reg)}")
    print(f"Missing values   : {missing_before}")

    filled_times = []

    for ts in df_reg.index[df_reg[value_col].isna()]:
        prev = ts - pd.Timedelta(days=1)
        if prev in df_reg.index and pd.notna(df_reg.loc[prev, value_col]):
            df_reg.loc[ts, value_col] = df_reg.loc[prev, value_col]
            filled_times.append(ts)

    missing_after = df_reg[value_col].isna().sum()

    print(f"\n=== {name}: after filling from previous day ===")
    print(f"Filled from previous day: {len(filled_times)}")
    print(f"Remaining missing      : {missing_after}")

    if filled_times:
        print("First filled timestamps (copied from t-24h):")
        print(filled_times[:10])

    if missing_after > 0:
        still_missing = df_reg.index[df_reg[value_col].isna()]
        print("First still-missing timestamps:")
        print(still_missing[:10])

    return df_reg


# ============================
# 1) Read raw f3 and f6 files
# ============================
f3 = pd.read_csv(PATH_F3)
f6 = pd.read_csv(PATH_F6)

# Build datetime_utc from "day" + "valid_time"
f3["datetime_utc"] = pd.to_datetime(
    f3["day"].astype(str) + " " + f3["valid_time"].astype(str),
    utc=True
)
f6["datetime_utc"] = pd.to_datetime(
    f6["day"].astype(str) + " " + f6["valid_time"].astype(str),
    utc=True
)

# Keep only needed columns and set index
f3 = f3[["datetime_utc", "dswrf"]].rename(columns={"dswrf": "A3"}).set_index("datetime_utc")
f6 = f6[["datetime_utc", "dswrf"]].rename(columns={"dswrf": "A6"}).set_index("datetime_utc")

# Enforce the correct expected grids (4 points/day) and fill real gaps
f3_reg = fix_by_previous_day_for_index(
    f3, "A3", "F3 DSWRF", EXPECTED_F3_INDEX
)
f6_reg = fix_by_previous_day_for_index(
    f6, "A6", "F6 DSWRF", EXPECTED_F6_INDEX
)

# ============================
# 2) Reconstruct full 3-hourly series (A3_2 = 2*A6 − A3_1)
# ============================

s3 = f3_reg["A3"].sort_index()
s6 = f6_reg["A6"].sort_index()

reconstructed = {}

for t6, A6 in s6.items():
    t3_1 = t6 - pd.Timedelta(hours=3)
    if t3_1 in s3.index and pd.notna(s3.loc[t3_1]):
        A3_1 = s3.loc[t3_1]
        A3_2 = 2.0 * A6 - A3_1
        # second 3-hour block ends at t6 → assign to t6
        reconstructed[t6] = A3_2

s_rec = pd.Series(reconstructed, name="A3_rec").sort_index()

# Combine original and reconstructed A3 values
s3_full = pd.concat([s3.rename("A3_orig"), s_rec]).sort_index()
s3_full = s3_full.groupby(level=0).mean()   # handle any duplicates

df3h = s3_full.to_frame(name="dswrf_3h")
df3h.index.name = "datetime_utc"

# Force full 3-hour grid for 2024
expected_3h = pd.date_range(
    start=YEAR_START,
    end=YEAR_END_3H,
    freq="3h",
    tz="UTC"
)
df3h = df3h.reindex(expected_3h)

missing_3h = df3h["dswrf_3h"].isna().sum()
print("\n=== Reconstructed 3-hour DSWRF grid (full 2024) ===")
print(f"Start: {df3h.index.min()}")
print(f"End  : {df3h.index.max()}")
print(f"Total 3h points : {len(df3h)}")
print(f"Missing 3h points before hourly step: {missing_3h}")

if missing_3h > 0:
    miss_times_3h = df3h.index[df3h["dswrf_3h"].isna()]
    print("First missing 3h timestamps:")
    print(miss_times_3h[:10])

# Save 3-hour reconstructed file (complete 3-hour time steps for 2024)
df3h_reset = df3h.reset_index().rename(columns={"dswrf_3h": "dswrf"})
df3h_reset.to_csv(OUT_3H, index=False)
print(f"\nSaved 3-hour reconstructed file to: {OUT_3H}")

# ============================
# ============================
# 3) Build hourly DSWRF by block-wise constant disaggregation
#    (each 3h value applies to the PREVIOUS 3 hours)
# ============================

# Full hourly index for 2024
hourly_index = pd.date_range(
    start=YEAR_START,
    end=YEAR_END_HOURLY,
    freq="1h",
    tz="UTC"
)

df1h = pd.DataFrame(index=hourly_index, columns=["dswrf"], dtype=float)

# Fill each 3-hour block: value at time t is average of [t-3, t)
for t, v in df3h["dswrf_3h"].items():
    if pd.isna(v):
        continue
    block_start = t - pd.Timedelta(hours=3)
    block_hours = pd.date_range(
        start=block_start,
        periods=3,
        freq="1h",
        tz="UTC"
    )
    for h in block_hours:
        if h in df1h.index:
            df1h.loc[h, "dswrf"] = v

# At this point, all hours from 2024-01-01 00:00 to 2024-12-31 20:00
# should be filled. Only possible NaNs are typically the very last hours
# (21:00–23:00 on Dec 31) if the last 3h block ends at 21:00.

missing_hours = df1h["dswrf"].isna().sum()
print("\n=== Hourly DSWRF after block-wise fill (before final edge fix) ===")
print(f"Total hourly timestamps : {len(df1h)}")
print(f"Missing dswrf values    : {missing_hours}")

if missing_hours > 0:
    missing_times = df1h.index[df1h["dswrf"].isna()]
    print("Missing hourly timestamps:")
    print(missing_times)

    # For the trailing night hours, it is safe to set them to 0
    # (DSWRF is zero at night).
    df1h.loc[missing_times, "dswrf"] = 0.0

# Final sanity check
final_missing = df1h["dswrf"].isna().sum()
print(f"\nMissing values AFTER block-wise fill + edge fix: {final_missing}")

# Save final hourly file
df1h_final = df1h.reset_index().rename(columns={"index": "datetime_utc"})
df1h_final.to_csv(OUT_1H, index=False)
print(f"Final hourly DSWRF (block-wise) saved to: {OUT_1H}")

df1h_final.head(10)


# 4. model-ready input preparation

This utility/precompute cell organizes the demand profiles, grid prices, GFS-based PV/wind availability, battery parameters, and emission signal files into the model-ready inputs used by M0, M1, and M2.


## 4.1 utility / precompute


In [ ]:
# =====================  Utility / Precompute
# Reads (time series):
#   /content/data/da_grid_elec_price_2024.csv
#   /content/data/localRES_PV_dswrf_hourly_GFS_2024.csv
#   /content/data/localRES_PV_t2m_hourly_GFS_2024.csv
#   /content/data/localRES_WT_u100_hourly_GFS_2024.csv
#   /content/data/localRES_WT_v100_hourly_GFS_2024.csv
#   /content/data/stage1_tier2.1_gridAEF_hourly_predicted_2024.csv
#   /content/data/stage1_tier2.2_gridMEF_hourly_predicted_2024.csv
#   /content/data/stage1_tier2.3_gridMEF_hourly_predicted_2024.csv
# Writes:
#   /content/data/model_ready/*.csv
#   /content/data/model_ready/plots/*.png
#   /content/data/model_ready/params_scenario_all.json

import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------- GLOBAL CONTROLS ----------------
YEAR = 2024
N_DAYS_TO_RUN = 7
START_DAY_OF_YEAR = 1
LAMBDA_EMIS = 1.0 #ignore this

DATA_DIR = Path("/content/data")
OUT_DIR  = DATA_DIR / "model_ready"
PLOT_DIR = OUT_DIR / "plots"
OUT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

idx_2024 = pd.date_range(
    "2024-01-01 00:00:00+00:00",
    "2024-12-31 23:00:00+00:00",
    freq="h",
    tz="UTC"
)

# ---------------- HELPERS ----------------
def read_hourly_csv(path: Path, time_col="datetime_utc") -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    df = pd.read_csv(path)
    if time_col not in df.columns:
        raise ValueError(f"{path.name}: missing time column '{time_col}'. Found: {list(df.columns)}")
    df[time_col] = pd.to_datetime(df[time_col], utc=True)
    df = df.sort_values(time_col).set_index(time_col)
    df = df.tz_convert("UTC") if df.index.tz is not None else df.tz_localize("UTC")
    return df

def assert_full_hourly(df: pd.DataFrame, name: str) -> pd.DataFrame:
    if df.index.duplicated().any():
        d = df.index[df.index.duplicated()][0]
        raise ValueError(f"{name}: duplicated timestamp example: {d}")
    missing = idx_2024.difference(df.index)
    extra   = df.index.difference(idx_2024)
    if len(missing) > 0:
        raise ValueError(f"{name}: missing {len(missing)} hours (first 5: {missing[:5].tolist()})")
    if len(extra) > 0:
        raise ValueError(f"{name}: extra {len(extra)} hours outside 2024 (first 5: {extra[:5].tolist()})")
    df2 = df.reindex(idx_2024)
    if df2.isna().any().any():
        cols = df2.columns[df2.isna().any()].tolist()
        where = df2[df2[cols].isna().any(axis=1)].index[:5].tolist()
        raise ValueError(f"{name}: NaNs after reindex in cols {cols}; example {where}")
    return df2

def save_lineplot_png(df: pd.DataFrame, cols, title: str, ylabel: str, outpath: Path):
    plt.figure(figsize=(14,4))
    for c in cols:
        plt.plot(df.index, df[c], label=c)
    plt.title(title)
    plt.ylabel(ylabel)
    plt.legend()
    plt.tight_layout()
    plt.savefig(outpath, dpi=150)
    plt.close()

# =============================================================================
# ---------------- SCALARS
# =============================================================================

# ---- Site / PV geometry ----
phi_deg   = 40.25      # latitude [deg]
lam_deg   = -3.75      # longitude [deg] (east positive, west negative)
beta_deg  = 34  # tilt [deg]
gamma_deg = 0  # kept for completeness

rho_g     = 0.2  # ground albedo [-]
eta_inv   = 0.96   # inverter efficiency [-]
G_sc      = 1366   # solar constant [W/m^2]

# ---- PV efficiency model coefficients ----
p_pv = 23.62
q_pv = -0.2983
r_pv = -0.09307
s_pv = -0.9795
m_pv = 0.1912
u_pv = 0.9865
h_pv = 0.028       # cell temp coefficient [°C/(W/m^2)]
ETA_PV_MAX = 0.2
# ---- PV sizing ----
installed_pv_kwp = 120000   # [kWp]
A_PV_unit        = 7   # [m^2/kWp]

# ---- Wind model scalars ----
Z_ref   = 100    # reference height for wind speed [m]
Z_wt    = 117     # hub height [m]
alpha   = 0.207     # shear exponent [-]
rho_air = 1.225    # air density [kg/m^3]
P_r_W   = 3450000     # rated power per turbine [W]
A_WT    = 12469     # rotor swept area [m^2]
Cp_eff  = 0.29    # effective Cp [-]
eta_wf  = 0.95    # wind farm efficiency [-]
wf_avail= 0.95    # availability [-]

v_in    = 3      # cut-in [m/s]
v_rated = 11.5     # rated speed [m/s]
v_out   = 22.5    # cut-out [m/s]
N_WT    = 35   # number of turbines [int]

# ---- Plant capacities  ----
P_ELZ_MAX_MW        = 209 #mw
P_GRID_MAX_MW       = 295.58 #mw
H2_MAX_KGPH         = 3657.567
NH3_CAP_KGPH        = 13472.036
MEOH_CAP_KGPH       = 5818.856
NH3_CRACK_CAP_KGPH  = 6062.416

# ---- Demand profile parameters  ----
A_DAY_NH3  = 0.06
A_DAY_H2   = 0.1
A_DAY_MEOH = 0.05
WEEKEND_FACTOR   = 0.90
DEMAND_CLIP_FRAC = 0.90
# =============================================================================
# ---------------- LOAD INPUT SERIES ----------------
# =============================================================================
price = assert_full_hourly(
    read_hourly_csv(DATA_DIR / "da_grid_elec_price_2024.csv")[["da_elec_price_EURperMWh"]],
    "price"
)

aef = assert_full_hourly(
    read_hourly_csv(DATA_DIR / "stage1_tier2.1_gridAEF_hourly_predicted_2024.csv")[["grid_aef_predicted_kg_per_kwh"]],
    "AEF"
)

AEF_COL = "grid_aef_predicted_kg_per_kwh"
MEF_UP_COL = "grid_mef_upramp_pred_kg_per_kwh"
MEF_SOFT_COL = "grid_mef_soft_kg_per_kwh"

# ---- Tier 2.2 MEF (upramp) fill with AEF ----
mef_upramp_raw = read_hourly_csv(
    DATA_DIR / "stage1_tier2.2_gridMEF_hourly_predicted_2024.csv"
)[[MEF_UP_COL]]
mef_upramp = mef_upramp_raw.reindex(idx_2024)

missing_mask_up = mef_upramp[MEF_UP_COL].isna()
missing_hours_up = mef_upramp.index[missing_mask_up]
print(f"[MEF-UP FILL] Missing MEF-upramp hours: {int(missing_mask_up.sum())} / {len(mef_upramp)}")
if missing_mask_up.sum() > 0:
    print("[MEF-UP FILL] First 10 missing timestamps:", missing_hours_up[:10].tolist())

mef_upramp_fill_audit = pd.DataFrame({
    "datetime_utc": missing_hours_up,
    "aef_used": aef.loc[missing_mask_up, AEF_COL].values,
    "fill_reason": "MEF upramp missing/undefined -> filled with AEF"
})
mef_upramp.loc[missing_mask_up, MEF_UP_COL] = aef.loc[missing_mask_up, AEF_COL].values

mef_upramp_fill_audit.to_csv(OUT_DIR / "mef_upramp_filled_with_aef_hours_2024.csv", index=False)
print(f"[MEF-UP FILL] Saved audit: {OUT_DIR / 'mef_upramp_filled_with_aef_hours_2024.csv'}")

if mef_upramp.isna().any().any():
    where = mef_upramp[mef_upramp.isna().any(axis=1)].index[:5].tolist()
    raise ValueError(f"MEF_UPRAMP: still has NaNs after fill. Example times: {where}")

# ---- Tier 2.3 MEF (soft) fill with AEF ----
mef_soft_raw = read_hourly_csv(
    DATA_DIR / "stage1_tier2.3_gridMEF_hourly_predicted_2024.csv"
)[[MEF_SOFT_COL]]
mef_soft = mef_soft_raw.reindex(idx_2024)

missing_mask_soft = mef_soft[MEF_SOFT_COL].isna()
missing_hours_soft = mef_soft.index[missing_mask_soft]
print(f"[MEF-SOFT FILL] Missing MEF-soft hours: {int(missing_mask_soft.sum())} / {len(mef_soft)}")
if missing_mask_soft.sum() > 0:
    print("[MEF-SOFT FILL] First 10 missing timestamps:", missing_hours_soft[:10].tolist())

mef_soft_fill_audit = pd.DataFrame({
    "datetime_utc": missing_hours_soft,
    "aef_used": aef.loc[missing_mask_soft, AEF_COL].values,
    "fill_reason": "MEF soft missing/undefined -> filled with AEF"
})
mef_soft.loc[missing_mask_soft, MEF_SOFT_COL] = aef.loc[missing_mask_soft, AEF_COL].values

mef_soft_fill_audit.to_csv(OUT_DIR / "mef_soft_filled_with_aef_hours_2024.csv", index=False)
print(f"[MEF-SOFT FILL] Saved audit: {OUT_DIR / 'mef_soft_filled_with_aef_hours_2024.csv'}")

if mef_soft.isna().any().any():
    where = mef_soft[mef_soft.isna().any(axis=1)].index[:5].tolist()
    raise ValueError(f"MEF_SOFT: still has NaNs after fill. Example times: {where}")


dswrf = assert_full_hourly(
    read_hourly_csv(DATA_DIR / "localRES_PV_dswrf_hourly_GFS_2024.csv")[["dswrf"]],
    "DSWRF"
)
t2m = assert_full_hourly(
    read_hourly_csv(DATA_DIR / "localRES_PV_t2m_hourly_GFS_2024.csv")[["t2m"]],
    "T2M"
)
u100 = assert_full_hourly(
    read_hourly_csv(DATA_DIR / "localRES_WT_u100_hourly_GFS_2024.csv")[["u100"]],
    "U100"
)
v100 = assert_full_hourly(
    read_hourly_csv(DATA_DIR / "localRES_WT_v100_hourly_GFS_2024.csv")[["v100"]],
    "V100"
)

print("All input CSVs validated.")

# =============================================================================
# ---------------- PV MODEL (with EoT; efficiency clip [0,1]) -------
# =============================================================================

# sizing
A_PV             = float(A_PV_unit) * float(installed_pv_kwp)
PV_NAMEPLATE_MWp = float(installed_pv_kwp) / 1000.0

# inputs
G_t   = np.clip(dswrf["dswrf"].astype(float).values, 0.0, None)
T_a_C = (t2m["t2m"].astype(float).values - 273.15)

# radians
phi  = np.deg2rad(float(phi_deg))
beta = np.deg2rad(float(beta_deg))

# day of year
n = idx_2024.dayofyear.values.astype(float)

# declination
delta = np.deg2rad(23.45 * np.sin(np.deg2rad(360.0*(284.0 + n)/365.0)))

# ---- Equation of Time ----
# B = (360/365)*(d - 81)   [deg]
# EoT = 9.87 sin(2B) - 7.53 cos(B) - 1.5 sin(B)   [minutes]
B_deg = (360.0/365.0) * (n - 81.0)
B_rad = np.deg2rad(B_deg)
EoT_min = 9.87*np.sin(2.0*B_rad) - 7.53*np.cos(B_rad) - 1.5*np.sin(B_rad)

# solar time ts: UTC hour + longitude correction + EoT correction
ts = idx_2024.hour.values.astype(float) + (float(lam_deg)/15.0) + (EoT_min/60.0)

# hour angle
omega = np.deg2rad(15.0*(ts - 12.0))

# zenith
cos_theta_z = np.sin(phi)*np.sin(delta) + np.cos(phi)*np.cos(delta)*np.cos(omega)
cos_theta_z = np.clip(cos_theta_z, 0.0, 1.0)
theta_z = np.arccos(cos_theta_z)

# incidence on tilted plane
cos_theta_t = np.sin(delta)*np.sin(phi - beta) + np.cos(delta)*np.cos(phi - beta)*np.cos(omega)
cos_theta_t = np.clip(cos_theta_t, 0.0, 1.0)

# extraterrestrial horizontal proxy
H0_t = float(G_sc)*(1.0 + 0.033*np.cos(np.deg2rad(360.0*n/365.0))) * (
    np.cos(phi)*np.cos(delta)*np.cos(omega) + np.sin(phi)*np.sin(delta)
)
H0_t = np.clip(H0_t, 1e-6, None)

# clearness index
CI = np.clip(G_t / H0_t, 0.0, 2.0)

# diffuse fraction f(CI)
f = np.zeros_like(CI)
m1 = CI < 0.21
m2 = (CI >= 0.21) & (CI <= 0.76)
m3 = CI > 0.76
f[m1] = 0.995 - 0.081*CI[m1]
f[m2] = 0.724 + 2.738*CI[m2] - 8.32*(CI[m2]**2) + 4.967*(CI[m2]**3)
f[m3] = 0.18
f = np.clip(f, 0.0, 1.0)

G_d  = f * G_t
G_bh = np.clip(G_t - G_d, 0.0, None)
F_t = 1.0 - (f**2)

# diffuse on tilt
G_db = (
    G_d
    * 0.5 * (1.0 + np.cos(beta))
    * (1.0 + F_t * (np.sin(beta / 2.0) ** 3))
    * (1.0 + F_t * (cos_theta_t ** 2) * (np.sin(theta_z) ** 3))
)

# beam on tilt
G_bb = np.zeros_like(G_t)
beam_mask = cos_theta_z >= 0.05
G_bb[beam_mask] = G_bh[beam_mask] * (cos_theta_t[beam_mask] / cos_theta_z[beam_mask])
G_bb = np.clip(G_bb, 0.0, None)

# ground reflected
G_rp = float(rho_g) * G_t * (1.0 - np.cos(beta))/2.0
G_beta = np.clip(G_db + G_bb + G_rp, 0.0, None)

# air mass
AM = 1.0 / (np.cos(theta_z) + 0.50572*((96.07995 - np.rad2deg(theta_z))**(-1.6364)))
#AM = np.clip(AM, 1.0,2.0)

# cell temp
theta_cell = T_a_C + float(h_pv) * G_beta
G0, theta0, AM0 = 1000.0, 25.0, 1.5
xG = (G_beta / G0)
xAM = (AM / AM0)
# efficiency model (percent to fraction)
eta_pv_percent = float(p_pv) * (
    (float(q_pv) * xG + (xG ** float(m_pv))) *
    (1.0 + float(r_pv) * (theta_cell / theta0) + float(s_pv) * xAM + (xAM ** float(u_pv)))
)

# convert percent -> fraction and clip(clip is redundant)
eta_pv = np.clip(eta_pv_percent / 100.0, 0.0, ETA_PV_MAX)

# ---- power (paper Eq. 14) ----
P_pv_dc_W = G_beta * eta_pv * A_PV
P_pv_ac_W = P_pv_dc_W * float(eta_inv)

pv_mw_raw = np.maximum(P_pv_ac_W / 1e6, 0.0)

# NOTE: clipping to nameplate is an "available capacity cap" assumption.
# If you want "physics-only" output, do NOT cap here; instead cap only when aggregating system export.
pv_mw = np.minimum(pv_mw_raw, PV_NAMEPLATE_MWp)

# ---------------- WIND MODEL (piecewise: cubic to v_rated, then constant P_r to v_out) ----------------

# 1) wind speed magnitude at reference height
v_ref = np.sqrt(u100["u100"].values**2 + v100["v100"].values**2)

# 2) extrapolate to hub height using power law
v_hub = v_ref * (float(Z_wt) / float(Z_ref))**float(alpha)

# 3) aerodynamic cubic power (proxy) up to rated
P_aero = 0.5 * float(rho_air) * float(A_WT) * float(Cp_eff) * (v_hub**3)

# 4) piecewise turbine electrical power (before efficiencies)
P_raw = np.zeros_like(v_hub, dtype=float)

# Region A: cut-in to rated -> cubic
mask_cubic = (v_hub >= float(v_in)) & (v_hub < float(v_rated))
P_raw[mask_cubic] = P_aero[mask_cubic]

# Region B: rated to cut-out -> constant rated power
mask_rated = (v_hub >= float(v_rated)) & (v_hub < float(v_out))
P_raw[mask_rated] = float(P_r_W)

# Region C: below cut-in or above/equal cut-out -> already 0 by initialization

# 5) apply wind farm efficiency and availability
P_one = P_raw * float(eta_wf) * float(wf_avail)

# 6) aggregate turbines and cap at farm nameplate
wind_mw_raw = np.clip((float(N_WT) * P_one) / 1e6, 0.0, None)
WIND_NAMEPLATE_MW = float(N_WT) * (float(P_r_W) / 1e6)
wind_mw = np.minimum(wind_mw_raw, WIND_NAMEPLATE_MW)


# ---------------- SAVE RES TIMESERIES ----------------
res = pd.DataFrame(
    {"pv_mw": pv_mw, "wind_mw": wind_mw, "res_total_mw": pv_mw + wind_mw},
    index=idx_2024
)
res.to_csv(OUT_DIR / "res_hourly_2024.csv", index_label="datetime_utc")

# =============================================================================
# ---------------- CAPACITIES
# =============================================================================
pd.DataFrame([
    ["P_ELZ_MAX_MW", float(P_ELZ_MAX_MW), "MW"],
    ["P_GRID_MAX_MW", float(P_GRID_MAX_MW), "MW"],
    ["H2_MAX_KGPH", float(H2_MAX_KGPH), "kg/h"],
    ["NH3_CAP_KGPH", float(NH3_CAP_KGPH), "kg/h"],
    ["MEOH_CAP_KGPH", float(MEOH_CAP_KGPH), "kg/h"],
    ["NH3_CRACK_CAP_KGPH", float(NH3_CRACK_CAP_KGPH), "kg/h"],
], columns=["Symbol","Value","Unit"]).to_csv(OUT_DIR / "derived_capacities.csv", index=False)


# =============================================================================
# ---------------- DEMAND PROFILES (EUROSTAT BACKBONE + SMOOTH RAMPS) ---------
# =============================================================================

rng = np.random.default_rng(42)

def ar1(n, phi=0.97, sigma=0.02, rng=None):
    if rng is None:
        rng = np.random.default_rng(0)
    e = rng.normal(0.0, sigma, size=n)
    x = np.zeros(n)
    for i in range(1, n):
        x[i] = phi * x[i-1] + e[i]
    return x

def spikes(n, p=0.002, mag=0.08, rng=None):
    if rng is None:
        rng = np.random.default_rng(0)
    u = rng.random(n)
    s = np.zeros(n)
    hit = u < p
    s[hit] = rng.uniform(0.0, mag, size=hit.sum())
    return s

def enforce_min_mean_max(x, target_min, target_mean, target_max, max_iter=60):
    """
    Enforce exact min/mean/max (within float tolerance) by:
    1) mapping x -> z in [0,1] using normalization
    2) applying z^p (monotone) to tune the mean without changing min/max
    3) scaling to [target_min, target_max]
    """
    x = np.asarray(x, dtype=float)

    xmin = float(np.min(x))
    xmax = float(np.max(x))
    if xmax - xmin < 1e-12:
        z = np.full_like(x, 0.5)
    else:
        z = (x - xmin) / (xmax - xmin)
    z = np.clip(z, 0.0, 1.0)

    lo, hi = 0.05, 20.0
    for _ in range(max_iter):
        mid = 0.5 * (lo + hi)
        y_mid = target_min + (target_max - target_min) * (z ** mid)
        m_mid = float(np.mean(y_mid))
        if m_mid > target_mean:
            lo = mid
        else:
            hi = mid

    p_star = 0.5 * (lo + hi)
    y = target_min + (target_max - target_min) * (z ** p_star)

    i_min = int(np.argmin(y))
    i_max = int(np.argmax(y))
    y[i_min] = target_min
    y[i_max] = target_max

    mean_err = target_mean - float(np.mean(y))
    y = np.clip(y + mean_err, target_min, target_max)
    y[i_min] = target_min
    y[i_max] = target_max

    return y

def block_mean_repeat(x, block=3):
    """
    Enforce 'ramps' by removing hour-to-hour jitter:
    - average each block of 'block' hours
    - repeat that average across the block
    2024 has 8784 hours; 8784 % 3 == 0 so block=3 is safe.
    """
    x = np.asarray(x, dtype=float)
    n = len(x)
    if n % block != 0:
        # fallback: trim last remainder (should not happen for 2024 with block=3)
        n2 = n - (n % block)
        x = x[:n2]
        n = n2
    xb = x.reshape(-1, block).mean(axis=1)
    return np.repeat(xb, block)

# ---------------- EUROSTAT NORMALIZED BACKBONES (hard-coded) ----------------
# Extracted from:
#  - csv_eurostat-passengers port.csv (quarterly, Spain, 2024)
#  - csv_eurostat-gas consumbtion inland.csv (monthly, Spain, 2024)
# Normalization: mean = 1.0, then damped towards 1.0 to avoid extreme swings.

# Q1..Q4 multipliers for NH3 (port passengers), damped
NH3_PORT_Q_MULT_2024 = np.array([0.822047, 0.896977, 1.340477, 0.940499], dtype=float)

# Jan..Dec multipliers for H2 (inland gas consumption), damped
H2_GAS_M_MULT_2024 = np.array([
    0.985216, 0.984325, 1.025127, 1.014627, 0.948715, 0.952309,
    0.952821, 0.958588, 0.993616, 1.021583, 1.076420, 1.086653
], dtype=float)

# Hourly backbones
months = idx_2024.month.values.astype(int)
quarters = ((months - 1) // 3).astype(int)  # 0..3
nh3_backbone = NH3_PORT_Q_MULT_2024[quarters]
nh3_backbone = pd.Series(nh3_backbone, index=idx_2024).rolling(24*30, min_periods=1, center=True).mean().values

h2_backbone  = H2_GAS_M_MULT_2024[months - 1]
meoh_backbone = np.ones(len(idx_2024), dtype=float)  # explicitly flat (per your instruction)

# ---------------- base realism drivers  -------
wfac = np.where(idx_2024.weekday.values >= 5, WEEKEND_FACTOR, 1.0)

Ta_C = pd.Series((t2m["t2m"].astype(float).values - 273.15), index=idx_2024)
Ta_C_smooth = Ta_C.rolling(24*7, center=True, min_periods=1).mean()
T_ref = float(Ta_C_smooth.quantile(0.75))
H = np.clip((T_ref - Ta_C_smooth) / 15.0, 0.0, 1.0)

uplift_max = 0.12
temp_uplift = (1.0 + uplift_max * H.values)

t = np.arange(len(idx_2024))
daily = np.sin(2*np.pi*t/24.0)

shape_nh3  = nh3_backbone * wfac * temp_uplift * (1.0 + float(A_DAY_NH3)*daily)
shape_h2   = h2_backbone  * wfac * temp_uplift * (1.0 + float(A_DAY_H2)*daily)
shape_meoh = meoh_backbone * wfac * temp_uplift * (1.0 + float(A_DAY_MEOH)*daily)

# ---------------- low-frequency stochasticity + smoothing to enforce ramps ----
# Noise is added, then block-smoothed (3h) to avoid hour-to-hour “teleporting”.
eps_nh3  = ar1(len(idx_2024), phi=0.990, sigma=0.008, rng=rng) + spikes(len(idx_2024), p=0.0015, mag=0.05, rng=rng)
eps_h2   = ar1(len(idx_2024), phi=0.985, sigma=0.010, rng=rng) + spikes(len(idx_2024), p=0.0015, mag=0.06, rng=rng)
eps_meoh = ar1(len(idx_2024), phi=0.985, sigma=0.010, rng=rng) + spikes(len(idx_2024), p=0.0015, mag=0.06, rng=rng)

nh3_raw = shape_nh3  * (1.0 + eps_nh3)
h2_raw  = shape_h2   * (1.0 + eps_h2)
me_raw  = shape_meoh * (1.0 + eps_meoh)

# Enforce “ramps” by 3-hour blocking + light rolling smoothing
BLOCK_H = 3

nh3_s = pd.Series(block_mean_repeat(nh3_raw, BLOCK_H), index=idx_2024).rolling(BLOCK_H, min_periods=1).mean().values
h2_s  = pd.Series(block_mean_repeat(h2_raw,  BLOCK_H), index=idx_2024).rolling(BLOCK_H, min_periods=1).mean().values
me_s  = pd.Series(block_mean_repeat(me_raw,  BLOCK_H), index=idx_2024).rolling(BLOCK_H, min_periods=1).mean().values

# ---------------- FINAL TARGETS (kg/h) --------------------------------------
H2_MIN,   H2_MEAN,   H2_MAX   = 700,  1400, 2000
NH3_MIN,  NH3_MEAN,  NH3_MAX  = 800,  1500, 3000
MEOH_MIN, MEOH_MEAN, MEOH_MAX = 600,  1200, 1800

h2  = enforce_min_mean_max(h2_s,  H2_MIN,  H2_MEAN,  H2_MAX)
nh3 = enforce_min_mean_max(nh3_s, NH3_MIN, NH3_MEAN, NH3_MAX)
me  = enforce_min_mean_max(me_s,  MEOH_MIN, MEOH_MEAN, MEOH_MAX)

dem = pd.DataFrame({
    "nh3_demand_kgph": nh3,
    "h2_demand_kgph":  h2,
    "meoh_demand_kgph": me,
}, index=idx_2024)

dem.to_csv(OUT_DIR / "demand_hourly_2024.csv", index_label="datetime_utc")


def _chk(col, mn, me, mx):
    v = dem[col].values.astype(float)
    if not (abs(v.min()-mn) < 1e-6 and abs(v.mean()-me) < 1e-3 and abs(v.max()-mx) < 1e-6):
        raise ValueError(f"{col}: got min/mean/max = {v.min():.6f}/{v.mean():.6f}/{v.max():.6f}, expected {mn}/{me}/{mx}")

_chk("h2_demand_kgph",   H2_MIN,   H2_MEAN,   H2_MAX)
_chk("nh3_demand_kgph",  NH3_MIN,  NH3_MEAN,  NH3_MAX)
_chk("meoh_demand_kgph", MEOH_MIN, MEOH_MEAN, MEOH_MAX)



# ---------------- SAVE SERIES ----------------
price.to_csv(OUT_DIR / "grid_price_hourly_2024.csv", index_label="datetime_utc")
aef.to_csv(OUT_DIR / "grid_aef_hourly_2024.csv", index_label="datetime_utc")
mef_upramp.to_csv(OUT_DIR / "grid_mef_upramp_hourly_2024.csv", index_label="datetime_utc")
mef_soft.to_csv(OUT_DIR / "grid_mef_soft_hourly_2024.csv", index_label="datetime_utc")

# ---------------- PLOTS (PNG) ----------------
# ---------------- PLOTS (PNG) ----------------

def save_lineplot_png_resampled(df: pd.DataFrame, col: str, rule: str, title: str, ylabel: str, outpath: Path):
    s = df[col].resample(rule).mean()
    plt.figure(figsize=(14,4))
    plt.plot(s.index, s.values, label=f"{col} ({rule} mean)")
    plt.title(title)
    plt.ylabel(ylabel)
    plt.legend()
    plt.tight_layout()
    plt.savefig(outpath, dpi=150)
    plt.close()

def save_multi_lineplot_png_resampled(df: pd.DataFrame, cols, rule: str, title: str, ylabel: str, outpath: Path):
    df_r = df[cols].resample(rule).mean()
    plt.figure(figsize=(14,4))
    for c in cols:
        plt.plot(df_r.index, df_r[c].values, label=f"{c} ({rule} mean)")
    plt.title(title)
    plt.ylabel(ylabel)
    plt.legend()
    plt.tight_layout()
    plt.savefig(outpath, dpi=150)
    plt.close()

# ---- base series (hourly) ----
# res: pv_mw, wind_mw, res_total_mw  (already hourly)
# dem: h2_demand_kgph, nh3_demand_kgph, meoh_demand_kgph (already hourly)

# ---- PV diagnostic series (hourly) ----
pv_diag = pd.DataFrame({
    "dswrf_Wm2": dswrf["dswrf"].astype(float),
    "G_beta_Wm2": pd.Series(G_beta, index=idx_2024),
    "eta_pv_frac": pd.Series(eta_pv, index=idx_2024),
    "theta_cell_C": pd.Series(theta_cell, index=idx_2024),
    "AM": pd.Series(AM, index=idx_2024),
}, index=idx_2024)

# =========================
# 1) RES plots (hourly)
# =========================
save_lineplot_png(res, ["pv_mw"], "Available PV generation (full year, hourly)", "MW", PLOT_DIR / "pv_hourly_full_year.png")
save_lineplot_png(res, ["wind_mw"], "Available wind generation (full year, hourly)", "MW", PLOT_DIR / "wind_hourly_full_year.png")
save_lineplot_png(res, ["res_total_mw"], "Available total RES generation (full year, hourly)", "MW", PLOT_DIR / "res_total_hourly_full_year.png")


save_lineplot_png(res, ["pv_mw", "wind_mw", "res_total_mw"], "Available RES generation (full year, hourly)", "MW", PLOT_DIR / "res_all_hourly_full_year.png")

# 2-week zoom (hourly)
zoom = slice(res.index[0], res.index[0] + pd.Timedelta(days=14))
save_lineplot_png(res.loc[zoom], ["pv_mw"], "Available PV generation (first 2 weeks, hourly)", "MW", PLOT_DIR / "pv_hourly_first_2_weeks.png")
save_lineplot_png(res.loc[zoom], ["wind_mw"], "Available wind generation (first 2 weeks, hourly)", "MW", PLOT_DIR / "wind_hourly_first_2_weeks.png")
save_lineplot_png(res.loc[zoom], ["res_total_mw"], "Available total RES generation (first 2 weeks, hourly)", "MW", PLOT_DIR / "res_total_hourly_first_2_weeks.png")
save_lineplot_png(res.loc[zoom], ["pv_mw", "wind_mw", "res_total_mw"], "Available RES generation (first 2 weeks, hourly)", "MW", PLOT_DIR / "res_all_hourly_first_2_weeks.png")

# =========================
# 2) RES plots (daily mean)
# =========================
for c, fname, title in [
    ("pv_mw", "pv_daily_mean.png", "PV generation (daily mean)"),
    ("wind_mw", "wind_daily_mean.png", "Wind generation (daily mean)"),
    ("res_total_mw", "res_total_daily_mean.png", "Total RES generation (daily mean)"),
]:
    save_lineplot_png_resampled(res, c, "D", title, "MW", PLOT_DIR / fname)


save_multi_lineplot_png_resampled(
    res, ["pv_mw", "wind_mw", "res_total_mw"], "D",
    "RES generation (daily mean)", "MW",
    PLOT_DIR / "res_all_daily_mean.png"
)

# =========================
# 3) RES plots (monthly mean)
# =========================
for c, fname, title in [
    ("pv_mw", "pv_monthly_mean.png", "PV generation (monthly mean)"),
    ("wind_mw", "wind_monthly_mean.png", "Wind generation (monthly mean)"),
    ("res_total_mw", "res_total_monthly_mean.png", "Total RES generation (monthly mean)"),
]:
    save_lineplot_png_resampled(res, c, "MS", title, "MW", PLOT_DIR / fname)


save_multi_lineplot_png_resampled(
    res, ["pv_mw", "wind_mw", "res_total_mw"], "MS",
    "RES generation (monthly mean)", "MW",
    PLOT_DIR / "res_all_monthly_mean.png"
)

# =========================
# 4) Demand plots (hourly full-year) — separate per product + combined
# =========================
save_lineplot_png(dem, ["h2_demand_kgph"],  "H2 demand (full year, hourly)",  "kg/h", PLOT_DIR / "demand_h2_hourly_full_year.png")
save_lineplot_png(dem, ["nh3_demand_kgph"], "NH3 demand (full year, hourly)", "kg/h", PLOT_DIR / "demand_nh3_hourly_full_year.png")
save_lineplot_png(dem, ["meoh_demand_kgph"],"MeOH demand (full year, hourly)","kg/h", PLOT_DIR / "demand_meoh_hourly_full_year.png")


save_lineplot_png(
    dem, ["h2_demand_kgph", "nh3_demand_kgph", "meoh_demand_kgph"],
    "Demand profiles (full year, hourly)", "kg/h",
    PLOT_DIR / "demand_all_hourly_full_year.png"
)

# =========================
# 5) Demand plots (daily mean)
# =========================
for c, fname, title in [
    ("h2_demand_kgph",  "demand_h2_daily_mean.png",  "H2 demand (daily mean)"),
    ("nh3_demand_kgph", "demand_nh3_daily_mean.png", "NH3 demand (daily mean)"),
    ("meoh_demand_kgph","demand_meoh_daily_mean.png","MeOH demand (daily mean)"),
]:
    save_lineplot_png_resampled(dem, c, "D", title, "kg/h", PLOT_DIR / fname)

save_multi_lineplot_png_resampled(
    dem, ["h2_demand_kgph", "nh3_demand_kgph", "meoh_demand_kgph"], "D",
    "Demand profiles (daily mean)", "kg/h",
    PLOT_DIR / "demand_all_daily_mean.png"
)

# =========================
# 6) Demand plots (monthly mean)
# =========================
for c, fname, title in [
    ("h2_demand_kgph",  "demand_h2_monthly_mean.png",  "H2 demand (monthly mean)"),
    ("nh3_demand_kgph", "demand_nh3_monthly_mean.png", "NH3 demand (monthly mean)"),
    ("meoh_demand_kgph","demand_meoh_monthly_mean.png","MeOH demand (monthly mean)"),
]:
    save_lineplot_png_resampled(dem, c, "MS", title, "kg/h", PLOT_DIR / fname)

save_multi_lineplot_png_resampled(
    dem, ["h2_demand_kgph", "nh3_demand_kgph", "meoh_demand_kgph"], "MS",
    "Demand profiles (monthly mean)", "kg/h",
    PLOT_DIR / "demand_all_monthly_mean.png"
)

# =========================
# 7) PV model sanity plots (hourly / daily / monthly)
# =========================
# Hourly full-year
for c, fname, ylabel, title in [
    ("dswrf_Wm2",     "pv_diag_dswrf_hourly_full_year.png", "W/m²", "DSWRF input (full year, hourly)"),
    ("G_beta_Wm2",    "pv_diag_Gbeta_hourly_full_year.png", "W/m²", "Plane-of-array irradiance G_beta (full year, hourly)"),
    ("eta_pv_frac",   "pv_diag_eta_hourly_full_year.png",   "-",    "PV efficiency eta_pv (fraction, full year, hourly)"),
    ("theta_cell_C",  "pv_diag_tcell_hourly_full_year.png", "°C",   "Cell temperature theta_cell (full year, hourly)"),
    ("AM",            "pv_diag_AM_hourly_full_year.png",    "-",    "Air mass AM (full year, hourly)"),
]:
    save_lineplot_png(pv_diag, [c], title, ylabel, PLOT_DIR / fname)

# Daily mean
for c, fname, ylabel, title in [
    ("dswrf_Wm2",     "pv_diag_dswrf_daily_mean.png", "W/m²", "DSWRF input (daily mean)"),
    ("G_beta_Wm2",    "pv_diag_Gbeta_daily_mean.png", "W/m²", "G_beta (daily mean)"),
    ("eta_pv_frac",   "pv_diag_eta_daily_mean.png",   "-",    "eta_pv (daily mean)"),
    ("theta_cell_C",  "pv_diag_tcell_daily_mean.png", "°C",   "theta_cell (daily mean)"),
    ("AM",            "pv_diag_AM_daily_mean.png",    "-",    "AM (daily mean)"),
]:
    save_lineplot_png_resampled(pv_diag, c, "D", title, ylabel, PLOT_DIR / fname)

# Monthly mean
for c, fname, ylabel, title in [
    ("dswrf_Wm2",     "pv_diag_dswrf_monthly_mean.png", "W/m²", "DSWRF input (monthly mean)"),
    ("G_beta_Wm2",    "pv_diag_Gbeta_monthly_mean.png", "W/m²", "G_beta (monthly mean)"),
    ("eta_pv_frac",   "pv_diag_eta_monthly_mean.png",   "-",    "eta_pv (monthly mean)"),
    ("theta_cell_C",  "pv_diag_tcell_monthly_mean.png", "°C",   "theta_cell (monthly mean)"),
    ("AM",            "pv_diag_AM_monthly_mean.png",    "-",    "AM (monthly mean)"),
]:
    save_lineplot_png_resampled(pv_diag, c, "MS", title, ylabel, PLOT_DIR / fname)


# ---------------- PARAMS JSON (for scenario cells) ----------------
# Note: removed SCALARS_FROM_UTILITY_INFO_FILE because Excel read is removed.
params = {
    "YEAR": YEAR,
    "N_DAYS_TO_RUN": N_DAYS_TO_RUN,
    "START_DAY_OF_YEAR": START_DAY_OF_YEAR,
    "LAMBDA_EMIS": LAMBDA_EMIS,
    "KEY_CAPS": {
        "P_ELZ_MAX_MW": float(P_ELZ_MAX_MW),
        "P_GRID_MAX_MW": float(P_GRID_MAX_MW),
        "H2_MAX_KGPH": float(H2_MAX_KGPH),
        "NH3_CAP_KGPH": float(NH3_CAP_KGPH),
        "MEOH_CAP_KGPH": float(MEOH_CAP_KGPH),
        "NH3_CRACK_CAP_KGPH": float(NH3_CRACK_CAP_KGPH),
    },
    "PV": {
        "A_PV_m2": float(A_PV),
        "PV_NAMEPLATE_MWp": float(PV_NAMEPLATE_MWp),
    },
    "WIND": {
        "N_WT": int(float(N_WT)),
        "WIND_NAMEPLATE_MW": float(WIND_NAMEPLATE_MW),
    },
    "FILES_WRITTEN": {
            "res_hourly_2024": str(OUT_DIR / "res_hourly_2024.csv"),
            "demand_hourly_2024": str(OUT_DIR / "demand_hourly_2024.csv"),
            "grid_price_hourly_2024": str(OUT_DIR / "grid_price_hourly_2024.csv"),
            "grid_aef_hourly_2024": str(OUT_DIR / "grid_aef_hourly_2024.csv"),
            "grid_mef_upramp_hourly_2024": str(OUT_DIR / "grid_mef_upramp_hourly_2024.csv"),
            "grid_mef_soft_hourly_2024": str(OUT_DIR / "grid_mef_soft_hourly_2024.csv"),
            "mef_upramp_fill_audit": str(OUT_DIR / "mef_upramp_filled_with_aef_hours_2024.csv"),
            "mef_soft_fill_audit": str(OUT_DIR / "mef_soft_filled_with_aef_hours_2024.csv"),
            "plots_dir": str(PLOT_DIR),
    }
}

with open(OUT_DIR / "params_scenario_all.json", "w") as f:
    json.dump(params, f, indent=2)

print(f"[JSON] Saved: {OUT_DIR / 'params_scenario_all.json'}")

# ---------------- SANITY PRINTS ----------------
print("\n========== Utility V3 summary ==========")
print("Saved to:", OUT_DIR)
print(" - res_hourly_2024.csv")
print(" - demand_hourly_2024.csv")
print(" - grid_price_hourly_2024.csv")
print(" - grid_aef_hourly_2024.csv")
print(" - grid_mef_upramp_hourly_2024.csv")
print(" - grid_mef_soft_hourly_2024.csv")
print(" - derived_capacities.csv")
print(" - params_scenario_all.json")
print(" - plots/*.png")

print("\nRES stats (MW):")
print(res[["pv_mw","wind_mw","res_total_mw"]].describe().loc[["min","mean","max"]])

tmp = pd.DataFrame({"G_t": G_t, "G_beta": G_beta, "eta_pv": eta_pv}, index=idx_2024)
print("\nPV intermediate sanity:")
print(tmp.describe().loc[["min","mean","max"]])

print("\nDemand stats (kg/h):")
print(dem.describe().loc[["min","mean","max"]])
print("\nScale check:")
print("PV nameplate (MWp):", PV_NAMEPLATE_MWp, " PV max after cap (MW):", float(np.max(pv_mw)))
print("Wind nameplate (MW):", WIND_NAMEPLATE_MW, " Wind max after cap (MW):", float(np.max(wind_mw)))
print("Electrolyzer max (MW):", float(P_ELZ_MAX_MW))
print("Grid import cap (MW):", float(P_GRID_MAX_MW))
clip_frac = np.mean(pv_mw_raw >= PV_NAMEPLATE_MWp)
print("Fraction of hours clipped:", clip_frac)
cf = np.mean(pv_mw) / PV_NAMEPLATE_MWp
print("PV capacity factor:", cf)
print("A_PV (m2):", A_PV)
print("A_PV_unit (m2/kWp):", A_PV_unit)
print("installed_pv_kwp:", installed_pv_kwp)
print("eta_pv min/mean/max:", float(np.min(eta_pv)), float(np.mean(eta_pv)), float(np.max(eta_pv)))
print("G_beta min/mean/max:", float(np.min(G_beta)), float(np.mean(G_beta)), float(np.max(G_beta)))
print("PV nameplate (MWp):", PV_NAMEPLATE_MWp)
print("pv_mw max:", float(np.max(pv_mw)))
print("pv_mw_raw max:", float(np.max(pv_mw_raw)))

import zipfile

ZIP_PATH = PLOT_DIR / "all_plots.zip"

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for png in PLOT_DIR.glob("*.png"):
        zf.write(png, arcname=png.name)

print(f"Plots zipped to: {ZIP_PATH}")

print("\nDone.")


# 5. Rolling horizon optimization

This section keeps one final code block per model class:

- **M0:** cost baseline.
- **M1:** signal specific emission reference
- **M2:** cost minimization under the daily emission cap.

For M1 and M2, change the emission mode near the top of the cell when running AEF, UpRamp MEF, or Soft MEF cases.


## 5.1 M0 cost baseline model


In [ ]:
# ===================== M0 =====================
# Assumes utility/precompute cell has already run and wrote /content/data/model_ready/*.csv and params_scenario_all.json

MODEL_TAG = "M0"
FORMULATION_TAG = "M0"
SCENARIO_TAG = "S1"# we only used one scenario and demand in the work
DEMAND_TAG = "D1"
GREEN_TAG = "S1_green_split_with_rfnbo_hourly"

import os, json, math, zipfile, shutil, time, re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gurobipy as gp
from gurobipy import GRB

DATA_DIR = Path("/content/data")
MR = DATA_DIR / "model_ready"

# Required model-ready files (from utility cell)
RES_CSV         = MR / "res_hourly_2024.csv"
DEM_CSV         = MR / "demand_hourly_2024.csv"
PRICE_CSV       = MR / "grid_price_hourly_2024.csv"
AEF_CSV         = MR / "grid_aef_hourly_2024.csv"
MEF_UPRAMP_CSV  = MR / "grid_mef_upramp_hourly_2024.csv"
MEF_SOFT_CSV    = MR / "grid_mef_soft_hourly_2024.csv"
PARAMS_JSON     = MR / "params_scenario_all.json"

# column names from utility cell
AEF_COL        = "grid_aef_predicted_kg_per_kwh"
MEF_UPRAMP_COL = "grid_mef_upramp_pred_kg_per_kwh"
MEF_SOFT_COL   = "grid_mef_soft_kg_per_kwh"

# Run controls
N_DAYS_TO_RUN     = 366
START_DAY_OF_YEAR = 1
MIPGAP            = 0.01
TIME_LIMIT_S      = 350
OUTPUTFLAG        = 1

for lic_path in [Path("/content/data/gurobi.lic")]:
    if lic_path.exists():
        os.environ["GRB_LICENSE_FILE"] = str(lic_path)
        break

def build_run_id():
    stamp = pd.Timestamp.utcnow().strftime("%Y%m%dT%H%M%SZ")
    return f"{MODEL_TAG}__{SCENARIO_TAG}__{DEMAND_TAG}__start{START_DAY_OF_YEAR:03d}__ndays{N_DAYS_TO_RUN:03d}__{stamp}"

RUN_ID = build_run_id()

OUT_BASE  = DATA_DIR / "stage2_runs"
OUT_RUN   = OUT_BASE / RUN_ID
OUT_DAILY = OUT_RUN / "daily"
OUT_POST  = OUT_RUN / "post"
OUT_PLOTS = OUT_RUN / "plots"
OUT_RUN.mkdir(parents=True, exist_ok=True)
OUT_DAILY.mkdir(parents=True, exist_ok=True)
OUT_POST.mkdir(parents=True, exist_ok=True)
OUT_PLOTS.mkdir(parents=True, exist_ok=True)

def read_hourly_csv(path: Path, time_col="datetime_utc") -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    df = pd.read_csv(path)
    if time_col not in df.columns:
        raise ValueError(f"{path.name}: missing '{time_col}'. Found: {list(df.columns)}")
    df[time_col] = pd.to_datetime(df[time_col], utc=True)
    df = df.sort_values(time_col).set_index(time_col)
    df = df.tz_convert("UTC") if df.index.tz is not None else df.tz_localize("UTC")
    return df

res        = read_hourly_csv(RES_CSV)[["pv_mw","wind_mw","res_total_mw"]]
dem        = read_hourly_csv(DEM_CSV)[["h2_demand_kgph","nh3_demand_kgph","meoh_demand_kgph"]]
price      = read_hourly_csv(PRICE_CSV)[["da_elec_price_EURperMWh"]]
aef        = read_hourly_csv(AEF_CSV)[[AEF_COL]]
mef_upramp = read_hourly_csv(MEF_UPRAMP_CSV)[[MEF_UPRAMP_COL]]
mef_soft   = read_hourly_csv(MEF_SOFT_CSV)[[MEF_SOFT_COL]]

with open(PARAMS_JSON, "r") as f:
    params = json.load(f)

idx_2024 = pd.date_range("2024-01-01 00:00:00+00:00", "2024-12-31 23:00:00+00:00", freq="h", tz="UTC")

def assert_full_hourly(df: pd.DataFrame, name: str) -> pd.DataFrame:
    if df.index.duplicated().any():
        dup = df.index[df.index.duplicated()][0]
        raise ValueError(f"{name}: duplicated timestamp example: {dup}")
    missing = idx_2024.difference(df.index)
    extra   = df.index.difference(idx_2024)
    if len(missing) > 0:
        raise ValueError(f"{name}: missing {len(missing)} hours (first 5: {missing[:5].tolist()})")
    if len(extra) > 0:
        raise ValueError(f"{name}: extra {len(extra)} hours outside 2024 (first 5: {extra[:5].tolist()})")

    df2 = df.reindex(idx_2024)
    df2.index.name = "datetime_utc"

    if df2.isna().any().any():
        cols = df2.columns[df2.isna().any()].tolist()
        where = df2[df2[cols].isna().any(axis=1)].index[:5].tolist()
        raise ValueError(f"{name}: NaNs after reindex in cols {cols}; example {where}")
    return df2

res        = assert_full_hourly(res, "res")
dem        = assert_full_hourly(dem, "dem")
price      = assert_full_hourly(price, "price")
aef        = assert_full_hourly(aef, "aef")
mef_upramp = assert_full_hourly(mef_upramp, "mef_upramp")
mef_soft   = assert_full_hourly(mef_soft, "mef_soft")

# ---------------- Scalars ----------------
P_GRID_MAX_MW = 295.58

P_BAT_MAX_MW = 50.0
E_BAT_MAX_MWH = 100.32
ETA_CH = 0.964
ETA_DIS = 0.964
SOC_MIN = 0.15
SOC_MAX = 0.95
SOC_INIT = 0.5
#elz
P_ELZ_MIN = 8 #mw
P_ELZ_MAX = 52.25
H_ELZ_MAX = 914.39161 #kg/h
A_elz = -0.075045052
B_elz = 21.73040904
C_elz = -16.14458074
N_stack = 4 #four modules
omega_H2O = 0.015     # m3/kgH2
c_H2O = 2.24          # EUR/m3
C_su_elz  = 1332     # for each module, EUR/staRt
P_ELZ_MAX_TOT = N_stack * P_ELZ_MAX
H_ELZ_MAX_TOT = N_stack * H_ELZ_MAX
# Electrolyzer linear under-estimator
H_min = A_elz*(P_ELZ_MIN**2) + B_elz*P_ELZ_MIN + C_elz
H_max = A_elz*(P_ELZ_MAX**2) + B_elz*P_ELZ_MAX + C_elz
Q1 = (H_max - H_min) / (P_ELZ_MAX - P_ELZ_MIN)
Q0 = H_min - Q1 * P_ELZ_MIN
#nh3
Q_NH3_CAP = 13472.04
f_NH3_min = 0.61
RR_NH3 = 0.2
mH2_to_NH3 = 0.18
mN2_to_NH3 = 0.87
SEC_HB = 0.36
SEC_ASU = 0.068
C_su_NH3 = 2397.46 # EUR/start
de_NH3 = 0.00519
#meoh
Q_MEOH_CAP = 5818.856
f_MEOH_min = 0.1
R_MEOH = 0.5 * Q_MEOH_CAP
mH2_to_MEOH = 0.22
SEC_MEOH = 161.75/1000.0
mCO2_to_MEOH = 1.59
c_CO2 = 0.2
C_su_MEOH = 1159.37  # EUR/start
de_MEOH = 0.08
#crk
Q_CRK_CAP = 6062.416
f_CRK_min = 0.05
R_CRK = 0.5 * Q_CRK_CAP
kH2_out = 0.12024
kNH3_rec = 0.11848
kH2_fuel = 0.035294
SEC_CRK_E = 0.524667
kSteam = 0.941176
eSteam = 0.75
omega_steam = 0.001

C_su_CRK  = 1000.0      # EUR/start
#tanks+EOD
H_TANK = 12.0
ALPHA_TANK_FILL = 0.15

S_H2_CAP   = H_TANK * H_ELZ_MAX_TOT #kg
S_NH3_CAP  = H_TANK * Q_NH3_CAP
S_MEOH_CAP = H_TANK * Q_MEOH_CAP

S_H2_INIT   = ALPHA_TANK_FILL * S_H2_CAP
S_NH3_INIT  = ALPHA_TANK_FILL * S_NH3_CAP
S_MEOH_INIT = ALPHA_TANK_FILL * S_MEOH_CAP

def day_to_time_slice(day_of_year: int):
    start_ts = idx_2024[0] + pd.Timedelta(days=day_of_year-1)
    end_ts = start_ts + pd.Timedelta(hours=24)
    return pd.date_range(start_ts, end_ts - pd.Timedelta(hours=1), freq="h", tz="UTC")

def save_lineplot(df, xcol, ycols, title, ylabel, outpath):
    plt.figure(figsize=(14, 4))
    x = df[xcol] if xcol in df.columns else df.index
    for c in ycols:
        plt.plot(x, df[c], label=c)
    plt.title(title)
    plt.ylabel(ylabel)
    plt.legend()
    plt.tight_layout()
    plt.savefig(outpath, dpi=150)
    plt.close()

def save_stacked_bar_by_day(df, date_col, components, title, ylabel, outpath):
    x = pd.to_datetime(df[date_col])
    plt.figure(figsize=(14,5))
    bottom = np.zeros(len(df), dtype=float)
    for col, label in components:
        vals = df[col].values.astype(float)
        plt.bar(x, vals, bottom=bottom, label=label)
        bottom += vals
    plt.title(title)
    plt.ylabel(ylabel)
    plt.legend()
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(outpath, dpi=150)
    plt.close()

def safe_get_attr(model, attr, default=np.nan):
    try:
        return getattr(model, attr)
    except Exception:
        return default

def status_name(code: int) -> str:
    mp = {
        GRB.LOADED: "LOADED",
        GRB.OPTIMAL: "OPTIMAL",
        GRB.INFEASIBLE: "INFEASIBLE",
        GRB.INF_OR_UNBD: "INF_OR_UNBD",
        GRB.UNBOUNDED: "UNBOUNDED",
        GRB.CUTOFF: "CUTOFF",
        GRB.ITERATION_LIMIT: "ITERATION_LIMIT",
        GRB.NODE_LIMIT: "NODE_LIMIT",
        GRB.TIME_LIMIT: "TIME_LIMIT",
        GRB.SOLUTION_LIMIT: "SOLUTION_LIMIT",
        GRB.INTERRUPTED: "INTERRUPTED",
        GRB.NUMERIC: "NUMERIC",
        GRB.SUBOPTIMAL: "SUBOPTIMAL",
        GRB.INPROGRESS: "INPROGRESS",
        GRB.USER_OBJ_LIMIT: "USER_OBJ_LIMIT",
    }
    return mp.get(int(code), f"STATUS_{int(code)}")

def get_objpass_metrics(model, obj_idx: int):
    out = {
        "obj_idx": int(obj_idx),
        "pass_status": np.nan,
        "pass_status_name": "NA",
        "pass_runtime": np.nan,
        "pass_objval": np.nan,
        "pass_objbound": np.nan,
        "pass_mipgap": np.nan,
    }
    try:
        prev_objnum  = getattr(model.Params, "ObjNumber", None)
        prev_passnum = getattr(model.Params, "ObjPassNumber", None)

        model.Params.ObjNumber = int(obj_idx)
        model.Params.ObjPassNumber = -1

        st = safe_get_attr(model, "ObjPassNStatus", np.nan)
        out["pass_status"] = float(st) if not np.isnan(st) else np.nan
        out["pass_status_name"] = status_name(int(st)) if not np.isnan(st) else "NA"
        out["pass_runtime"]  = float(safe_get_attr(model, "ObjPassNRuntime", np.nan))
        out["pass_objval"]   = float(safe_get_attr(model, "ObjPassNObjVal", np.nan))
        out["pass_objbound"] = float(safe_get_attr(model, "ObjPassNObjBound", np.nan))
        out["pass_mipgap"]   = float(safe_get_attr(model, "ObjPassNMipGap", np.nan))

        if prev_objnum is not None:
            model.Params.ObjNumber = prev_objnum
        if prev_passnum is not None:
            model.Params.ObjPassNumber = prev_passnum
    except Exception:
        pass
    return out

carry = {
    "soc": SOC_INIT,
    "S_H2_g": 0.5*S_H2_INIT,
    "S_H2_ng": 0.5*S_H2_INIT,
    "S_NH3_g": 0.5*S_NH3_INIT,
    "S_NH3_ng": 0.5*S_NH3_INIT,
    "S_MEOH_g": 0.5*S_MEOH_INIT,
    "S_MEOH_ng": 0.5*S_MEOH_INIT,
    "y_elz_prev_mod": [0]*N_stack,
    "y_nh3_prev": 0,
    "y_meoh_prev": 0,
    "y_crk_prev": 0,
    "Q_nh3_prev": 0.0,
    "Q_meoh_prev": 0.0,
    "Q_crk_prev": 0.0,
}

daily_rows = []
solver_rows = []
state_rows = []
diag_rows = []
all_hourly_frames = []
# run loop
t0_global = time.time()

for d in range(START_DAY_OF_YEAR, START_DAY_OF_YEAR + N_DAYS_TO_RUN):
    day_hours = day_to_time_slice(d)

    if day_hours[0] < idx_2024[0] or day_hours[-1] > idx_2024[-1]:
        break

    day_dir = OUT_DAILY / f"day_{d:03d}"
    day_dir.mkdir(parents=True, exist_ok=True)

    pv_av    = res.loc[day_hours, "pv_mw"].values.astype(float)
    wd_av    = res.loc[day_hours, "wind_mw"].values.astype(float)
    pr_eur   = price.loc[day_hours, "da_elec_price_EURperMWh"].values.astype(float)
    dem_h2   = dem.loc[day_hours, "h2_demand_kgph"].values.astype(float)
    dem_nh3  = dem.loc[day_hours, "nh3_demand_kgph"].values.astype(float)
    dem_meoh = dem.loc[day_hours, "meoh_demand_kgph"].values.astype(float)

    aef_k        = aef.loc[day_hours, AEF_COL].values.astype(float)
    mef_upramp_k = mef_upramp.loc[day_hours, MEF_UPRAMP_COL].values.astype(float)
    mef_soft_k   = mef_soft.loc[day_hours, MEF_SOFT_COL].values.astype(float)

    m = gp.Model(f"{MODEL_TAG}_day{d:03d}")
    m.Params.MIPGap = MIPGAP
    m.Params.TimeLimit = TIME_LIMIT_S
    m.Params.OutputFlag = OUTPUTFLAG
    m.Params.NumericFocus = 1
    m.Params.MultiObjMethod = 2
    m.Params.MultiObjPre = 1

    T = range(24) #flows
    T2 = range(25) #states

    S_sources = ["pv", "wd", "grid", "bat"]
    K_sinks = ["elz", "NH3", "MeOH", "crk", "bch"]
    S_res = ["pv", "wd"] # S1: battery discharge does not counted green (accounting only)
    # Decision variables
    P = m.addVars(S_sources, K_sinks, T, lb=0.0, name="P_s2k")

    P_pv_cur = m.addVars(T, lb=0.0, name="P_pv_cur")
    P_wd_cur = m.addVars(T, lb=0.0, name="P_wd_cur")

    P_grid = m.addVars(T, lb=0.0, ub=P_GRID_MAX_MW, name="P_grid")

    P_ch  = m.addVars(T, lb=0.0, ub=P_BAT_MAX_MW, name="P_ch")
    P_dis = m.addVars(T, lb=0.0, ub=P_BAT_MAX_MW, name="P_dis")
    SOC   = m.addVars(T2, lb=SOC_MIN, ub=SOC_MAX, name="SOC")

    Pk_g  = m.addVars(K_sinks, T, lb=0.0, name="P_sink_g")
    Pk_ng = m.addVars(K_sinks, T, lb=0.0, name="P_sink_ng")

    I_elz = range(N_stack)

    y_elz_m  = m.addVars(I_elz, T, vtype=GRB.BINARY, name="y_elz_m")
    su_elz_m = m.addVars(I_elz, T, vtype=GRB.BINARY, name="su_elz_m")
    # per-module "color choice": 1 => green-powered, 0 => non-green-powered (grid/bat depending on scenario definition)
    b_elz_g  = m.addVars(I_elz, T, vtype=GRB.BINARY, name="b_elz_g")

    P_elz_m_g   = m.addVars(I_elz, T, lb=0.0, ub=P_ELZ_MAX, name="P_elz_m_g")
    P_elz_m_ng  = m.addVars(I_elz, T, lb=0.0, ub=P_ELZ_MAX, name="P_elz_m_ng")
    P_elz_m_tot = m.addVars(I_elz, T, lb=0.0, ub=P_ELZ_MAX, name="P_elz_m_tot")

    H_elz_m    = m.addVars(I_elz, T, lb=0.0, ub=H_ELZ_MAX, name="H_elz_m")
    H_elz_m_g  = m.addVars(I_elz, T, lb=0.0, ub=H_ELZ_MAX, name="H_elz_m_g")
    H_elz_m_ng = m.addVars(I_elz, T, lb=0.0, ub=H_ELZ_MAX, name="H_elz_m_ng")

    y_elz    = m.addVars(T, vtype=GRB.BINARY, name="y_elz")
    n_elz_on = m.addVars(T, vtype=GRB.INTEGER, lb=0, ub=N_stack, name="n_elz_on")
    su_elz   = m.addVars(T, vtype=GRB.INTEGER, lb=0, ub=N_stack, name="su_elz")

    P_elz_g   = m.addVars(T, lb=0.0, ub=N_stack*P_ELZ_MAX, name="P_elz_g")
    P_elz_ng  = m.addVars(T, lb=0.0, ub=N_stack*P_ELZ_MAX, name="P_elz_ng")
    P_elz_tot = m.addVars(T, lb=0.0, ub=N_stack*P_ELZ_MAX, name="P_elz_tot")
    H_prod    = m.addVars(T, lb=0.0, ub=N_stack*H_ELZ_MAX, name="H_prod")
    H_prod_g  = m.addVars(T, lb=0.0, ub=N_stack*H_ELZ_MAX, name="H_prod_g")
    H_prod_ng = m.addVars(T, lb=0.0, ub=N_stack*H_ELZ_MAX, name="H_prod_ng")
    W_elz     = m.addVars(T, lb=0.0, name="W_elz_m3ph")

    y_nh3   = m.addVars(T, vtype=GRB.BINARY, name="y_nh3")
    su_nh3  = m.addVars(T, vtype=GRB.BINARY, name="su_nh3")
    Q_nh3_g = m.addVars(T, lb=0.0, name="Q_nh3_g")
    Q_nh3_ng = m.addVars(T, lb=0.0, name="Q_nh3_ng")
    Q_nh3   = m.addVars(T, lb=0.0, name="Q_nh3")
    P_nh3_g = m.addVars(T, lb=0.0, name="P_nh3_g")
    P_nh3_ng = m.addVars(T, lb=0.0, name="P_nh3_ng")

    y_meoh   = m.addVars(T, vtype=GRB.BINARY, name="y_meoh")
    su_meoh  = m.addVars(T, vtype=GRB.BINARY, name="su_meoh")
    Q_meoh_g = m.addVars(T, lb=0.0, name="Q_meoh_g")
    Q_meoh_ng = m.addVars(T, lb=0.0, name="Q_meoh_ng")
    Q_meoh   = m.addVars(T, lb=0.0, name="Q_meoh")
    P_meoh_g = m.addVars(T, lb=0.0, name="P_meoh_g")
    P_meoh_ng = m.addVars(T, lb=0.0, name="P_meoh_ng")
    CO2_buy  = m.addVars(T, lb=0.0, name="CO2_buy_kgph")

    y_crk    = m.addVars(T, vtype=GRB.BINARY, name="y_crk")
    su_crk   = m.addVars(T, vtype=GRB.BINARY, name="su_crk")
    Q_crk_in_g = m.addVars(T, lb=0.0, name="Q_crk_in_g")
    Q_crk_in_ng = m.addVars(T, lb=0.0, name="Q_crk_in_ng")
    Q_crk_in = m.addVars(T, lb=0.0, name="Q_crk_in")

    H_crk_out_g  = m.addVars(T, lb=0.0, name="H_crk_out_g")
    H_crk_out_ng = m.addVars(T, lb=0.0, name="H_crk_out_ng")
    NH3_rec_g    = m.addVars(T, lb=0.0, name="NH3_rec_g")
    NH3_rec_ng   = m.addVars(T, lb=0.0, name="NH3_rec_ng")
    H_crk_fuel_g  = m.addVars(T, lb=0.0, name="H_crk_fuel_g")
    H_crk_fuel_ng = m.addVars(T, lb=0.0, name="H_crk_fuel_ng")
    Steam_g      = m.addVars(T, lb=0.0, name="Steam_g_kgph")
    Steam_ng     = m.addVars(T, lb=0.0, name="Steam_ng_kgph")
    P_crk_g      = m.addVars(T, lb=0.0, name="P_crk_g")
    P_crk_ng     = m.addVars(T, lb=0.0, name="P_crk_ng")
    W_steam      = m.addVars(T, lb=0.0, name="W_steam_m3ph")
    # Tanks (states, t=0..24)
    # one physical tank per product.
    # We track green/non-green "labels" as bookkeeping, but only the TOTAL is capacity constrained.
    S_H2_g   = m.addVars(T2, lb=0.0, name="S_H2_g")
    S_H2_ng  = m.addVars(T2, lb=0.0, name="S_H2_ng")
    S_NH3_g  = m.addVars(T2, lb=0.0, name="S_NH3_g")
    S_NH3_ng = m.addVars(T2, lb=0.0, name="S_NH3_ng")
    S_MEOH_g = m.addVars(T2, lb=0.0, name="S_MEOH_g")
    S_MEOH_ng = m.addVars(T2, lb=0.0, name="S_MEOH_ng")

    H_sell_g = m.addVars(T, lb=0.0, name="H_sell_g")
    H_sell_ng = m.addVars(T, lb=0.0, name="H_sell_ng")
    NH3_sell_g = m.addVars(T, lb=0.0, name="NH3_sell_g")
    NH3_sell_ng = m.addVars(T, lb=0.0, name="NH3_sell_ng")
    MEOH_sell_g = m.addVars(T, lb=0.0, name="MEOH_sell_g")
    MEOH_sell_ng = m.addVars(T, lb=0.0, name="MEOH_sell_ng")

    H_to_NH3_g = m.addVars(T, lb=0.0, name="H_to_NH3_g")
    H_to_NH3_ng = m.addVars(T, lb=0.0, name="H_to_NH3_ng")
    H_to_MEOH_g = m.addVars(T, lb=0.0, name="H_to_MEOH_g")
    H_to_MEOH_ng = m.addVars(T, lb=0.0, name="H_to_MEOH_ng")

    u_H2   = m.addVars(T, lb=0.0, name="u_H2")
    u_NH3  = m.addVars(T, lb=0.0, name="u_NH3")
    u_MEOH = m.addVars(T, lb=0.0, name="u_MEOH")
    #bounds
    for t in T:
        m.addConstr(P_pv_cur[t] <= pv_av[t], name=f"ub_pv_cur[{t}]")
        m.addConstr(P_wd_cur[t] <= wd_av[t], name=f"ub_wd_cur[{t}]")

    for t in T:
        for k in K_sinks:
            m.addConstr(P["pv", k, t]   <= pv_av[t], name=f"ub_P_pv_{k}[{t}]")
            m.addConstr(P["wd", k, t]   <= wd_av[t], name=f"ub_P_wd_{k}[{t}]")
            m.addConstr(P["grid", k, t] <= P_GRID_MAX_MW, name=f"ub_P_grid_{k}[{t}]")
            m.addConstr(P["bat", k, t]  <= P_BAT_MAX_MW, name=f"ub_P_bat_{k}[{t}]")

    for t in T:
        m.addConstr(gp.quicksum(P["pv",k,t] for k in K_sinks) + P_pv_cur[t] == pv_av[t], name=f"pv_balance[{t}]")
        m.addConstr(gp.quicksum(P["wd",k,t] for k in K_sinks) + P_wd_cur[t] == wd_av[t], name=f"wd_balance[{t}]")

        m.addConstr(gp.quicksum(P["grid",k,t] for k in K_sinks) == P_grid[t], name=f"grid_alloc[{t}]")
        m.addConstr(gp.quicksum(P["bat",k,t] for k in K_sinks) == P_dis[t], name=f"bat_alloc[{t}]")
        m.addConstr(gp.quicksum(P[s,"bch",t] for s in S_sources) == P_ch[t], name=f"bch_sink[{t}]")
        # Baseline rule: grid cannot charge battery; battery cannot charge battery
        m.addConstr(P["grid","bch",t] == 0.0, name=f"no_grid_to_bch[{t}]")
        m.addConstr(P["bat","bch",t] == 0.0, name=f"no_bat_to_bch[{t}]")
        m.addConstr(P["pv","bch",t] + P["wd","bch",t] == P_ch[t], name=f"bch_from_res[{t}]")

        m.addSOS(GRB.SOS_TYPE1, [P_ch[t], P_dis[t]])
        m.addConstr(P_dis[t] <= ETA_DIS * (SOC[t] - SOC_MIN) * E_BAT_MAX_MWH, name=f"ub_Pdis_soc[{t}]")
        m.addConstr(P_ch[t]  <= (SOC_MAX - SOC[t]) * E_BAT_MAX_MWH / ETA_CH, name=f"ub_Pch_soc[{t}]")

        for k in K_sinks:
            m.addConstr(Pk_ng[k,t] == P["grid",k,t] + P["bat",k,t], name=f"Pk_ng_def[{k},{t}]")
            m.addConstr(Pk_g[k,t] == gp.quicksum(P[s,k,t] for s in S_res), name=f"Pk_g_def[{k},{t}]")

    m.addConstr(SOC[0] == float(carry["soc"]), name="SOC_init_carry")
    for t in T:
        m.addConstr(
            SOC[t+1] == SOC[t] + (ETA_CH*P_ch[t] - (1.0/ETA_DIS)*P_dis[t]) / E_BAT_MAX_MWH,
            name=f"SOC_dyn[{t}]"
        )

    m.addConstrs((y_elz_m[i,t] >= y_elz_m[i+1,t] for i in range(N_stack-1) for t in T), name="elz_module_order")

    # ---------------- ELZ ----------------
    for t in T:
        m.addConstr(P_elz_g[t]  == Pk_g["elz",t],  name=f"P_elz_g_link[{t}]")
        m.addConstr(P_elz_ng[t] == Pk_ng["elz",t], name=f"P_elz_ng_link[{t}]")
        m.addConstr(P_elz_g[t]  == gp.quicksum(P_elz_m_g[i,t] for i in I_elz), name=f"P_elz_g_sum_mod[{t}]")
        m.addConstr(P_elz_ng[t] == gp.quicksum(P_elz_m_ng[i,t] for i in I_elz), name=f"P_elz_ng_sum_mod[{t}]")
        m.addConstr(P_elz_tot[t] == P_elz_g[t] + P_elz_ng[t], name=f"P_elz_tot_sum[{t}]")

        m.addConstr(n_elz_on[t] == gp.quicksum(y_elz_m[i,t] for i in I_elz), name=f"n_elz_on_def[{t}]")
        m.addConstr(n_elz_on[t] >= y_elz[t], name=f"y_elz_lb[{t}]")
        m.addConstr(n_elz_on[t] <= N_stack * y_elz[t], name=f"y_elz_ub[{t}]")
        m.addConstr(su_elz[t] == gp.quicksum(su_elz_m[i,t] for i in I_elz), name=f"su_elz_sum[{t}]")

        m.addConstr(H_prod[t]    == gp.quicksum(H_elz_m[i,t] for i in I_elz), name=f"H_prod_sum_mod[{t}]")
        m.addConstr(H_prod_g[t]  == gp.quicksum(H_elz_m_g[i,t] for i in I_elz), name=f"H_prod_g_sum_mod[{t}]")
        m.addConstr(H_prod_ng[t] == gp.quicksum(H_elz_m_ng[i,t] for i in I_elz), name=f"H_prod_ng_sum_mod[{t}]")
        m.addConstr(H_prod[t] == H_prod_g[t] + H_prod_ng[t], name=f"H_prod_split[{t}]")
        m.addConstr(W_elz[t] == omega_H2O * H_prod[t], name=f"W_elz[{t}]")

        for i in range(N_stack - 1):
            m.addConstr(b_elz_g[i,t] >= b_elz_g[i+1,t], name=f"elz_bg_order[{i},{t}]")

        for i in I_elz:
            m.addConstr(P_elz_m_tot[i,t] == P_elz_m_g[i,t] + P_elz_m_ng[i,t], name=f"P_elz_m_tot_def[{i},{t}]")
            m.addConstr(P_elz_m_tot[i,t] <= P_ELZ_MAX * y_elz_m[i,t], name=f"elz_m_max[{i},{t}]")
            m.addConstr(P_elz_m_tot[i,t] >= P_ELZ_MIN * y_elz_m[i,t], name=f"elz_m_min[{i},{t}]")
            m.addConstr(b_elz_g[i,t] <= y_elz_m[i,t], name=f"b_elz_g_gate[{i},{t}]")
            m.addConstr(P_elz_m_g[i,t]  <= P_ELZ_MAX * b_elz_g[i,t], name=f"P_elz_m_g_cap[{i},{t}]")
            m.addConstr(P_elz_m_ng[i,t] <= P_ELZ_MAX * (y_elz_m[i,t] - b_elz_g[i,t]), name=f"P_elz_m_ng_cap[{i},{t}]")
            m.addConstr(H_elz_m[i,t] == H_elz_m_g[i,t] + H_elz_m_ng[i,t], name=f"H_elz_m_split[{i},{t}]")

            m.addQConstr(
                H_elz_m[i,t] <= A_elz*(P_elz_m_tot[i,t]*P_elz_m_tot[i,t]) + B_elz*P_elz_m_tot[i,t] + C_elz*y_elz_m[i,t],
                name=f"elz_m_quad[{i},{t}]"
            )
            m.addConstr(H_elz_m[i,t] >= Q1*P_elz_m_tot[i,t] + Q0*y_elz_m[i,t], name=f"elz_m_lin[{i},{t}]")

            m.addConstr(H_elz_m_g[i,t] <= H_ELZ_MAX * b_elz_g[i,t], name=f"H_elz_m_g_ub1[{i},{t}]")
            m.addConstr(H_elz_m_g[i,t] <= H_elz_m[i,t], name=f"H_elz_m_g_ub2[{i},{t}]")
            m.addConstr(H_elz_m_g[i,t] >= H_elz_m[i,t] - H_ELZ_MAX*(1 - b_elz_g[i,t]), name=f"H_elz_m_g_lb[{i},{t}]")

            m.addConstr(H_elz_m_ng[i,t] <= H_ELZ_MAX * (y_elz_m[i,t] - b_elz_g[i,t]), name=f"H_elz_m_ng_ub1[{i},{t}]")
            m.addConstr(H_elz_m_ng[i,t] <= H_elz_m[i,t], name=f"H_elz_m_ng_ub2[{i},{t}]")
            m.addConstr(H_elz_m_ng[i,t] >= H_elz_m[i,t] - H_ELZ_MAX*b_elz_g[i,t], name=f"H_elz_m_ng_lb[{i},{t}]")
            # startup linkage per module
            if t >= 1:
                m.addConstr(su_elz_m[i,t] <= y_elz_m[i,t], name=f"su_elz_m1[{i},{t}]")
                m.addConstr(su_elz_m[i,t] <= 1 - y_elz_m[i,t-1], name=f"su_elz_m2[{i},{t}]")
                m.addConstr(su_elz_m[i,t] >= y_elz_m[i,t] - y_elz_m[i,t-1], name=f"su_elz_m3[{i},{t}]")
            else:
                y_prev_i = int(carry["y_elz_prev_mod"][i])
                m.addConstr(su_elz_m[i,t] <= y_elz_m[i,t], name=f"su_elz_m1[{i},{t}]")
                m.addConstr(su_elz_m[i,t] <= 1 - y_prev_i, name=f"su_elz_m2[{i},{t}]")
                m.addConstr(su_elz_m[i,t] >= y_elz_m[i,t] - y_prev_i, name=f"su_elz_m3[{i},{t}]")

    # ---------------- NH3 ----------------
    for t in T:
        m.addConstr(Q_nh3[t] == Q_nh3_g[t] + Q_nh3_ng[t], name=f"Q_nh3_sum[{t}]")
        m.addConstr(Q_nh3[t] <= Q_NH3_CAP * y_nh3[t], name=f"nh3_max[{t}]")
        m.addConstr(Q_nh3[t] >= f_NH3_min * Q_NH3_CAP * y_nh3[t], name=f"nh3_min[{t}]")
        # Startup linkage
        if t >= 1:
            m.addConstr(su_nh3[t] <= y_nh3[t])
            m.addConstr(su_nh3[t] <= 1 - y_nh3[t-1])
            m.addConstr(su_nh3[t] >= y_nh3[t] - y_nh3[t-1])
        else:
            y_prev = int(carry["y_nh3_prev"])
            m.addConstr(su_nh3[t] <= y_nh3[t])
            m.addConstr(su_nh3[t] <= 1 - y_prev)
            m.addConstr(su_nh3[t] >= y_nh3[t] - y_prev)

        m.addConstr(
            Q_nh3[t] <= f_NH3_min * Q_NH3_CAP + (Q_NH3_CAP - f_NH3_min * Q_NH3_CAP) * (1 - su_nh3[t]),
            name=f"nh3_start_minload[{t}]"
        )

        ramp_nh3 = RR_NH3 * Q_NH3_CAP
        if t == 0:
            Q_prev = float(carry["Q_nh3_prev"])
            y_prev = int(carry["y_nh3_prev"])
            m.addConstr(Q_nh3[t] - Q_prev <= ramp_nh3 + (Q_NH3_CAP - ramp_nh3) * (2 - y_nh3[t] - y_prev), name=f"nh3_rup0[{t}]")
            m.addConstr(Q_prev - Q_nh3[t] <= ramp_nh3 + (Q_NH3_CAP - ramp_nh3) * (2 - y_nh3[t] - y_prev), name=f"nh3_rdn0[{t}]")
        else:
            m.addConstr(Q_nh3[t] - Q_nh3[t-1] <= ramp_nh3 + (Q_NH3_CAP - ramp_nh3) * (2 - y_nh3[t] - y_nh3[t-1]), name=f"nh3_rup[{t}]")
            m.addConstr(Q_nh3[t-1] - Q_nh3[t] <= ramp_nh3 + (Q_NH3_CAP - ramp_nh3) * (2 - y_nh3[t] - y_nh3[t-1]), name=f"nh3_rdn[{t}]")

        m.addConstr(P_nh3_g[t]  == (SEC_HB*Q_nh3_g[t]  + SEC_ASU*(mN2_to_NH3*Q_nh3_g[t]))  / 1000.0, name=f"P_nh3_g_map[{t}]")
        m.addConstr(P_nh3_ng[t] == (SEC_HB*Q_nh3_ng[t] + SEC_ASU*(mN2_to_NH3*Q_nh3_ng[t])) / 1000.0, name=f"P_nh3_ng_map[{t}]")

        m.addConstr(P_nh3_g[t]  == Pk_g["NH3",t],  name=f"P_nh3_g_link[{t}]")
        m.addConstr(P_nh3_ng[t] == Pk_ng["NH3",t], name=f"P_nh3_ng_link[{t}]")

        m.addConstr(H_to_NH3_g[t]  == mH2_to_NH3 * Q_nh3_g[t],  name=f"H_to_NH3_g[{t}]")
        m.addConstr(H_to_NH3_ng[t] == mH2_to_NH3 * Q_nh3_ng[t], name=f"H_to_NH3_ng[{t}]")

    # ---------------- MeOH ----------------
    for t in T:
        m.addConstr(Q_meoh[t] == Q_meoh_g[t] + Q_meoh_ng[t], name=f"Q_meoh_sum[{t}]")
        m.addConstr(Q_meoh[t] <= Q_MEOH_CAP * y_meoh[t], name=f"meoh_max[{t}]")
        m.addConstr(Q_meoh[t] >= f_MEOH_min * Q_MEOH_CAP * y_meoh[t], name=f"meoh_min[{t}]")

        if t >= 1:
            m.addConstr(su_meoh[t] <= y_meoh[t])
            m.addConstr(su_meoh[t] <= 1 - y_meoh[t-1])
            m.addConstr(su_meoh[t] >= y_meoh[t] - y_meoh[t-1])
        else:
            y_prev = int(carry["y_meoh_prev"])
            m.addConstr(su_meoh[t] <= y_meoh[t])
            m.addConstr(su_meoh[t] <= 1 - y_prev)
            m.addConstr(su_meoh[t] >= y_meoh[t] - y_prev)

        m.addConstr(
            Q_meoh[t] <= f_MEOH_min * Q_MEOH_CAP + (Q_MEOH_CAP - f_MEOH_min * Q_MEOH_CAP) * (1 - su_meoh[t]),
            name=f"meoh_start_minload[{t}]"
        )

        ramp_meoh = R_MEOH
        if t == 0:
            Q_prev = float(carry["Q_meoh_prev"])
            y_prev = int(carry["y_meoh_prev"])
            m.addConstr(Q_meoh[t] - Q_prev <= ramp_meoh + (Q_MEOH_CAP - ramp_meoh) * (2 - y_meoh[t] - y_prev), name=f"meoh_rup0[{t}]")
            m.addConstr(Q_prev - Q_meoh[t] <= ramp_meoh + (Q_MEOH_CAP - ramp_meoh) * (2 - y_meoh[t] - y_prev), name=f"meoh_rdn0[{t}]")
        else:
            m.addConstr(Q_meoh[t] - Q_meoh[t-1] <= ramp_meoh + (Q_MEOH_CAP - ramp_meoh) * (2 - y_meoh[t] - y_meoh[t-1]), name=f"meoh_rup[{t}]")
            m.addConstr(Q_meoh[t-1] - Q_meoh[t] <= ramp_meoh + (Q_MEOH_CAP - ramp_meoh) * (2 - y_meoh[t] - y_meoh[t-1]), name=f"meoh_rdn[{t}]")

        m.addConstr(P_meoh_g[t]  == (SEC_MEOH * Q_meoh_g[t])  / 1000.0, name=f"P_meoh_g_map[{t}]")
        m.addConstr(P_meoh_ng[t] == (SEC_MEOH * Q_meoh_ng[t]) / 1000.0, name=f"P_meoh_ng_map[{t}]")

        m.addConstr(P_meoh_g[t]  == Pk_g["MeOH",t],  name=f"P_meoh_g_link[{t}]")
        m.addConstr(P_meoh_ng[t] == Pk_ng["MeOH",t], name=f"P_meoh_ng_link[{t}]")

        m.addConstr(CO2_buy[t] == mCO2_to_MEOH * Q_meoh[t], name=f"CO2_buy[{t}]")
        m.addConstr(H_to_MEOH_g[t]  == mH2_to_MEOH * Q_meoh_g[t],  name=f"H_to_MEOH_g[{t}]")
        m.addConstr(H_to_MEOH_ng[t] == mH2_to_MEOH * Q_meoh_ng[t], name=f"H_to_MEOH_ng[{t}]")

    # ---------------- Cracker ----------------
    for t in T:
        m.addConstr(Q_crk_in[t] == Q_crk_in_g[t] + Q_crk_in_ng[t], name=f"Q_crk_sum[{t}]")
        m.addConstr(Q_crk_in[t] <= Q_CRK_CAP * y_crk[t], name=f"crk_max[{t}]")
        m.addConstr(Q_crk_in[t] >= f_CRK_min * Q_CRK_CAP * y_crk[t], name=f"crk_min[{t}]")

        if t >= 1:
            m.addConstr(su_crk[t] <= y_crk[t])
            m.addConstr(su_crk[t] <= 1 - y_crk[t-1])
            m.addConstr(su_crk[t] >= y_crk[t] - y_crk[t-1])
        else:
            y_prev = int(carry["y_crk_prev"])
            m.addConstr(su_crk[t] <= y_crk[t])
            m.addConstr(su_crk[t] <= 1 - y_prev)
            m.addConstr(su_crk[t] >= y_crk[t] - y_prev)

        m.addConstr(
            Q_crk_in[t] <= f_CRK_min * Q_CRK_CAP + (Q_CRK_CAP - f_CRK_min * Q_CRK_CAP) * (1 - su_crk[t]),
            name=f"crk_start_minload[{t}]"
        )

        ramp_crk = R_CRK
        if t == 0:
            Q_prev = float(carry["Q_crk_prev"])
            y_prev = int(carry["y_crk_prev"])
            m.addConstr(Q_crk_in[t] - Q_prev <= ramp_crk + (Q_CRK_CAP - ramp_crk) * (2 - y_crk[t] - y_prev), name=f"crk_rup0[{t}]")
            m.addConstr(Q_prev - Q_crk_in[t] <= ramp_crk + (Q_CRK_CAP - ramp_crk) * (2 - y_crk[t] - y_prev), name=f"crk_rdn0[{t}]")
        else:
            m.addConstr(Q_crk_in[t] - Q_crk_in[t-1] <= ramp_crk + (Q_CRK_CAP - ramp_crk) * (2 - y_crk[t] - y_crk[t-1]), name=f"crk_rup[{t}]")
            m.addConstr(Q_crk_in[t-1] - Q_crk_in[t] <= ramp_crk + (Q_CRK_CAP - ramp_crk) * (2 - y_crk[t] - y_crk[t-1]), name=f"crk_rdn[{t}]")

        m.addConstr(H_crk_out_g[t]  == kH2_out * Q_crk_in_g[t],  name=f"Hcrk_out_g[{t}]")
        m.addConstr(H_crk_out_ng[t] == kH2_out * Q_crk_in_ng[t], name=f"Hcrk_out_ng[{t}]")
        m.addConstr(NH3_rec_g[t]    == kNH3_rec * Q_crk_in_g[t],  name=f"NH3rec_g[{t}]")
        m.addConstr(NH3_rec_ng[t]   == kNH3_rec * Q_crk_in_ng[t], name=f"NH3rec_ng[{t}]")
        m.addConstr(H_crk_fuel_g[t]  == kH2_fuel * Q_crk_in_g[t],  name=f"Hcrk_fuel_g[{t}]")
        m.addConstr(H_crk_fuel_ng[t] == kH2_fuel * Q_crk_in_ng[t], name=f"Hcrk_fuel_ng[{t}]")
        m.addConstr(Steam_g[t]  == kSteam * Q_crk_in_g[t],  name=f"Steam_g[{t}]")
        m.addConstr(Steam_ng[t] == kSteam * Q_crk_in_ng[t], name=f"Steam_ng[{t}]")
        m.addConstr(W_steam[t]  == omega_steam * (Steam_g[t] + Steam_ng[t]), name=f"W_steam[{t}]")
        m.addConstr(P_crk_g[t]  == (SEC_CRK_E*Q_crk_in_g[t]  + eSteam*Steam_g[t])  / 1000.0, name=f"P_crk_g_map[{t}]")
        m.addConstr(P_crk_ng[t] == (SEC_CRK_E*Q_crk_in_ng[t] + eSteam*Steam_ng[t]) / 1000.0, name=f"P_crk_ng_map[{t}]")
        m.addConstr(P_crk_g[t]  == Pk_g["crk",t],  name=f"P_crk_g_link[{t}]")
        m.addConstr(P_crk_ng[t] == Pk_ng["crk",t], name=f"P_crk_ng_link[{t}]")

    # ---------------- Tanks and demand ----------------
    m.addConstr(S_H2_g[0]   == float(carry["S_H2_g"]), name="S_H2_g_init")
    m.addConstr(S_H2_ng[0]  == float(carry["S_H2_ng"]), name="S_H2_ng_init")
    m.addConstr(S_NH3_g[0]  == float(carry["S_NH3_g"]), name="S_NH3_g_init")
    m.addConstr(S_NH3_ng[0] == float(carry["S_NH3_ng"]), name="S_NH3_ng_init")
    m.addConstr(S_MEOH_g[0] == float(carry["S_MEOH_g"]), name="S_MEOH_g_init")
    m.addConstr(S_MEOH_ng[0] == float(carry["S_MEOH_ng"]), name="S_MEOH_ng_init")

    for tt in T2:
        m.addConstr(S_H2_g[tt] + S_H2_ng[tt] <= S_H2_CAP, name=f"S_H2_cap[{tt}]")
        m.addConstr(S_NH3_g[tt] + S_NH3_ng[tt] <= S_NH3_CAP, name=f"S_NH3_cap[{tt}]")
        m.addConstr(S_MEOH_g[tt] + S_MEOH_ng[tt] <= S_MEOH_CAP, name=f"S_MEOH_cap[{tt}]")
    # ---------------- Hard EOD minimum inventory (fraction of tank capacity) ----------------
    m.addConstr(S_H2_g[24] + S_H2_ng[24] >= ALPHA_TANK_FILL * S_H2_CAP, name="eod_hard_H2_frac")
    m.addConstr(S_NH3_g[24] + S_NH3_ng[24] >= ALPHA_TANK_FILL * S_NH3_CAP, name="eod_hard_NH3_frac")
    m.addConstr(S_MEOH_g[24] + S_MEOH_ng[24] >= ALPHA_TANK_FILL * S_MEOH_CAP, name="eod_hard_MeOH_frac")

    for t in T:
        m.addConstr(H_sell_g[t] + H_sell_ng[t] + u_H2[t] == dem_h2[t], name=f"dem_H2[{t}]")
        m.addConstr(NH3_sell_g[t] + NH3_sell_ng[t] + u_NH3[t] == dem_nh3[t], name=f"dem_NH3[{t}]")
        m.addConstr(MEOH_sell_g[t] + MEOH_sell_ng[t] + u_MEOH[t] == dem_meoh[t], name=f"dem_MEOH[{t}]")

        m.addConstr(S_H2_g[t+1] == S_H2_g[t] + H_prod_g[t] + H_crk_out_g[t] - H_to_NH3_g[t] - H_to_MEOH_g[t] - H_sell_g[t] - H_crk_fuel_g[t], name=f"S_H2_g_dyn[{t}]")
        m.addConstr(S_H2_ng[t+1] == S_H2_ng[t] + H_prod_ng[t] + H_crk_out_ng[t] - H_to_NH3_ng[t] - H_to_MEOH_ng[t] - H_sell_ng[t] - H_crk_fuel_ng[t], name=f"S_H2_ng_dyn[{t}]")

        m.addConstr(S_NH3_g[t+1] == S_NH3_g[t] + Q_nh3_g[t] + NH3_rec_g[t] - NH3_sell_g[t] - Q_crk_in_g[t], name=f"S_NH3_g_dyn[{t}]")
        m.addConstr(S_NH3_ng[t+1] == S_NH3_ng[t] + Q_nh3_ng[t] + NH3_rec_ng[t] - NH3_sell_ng[t] - Q_crk_in_ng[t], name=f"S_NH3_ng_dyn[{t}]")

        m.addConstr(S_MEOH_g[t+1] == S_MEOH_g[t] + Q_meoh_g[t] - MEOH_sell_g[t], name=f"S_MEOH_g_dyn[{t}]")
        m.addConstr(S_MEOH_ng[t+1] == S_MEOH_ng[t] + Q_meoh_ng[t] - MEOH_sell_ng[t], name=f"S_MEOH_ng_dyn[{t}]")

        m.addConstr(H_to_NH3_g[t] + H_to_MEOH_g[t] + H_sell_g[t] + H_crk_fuel_g[t] <= S_H2_g[t], name=f"dir_H2_g[{t}]")
        m.addConstr(H_to_NH3_ng[t] + H_to_MEOH_ng[t] + H_sell_ng[t] + H_crk_fuel_ng[t] <= S_H2_ng[t], name=f"dir_H2_ng[{t}]")
        m.addConstr(NH3_sell_g[t] + Q_crk_in_g[t] <= S_NH3_g[t], name=f"dir_NH3_g[{t}]")
        m.addConstr(NH3_sell_ng[t] + Q_crk_in_ng[t] <= S_NH3_ng[t], name=f"dir_NH3_ng[{t}]")
        m.addConstr(MEOH_sell_g[t] <= S_MEOH_g[t], name=f"dir_MEOH_g[{t}]")
        m.addConstr(MEOH_sell_ng[t] <= S_MEOH_ng[t], name=f"dir_MEOH_ng[{t}]")
    # Priority 1: minimize total unmet demand (kg over the day)
    unmet_total = gp.quicksum(u_H2[t] + u_NH3[t] + u_MEOH[t] for t in T)
    # Priority 2: minimize operating cost (EUR) among solutions with minimal unmet
    cost_total = gp.LinExpr()
    for t in T:
        cost_total += pr_eur[t] * P_grid[t]
        cost_total += c_H2O * (W_elz[t] + W_steam[t])
        cost_total += c_CO2 * CO2_buy[t]
        cost_total += C_su_elz * gp.quicksum(su_elz_m[i,t] for i in I_elz)
        cost_total += C_su_NH3 * su_nh3[t]
        cost_total += C_su_MEOH * su_meoh[t]
        cost_total += C_su_CRK * su_crk[t]

    m.setObjectiveN(unmet_total, index=0, priority=2, reltol=1e-4, abstol=1e-6, name="OBJ1_min_unmet")
    m.setObjectiveN(cost_total, index=1, priority=1, reltol=1e-4, abstol=1e-6, name="OBJ2_min_cost")

    m.optimize()

    status = int(safe_get_attr(m, "Status", -1))
    runtime = float(safe_get_attr(m, "Runtime", np.nan))
    nodecnt = float(safe_get_attr(m, "NodeCount", np.nan))
    solcnt  = int(safe_get_attr(m, "SolCount", -1))
    # index=0 is unmet, index=1 is cost (for M0)
    pass0 = get_objpass_metrics(m, 0)
    pass1 = get_objpass_metrics(m, 1)

    mip_gap = pass1["pass_mipgap"]
    obj_val = pass1["pass_objval"]
    obj_bnd = pass1["pass_objbound"]

    itercnt = safe_get_attr(m, "IterCount", np.nan)
    bariter = safe_get_attr(m, "BarIterCount", np.nan)
    work = safe_get_attr(m, "Work", np.nan)

    obj1_val = np.nan
    obj2_val = np.nan
    try:
        obj1_val = float(unmet_total.getValue())
    except Exception:
        pass
    try:
        obj2_val = float(cost_total.getValue())
    except Exception:
        pass

    is_optimal = (status == GRB.OPTIMAL)
    hit_time_limit = (status == GRB.TIME_LIMIT)
    has_incumbent = (solcnt is not None) and (solcnt >= 1)
    is_within_target_gap = bool(has_incumbent) and (mip_gap is not None) and (not np.isnan(mip_gap)) and (float(mip_gap) <= float(MIPGAP) + 1e-12)
    is_acceptable = bool(is_optimal or is_within_target_gap)

    solver_rows.append({
        "run_id": RUN_ID,
        "model_tag": MODEL_TAG,
        "scenario_tag": SCENARIO_TAG,
        "demand_tag": DEMAND_TAG,
        "day_of_year": d,
        "date_utc": str(day_hours[0].date()),
        "solver_status": status,
        "solver_status_name": status_name(status),
        "runtime_seconds": runtime,
        "time_limit_s": TIME_LIMIT_S,
        "mipgap_target": float(MIPGAP),
        "mip_gap": pass1["pass_mipgap"],
        "obj0_pass_status": pass0["pass_status"],
        "obj0_pass_status_name": pass0["pass_status_name"],
        "obj0_pass_runtime_s": pass0["pass_runtime"],
        "obj0_pass_objval": pass0["pass_objval"],
        "obj0_pass_objbound": pass0["pass_objbound"],
        "obj0_pass_mipgap": pass0["pass_mipgap"],
        "obj1_pass_status": pass1["pass_status"],
        "obj1_pass_status_name": pass1["pass_status_name"],
        "obj1_pass_runtime_s": pass1["pass_runtime"],
        "obj1_pass_objval": pass1["pass_objval"],
        "obj1_pass_objbound": pass1["pass_objbound"],
        "obj1_pass_mipgap": pass1["pass_mipgap"],
        "has_incumbent": bool(has_incumbent),
        "is_optimal": bool(is_optimal),
        "hit_time_limit": bool(hit_time_limit),
        "is_within_target_gap": bool(is_within_target_gap),
        "is_acceptable": bool(is_acceptable),
        "obj_val": float(obj_val) if not np.isnan(obj_val) else np.nan,
        "obj_bound": float(obj_bnd) if not np.isnan(obj_bnd) else np.nan,
        "node_count": nodecnt,
        "sol_count": solcnt,
        "iter_count": float(itercnt) if not np.isnan(itercnt) else np.nan,
        "bar_iter_count": float(bariter) if not np.isnan(bariter) else np.nan,
        "work": float(work) if not np.isnan(work) else np.nan,
        "obj1_unmet_value": float(obj1_val) if not np.isnan(obj1_val) else np.nan,
        "obj2_cost_value": float(obj2_val) if not np.isnan(obj2_val) else np.nan,
    })

    if status not in [GRB.OPTIMAL, GRB.TIME_LIMIT]:
        try:
            m.computeIIS()
            iis_path = day_dir / f"{MODEL_TAG}_day{d:03d}_iis.ilp"
            m.write(str(iis_path))
        except Exception:
            pass
        raise RuntimeError(f"Day {d}: solver status {status} (not optimal/time_limit).")

    def val(v):
        return float(v.X)

    rows = []
    unmet_rows = []

    for t in T:
        ts = day_hours[t]
        pv_used = pv_av[t] - val(P_pv_cur[t])
        wd_used = wd_av[t] - val(P_wd_cur[t])
        grid_mw = val(P_grid[t])
        res_used_mw = pv_used + wd_used
        res_curt_mw = val(P_pv_cur[t]) + val(P_wd_cur[t])

        P_elz = val(P_elz_g[t]) + val(P_elz_ng[t])
        P_nh3 = val(P_nh3_g[t]) + val(P_nh3_ng[t])
        P_meo = val(P_meoh_g[t]) + val(P_meoh_ng[t])
        P_crk = val(P_crk_g[t]) + val(P_crk_ng[t])

        H_total = val(H_prod[t])
        H_g = val(H_prod_g[t])
        H_ng = val(H_prod_ng[t])

        nh3_total = val(Q_nh3[t])
        nh3_g = val(Q_nh3_g[t])
        nh3_ng = val(Q_nh3_ng[t])

        me_total = val(Q_meoh[t])
        me_g = val(Q_meoh_g[t])
        me_ng = val(Q_meoh_ng[t])

        h2_sell_g = val(H_sell_g[t])
        h2_sell_ng = val(H_sell_ng[t])
        nh3_sell_g = val(NH3_sell_g[t])
        nh3_sell_ng = val(NH3_sell_ng[t])
        meoh_sell_g = val(MEOH_sell_g[t])
        meoh_sell_ng = val(MEOH_sell_ng[t])

        u1 = val(u_H2[t])
        u2 = val(u_NH3[t])
        u3 = val(u_MEOH[t])

        def share(g, ng):
            tot = g + ng
            return (g / tot) if tot > 1e-9 else np.nan

        H2_min_to_NH3  = mH2_to_NH3 * f_NH3_min * Q_NH3_CAP * int(round(val(y_nh3[t])))
        H2_min_to_MeOH = mH2_to_MEOH * f_MEOH_min * Q_MEOH_CAP * int(round(val(y_meoh[t])))
        H2_min_to_CRK  = kH2_fuel * f_CRK_min * Q_CRK_CAP * int(round(val(y_crk[t])))
        H2_min_committed = H2_min_to_NH3 + H2_min_to_MeOH + H2_min_to_CRK
        H2_tank_start = val(S_H2_g[t]) + val(S_H2_ng[t])
        H2_free_margin = H2_tank_start - H2_min_committed

        elz_mod = {}
        for i in I_elz:
            elz_mod[f"elz_m{i}_y"]     = int(round(val(y_elz_m[i,t])))
            elz_mod[f"elz_m{i}_su"]    = int(round(val(su_elz_m[i,t])))
            elz_mod[f"elz_m{i}_b_g"]   = int(round(val(b_elz_g[i,t])))
            elz_mod[f"elz_m{i}_P_tot"] = float(val(P_elz_m_tot[i,t]))
            elz_mod[f"elz_m{i}_P_g"]   = float(val(P_elz_m_g[i,t]))
            elz_mod[f"elz_m{i}_P_ng"]  = float(val(P_elz_m_ng[i,t]))
            elz_mod[f"elz_m{i}_H_tot"] = float(val(H_elz_m[i,t]))
            elz_mod[f"elz_m{i}_H_g"]   = float(val(H_elz_m_g[i,t]))
            elz_mod[f"elz_m{i}_H_ng"]  = float(val(H_elz_m_ng[i,t]))

        rows.append({
            "run_id": RUN_ID,
            "model_tag": MODEL_TAG,
            "scenario_tag": SCENARIO_TAG,
            "demand_tag": DEMAND_TAG,
            "datetime_utc": ts.isoformat(),
            "day_of_year": d,
            "hour_in_day": t,
            "H2_tank_start_kg": H2_tank_start,
            "H2_min_committed_kgph": H2_min_committed,
            "H2_free_margin_kg": H2_free_margin,
            "pv_avail_mw": pv_av[t],
            "wind_avail_mw": wd_av[t],
            "pv_curt_mw": val(P_pv_cur[t]),
            "wind_curt_mw": val(P_wd_cur[t]),
            "res_used_mw": res_used_mw,
            "res_curtailed_mw": res_curt_mw,
            "grid_import_mw": grid_mw,
            "da_price_eur_per_mwh": pr_eur[t],
            "bat_ch_mw": val(P_ch[t]),
            "bat_dis_mw": val(P_dis[t]),
            "battery_soc": val(SOC[t]),
            "battery_soc_next": val(SOC[t+1]),
            "elz_grid_mw": val(P_elz_ng[t]),
            "nh3_grid_mw": val(P_nh3_ng[t]),
            "meoh_grid_mw": val(P_meoh_ng[t]),
            "crk_grid_mw": val(P_crk_ng[t]),
            "P_elz_mw": P_elz,
            "P_nh3_mw": P_nh3,
            "P_meoh_mw": P_meo,
            "P_crk_mw": P_crk,
            "elz_power_green_share": share(val(P_elz_g[t]), val(P_elz_ng[t])),
            "nh3_power_green_share": share(val(P_nh3_g[t]), val(P_nh3_ng[t])),
            "meoh_power_green_share": share(val(P_meoh_g[t]), val(P_meoh_ng[t])),
            "crk_power_green_share": share(val(P_crk_g[t]), val(P_crk_ng[t])),
            "H2_prod_kgph": H_total,
            "H2_prod_g_kgph": H_g,
            "H2_prod_ng_kgph": H_ng,
            "NH3_prod_kgph": nh3_total,
            "NH3_prod_g_kgph": nh3_g,
            "NH3_prod_ng_kgph": nh3_ng,
            "MeOH_prod_kgph": me_total,
            "MeOH_prod_g_kgph": me_g,
            "MeOH_prod_ng_kgph": me_ng,
            "crk_nh3_in_kgph": val(Q_crk_in[t]),
            "crk_h2_out_kgph": val(H_crk_out_g[t]) + val(H_crk_out_ng[t]),
            "crk_h2_fuel_kgph": val(H_crk_fuel_g[t]) + val(H_crk_fuel_ng[t]),
            "steam_kgph": val(Steam_g[t]) + val(Steam_ng[t]),
            "water_elz_m3ph": val(W_elz[t]),
            "water_steam_m3ph": val(W_steam[t]),
            "co2_buy_kgph": val(CO2_buy[t]),

            "H2_sell_g_kgph": h2_sell_g,
            "H2_sell_ng_kgph": h2_sell_ng,
            "H2_sell_kgph": h2_sell_g + h2_sell_ng,

            "NH3_sell_g_kgph": nh3_sell_g,
            "NH3_sell_ng_kgph": nh3_sell_ng,
            "NH3_sell_kgph": nh3_sell_g + nh3_sell_ng,

            "MeOH_sell_g_kgph": meoh_sell_g,
            "MeOH_sell_ng_kgph": meoh_sell_ng,
            "MeOH_sell_kgph": meoh_sell_g + meoh_sell_ng,

            "unmet_h2_kgph": u1,
            "unmet_nh3_kgph": u2,
            "unmet_meoh_kgph": u3,
            "tank_H2_g_kg": val(S_H2_g[t]),
            "tank_H2_ng_kg": val(S_H2_ng[t]),
            "tank_NH3_g_kg": val(S_NH3_g[t]),
            "tank_NH3_ng_kg": val(S_NH3_ng[t]),
            "tank_MeOH_g_kg": val(S_MEOH_g[t]),
            "tank_MeOH_ng_kg": val(S_MEOH_ng[t]),
            "b_ch": int(val(P_ch[t]) > 1e-6),
            "n_elz_on": int(round(val(n_elz_on[t]))),
            "y_elz": int(round(val(y_elz[t]))),
            "su_elz": int(round(val(su_elz[t]))),
            "y_nh3": int(round(val(y_nh3[t]))),
            "su_nh3": int(round(val(su_nh3[t]))),
            "y_meoh": int(round(val(y_meoh[t]))),
            "su_meoh": int(round(val(su_meoh[t]))),
            "y_crk": int(round(val(y_crk[t]))),
            "su_crk": int(round(val(su_crk[t]))),
            **elz_mod,
        })

        unmet_rows.append({
            "run_id": RUN_ID,
            "model_tag": MODEL_TAG,
            "scenario_tag": SCENARIO_TAG,
            "demand_tag": DEMAND_TAG,
            "datetime_utc": ts.isoformat(),
            "day_of_year": d,
            "hour_in_day": t,
            "unmet_h2_kgph": u1,
            "unmet_nh3_kgph": u2,
            "unmet_meoh_kgph": u3,
            "da_price_eur_per_mwh": pr_eur[t],
            "grid_import_mw": grid_mw,
            "res_curtailed_mw": res_curt_mw,
        })

    day_hourly = pd.DataFrame(rows)
    day_unmet  = pd.DataFrame(unmet_rows)

    day_hourly.to_csv(day_dir / "hourly_timeseries.csv", index=False)
    day_unmet.to_csv(day_dir / "hourly_unmet_demand.csv", index=False)

    day_hourly_all = day_hourly.copy()
    day_hourly_all["day_id"] = int(d)
    all_hourly_frames.append(day_hourly_all)

    # ---------------- Daily post-processing ----------------
    grid_cost_eur   = float(np.sum([pr_eur[t] * val(P_grid[t]) for t in T]))
    water_cost_eur  = float(c_H2O * np.sum([val(W_elz[t]) + val(W_steam[t]) for t in T]))
    co2_cost_eur    = float(c_CO2 * np.sum([val(CO2_buy[t]) for t in T]))
    startup_elz_eur = float(C_su_elz * np.sum([val(su_elz[t]) for t in T]))
    startup_nh3_eur = float(C_su_NH3 * np.sum([val(su_nh3[t]) for t in T]))
    startup_meoh_eur = float(C_su_MEOH * np.sum([val(su_meoh[t]) for t in T]))
    startup_crk_eur = float(C_su_CRK * np.sum([val(su_crk[t]) for t in T]))

    total_unmet_kg = float(np.sum(day_hourly["unmet_h2_kgph"].values + day_hourly["unmet_nh3_kgph"].values + day_hourly["unmet_meoh_kgph"].values))
    total_cost = float(obj2_val) if not np.isnan(obj2_val) else float(grid_cost_eur + water_cost_eur + co2_cost_eur + startup_elz_eur + startup_nh3_eur + startup_meoh_eur + startup_crk_eur)

    grid_emis_aef = float(np.sum([val(P_grid[t]) * 1000.0 * aef_k[t] for t in T]))
    grid_emis_mef_upramp = float(np.sum([val(P_grid[t]) * 1000.0 * mef_upramp_k[t] for t in T]))
    grid_emis_mef_soft = float(np.sum([val(P_grid[t]) * 1000.0 * mef_soft_k[t] for t in T]))

    direct_emis_nh3 = float(np.sum([de_NH3 * val(Q_nh3[t]) for t in T]))
    direct_emis_meoh = float(np.sum([de_MEOH * val(Q_meoh[t]) for t in T]))
    direct_emis_total = direct_emis_nh3 + direct_emis_meoh

    demand_H2_kg_day   = float(np.sum(dem_h2))
    demand_NH3_kg_day  = float(np.sum(dem_nh3))
    demand_MeOH_kg_day = float(np.sum(dem_meoh))

    elz_kwh_day  = float(np.sum(day_hourly["P_elz_mw"].values * 1000.0))
    nh3_kwh_day  = float(np.sum(day_hourly["P_nh3_mw"].values * 1000.0))
    meoh_kwh_day = float(np.sum(day_hourly["P_meoh_mw"].values * 1000.0))
    crk_kwh_day  = float(np.sum(day_hourly["P_crk_mw"].values * 1000.0))
    units_kwh_day = elz_kwh_day + nh3_kwh_day + meoh_kwh_day + crk_kwh_day

    res_gen_kwh = float(np.sum((pv_av + wd_av) * 1000.0))
    res_used_kwh = float(np.sum((pv_av - day_hourly["pv_curt_mw"].values + wd_av - day_hourly["wind_curt_mw"].values) * 1000.0))
    res_curt_kwh = float(np.sum((day_hourly["pv_curt_mw"].values + day_hourly["wind_curt_mw"].values) * 1000.0))
    grid_import_kwh = float(np.sum(day_hourly["grid_import_mw"].values * 1000.0))
    grid_share_percent = 100.0 * grid_import_kwh / max(grid_import_kwh + res_used_kwh, 1e-9)

    H2_g_sum = float(np.sum(day_hourly["H2_prod_g_kgph"].values))
    H2_ng_sum = float(np.sum(day_hourly["H2_prod_ng_kgph"].values))
    NH3_g_sum = float(np.sum(day_hourly["NH3_prod_g_kgph"].values))
    NH3_ng_sum = float(np.sum(day_hourly["NH3_prod_ng_kgph"].values))
    Me_g_sum = float(np.sum(day_hourly["MeOH_prod_g_kgph"].values))
    Me_ng_sum = float(np.sum(day_hourly["MeOH_prod_ng_kgph"].values))

    def green_share(sum_g, sum_ng):
        tot = sum_g + sum_ng
        return (sum_g / tot) if tot > 1e-9 else np.nan

    daily_rows.append({
        "run_id": RUN_ID,
        "model_tag": MODEL_TAG,
        "scenario_tag": SCENARIO_TAG,
        "demand_tag": DEMAND_TAG,
        "day_of_year": d,
        "date_utc": str(day_hours[0].date()),
        "has_incumbent": bool(has_incumbent),
        "is_optimal": bool(is_optimal),
        "hit_time_limit": bool(hit_time_limit),
        "mip_gap": float(mip_gap) if mip_gap is not None else np.nan,
        "is_within_target_gap": bool(is_within_target_gap),
        "is_acceptable": bool(is_acceptable),

        "total_cost_eur": total_cost,
        "obj1_total_unmet_kg": total_unmet_kg,
        "obj2_operating_cost_eur": total_cost,

        "res_generation_kWh": res_gen_kwh,
        "res_used_kWh": res_used_kwh,
        "res_curtailed_kWh": res_curt_kwh,
        "grid_import_kWh": grid_import_kwh,
        "grid_share_percent": grid_share_percent,

        "battery_soc_avg": float(np.mean(day_hourly["battery_soc"].values)),
        "tank_H2_level_avg_kg": float(np.mean(day_hourly["tank_H2_g_kg"].values + day_hourly["tank_H2_ng_kg"].values)),
        "tank_NH3_level_avg_kg": float(np.mean(day_hourly["tank_NH3_g_kg"].values + day_hourly["tank_NH3_ng_kg"].values)),
        "tank_MeOH_level_avg_kg": float(np.mean(day_hourly["tank_MeOH_g_kg"].values + day_hourly["tank_MeOH_ng_kg"].values)),

        "H2_prod_green_share": green_share(H2_g_sum, H2_ng_sum),
        "NH3_prod_green_share": green_share(NH3_g_sum, NH3_ng_sum),
        "MeOH_prod_green_share": green_share(Me_g_sum, Me_ng_sum),

        # grid emissions only, by signal
        "emissions_grid_aef_kg": grid_emis_aef,
        "emissions_grid_mef_upramp_kg": grid_emis_mef_upramp,
        "emissions_grid_mef_soft_kg": grid_emis_mef_soft,

        # direct emissions separate
        "emissions_direct_nh3_kg": direct_emis_nh3,
        "emissions_direct_meoh_kg": direct_emis_meoh,
        "emissions_direct_total_kg": direct_emis_total,

        "unmet_H2_kg": float(np.sum(day_hourly["unmet_h2_kgph"].values)),
        "unmet_NH3_kg": float(np.sum(day_hourly["unmet_nh3_kgph"].values)),
        "unmet_MeOH_kg": float(np.sum(day_hourly["unmet_meoh_kgph"].values)),

        "demand_H2_kg_day": demand_H2_kg_day,
        "demand_NH3_kg_day": demand_NH3_kg_day,
        "demand_MeOH_kg_day": demand_MeOH_kg_day,

        "elz_consumption_kWh_day": elz_kwh_day,
        "nh3_consumption_kWh_day": nh3_kwh_day,
        "meoh_consumption_kWh_day": meoh_kwh_day,
        "crk_consumption_kWh_day": crk_kwh_day,
        "units_total_consumption_kWh_day": units_kwh_day,

        # sales accounting
        "H2_sell_g_kg_day": float(np.sum(day_hourly["H2_sell_g_kgph"].values)),
        "H2_sell_ng_kg_day": float(np.sum(day_hourly["H2_sell_ng_kgph"].values)),
        "H2_sell_total_kg_day": float(np.sum(day_hourly["H2_sell_kgph"].values)),

        "NH3_sell_g_kg_day": float(np.sum(day_hourly["NH3_sell_g_kgph"].values)),
        "NH3_sell_ng_kg_day": float(np.sum(day_hourly["NH3_sell_ng_kgph"].values)),
        "NH3_sell_total_kg_day": float(np.sum(day_hourly["NH3_sell_kgph"].values)),

        "MeOH_sell_g_kg_day": float(np.sum(day_hourly["MeOH_sell_g_kgph"].values)),
        "MeOH_sell_ng_kg_day": float(np.sum(day_hourly["MeOH_sell_ng_kgph"].values)),
        "MeOH_sell_total_kg_day": float(np.sum(day_hourly["MeOH_sell_kgph"].values)),

        "eod_H2_total_kg": float(val(S_H2_g[24]) + val(S_H2_ng[24])),
        "eod_NH3_total_kg": float(val(S_NH3_g[24]) + val(S_NH3_ng[24])),
        "eod_MeOH_total_kg": float(val(S_MEOH_g[24]) + val(S_MEOH_ng[24])),
        "eod_req_H2_kg": float(ALPHA_TANK_FILL * S_H2_CAP),
        "eod_req_NH3_kg": float(ALPHA_TANK_FILL * S_NH3_CAP),
        "eod_req_MeOH_kg": float(ALPHA_TANK_FILL * S_MEOH_CAP),
        "eod_margin_H2_kg": float((val(S_H2_g[24]) + val(S_H2_ng[24])) - ALPHA_TANK_FILL * S_H2_CAP),
        "eod_margin_NH3_kg": float((val(S_NH3_g[24]) + val(S_NH3_ng[24])) - ALPHA_TANK_FILL * S_NH3_CAP),
        "eod_margin_MeOH_kg": float((val(S_MEOH_g[24]) + val(S_MEOH_ng[24])) - ALPHA_TANK_FILL * S_MEOH_CAP),

        "cost_grid_eur": grid_cost_eur,
        "cost_water_eur": water_cost_eur,
        "cost_co2_eur": co2_cost_eur,
        "cost_startup_elz_eur": startup_elz_eur,
        "cost_startup_nh3_eur": startup_nh3_eur,
        "cost_startup_meoh_eur": startup_meoh_eur,
        "cost_startup_crk_eur": startup_crk_eur,
    })

    carry["soc"] = val(SOC[24])
    carry["S_H2_g"] = val(S_H2_g[24]); carry["S_H2_ng"] = val(S_H2_ng[24])
    carry["S_NH3_g"] = val(S_NH3_g[24]); carry["S_NH3_ng"] = val(S_NH3_ng[24])
    carry["S_MEOH_g"] = val(S_MEOH_g[24]); carry["S_MEOH_ng"] = val(S_MEOH_ng[24])
    carry["y_elz_prev_mod"] = [int(round(val(y_elz_m[i,23]))) for i in I_elz]
    carry["y_nh3_prev"]  = int(round(val(y_nh3[23])))
    carry["y_meoh_prev"] = int(round(val(y_meoh[23])))
    carry["y_crk_prev"]  = int(round(val(y_crk[23])))
    carry["Q_nh3_prev"]  = float(val(Q_nh3[23]))
    carry["Q_meoh_prev"] = float(val(Q_meoh[23]))
    carry["Q_crk_prev"]  = float(val(Q_crk_in[23]))

    state_rows.append({
        "run_id": RUN_ID,
        "day_of_year": d,
        "date_utc": str(day_hours[0].date()),
        "SOC_end": carry["soc"],
        "S_H2_g_end": carry["S_H2_g"],
        "S_H2_ng_end": carry["S_H2_ng"],
        "S_NH3_g_end": carry["S_NH3_g"],
        "S_NH3_ng_end": carry["S_NH3_ng"],
        "S_MEOH_g_end": carry["S_MEOH_g"],
        "S_MEOH_ng_end": carry["S_MEOH_ng"],
        "y_elz_prev_mod_end": carry["y_elz_prev_mod"],
        "y_nh3_end": carry["y_nh3_prev"],
        "y_meoh_end": carry["y_meoh_prev"],
        "y_crk_end": carry["y_crk_prev"],
        "Q_nh3_end": carry["Q_nh3_prev"],
        "Q_meoh_end": carry["Q_meoh_prev"],
        "Q_crk_end": carry["Q_crk_prev"],
    })

    eps = 1e-6
    sim_cd = float(np.sum((day_hourly["bat_ch_mw"].values > eps) & (day_hourly["bat_dis_mw"].values > eps)))
    soc_viol = float(np.sum((day_hourly["battery_soc"].values < SOC_MIN - 1e-6) | (day_hourly["battery_soc"].values > SOC_MAX + 1e-6)))
    unmet_any = float(np.sum((day_hourly["unmet_h2_kgph"].values > eps) | (day_hourly["unmet_nh3_kgph"].values > eps) | (day_hourly["unmet_meoh_kgph"].values > eps)))

    diag_rows.append({
        "run_id": RUN_ID,
        "day_of_year": d,
        "date_utc": str(day_hours[0].date()),
        "flag_simultaneous_charge_discharge_hours": sim_cd,
        "flag_soc_bound_violation_hours": soc_viol,
        "flag_unmet_demand_hours": unmet_any,
    })

# ---------------- End of run aggregation ----------------
daily_df = pd.DataFrame(daily_rows)
if daily_df.empty:
    raise RuntimeError("No days were solved (daily_df is empty).")

solver_df = pd.DataFrame(solver_rows)
state_df  = pd.DataFrame(state_rows)
diag_df   = pd.DataFrame(diag_rows)

if len(all_hourly_frames) > 0:
    hourly_all_df = pd.concat(all_hourly_frames, ignore_index=True)
    hourly_all_df.to_csv(OUT_POST / "hourly_timeseries_all_days.csv", index=False)
else:
    hourly_all_df = pd.DataFrame()

daily_df.to_csv(OUT_POST / "daily_summary.csv", index=False)
solver_df.to_csv(OUT_POST / "solver_log_daily.csv", index=False)
state_df.to_csv(OUT_POST / "state_trace.csv", index=False)
diag_df.to_csv(OUT_POST / "daily_diagnostics_flags.csv", index=False)

annual_summary = {
    "run_id": RUN_ID,
    "model_tag": MODEL_TAG,
    "scenario_tag": SCENARIO_TAG,
    "demand_tag": DEMAND_TAG,
    "start_day_of_year": START_DAY_OF_YEAR,
    "n_days_run": int(len(daily_df)),

    "total_cost_eur": float(daily_df["total_cost_eur"].sum()),

    "total_grid_emissions_aef_kg": float(daily_df["emissions_grid_aef_kg"].sum()),
    "total_grid_emissions_mef_upramp_kg": float(daily_df["emissions_grid_mef_upramp_kg"].sum()),
    "total_grid_emissions_mef_soft_kg": float(daily_df["emissions_grid_mef_soft_kg"].sum()),

    "total_direct_emissions_nh3_kg": float(daily_df["emissions_direct_nh3_kg"].sum()),
    "total_direct_emissions_meoh_kg": float(daily_df["emissions_direct_meoh_kg"].sum()),
    "total_direct_emissions_kg": float(daily_df["emissions_direct_total_kg"].sum()),

    "total_grid_import_kWh": float(daily_df["grid_import_kWh"].sum()),
    "total_res_used_kWh": float(daily_df["res_used_kWh"].sum()),
    "total_res_curtailed_kWh": float(daily_df["res_curtailed_kWh"].sum()),
    "avg_grid_share_percent": float(daily_df["grid_share_percent"].mean()),

    "total_unmet_H2_kg": float(daily_df["unmet_H2_kg"].sum()),
    "total_unmet_NH3_kg": float(daily_df["unmet_NH3_kg"].sum()),
    "total_unmet_MeOH_kg": float(daily_df["unmet_MeOH_kg"].sum()),

    "total_H2_sell_g_kg": float(daily_df["H2_sell_g_kg_day"].sum()),
    "total_H2_sell_ng_kg": float(daily_df["H2_sell_ng_kg_day"].sum()),
    "total_H2_sell_kg": float(daily_df["H2_sell_total_kg_day"].sum()),

    "total_NH3_sell_g_kg": float(daily_df["NH3_sell_g_kg_day"].sum()),
    "total_NH3_sell_ng_kg": float(daily_df["NH3_sell_ng_kg_day"].sum()),
    "total_NH3_sell_kg": float(daily_df["NH3_sell_total_kg_day"].sum()),

    "total_MeOH_sell_g_kg": float(daily_df["MeOH_sell_g_kg_day"].sum()),
    "total_MeOH_sell_ng_kg": float(daily_df["MeOH_sell_ng_kg_day"].sum()),
    "total_MeOH_sell_kg": float(daily_df["MeOH_sell_total_kg_day"].sum()),
}
pd.DataFrame([annual_summary]).to_csv(OUT_POST / "annual_summary.csv", index=False)

# ---------------- Plots ----------------
daily_df = daily_df.copy()
daily_df["date"] = pd.to_datetime(daily_df["date_utc"])

save_lineplot(daily_df, "date", ["total_cost_eur"], f"{RUN_ID} • Daily total cost", "EUR", OUT_PLOTS / "P01_daily_total_cost.png")

if "obj1_total_unmet_kg" in daily_df.columns:
    save_lineplot(daily_df, "date", ["obj1_total_unmet_kg"], f"{RUN_ID} • Objective-1 total unmet (daily)", "kg/day", OUT_PLOTS / "P07_obj1_total_unmet_daily.png")

# daily grid emissions only, 3 signals
tmp = daily_df[["date","emissions_grid_aef_kg","emissions_grid_mef_upramp_kg","emissions_grid_mef_soft_kg"]]
save_lineplot(tmp, "date", ["emissions_grid_aef_kg","emissions_grid_mef_upramp_kg","emissions_grid_mef_soft_kg"],
              f"{RUN_ID} • Daily grid emissions by signal", "kg CO2/day",
              OUT_PLOTS / "P02_daily_grid_emissions_three_signals.png")

tmp = daily_df[["date","emissions_direct_nh3_kg","emissions_direct_meoh_kg"]]
save_lineplot(tmp, "date", ["emissions_direct_nh3_kg","emissions_direct_meoh_kg"],
              f"{RUN_ID} • Daily direct emissions by unit", "kg CO2/day",
              OUT_PLOTS / "P17_direct_emissions_daily_total.png")

tmp = daily_df[["date","grid_import_kWh"]]
save_lineplot(tmp, "date", ["grid_import_kWh"], f"{RUN_ID} • Daily grid import", "kWh", OUT_PLOTS / "P03_daily_grid_import_kwh.png")

tmp = daily_df[["date","res_used_kWh","res_curtailed_kWh"]]
save_lineplot(tmp, "date", ["res_used_kWh","res_curtailed_kWh"],
              f"{RUN_ID} • Daily RES used vs curtailed", "kWh",
              OUT_PLOTS / "P04_daily_res_used_vs_curtailed.png")

solver_df = solver_df.copy()
solver_df["date"] = pd.to_datetime(solver_df["date_utc"])
tmp = solver_df[["date","runtime_seconds","mip_gap"]]
save_lineplot(tmp, "date", ["runtime_seconds"], f"{RUN_ID} • Daily solver runtime", "seconds", OUT_PLOTS / "P05_daily_solver_runtime.png")
save_lineplot(tmp, "date", ["mip_gap"], f"{RUN_ID} • Daily solver MIP gap", "-", OUT_PLOTS / "P06_daily_mip_gap.png")

if hourly_all_df is not None and len(hourly_all_df) > 0:
    hdf = hourly_all_df.copy()
    hdf["dt"] = pd.to_datetime(hdf["datetime_utc"], utc=True)
    hdf = hdf.sort_values("dt")
else:
    hdf = pd.DataFrame()

if len(hdf) > 0:
    hh = hdf.copy()
    hh["tank_H2_total_kg"]   = hh["tank_H2_g_kg"].values + hh["tank_H2_ng_kg"].values
    hh["tank_NH3_total_kg"]  = hh["tank_NH3_g_kg"].values + hh["tank_NH3_ng_kg"].values
    hh["tank_MeOH_total_kg"] = hh["tank_MeOH_g_kg"].values + hh["tank_MeOH_ng_kg"].values

    hh["demand_H2_kgph"]   = hh["H2_sell_kgph"].values + hh["unmet_h2_kgph"].values
    hh["demand_NH3_kgph"]  = hh["NH3_sell_kgph"].values + hh["unmet_nh3_kgph"].values
    hh["demand_MeOH_kgph"] = hh["MeOH_sell_kgph"].values + hh["unmet_meoh_kgph"].values

    hh["pct_demand_met_H2"] = np.where(hh["demand_H2_kgph"].values > 1e-9, 100.0 * (1.0 - hh["unmet_h2_kgph"].values / hh["demand_H2_kgph"].values), 100.0)
    hh["pct_demand_met_NH3"] = np.where(hh["demand_NH3_kgph"].values > 1e-9, 100.0 * (1.0 - hh["unmet_nh3_kgph"].values / hh["demand_NH3_kgph"].values), 100.0)
    hh["pct_demand_met_MeOH"] = np.where(hh["demand_MeOH_kgph"].values > 1e-9, 100.0 * (1.0 - hh["unmet_meoh_kgph"].values / hh["demand_MeOH_kgph"].values), 100.0)

    for c in ["pct_demand_met_H2","pct_demand_met_NH3","pct_demand_met_MeOH"]:
        hh[c] = hh[c].clip(lower=0.0, upper=100.0)

    hh["pct_tank_fill_H2"] = 100.0 * hh["tank_H2_total_kg"].values / float(S_H2_CAP)
    hh["pct_tank_fill_NH3"] = 100.0 * hh["tank_NH3_total_kg"].values / float(S_NH3_CAP)
    hh["pct_tank_fill_MeOH"] = 100.0 * hh["tank_MeOH_total_kg"].values / float(S_MEOH_CAP)

    for c in ["pct_tank_fill_H2","pct_tank_fill_NH3","pct_tank_fill_MeOH"]:
        hh[c] = hh[c].clip(lower=0.0, upper=100.0)

    binary_cols_present = [c for c in ["su_crk","y_crk","su_meoh","y_meoh","su_nh3","y_nh3","su_elz","y_elz","b_ch"] if c in hh.columns]

    ops_cols = (
        ["run_id","model_tag","scenario_tag","demand_tag","dt","day_id","day_of_year","hour_in_day"]
        + binary_cols_present
        + ["pct_demand_met_H2","pct_demand_met_NH3","pct_demand_met_MeOH",
           "pct_tank_fill_H2","pct_tank_fill_NH3","pct_tank_fill_MeOH"]
    )
    ops_cols = [c for c in ops_cols if c in hh.columns]
    hh[ops_cols].copy().to_csv(OUT_POST / "hourly_ops_signals.csv", index=False)

if len(hdf) > 0:
    for prod_col, fname_stub, title_stub, ylab in [
        ("H2_prod_kgph",  "H2",  "H2 production",  "kg/h"),
        ("NH3_prod_kgph", "NH3", "NH3 production", "kg/h"),
        ("MeOH_prod_kgph","MeOH","MeOH production","kg/h"),
    ]:
        tmp = hdf[["dt", prod_col]]
        save_lineplot(tmp, "dt", [prod_col], f"{RUN_ID} • {title_stub} (hourly)", ylab,
                      OUT_PLOTS / f"P08_prod_{fname_stub}_hourly.png")

    # hourly sell diagnostics
    for sell_cols, fname_stub, title_stub in [
        (["H2_sell_g_kgph","H2_sell_ng_kgph"], "H2", "H2 sales g/ng"),
        (["NH3_sell_g_kgph","NH3_sell_ng_kgph"], "NH3", "NH3 sales g/ng"),
        (["MeOH_sell_g_kgph","MeOH_sell_ng_kgph"], "MeOH", "MeOH sales g/ng"),
    ]:
        tmp = hdf[["dt"] + sell_cols].copy()
        save_lineplot(tmp, "dt", sell_cols, f"{RUN_ID} • {title_stub} (hourly)", "kg/h",
                      OUT_PLOTS / f"P08b_sales_{fname_stub}_hourly_gng.png")

    prod_daily = (
        hdf.groupby("day_id")[["H2_prod_kgph","NH3_prod_kgph","MeOH_prod_kgph"]]
        .sum()
        .rename(columns={
            "H2_prod_kgph":"H2_prod_kg_day",
            "NH3_prod_kgph":"NH3_prod_kg_day",
            "MeOH_prod_kgph":"MeOH_prod_kg_day",
        })
        .reset_index()
        .merge(daily_df[["day_of_year","date"]], left_on="day_id", right_on="day_of_year", how="left")
    )

    for col, fname_stub, title_stub in [
        ("H2_prod_kg_day",  "H2",  "H2 production"),
        ("NH3_prod_kg_day", "NH3", "NH3 production"),
        ("MeOH_prod_kg_day","MeOH","MeOH production"),
    ]:
        tmp = prod_daily[["date", col]].dropna()
        save_lineplot(tmp, "date", [col], f"{RUN_ID} • {title_stub} (daily total)", "kg/day",
                      OUT_PLOTS / f"P09_prod_{fname_stub}_daily_total.png")

    sell_daily = (
        hdf.groupby("day_id")[[
            "H2_sell_g_kgph","H2_sell_ng_kgph",
            "NH3_sell_g_kgph","NH3_sell_ng_kgph",
            "MeOH_sell_g_kgph","MeOH_sell_ng_kgph"
        ]]
        .sum()
        .rename(columns={
            "H2_sell_g_kgph":"H2_sell_g_kg_day",
            "H2_sell_ng_kgph":"H2_sell_ng_kg_day",
            "NH3_sell_g_kgph":"NH3_sell_g_kg_day",
            "NH3_sell_ng_kgph":"NH3_sell_ng_kg_day",
            "MeOH_sell_g_kgph":"MeOH_sell_g_kg_day",
            "MeOH_sell_ng_kgph":"MeOH_sell_ng_kg_day",
        })
        .reset_index()
        .merge(daily_df[["day_of_year","date"]], left_on="day_id", right_on="day_of_year", how="left")
    )

    for cols, fname_stub, title_stub in [
        (["H2_sell_g_kg_day","H2_sell_ng_kg_day"], "H2", "H2 sales g/ng"),
        (["NH3_sell_g_kg_day","NH3_sell_ng_kg_day"], "NH3", "NH3 sales g/ng"),
        (["MeOH_sell_g_kg_day","MeOH_sell_ng_kg_day"], "MeOH", "MeOH sales g/ng"),
    ]:
        tmp = sell_daily[["date"] + cols].dropna()
        save_lineplot(tmp, "date", cols, f"{RUN_ID} • {title_stub} (daily total)", "kg/day",
                      OUT_PLOTS / f"P09b_sales_{fname_stub}_daily_total_gng.png")

    hdf["tank_H2_total"] = hdf["tank_H2_g_kg"].values + hdf["tank_H2_ng_kg"].values
    hdf["tank_NH3_total"] = hdf["tank_NH3_g_kg"].values + hdf["tank_NH3_ng_kg"].values
    hdf["tank_MeOH_total"] = hdf["tank_MeOH_g_kg"].values + hdf["tank_MeOH_ng_kg"].values

    hdf["tank_H2_green_share"] = np.where(hdf["tank_H2_total"] > 1e-9, hdf["tank_H2_g_kg"] / hdf["tank_H2_total"], np.nan)
    hdf["tank_NH3_green_share"] = np.where(hdf["tank_NH3_total"] > 1e-9, hdf["tank_NH3_g_kg"] / hdf["tank_NH3_total"], np.nan)
    hdf["tank_MeOH_green_share"] = np.where(hdf["tank_MeOH_total"] > 1e-9, hdf["tank_MeOH_g_kg"] / hdf["tank_MeOH_total"], np.nan)

    for col, fname_stub, title_stub in [
        ("tank_H2_green_share", "H2", "Tank H2 green share"),
        ("tank_NH3_green_share", "NH3", "Tank NH3 green share"),
        ("tank_MeOH_green_share", "MeOH", "Tank MeOH green share"),
    ]:
        tmp = hdf[["dt", col]].copy()
        tmp[col] = tmp[col].fillna(0.0)
        save_lineplot(tmp, "dt", [col], f"{RUN_ID} • {title_stub} (hourly)", "share",
                      OUT_PLOTS / f"P10_tank_share_{fname_stub}_hourly.png")

for col, fname_stub, title_stub, ylab in [
    ("demand_H2_kg_day", "H2", "H2 demand", "kg/day"),
    ("demand_NH3_kg_day", "NH3", "NH3 demand", "kg/day"),
    ("demand_MeOH_kg_day", "MeOH", "MeOH demand", "kg/day"),
]:
    tmp = daily_df[["date", col]]
    save_lineplot(tmp, "date", [col], f"{RUN_ID} • {title_stub} (daily)", ylab,
                  OUT_PLOTS / f"P12_demand_{fname_stub}_daily.png")

for col, fname_stub, title_stub in [
    ("elz_consumption_kWh_day", "elz", "Electrolyzer electricity"),
    ("nh3_consumption_kWh_day", "nh3", "NH3 synthesis electricity"),
    ("meoh_consumption_kWh_day", "meoh", "MeOH synthesis electricity"),
    ("crk_consumption_kWh_day", "crk", "Cracker electricity"),
]:
    tmp = daily_df[["date", col]]
    save_lineplot(tmp, "date", [col], f"{RUN_ID} • {title_stub} (daily)", "kWh/day",
                  OUT_PLOTS / f"P14_power_{fname_stub}_daily.png")

tmp = daily_df[["date","elz_consumption_kWh_day","nh3_consumption_kWh_day","meoh_consumption_kWh_day","crk_consumption_kWh_day"]]
save_lineplot(tmp, "date", ["elz_consumption_kWh_day","nh3_consumption_kWh_day","meoh_consumption_kWh_day","crk_consumption_kWh_day"],
              f"{RUN_ID} • Unit electricity consumption (daily)", "kWh/day",
              OUT_PLOTS / "P15_power_units_combined_daily.png")

if len(hdf) > 0:
    hdf["direct_emis_nh3_kgph"]  = float(de_NH3) * hdf["NH3_prod_kgph"].values
    hdf["direct_emis_meoh_kgph"] = float(de_MEOH) * hdf["MeOH_prod_kgph"].values

    tmp = hdf[["dt","direct_emis_nh3_kgph","direct_emis_meoh_kgph"]]
    save_lineplot(tmp, "dt", ["direct_emis_nh3_kgph","direct_emis_meoh_kgph"],
                  f"{RUN_ID} • Direct emissions by unit (hourly)", "kg CO2/h",
                  OUT_PLOTS / "P16_direct_emissions_hourly.png")

    # merge three signals to hdf
    aef_series = aef[[AEF_COL]].reset_index().rename(columns={"datetime_utc":"dt", AEF_COL:"aef_kg_per_kwh"})
    mef_upramp_series = mef_upramp[[MEF_UPRAMP_COL]].reset_index().rename(columns={"datetime_utc":"dt", MEF_UPRAMP_COL:"mef_upramp_kg_per_kwh"})
    mef_soft_series = mef_soft[[MEF_SOFT_COL]].reset_index().rename(columns={"datetime_utc":"dt", MEF_SOFT_COL:"mef_soft_kg_per_kwh"})

    aef_series["dt"] = pd.to_datetime(aef_series["dt"], utc=True)
    mef_upramp_series["dt"] = pd.to_datetime(mef_upramp_series["dt"], utc=True)
    mef_soft_series["dt"] = pd.to_datetime(mef_soft_series["dt"], utc=True)

    hdf["dt"] = pd.to_datetime(hdf["dt"], utc=True)
    hdf = hdf.merge(aef_series[["dt","aef_kg_per_kwh"]], on="dt", how="left")
    hdf = hdf.merge(mef_upramp_series[["dt","mef_upramp_kg_per_kwh"]], on="dt", how="left")
    hdf = hdf.merge(mef_soft_series[["dt","mef_soft_kg_per_kwh"]], on="dt", how="left")

    # allocated grid emissions by unit for all 3 signals
    for tag, ef_col in [
        ("aef", "aef_kg_per_kwh"),
        ("mef_upramp", "mef_upramp_kg_per_kwh"),
        ("mef_soft", "mef_soft_kg_per_kwh"),
    ]:
        hdf[f"grid_emis_{tag}_elz_kgph"]  = 1000.0 * hdf["elz_grid_mw"].values  * hdf[ef_col].values
        hdf[f"grid_emis_{tag}_nh3_kgph"]  = 1000.0 * hdf["nh3_grid_mw"].values  * hdf[ef_col].values
        hdf[f"grid_emis_{tag}_meoh_kgph"] = 1000.0 * hdf["meoh_grid_mw"].values * hdf[ef_col].values
        hdf[f"grid_emis_{tag}_crk_kgph"]  = 1000.0 * hdf["crk_grid_mw"].values  * hdf[ef_col].values

        tmp = hdf[["dt",
                   f"grid_emis_{tag}_elz_kgph",
                   f"grid_emis_{tag}_nh3_kgph",
                   f"grid_emis_{tag}_meoh_kgph",
                   f"grid_emis_{tag}_crk_kgph"]].copy()

        save_lineplot(tmp, "dt",
                      [f"grid_emis_{tag}_elz_kgph", f"grid_emis_{tag}_nh3_kgph", f"grid_emis_{tag}_meoh_kgph", f"grid_emis_{tag}_crk_kgph"],
                      f"{RUN_ID} • Allocated grid emissions by unit (hourly, {tag})", "kg CO2/h",
                      OUT_PLOTS / f"P17b_grid_emissions_by_unit_hourly_{tag}.png")

        daily_alloc = (
            hdf.groupby("day_id")[
                [f"grid_emis_{tag}_elz_kgph", f"grid_emis_{tag}_nh3_kgph", f"grid_emis_{tag}_meoh_kgph", f"grid_emis_{tag}_crk_kgph"]
            ]
            .sum()
            .rename(columns={
                f"grid_emis_{tag}_elz_kgph": f"grid_emis_{tag}_elz_kg_day",
                f"grid_emis_{tag}_nh3_kgph": f"grid_emis_{tag}_nh3_kg_day",
                f"grid_emis_{tag}_meoh_kgph": f"grid_emis_{tag}_meoh_kg_day",
                f"grid_emis_{tag}_crk_kgph": f"grid_emis_{tag}_crk_kg_day",
            })
            .reset_index()
            .merge(daily_df[["day_of_year","date"]], left_on="day_id", right_on="day_of_year", how="left")
        )

        tmp = daily_alloc[["date",
                           f"grid_emis_{tag}_elz_kg_day",
                           f"grid_emis_{tag}_nh3_kg_day",
                           f"grid_emis_{tag}_meoh_kg_day",
                           f"grid_emis_{tag}_crk_kg_day"]].dropna()

        save_lineplot(tmp, "date",
                      [f"grid_emis_{tag}_elz_kg_day", f"grid_emis_{tag}_nh3_kg_day", f"grid_emis_{tag}_meoh_kg_day", f"grid_emis_{tag}_crk_kg_day"],
                      f"{RUN_ID} • Allocated grid emissions by unit (daily total, {tag})", "kg CO2/day",
                      OUT_PLOTS / f"P17c_grid_emissions_by_unit_daily_{tag}.png")

    # one compact signal comparison diagnostics CSV only
    eps = 1e-9
    hdf["grid_emis_aef_total_kgph"]        = 1000.0 * hdf["grid_import_mw"].values * hdf["aef_kg_per_kwh"].values
    hdf["grid_emis_mef_upramp_total_kgph"] = 1000.0 * hdf["grid_import_mw"].values * hdf["mef_upramp_kg_per_kwh"].values
    hdf["grid_emis_mef_soft_total_kgph"]   = 1000.0 * hdf["grid_import_mw"].values * hdf["mef_soft_kg_per_kwh"].values

    hdf["ef_delta_upramp_minus_aef"] = hdf["mef_upramp_kg_per_kwh"].values - hdf["aef_kg_per_kwh"].values
    hdf["ef_delta_soft_minus_aef"]   = hdf["mef_soft_kg_per_kwh"].values   - hdf["aef_kg_per_kwh"].values

    hdf["grid_emis_delta_upramp_minus_aef_kgph"] = hdf["grid_emis_mef_upramp_total_kgph"].values - hdf["grid_emis_aef_total_kgph"].values
    hdf["grid_emis_delta_soft_minus_aef_kgph"]   = hdf["grid_emis_mef_soft_total_kgph"].values   - hdf["grid_emis_aef_total_kgph"].values

    hdf["cum_delta_upramp_minus_aef_kg"] = np.cumsum(hdf["grid_emis_delta_upramp_minus_aef_kgph"].fillna(0.0).values)
    hdf["cum_delta_soft_minus_aef_kg"]   = np.cumsum(hdf["grid_emis_delta_soft_minus_aef_kgph"].fillna(0.0).values)

    tmp = hdf[["dt","cum_delta_upramp_minus_aef_kg","cum_delta_soft_minus_aef_kg"]]
    save_lineplot(tmp, "dt", ["cum_delta_upramp_minus_aef_kg","cum_delta_soft_minus_aef_kg"],
                  f"{RUN_ID} • Cumulative Δ grid emissions vs AEF", "kg CO2",
                  OUT_PLOTS / "P02e_cumulative_delta_grid_emissions_vs_aef.png")

    hdf[[
        "run_id","model_tag","scenario_tag","demand_tag","dt","day_id","day_of_year","hour_in_day",
        "grid_import_mw",
        "aef_kg_per_kwh","mef_upramp_kg_per_kwh","mef_soft_kg_per_kwh",
        "grid_emis_aef_total_kgph","grid_emis_mef_upramp_total_kgph","grid_emis_mef_soft_total_kgph",
        "ef_delta_upramp_minus_aef","ef_delta_soft_minus_aef",
        "grid_emis_delta_upramp_minus_aef_kgph","grid_emis_delta_soft_minus_aef_kgph",
        "cum_delta_upramp_minus_aef_kg","cum_delta_soft_minus_aef_kg"
    ]].to_csv(OUT_POST / "hourly_emission_signal_diagnostics.csv", index=False)

if not hdf.empty:
    mod_ids = sorted({int(re.match(r"^elz_m(\d+)_y$", c).group(1)) for c in hdf.columns if re.match(r"^elz_m(\d+)_y$", c)})
    mod_ids = [i for i in mod_ids if (f"elz_m{i}_y" in hdf.columns) and (f"elz_m{i}_b_g" in hdf.columns)]

    if len(mod_ids) > 0:
        fig = plt.figure(figsize=(12, 6))
        ax1 = fig.add_subplot(2, 1, 1)
        for i in mod_ids:
            ax1.plot(hdf["dt"], hdf[f"elz_m{i}_y"].values, label=f"m{i}_y")
        ax1.set_title(f"{RUN_ID} • ELZ modules on/off (y)")
        ax1.set_ylabel("binary")
        ax1.legend(ncol=4, fontsize=8)

        ax2 = fig.add_subplot(2, 1, 2)
        for i in mod_ids:
            ax2.plot(hdf["dt"], hdf[f"elz_m{i}_b_g"].values, label=f"m{i}_b_g")
        ax2.set_title(f"{RUN_ID} • ELZ modules green-choice (b_g)")
        ax2.set_ylabel("binary")
        ax2.legend(ncol=4, fontsize=8)

        plt.tight_layout()
        plt.savefig(OUT_PLOTS / "P90_elz_modules_y_and_bg.png", dpi=150)
        plt.close()

        i0 = mod_ids[0]
        needed = [f"elz_m{i0}_P_tot", f"elz_m{i0}_P_g", f"elz_m{i0}_P_ng"]
        if all(c in hdf.columns for c in needed):
            tmp = hdf[["dt"] + needed].copy()
            save_lineplot(tmp, "dt", needed,
                          f"{RUN_ID} • ELZ module m{i0} power split (P_tot, P_g, P_ng)",
                          "MW",
                          OUT_PLOTS / f"P91_elz_module_m{i0}_power_split.png")

cost_cols = [
    ("cost_grid_eur", "Grid energy"),
    ("cost_water_eur", "Water"),
    ("cost_co2_eur", "CO2"),
    ("cost_startup_elz_eur", "Startup ELZ"),
    ("cost_startup_nh3_eur", "Startup NH3"),
    ("cost_startup_meoh_eur", "Startup MeOH"),
    ("cost_startup_crk_eur", "Startup CRK"),
]
save_stacked_bar_by_day(daily_df, "date", cost_cols,
                        f"{RUN_ID} • Daily cost breakdown (stacked)", "EUR/day",
                        OUT_PLOTS / "P18_daily_cost_breakdown_stacked.png")

res_cols = [
    ("res_used_kWh", "RES used"),
    ("res_curtailed_kWh", "RES curtailed"),
]
save_stacked_bar_by_day(daily_df, "date", res_cols,
                        f"{RUN_ID} • Daily RES split (used vs curtailed)", "kWh/day",
                        OUT_PLOTS / "P19_daily_res_used_vs_curtailed_stacked.png")

ZIP_PATH = OUT_BASE / f"{RUN_ID}.zip"
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in OUT_RUN.rglob("*"):
        if p.is_file():
            zf.write(p, arcname=str(p.relative_to(OUT_RUN)))

print("Run completed.")
print("Run folder:", OUT_RUN)
print("ZIP:", ZIP_PATH)
print("Total wall time (s):", time.time() - t0_global)

## 5.2 M1 emission reference model


In [ ]:
# ===================== M1 ====================
# Assumes utility/precompute cell has already run and wrote /content/data/model_ready/*.csv and params_scenario_all.json

# min unmet >> min emissions, for lower bound of emission for M2

# ---------------- Header / metadata ----------------
MODEL_TAG = "M1"
FORMULATION_TAG = "M1_final_v2"
SCENARIO_TAG = "S0"
DEMAND_TAG = "D1"
GREEN_TAG = "S0_green_split_with_rfnbo_hourly"

# Choose which EF to use for the emissions objective:
# "aef" or "mef_upramp" or "mef_soft"
EMIS_EF_MODE = "mef_upramp"

# ---------------- Imports ----------------
import os, json, math, zipfile, shutil, time, re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gurobipy as gp
from gurobipy import GRB

# ---------------- Global configuration ----------------
DATA_DIR = Path("/content/data")
MR = DATA_DIR / "model_ready"

# Required model-ready files (from utility cell)
RES_CSV          = MR / "res_hourly_2024.csv"
DEM_CSV          = MR / "demand_hourly_2024.csv"
PRICE_CSV        = MR / "grid_price_hourly_2024.csv"
AEF_CSV          = MR / "grid_aef_hourly_2024.csv"
MEF_UPRAMP_CSV   = MR / "grid_mef_upramp_hourly_2024.csv"
MEF_SOFT_CSV     = MR / "grid_mef_soft_hourly_2024.csv"
PARAMS_JSON      = MR / "params_scenario_all.json"

AEF_COL          = "grid_aef_predicted_kg_per_kwh"
MEF_UPRAMP_COL   = "grid_mef_upramp_pred_kg_per_kwh"
MEF_SOFT_COL     = "grid_mef_soft_kg_per_kwh"

# ---------------- Run controls ----------------
N_DAYS_TO_RUN     = 366
START_DAY_OF_YEAR = 1
MIPGAP            = 0.01
TIME_LIMIT_S      = 350
OUTPUTFLAG        = 1

# ---------------- License handling ----------------
for lic_path in [Path("/content/data/gurobi.lic")]:
    if lic_path.exists():
        os.environ["GRB_LICENSE_FILE"] = str(lic_path)
        break

# ---------------- Run ID builder ----------------
def build_run_id():
    stamp = pd.Timestamp.utcnow().strftime("%Y%m%dT%H%M%SZ")
    return f"{MODEL_TAG}__{SCENARIO_TAG}__{DEMAND_TAG}__start{START_DAY_OF_YEAR:03d}__ndays{N_DAYS_TO_RUN:03d}__{stamp}"

RUN_ID = build_run_id()

# ---------------- Paths for outputs ----------------
OUT_BASE  = DATA_DIR / "stage2_runs"
OUT_RUN   = OUT_BASE / RUN_ID
OUT_DAILY = OUT_RUN / "daily"
OUT_POST  = OUT_RUN / "post"
OUT_PLOTS = OUT_RUN / "plots"
OUT_RUN.mkdir(parents=True, exist_ok=True)
OUT_DAILY.mkdir(parents=True, exist_ok=True)
OUT_POST.mkdir(parents=True, exist_ok=True)
OUT_PLOTS.mkdir(parents=True, exist_ok=True)

# ---------------- Input loading ----------------
def read_hourly_csv(path: Path, time_col="datetime_utc") -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    df = pd.read_csv(path)
    if time_col not in df.columns:
        raise ValueError(f"{path.name}: missing '{time_col}'. Found: {list(df.columns)}")
    df[time_col] = pd.to_datetime(df[time_col], utc=True)
    df = df.sort_values(time_col).set_index(time_col)
    df = df.tz_convert("UTC") if df.index.tz is not None else df.tz_localize("UTC")
    return df

res        = read_hourly_csv(RES_CSV)[["pv_mw","wind_mw","res_total_mw"]]
dem        = read_hourly_csv(DEM_CSV)[["h2_demand_kgph","nh3_demand_kgph","meoh_demand_kgph"]]
price      = read_hourly_csv(PRICE_CSV)[["da_elec_price_EURperMWh"]]
aef        = read_hourly_csv(AEF_CSV)[[AEF_COL]]
mef_upramp = read_hourly_csv(MEF_UPRAMP_CSV)[[MEF_UPRAMP_COL]]
mef_soft   = read_hourly_csv(MEF_SOFT_CSV)[[MEF_SOFT_COL]]

with open(PARAMS_JSON, "r") as f:
    params = json.load(f)

# ---------------- Input validation & alignment ----------------
idx_2024 = pd.date_range("2024-01-01 00:00:00+00:00", "2024-12-31 23:00:00+00:00", freq="h", tz="UTC")

def assert_full_hourly(df: pd.DataFrame, name: str) -> pd.DataFrame:
    if df.index.duplicated().any():
        dup = df.index[df.index.duplicated()][0]
        raise ValueError(f"{name}: duplicated timestamp example: {dup}")
    missing = idx_2024.difference(df.index)
    extra   = df.index.difference(idx_2024)
    if len(missing) > 0:
        raise ValueError(f"{name}: missing {len(missing)} hours (first 5: {missing[:5].tolist()})")
    if len(extra) > 0:
        raise ValueError(f"{name}: extra {len(extra)} hours outside 2024 (first 5: {extra[:5].tolist()})")

    df2 = df.reindex(idx_2024)
    df2.index.name = "datetime_utc"

    if df2.isna().any().any():
        cols = df2.columns[df2.isna().any()].tolist()
        where = df2[df2[cols].isna().any(axis=1)].index[:5].tolist()
        raise ValueError(f"{name}: NaNs after reindex in cols {cols}; example {where}")
    return df2

res        = assert_full_hourly(res, "res")
dem        = assert_full_hourly(dem, "dem")
price      = assert_full_hourly(price, "price")
aef        = assert_full_hourly(aef, "aef")
mef_upramp = assert_full_hourly(mef_upramp, "mef_upramp")
mef_soft   = assert_full_hourly(mef_soft, "mef_soft")

# ---------------- Scalar parameters (aligned with M0-V2) ----------------
P_GRID_MAX_MW = 295.58

# Battery
P_BAT_MAX_MW  = 50.0
E_BAT_MAX_MWH = 100.32
ETA_CH        = 0.964
ETA_DIS       = 0.964
SOC_MIN       = 0.15
SOC_MAX       = 0.95
SOC_INIT      = 0.5

# Electrolyzer
P_ELZ_MIN = 8
P_ELZ_MAX = 52.25
H_ELZ_MAX = 914.39161
A_elz = -0.075045052
B_elz = 21.73040904
C_elz = -16.14458074
N_stack = 4
omega_H2O = 0.015
c_H2O = 2.24
C_su_elz = 1392

P_ELZ_MAX_TOT = N_stack * P_ELZ_MAX
H_ELZ_MAX_TOT = N_stack * H_ELZ_MAX

H_min = A_elz*(P_ELZ_MIN**2) + B_elz*P_ELZ_MIN + C_elz
H_max = A_elz*(P_ELZ_MAX**2) + B_elz*P_ELZ_MAX + C_elz
Q1 = (H_max - H_min) / (P_ELZ_MAX - P_ELZ_MIN)
Q0 = H_min - Q1 * P_ELZ_MIN

# NH3
Q_NH3_CAP = 13472.04
f_NH3_min = 0.61
RR_NH3 = 0.2
mH2_to_NH3 = 0.18
mN2_to_NH3 = 0.87
SEC_HB = 0.36
SEC_ASU = 0.068
C_su_NH3 = 2397.46
de_NH3 = 0.00519

# MeOH
Q_MEOH_CAP = 5818.856
f_MEOH_min = 0.1
R_MEOH = 0.5 * Q_MEOH_CAP
mH2_to_MEOH = 0.22
SEC_MEOH = 161.75/1000.0
mCO2_to_MEOH = 1.59
c_CO2 = 0.2
C_su_MEOH = 1159.37
de_MEOH = 0.08

# Cracker
Q_CRK_CAP = 6062.416
f_CRK_min = 0.05
R_CRK = 0.5 * Q_CRK_CAP
kH2_out = 0.12024
kNH3_rec = 0.11848
kH2_fuel = 0.035294
SEC_CRK_E = 0.524667
kSteam = 0.941176
eSteam = 0.75
omega_steam = 0.001
C_su_CRK = 1000.0

# Tanks
H_TANK = 12.0
ALPHA_TANK_FILL = 0.15

S_H2_CAP   = H_TANK * H_ELZ_MAX_TOT
S_NH3_CAP  = H_TANK * Q_NH3_CAP
S_MEOH_CAP = H_TANK * Q_MEOH_CAP

S_H2_INIT   = ALPHA_TANK_FILL * S_H2_CAP
S_NH3_INIT  = ALPHA_TANK_FILL * S_NH3_CAP
S_MEOH_INIT = ALPHA_TANK_FILL * S_MEOH_CAP

# ---------------- Helpers ----------------
def day_to_time_slice(day_of_year: int):
    start_ts = idx_2024[0] + pd.Timedelta(days=day_of_year-1)
    end_ts = start_ts + pd.Timedelta(hours=24)
    return pd.date_range(start_ts, end_ts - pd.Timedelta(hours=1), freq="h", tz="UTC")

def save_lineplot(df, xcol, ycols, title, ylabel, outpath):
    plt.figure(figsize=(14, 4))
    x = df[xcol] if xcol in df.columns else df.index
    for c in ycols:
        plt.plot(x, df[c], label=c)
    plt.title(title)
    plt.ylabel(ylabel)
    plt.legend()
    plt.tight_layout()
    plt.savefig(outpath, dpi=150)
    plt.close()

def save_stacked_bar_by_day(df, date_col, components, title, ylabel, outpath):
    x = pd.to_datetime(df[date_col])
    plt.figure(figsize=(14,5))
    bottom = np.zeros(len(df), dtype=float)
    for col, label in components:
        vals = df[col].values.astype(float)
        plt.bar(x, vals, bottom=bottom, label=label)
        bottom += vals
    plt.title(title)
    plt.ylabel(ylabel)
    plt.legend()
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(outpath, dpi=150)
    plt.close()

def safe_get_attr(model, attr, default=np.nan):
    try:
        return getattr(model, attr)
    except Exception:
        return default

def status_name(code: int) -> str:
    mp = {
        GRB.LOADED: "LOADED",
        GRB.OPTIMAL: "OPTIMAL",
        GRB.INFEASIBLE: "INFEASIBLE",
        GRB.INF_OR_UNBD: "INF_OR_UNBD",
        GRB.UNBOUNDED: "UNBOUNDED",
        GRB.CUTOFF: "CUTOFF",
        GRB.ITERATION_LIMIT: "ITERATION_LIMIT",
        GRB.NODE_LIMIT: "NODE_LIMIT",
        GRB.TIME_LIMIT: "TIME_LIMIT",
        GRB.SOLUTION_LIMIT: "SOLUTION_LIMIT",
        GRB.INTERRUPTED: "INTERRUPTED",
        GRB.NUMERIC: "NUMERIC",
        GRB.SUBOPTIMAL: "SUBOPTIMAL",
        GRB.INPROGRESS: "INPROGRESS",
        GRB.USER_OBJ_LIMIT: "USER_OBJ_LIMIT",
    }
    return mp.get(int(code), f"STATUS_{int(code)}")

def get_objpass_metrics(model, obj_idx: int):
    out = {
        "obj_idx": int(obj_idx),
        "pass_status": np.nan,
        "pass_status_name": "NA",
        "pass_runtime": np.nan,
        "pass_objval": np.nan,
        "pass_objbound": np.nan,
        "pass_mipgap": np.nan,
    }
    try:
        prev_objnum  = getattr(model.Params, "ObjNumber", None)
        prev_passnum = getattr(model.Params, "ObjPassNumber", None)

        model.Params.ObjNumber = int(obj_idx)
        model.Params.ObjPassNumber = -1

        st = safe_get_attr(model, "ObjPassNStatus", np.nan)
        out["pass_status"] = float(st) if not np.isnan(st) else np.nan
        out["pass_status_name"] = status_name(int(st)) if not np.isnan(st) else "NA"
        out["pass_runtime"]  = float(safe_get_attr(model, "ObjPassNRuntime", np.nan))
        out["pass_objval"]   = float(safe_get_attr(model, "ObjPassNObjVal", np.nan))
        out["pass_objbound"] = float(safe_get_attr(model, "ObjPassNObjBound", np.nan))
        out["pass_mipgap"]   = float(safe_get_attr(model, "ObjPassNMipGap", np.nan))

        if prev_objnum is not None:
            model.Params.ObjNumber = prev_objnum
        if prev_passnum is not None:
            model.Params.ObjPassNumber = prev_passnum
    except Exception:
        pass
    return out

# ---------------- Carry-over initialization ----------------
carry = {
    "soc": SOC_INIT,
    "S_H2_g": 0.5*S_H2_INIT,
    "S_H2_ng": 0.5*S_H2_INIT,
    "S_NH3_g": 0.5*S_NH3_INIT,
    "S_NH3_ng": 0.5*S_NH3_INIT,
    "S_MEOH_g": 0.5*S_MEOH_INIT,
    "S_MEOH_ng": 0.5*S_MEOH_INIT,
    "y_elz_prev_mod": [0]*N_stack,
    "y_nh3_prev": 0,
    "y_meoh_prev": 0,
    "y_crk_prev": 0,
    "Q_nh3_prev": 0.0,
    "Q_meoh_prev": 0.0,
    "Q_crk_prev": 0.0,
}

daily_rows = []
solver_rows = []
state_rows = []
diag_rows = []
all_hourly_frames = []

# ---------------- Run loop ----------------
t0_global = time.time()

for d in range(START_DAY_OF_YEAR, START_DAY_OF_YEAR + N_DAYS_TO_RUN):
    day_hours = day_to_time_slice(d)

    if day_hours[0] < idx_2024[0] or day_hours[-1] > idx_2024[-1]:
        break

    day_dir = OUT_DAILY / f"day_{d:03d}"
    day_dir.mkdir(parents=True, exist_ok=True)

    pv_av    = res.loc[day_hours, "pv_mw"].values.astype(float)
    wd_av    = res.loc[day_hours, "wind_mw"].values.astype(float)
    pr_eur   = price.loc[day_hours, "da_elec_price_EURperMWh"].values.astype(float)
    dem_h2   = dem.loc[day_hours, "h2_demand_kgph"].values.astype(float)
    dem_nh3  = dem.loc[day_hours, "nh3_demand_kgph"].values.astype(float)
    dem_meoh = dem.loc[day_hours, "meoh_demand_kgph"].values.astype(float)

    aef_k        = aef.loc[day_hours, AEF_COL].values.astype(float)
    mef_upramp_k = mef_upramp.loc[day_hours, MEF_UPRAMP_COL].values.astype(float)
    mef_soft_k   = mef_soft.loc[day_hours, MEF_SOFT_COL].values.astype(float)

    if EMIS_EF_MODE.lower() == "aef":
        ef_obj_k = aef_k
    elif EMIS_EF_MODE.lower() == "mef_upramp":
        ef_obj_k = mef_upramp_k
    elif EMIS_EF_MODE.lower() == "mef_soft":
        ef_obj_k = mef_soft_k
    else:
        raise ValueError("EMIS_EF_MODE must be one of: 'aef', 'mef_upramp', 'mef_soft'")

    # ---------------- Model ----------------
    m = gp.Model(f"{MODEL_TAG}_day{d:03d}")
    m.Params.MIPGap = MIPGAP
    m.Params.TimeLimit = TIME_LIMIT_S
    m.Params.OutputFlag = OUTPUTFLAG
    m.Params.NumericFocus = 1
    m.Params.MultiObjMethod = 2
    m.Params.MultiObjPre = 1

    T = range(24)
    T2 = range(25)

    S_sources = ["pv", "wd", "grid", "bat"]
    K_sinks = ["elz", "NH3", "MeOH", "crk", "bch"]
    S_res = ["pv", "wd"]

    # ---------------- Variables ----------------
    P = m.addVars(S_sources, K_sinks, T, lb=0.0, name="P_s2k")
    P_pv_cur = m.addVars(T, lb=0.0, name="P_pv_cur")
    P_wd_cur = m.addVars(T, lb=0.0, name="P_wd_cur")
    P_grid = m.addVars(T, lb=0.0, ub=P_GRID_MAX_MW, name="P_grid")

    P_ch  = m.addVars(T, lb=0.0, ub=P_BAT_MAX_MW, name="P_ch")
    P_dis = m.addVars(T, lb=0.0, ub=P_BAT_MAX_MW, name="P_dis")
    SOC   = m.addVars(T2, lb=SOC_MIN, ub=SOC_MAX, name="SOC")

    Pk_g  = m.addVars(K_sinks, T, lb=0.0, name="P_sink_g")
    Pk_ng = m.addVars(K_sinks, T, lb=0.0, name="P_sink_ng")

    # Electrolyzer
    I_elz = range(N_stack)
    y_elz_m  = m.addVars(I_elz, T, vtype=GRB.BINARY, name="y_elz_m")
    su_elz_m = m.addVars(I_elz, T, vtype=GRB.BINARY, name="su_elz_m")
    b_elz_g  = m.addVars(I_elz, T, vtype=GRB.BINARY, name="b_elz_g")

    P_elz_m_g   = m.addVars(I_elz, T, lb=0.0, ub=P_ELZ_MAX, name="P_elz_m_g")
    P_elz_m_ng  = m.addVars(I_elz, T, lb=0.0, ub=P_ELZ_MAX, name="P_elz_m_ng")
    P_elz_m_tot = m.addVars(I_elz, T, lb=0.0, ub=P_ELZ_MAX, name="P_elz_m_tot")

    H_elz_m    = m.addVars(I_elz, T, lb=0.0, ub=H_ELZ_MAX, name="H_elz_m")
    H_elz_m_g  = m.addVars(I_elz, T, lb=0.0, ub=H_ELZ_MAX, name="H_elz_m_g")
    H_elz_m_ng = m.addVars(I_elz, T, lb=0.0, ub=H_ELZ_MAX, name="H_elz_m_ng")

    y_elz    = m.addVars(T, vtype=GRB.BINARY, name="y_elz")
    n_elz_on = m.addVars(T, vtype=GRB.INTEGER, lb=0, ub=N_stack, name="n_elz_on")
    su_elz   = m.addVars(T, vtype=GRB.INTEGER, lb=0, ub=N_stack, name="su_elz")

    P_elz_g   = m.addVars(T, lb=0.0, ub=N_stack*P_ELZ_MAX, name="P_elz_g")
    P_elz_ng  = m.addVars(T, lb=0.0, ub=N_stack*P_ELZ_MAX, name="P_elz_ng")
    P_elz_tot = m.addVars(T, lb=0.0, ub=N_stack*P_ELZ_MAX, name="P_elz_tot")
    H_prod    = m.addVars(T, lb=0.0, ub=N_stack*H_ELZ_MAX, name="H_prod")
    H_prod_g  = m.addVars(T, lb=0.0, ub=N_stack*H_ELZ_MAX, name="H_prod_g")
    H_prod_ng = m.addVars(T, lb=0.0, ub=N_stack*H_ELZ_MAX, name="H_prod_ng")
    W_elz     = m.addVars(T, lb=0.0, name="W_elz_m3ph")

    # NH3
    y_nh3    = m.addVars(T, vtype=GRB.BINARY, name="y_nh3")
    su_nh3   = m.addVars(T, vtype=GRB.BINARY, name="su_nh3")
    Q_nh3_g  = m.addVars(T, lb=0.0, name="Q_nh3_g")
    Q_nh3_ng = m.addVars(T, lb=0.0, name="Q_nh3_ng")
    Q_nh3    = m.addVars(T, lb=0.0, name="Q_nh3")
    P_nh3_g  = m.addVars(T, lb=0.0, name="P_nh3_g")
    P_nh3_ng = m.addVars(T, lb=0.0, name="P_nh3_ng")

    # MeOH
    y_meoh    = m.addVars(T, vtype=GRB.BINARY, name="y_meoh")
    su_meoh   = m.addVars(T, vtype=GRB.BINARY, name="su_meoh")
    Q_meoh_g  = m.addVars(T, lb=0.0, name="Q_meoh_g")
    Q_meoh_ng = m.addVars(T, lb=0.0, name="Q_meoh_ng")
    Q_meoh    = m.addVars(T, lb=0.0, name="Q_meoh")
    P_meoh_g  = m.addVars(T, lb=0.0, name="P_meoh_g")
    P_meoh_ng = m.addVars(T, lb=0.0, name="P_meoh_ng")
    CO2_buy   = m.addVars(T, lb=0.0, name="CO2_buy_kgph")

    # Cracker
    y_crk      = m.addVars(T, vtype=GRB.BINARY, name="y_crk")
    su_crk     = m.addVars(T, vtype=GRB.BINARY, name="su_crk")
    Q_crk_in_g = m.addVars(T, lb=0.0, name="Q_crk_in_g")
    Q_crk_in_ng= m.addVars(T, lb=0.0, name="Q_crk_in_ng")
    Q_crk_in   = m.addVars(T, lb=0.0, name="Q_crk_in")

    H_crk_out_g   = m.addVars(T, lb=0.0, name="H_crk_out_g")
    H_crk_out_ng  = m.addVars(T, lb=0.0, name="H_crk_out_ng")
    NH3_rec_g     = m.addVars(T, lb=0.0, name="NH3_rec_g")
    NH3_rec_ng    = m.addVars(T, lb=0.0, name="NH3_rec_ng")
    H_crk_fuel_g  = m.addVars(T, lb=0.0, name="H_crk_fuel_g")
    H_crk_fuel_ng = m.addVars(T, lb=0.0, name="H_crk_fuel_ng")
    Steam_g       = m.addVars(T, lb=0.0, name="Steam_g_kgph")
    Steam_ng      = m.addVars(T, lb=0.0, name="Steam_ng_kgph")
    P_crk_g       = m.addVars(T, lb=0.0, name="P_crk_g")
    P_crk_ng      = m.addVars(T, lb=0.0, name="P_crk_ng")
    W_steam       = m.addVars(T, lb=0.0, name="W_steam_m3ph")

    # Tanks
    S_H2_g    = m.addVars(T2, lb=0.0, name="S_H2_g")
    S_H2_ng   = m.addVars(T2, lb=0.0, name="S_H2_ng")
    S_NH3_g   = m.addVars(T2, lb=0.0, name="S_NH3_g")
    S_NH3_ng  = m.addVars(T2, lb=0.0, name="S_NH3_ng")
    S_MEOH_g  = m.addVars(T2, lb=0.0, name="S_MEOH_g")
    S_MEOH_ng = m.addVars(T2, lb=0.0, name="S_MEOH_ng")

    # Sales
    H_sell_g     = m.addVars(T, lb=0.0, name="H_sell_g")
    H_sell_ng    = m.addVars(T, lb=0.0, name="H_sell_ng")
    NH3_sell_g   = m.addVars(T, lb=0.0, name="NH3_sell_g")
    NH3_sell_ng  = m.addVars(T, lb=0.0, name="NH3_sell_ng")
    MEOH_sell_g  = m.addVars(T, lb=0.0, name="MEOH_sell_g")
    MEOH_sell_ng = m.addVars(T, lb=0.0, name="MEOH_sell_ng")

    # H2 to units
    H_to_NH3_g   = m.addVars(T, lb=0.0, name="H_to_NH3_g")
    H_to_NH3_ng  = m.addVars(T, lb=0.0, name="H_to_NH3_ng")
    H_to_MEOH_g  = m.addVars(T, lb=0.0, name="H_to_MEOH_g")
    H_to_MEOH_ng = m.addVars(T, lb=0.0, name="H_to_MEOH_ng")

    # Unmet
    u_H2   = m.addVars(T, lb=0.0, name="u_H2")
    u_NH3  = m.addVars(T, lb=0.0, name="u_NH3")
    u_MEOH = m.addVars(T, lb=0.0, name="u_MEOH")

    # ---------------- Tight bounds ----------------
    for t in T:
        m.addConstr(P_pv_cur[t] <= pv_av[t], name=f"ub_pv_cur[{t}]")
        m.addConstr(P_wd_cur[t] <= wd_av[t], name=f"ub_wd_cur[{t}]")

    for t in T:
        for k in K_sinks:
            m.addConstr(P["pv",k,t]   <= pv_av[t],      name=f"ub_P_pv_{k}[{t}]")
            m.addConstr(P["wd",k,t]   <= wd_av[t],      name=f"ub_P_wd_{k}[{t}]")
            m.addConstr(P["grid",k,t] <= P_GRID_MAX_MW, name=f"ub_P_grid_{k}[{t}]")
            m.addConstr(P["bat",k,t]  <= P_BAT_MAX_MW,  name=f"ub_P_bat_{k}[{t}]")

    # ---------------- General physics ----------------
    for t in T:
        m.addConstr(gp.quicksum(P["pv",k,t] for k in K_sinks) + P_pv_cur[t] == pv_av[t], name=f"pv_balance[{t}]")
        m.addConstr(gp.quicksum(P["wd",k,t] for k in K_sinks) + P_wd_cur[t] == wd_av[t], name=f"wd_balance[{t}]")

        m.addConstr(gp.quicksum(P["grid",k,t] for k in K_sinks) == P_grid[t], name=f"grid_alloc[{t}]")
        m.addConstr(gp.quicksum(P["bat",k,t] for k in K_sinks)  == P_dis[t],  name=f"bat_alloc[{t}]")

        m.addConstr(gp.quicksum(P[s,"bch",t] for s in S_sources) == P_ch[t], name=f"bch_sink[{t}]")

        m.addConstr(P["grid","bch",t] == 0.0, name=f"no_grid_to_bch[{t}]")
        m.addConstr(P["bat","bch",t]  == 0.0, name=f"no_bat_to_bch[{t}]")
        m.addConstr(P["pv","bch",t] + P["wd","bch",t] == P_ch[t], name=f"bch_from_res[{t}]")

        m.addSOS(GRB.SOS_TYPE1, [P_ch[t], P_dis[t]])
        m.addConstr(P_dis[t] <= ETA_DIS * (SOC[t] - SOC_MIN) * E_BAT_MAX_MWH, name=f"ub_Pdis_soc[{t}]")
        m.addConstr(P_ch[t]  <= (SOC_MAX - SOC[t]) * E_BAT_MAX_MWH / ETA_CH,  name=f"ub_Pch_soc[{t}]")

        for k in K_sinks:
            m.addConstr(Pk_ng[k,t] == P["grid",k,t] + P["bat",k,t], name=f"Pk_ng_def[{k},{t}]")
            m.addConstr(Pk_g[k,t]  == gp.quicksum(P[s,k,t] for s in S_res), name=f"Pk_g_def[{k},{t}]")

    m.addConstr(SOC[0] == float(carry["soc"]), name="SOC_init_carry")
    for t in T:
        m.addConstr(
            SOC[t+1] == SOC[t] + (ETA_CH*P_ch[t] - (1.0/ETA_DIS)*P_dis[t]) / E_BAT_MAX_MWH,
            name=f"SOC_dyn[{t}]"
        )

    m.addConstrs((y_elz_m[i,t] >= y_elz_m[i+1,t] for i in range(N_stack-1) for t in T), name="elz_module_order")

    # ---------------- Electrolyzer ----------------
    for t in T:
        m.addConstr(P_elz_g[t]  == Pk_g["elz",t],  name=f"P_elz_g_link[{t}]")
        m.addConstr(P_elz_ng[t] == Pk_ng["elz",t], name=f"P_elz_ng_link[{t}]")

        m.addConstr(P_elz_g[t]  == gp.quicksum(P_elz_m_g[i,t] for i in I_elz), name=f"P_elz_g_sum_mod[{t}]")
        m.addConstr(P_elz_ng[t] == gp.quicksum(P_elz_m_ng[i,t] for i in I_elz), name=f"P_elz_ng_sum_mod[{t}]")
        m.addConstr(P_elz_tot[t] == P_elz_g[t] + P_elz_ng[t], name=f"P_elz_tot_sum[{t}]")

        m.addConstr(n_elz_on[t] == gp.quicksum(y_elz_m[i,t] for i in I_elz), name=f"n_elz_on_def[{t}]")
        m.addConstr(n_elz_on[t] >= y_elz[t], name=f"y_elz_lb[{t}]")
        m.addConstr(n_elz_on[t] <= N_stack * y_elz[t], name=f"y_elz_ub[{t}]")
        m.addConstr(su_elz[t] == gp.quicksum(su_elz_m[i,t] for i in I_elz), name=f"su_elz_sum[{t}]")

        m.addConstr(H_prod[t]    == gp.quicksum(H_elz_m[i,t] for i in I_elz), name=f"H_prod_sum_mod[{t}]")
        m.addConstr(H_prod_g[t]  == gp.quicksum(H_elz_m_g[i,t] for i in I_elz), name=f"H_prod_g_sum_mod[{t}]")
        m.addConstr(H_prod_ng[t] == gp.quicksum(H_elz_m_ng[i,t] for i in I_elz), name=f"H_prod_ng_sum_mod[{t}]")
        m.addConstr(H_prod[t] == H_prod_g[t] + H_prod_ng[t], name=f"H_prod_split[{t}]")
        m.addConstr(W_elz[t] == omega_H2O * H_prod[t], name=f"W_elz[{t}]")

        for i in range(N_stack - 1):
            m.addConstr(b_elz_g[i,t] >= b_elz_g[i+1,t], name=f"elz_bg_order[{i},{t}]")

        for i in I_elz:
            m.addConstr(P_elz_m_tot[i,t] == P_elz_m_g[i,t] + P_elz_m_ng[i,t], name=f"P_elz_m_tot_def[{i},{t}]")
            m.addConstr(P_elz_m_tot[i,t] <= P_ELZ_MAX * y_elz_m[i,t], name=f"elz_m_max[{i},{t}]")
            m.addConstr(P_elz_m_tot[i,t] >= P_ELZ_MIN * y_elz_m[i,t], name=f"elz_m_min[{i},{t}]")
            m.addConstr(b_elz_g[i,t] <= y_elz_m[i,t], name=f"b_elz_g_gate[{i},{t}]")

            m.addConstr(P_elz_m_g[i,t]  <= P_ELZ_MAX * b_elz_g[i,t], name=f"P_elz_m_g_cap[{i},{t}]")
            m.addConstr(P_elz_m_ng[i,t] <= P_ELZ_MAX * (y_elz_m[i,t] - b_elz_g[i,t]), name=f"P_elz_m_ng_cap[{i},{t}]")

            m.addConstr(H_elz_m[i,t] == H_elz_m_g[i,t] + H_elz_m_ng[i,t], name=f"H_elz_m_split[{i},{t}]")

            m.addQConstr(
                H_elz_m[i,t] <= A_elz*(P_elz_m_tot[i,t]*P_elz_m_tot[i,t]) + B_elz*P_elz_m_tot[i,t] + C_elz*y_elz_m[i,t],
                name=f"elz_m_quad[{i},{t}]"
            )
            m.addConstr(H_elz_m[i,t] >= Q1*P_elz_m_tot[i,t] + Q0*y_elz_m[i,t], name=f"elz_m_lin[{i},{t}]")

            m.addConstr(H_elz_m_g[i,t] <= H_ELZ_MAX * b_elz_g[i,t], name=f"H_elz_m_g_ub1[{i},{t}]")
            m.addConstr(H_elz_m_g[i,t] <= H_elz_m[i,t], name=f"H_elz_m_g_ub2[{i},{t}]")
            m.addConstr(H_elz_m_g[i,t] >= H_elz_m[i,t] - H_ELZ_MAX*(1 - b_elz_g[i,t]), name=f"H_elz_m_g_lb[{i},{t}]")

            m.addConstr(H_elz_m_ng[i,t] <= H_ELZ_MAX * (y_elz_m[i,t] - b_elz_g[i,t]), name=f"H_elz_m_ng_ub1[{i},{t}]")
            m.addConstr(H_elz_m_ng[i,t] <= H_elz_m[i,t], name=f"H_elz_m_ng_ub2[{i},{t}]")
            m.addConstr(H_elz_m_ng[i,t] >= H_elz_m[i,t] - H_ELZ_MAX*b_elz_g[i,t], name=f"H_elz_m_ng_lb[{i},{t}]")

            if t >= 1:
                m.addConstr(su_elz_m[i,t] <= y_elz_m[i,t], name=f"su_elz_m1[{i},{t}]")
                m.addConstr(su_elz_m[i,t] <= 1 - y_elz_m[i,t-1], name=f"su_elz_m2[{i},{t}]")
                m.addConstr(su_elz_m[i,t] >= y_elz_m[i,t] - y_elz_m[i,t-1], name=f"su_elz_m3[{i},{t}]")
            else:
                y_prev_i = int(carry["y_elz_prev_mod"][i])
                m.addConstr(su_elz_m[i,t] <= y_elz_m[i,t], name=f"su_elz_m1[{i},{t}]")
                m.addConstr(su_elz_m[i,t] <= 1 - y_prev_i, name=f"su_elz_m2[{i},{t}]")
                m.addConstr(su_elz_m[i,t] >= y_elz_m[i,t] - y_prev_i, name=f"su_elz_m3[{i},{t}]")

    # ---------------- NH3 ----------------
    for t in T:
        m.addConstr(Q_nh3[t] == Q_nh3_g[t] + Q_nh3_ng[t], name=f"Q_nh3_sum[{t}]")
        m.addConstr(Q_nh3[t] <= Q_NH3_CAP * y_nh3[t], name=f"nh3_max[{t}]")
        m.addConstr(Q_nh3[t] >= f_NH3_min * Q_NH3_CAP * y_nh3[t], name=f"nh3_min[{t}]")

        if t >= 1:
            m.addConstr(su_nh3[t] <= y_nh3[t])
            m.addConstr(su_nh3[t] <= 1 - y_nh3[t-1])
            m.addConstr(su_nh3[t] >= y_nh3[t] - y_nh3[t-1])
        else:
            y_prev = int(carry["y_nh3_prev"])
            m.addConstr(su_nh3[t] <= y_nh3[t])
            m.addConstr(su_nh3[t] <= 1 - y_prev)
            m.addConstr(su_nh3[t] >= y_nh3[t] - y_prev)

        m.addConstr(
            Q_nh3[t] <= f_NH3_min * Q_NH3_CAP + (Q_NH3_CAP - f_NH3_min * Q_NH3_CAP) * (1 - su_nh3[t]),
            name=f"nh3_start_minload[{t}]"
        )

        ramp_nh3 = RR_NH3 * Q_NH3_CAP

        if t == 0:
            Q_prev = float(carry["Q_nh3_prev"])
            y_prev = int(carry["y_nh3_prev"])
            m.addConstr(Q_nh3[t] - Q_prev <= ramp_nh3 + (Q_NH3_CAP - ramp_nh3) * (2 - y_nh3[t] - y_prev), name=f"nh3_rup0[{t}]")
            m.addConstr(Q_prev - Q_nh3[t] <= ramp_nh3 + (Q_NH3_CAP - ramp_nh3) * (2 - y_nh3[t] - y_prev), name=f"nh3_rdn0[{t}]")
        else:
            m.addConstr(Q_nh3[t] - Q_nh3[t-1] <= ramp_nh3 + (Q_NH3_CAP - ramp_nh3) * (2 - y_nh3[t] - y_nh3[t-1]), name=f"nh3_rup[{t}]")
            m.addConstr(Q_nh3[t-1] - Q_nh3[t] <= ramp_nh3 + (Q_NH3_CAP - ramp_nh3) * (2 - y_nh3[t] - y_nh3[t-1]), name=f"nh3_rdn[{t}]")

        m.addConstr(P_nh3_g[t]  == (SEC_HB*Q_nh3_g[t]  + SEC_ASU*(mN2_to_NH3*Q_nh3_g[t]))  / 1000.0, name=f"P_nh3_g_map[{t}]")
        m.addConstr(P_nh3_ng[t] == (SEC_HB*Q_nh3_ng[t] + SEC_ASU*(mN2_to_NH3*Q_nh3_ng[t])) / 1000.0, name=f"P_nh3_ng_map[{t}]")

        m.addConstr(P_nh3_g[t]  == Pk_g["NH3",t],  name=f"P_nh3_g_link[{t}]")
        m.addConstr(P_nh3_ng[t] == Pk_ng["NH3",t], name=f"P_nh3_ng_link[{t}]")

        m.addConstr(H_to_NH3_g[t]  == mH2_to_NH3 * Q_nh3_g[t],  name=f"H_to_NH3_g[{t}]")
        m.addConstr(H_to_NH3_ng[t] == mH2_to_NH3 * Q_nh3_ng[t], name=f"H_to_NH3_ng[{t}]")

    # ---------------- MeOH ----------------
    for t in T:
        m.addConstr(Q_meoh[t] == Q_meoh_g[t] + Q_meoh_ng[t], name=f"Q_meoh_sum[{t}]")
        m.addConstr(Q_meoh[t] <= Q_MEOH_CAP * y_meoh[t], name=f"meoh_max[{t}]")
        m.addConstr(Q_meoh[t] >= f_MEOH_min * Q_MEOH_CAP * y_meoh[t], name=f"meoh_min[{t}]")

        if t >= 1:
            m.addConstr(su_meoh[t] <= y_meoh[t])
            m.addConstr(su_meoh[t] <= 1 - y_meoh[t-1])
            m.addConstr(su_meoh[t] >= y_meoh[t] - y_meoh[t-1])
        else:
            y_prev = int(carry["y_meoh_prev"])
            m.addConstr(su_meoh[t] <= y_meoh[t])
            m.addConstr(su_meoh[t] <= 1 - y_prev)
            m.addConstr(su_meoh[t] >= y_meoh[t] - y_prev)

        m.addConstr(
            Q_meoh[t] <= f_MEOH_min * Q_MEOH_CAP + (Q_MEOH_CAP - f_MEOH_min * Q_MEOH_CAP) * (1 - su_meoh[t]),
            name=f"meoh_start_minload[{t}]"
        )

        ramp_meoh = R_MEOH

        if t == 0:
            Q_prev = float(carry["Q_meoh_prev"])
            y_prev = int(carry["y_meoh_prev"])
            m.addConstr(Q_meoh[t] - Q_prev <= ramp_meoh + (Q_MEOH_CAP - ramp_meoh) * (2 - y_meoh[t] - y_prev), name=f"meoh_rup0[{t}]")
            m.addConstr(Q_prev - Q_meoh[t] <= ramp_meoh + (Q_MEOH_CAP - ramp_meoh) * (2 - y_meoh[t] - y_prev), name=f"meoh_rdn0[{t}]")
        else:
            m.addConstr(Q_meoh[t] - Q_meoh[t-1] <= ramp_meoh + (Q_MEOH_CAP - ramp_meoh) * (2 - y_meoh[t] - y_meoh[t-1]), name=f"meoh_rup[{t}]")
            m.addConstr(Q_meoh[t-1] - Q_meoh[t] <= ramp_meoh + (Q_MEOH_CAP - ramp_meoh) * (2 - y_meoh[t] - y_meoh[t-1]), name=f"meoh_rdn[{t}]")

        m.addConstr(P_meoh_g[t]  == (SEC_MEOH * Q_meoh_g[t])  / 1000.0, name=f"P_meoh_g_map[{t}]")
        m.addConstr(P_meoh_ng[t] == (SEC_MEOH * Q_meoh_ng[t]) / 1000.0, name=f"P_meoh_ng_map[{t}]")

        m.addConstr(P_meoh_g[t]  == Pk_g["MeOH",t],  name=f"P_meoh_g_link[{t}]")
        m.addConstr(P_meoh_ng[t] == Pk_ng["MeOH",t], name=f"P_meoh_ng_link[{t}]")

        m.addConstr(CO2_buy[t] == mCO2_to_MEOH * Q_meoh[t], name=f"CO2_buy[{t}]")

        m.addConstr(H_to_MEOH_g[t]  == mH2_to_MEOH * Q_meoh_g[t],  name=f"H_to_MEOH_g[{t}]")
        m.addConstr(H_to_MEOH_ng[t] == mH2_to_MEOH * Q_meoh_ng[t], name=f"H_to_MEOH_ng[{t}]")

    # ---------------- Cracker ----------------
    for t in T:
        m.addConstr(Q_crk_in[t] == Q_crk_in_g[t] + Q_crk_in_ng[t], name=f"Q_crk_sum[{t}]")
        m.addConstr(Q_crk_in[t] <= Q_CRK_CAP * y_crk[t], name=f"crk_max[{t}]")
        m.addConstr(Q_crk_in[t] >= f_CRK_min * Q_CRK_CAP * y_crk[t], name=f"crk_min[{t}]")

        if t >= 1:
            m.addConstr(su_crk[t] <= y_crk[t])
            m.addConstr(su_crk[t] <= 1 - y_crk[t-1])
            m.addConstr(su_crk[t] >= y_crk[t] - y_crk[t-1])
        else:
            y_prev = int(carry["y_crk_prev"])
            m.addConstr(su_crk[t] <= y_crk[t])
            m.addConstr(su_crk[t] <= 1 - y_prev)
            m.addConstr(su_crk[t] >= y_crk[t] - y_prev)

        m.addConstr(
            Q_crk_in[t] <= f_CRK_min * Q_CRK_CAP + (Q_CRK_CAP - f_CRK_min * Q_CRK_CAP) * (1 - su_crk[t]),
            name=f"crk_start_minload[{t}]"
        )

        ramp_crk = R_CRK

        if t == 0:
            Q_prev = float(carry["Q_crk_prev"])
            y_prev = int(carry["y_crk_prev"])
            m.addConstr(Q_crk_in[t] - Q_prev <= ramp_crk + (Q_CRK_CAP - ramp_crk) * (2 - y_crk[t] - y_prev), name=f"crk_rup0[{t}]")
            m.addConstr(Q_prev - Q_crk_in[t] <= ramp_crk + (Q_CRK_CAP - ramp_crk) * (2 - y_crk[t] - y_prev), name=f"crk_rdn0[{t}]")
        else:
            m.addConstr(Q_crk_in[t] - Q_crk_in[t-1] <= ramp_crk + (Q_CRK_CAP - ramp_crk) * (2 - y_crk[t] - y_crk[t-1]), name=f"crk_rup[{t}]")
            m.addConstr(Q_crk_in[t-1] - Q_crk_in[t] <= ramp_crk + (Q_CRK_CAP - ramp_crk) * (2 - y_crk[t] - y_crk[t-1]), name=f"crk_rdn[{t}]")

        m.addConstr(H_crk_out_g[t]  == kH2_out * Q_crk_in_g[t],  name=f"Hcrk_out_g[{t}]")
        m.addConstr(H_crk_out_ng[t] == kH2_out * Q_crk_in_ng[t], name=f"Hcrk_out_ng[{t}]")
        m.addConstr(NH3_rec_g[t]    == kNH3_rec * Q_crk_in_g[t],  name=f"NH3rec_g[{t}]")
        m.addConstr(NH3_rec_ng[t]   == kNH3_rec * Q_crk_in_ng[t], name=f"NH3rec_ng[{t}]")
        m.addConstr(H_crk_fuel_g[t]  == kH2_fuel * Q_crk_in_g[t],  name=f"Hcrk_fuel_g[{t}]")
        m.addConstr(H_crk_fuel_ng[t] == kH2_fuel * Q_crk_in_ng[t], name=f"Hcrk_fuel_ng[{t}]")

        m.addConstr(Steam_g[t]  == kSteam * Q_crk_in_g[t],  name=f"Steam_g[{t}]")
        m.addConstr(Steam_ng[t] == kSteam * Q_crk_in_ng[t], name=f"Steam_ng[{t}]")
        m.addConstr(W_steam[t]  == omega_steam * (Steam_g[t] + Steam_ng[t]), name=f"W_steam[{t}]")

        m.addConstr(P_crk_g[t]  == (SEC_CRK_E*Q_crk_in_g[t]  + eSteam*Steam_g[t])  / 1000.0, name=f"P_crk_g_map[{t}]")
        m.addConstr(P_crk_ng[t] == (SEC_CRK_E*Q_crk_in_ng[t] + eSteam*Steam_ng[t]) / 1000.0, name=f"P_crk_ng_map[{t}]")

        m.addConstr(P_crk_g[t]  == Pk_g["crk",t],  name=f"P_crk_g_link[{t}]")
        m.addConstr(P_crk_ng[t] == Pk_ng["crk",t], name=f"P_crk_ng_link[{t}]")

    # ---------------- Tanks / demand ----------------
    m.addConstr(S_H2_g[0]    == float(carry["S_H2_g"]),    name="S_H2_g_init")
    m.addConstr(S_H2_ng[0]   == float(carry["S_H2_ng"]),   name="S_H2_ng_init")
    m.addConstr(S_NH3_g[0]   == float(carry["S_NH3_g"]),   name="S_NH3_g_init")
    m.addConstr(S_NH3_ng[0]  == float(carry["S_NH3_ng"]),  name="S_NH3_ng_init")
    m.addConstr(S_MEOH_g[0]  == float(carry["S_MEOH_g"]),  name="S_MEOH_g_init")
    m.addConstr(S_MEOH_ng[0] == float(carry["S_MEOH_ng"]), name="S_MEOH_ng_init")

    for tt in T2:
        m.addConstr(S_H2_g[tt] + S_H2_ng[tt] <= S_H2_CAP,   name=f"S_H2_cap[{tt}]")
        m.addConstr(S_NH3_g[tt] + S_NH3_ng[tt] <= S_NH3_CAP, name=f"S_NH3_cap[{tt}]")
        m.addConstr(S_MEOH_g[tt] + S_MEOH_ng[tt] <= S_MEOH_CAP, name=f"S_MEOH_cap[{tt}]")

    m.addConstr(S_H2_g[24]   + S_H2_ng[24]   >= ALPHA_TANK_FILL * S_H2_CAP,   name="eod_hard_H2_frac")
    m.addConstr(S_NH3_g[24]  + S_NH3_ng[24]  >= ALPHA_TANK_FILL * S_NH3_CAP,  name="eod_hard_NH3_frac")
    m.addConstr(S_MEOH_g[24] + S_MEOH_ng[24] >= ALPHA_TANK_FILL * S_MEOH_CAP, name="eod_hard_MeOH_frac")

    for t in T:
        m.addConstr(H_sell_g[t] + H_sell_ng[t] + u_H2[t] == dem_h2[t], name=f"dem_H2[{t}]")
        m.addConstr(NH3_sell_g[t] + NH3_sell_ng[t] + u_NH3[t] == dem_nh3[t], name=f"dem_NH3[{t}]")
        m.addConstr(MEOH_sell_g[t] + MEOH_sell_ng[t] + u_MEOH[t] == dem_meoh[t], name=f"dem_MEOH[{t}]")

        m.addConstr(
            S_H2_g[t+1] == S_H2_g[t] + H_prod_g[t] + H_crk_out_g[t] - H_to_NH3_g[t] - H_to_MEOH_g[t] - H_sell_g[t] - H_crk_fuel_g[t],
            name=f"S_H2_g_dyn[{t}]"
        )
        m.addConstr(
            S_H2_ng[t+1] == S_H2_ng[t] + H_prod_ng[t] + H_crk_out_ng[t] - H_to_NH3_ng[t] - H_to_MEOH_ng[t] - H_sell_ng[t] - H_crk_fuel_ng[t],
            name=f"S_H2_ng_dyn[{t}]"
        )

        m.addConstr(
            S_NH3_g[t+1] == S_NH3_g[t] + Q_nh3_g[t] + NH3_rec_g[t] - NH3_sell_g[t] - Q_crk_in_g[t],
            name=f"S_NH3_g_dyn[{t}]"
        )
        m.addConstr(
            S_NH3_ng[t+1] == S_NH3_ng[t] + Q_nh3_ng[t] + NH3_rec_ng[t] - NH3_sell_ng[t] - Q_crk_in_ng[t],
            name=f"S_NH3_ng_dyn[{t}]"
        )

        m.addConstr(
            S_MEOH_g[t+1] == S_MEOH_g[t] + Q_meoh_g[t] - MEOH_sell_g[t],
            name=f"S_MEOH_g_dyn[{t}]"
        )
        m.addConstr(
            S_MEOH_ng[t+1] == S_MEOH_ng[t] + Q_meoh_ng[t] - MEOH_sell_ng[t],
            name=f"S_MEOH_ng_dyn[{t}]"
        )

        m.addConstr(H_to_NH3_g[t] + H_to_MEOH_g[t] + H_sell_g[t] + H_crk_fuel_g[t] <= S_H2_g[t], name=f"dir_H2_g[{t}]")
        m.addConstr(H_to_NH3_ng[t] + H_to_MEOH_ng[t] + H_sell_ng[t] + H_crk_fuel_ng[t] <= S_H2_ng[t], name=f"dir_H2_ng[{t}]")

        m.addConstr(NH3_sell_g[t] + Q_crk_in_g[t] <= S_NH3_g[t], name=f"dir_NH3_g[{t}]")
        m.addConstr(NH3_sell_ng[t] + Q_crk_in_ng[t] <= S_NH3_ng[t], name=f"dir_NH3_ng[{t}]")

        m.addConstr(MEOH_sell_g[t] <= S_MEOH_g[t], name=f"dir_MEOH_g[{t}]")
        m.addConstr(MEOH_sell_ng[t] <= S_MEOH_ng[t], name=f"dir_MEOH_ng[{t}]")

    # ---------------- Objectives ----------------
    unmet_total = gp.quicksum(u_H2[t] + u_NH3[t] + u_MEOH[t] for t in T)

    emis_total = gp.LinExpr()
    for t in T:
        emis_total += 1000.0 * ef_obj_k[t] * P_grid[t]
        emis_total += de_NH3 * Q_nh3[t]
        emis_total += de_MEOH * Q_meoh[t]

    cost_total = gp.LinExpr()
    for t in T:
        cost_total += pr_eur[t] * P_grid[t]
        cost_total += c_H2O * (W_elz[t] + W_steam[t])
        cost_total += c_CO2 * CO2_buy[t]
        cost_total += C_su_elz * gp.quicksum(su_elz_m[i,t] for i in I_elz)
        cost_total += C_su_NH3 * su_nh3[t]
        cost_total += C_su_MEOH * su_meoh[t]
        cost_total += C_su_CRK * su_crk[t]

    m.setObjectiveN(unmet_total, index=0, priority=2, reltol=1e-4, abstol=1e-6, name="OBJ1_min_unmet")
    m.setObjectiveN(emis_total,  index=1, priority=1, reltol=1e-4, abstol=1e-6, name=f"OBJ2_min_emis_{EMIS_EF_MODE.lower()}")

    # ---------------- Solve ----------------
    m.optimize()

    status = int(safe_get_attr(m, "Status", -1))
    runtime = float(safe_get_attr(m, "Runtime", np.nan))
    nodecnt = float(safe_get_attr(m, "NodeCount", np.nan))
    solcnt  = int(safe_get_attr(m, "SolCount", -1))

    pass0 = get_objpass_metrics(m, 0)
    pass1 = get_objpass_metrics(m, 1)

    mip_gap = pass1["pass_mipgap"]
    obj_val = pass1["pass_objval"]
    obj_bnd = pass1["pass_objbound"]

    itercnt = safe_get_attr(m, "IterCount", np.nan)
    bariter = safe_get_attr(m, "BarIterCount", np.nan)
    work = safe_get_attr(m, "Work", np.nan)

    obj1_val = np.nan
    obj2_val = np.nan
    obj3_val = np.nan
    try:
        obj1_val = float(unmet_total.getValue())
    except Exception:
        pass
    try:
        obj2_val = float(emis_total.getValue())
    except Exception:
        pass
    try:
        obj3_val = float(cost_total.getValue()) #there is no obj 3, its just arbitary
    except Exception:
        pass

    is_optimal = (status == GRB.OPTIMAL)
    hit_time_limit = (status == GRB.TIME_LIMIT)
    has_incumbent = (solcnt is not None) and (solcnt >= 1)
    is_within_target_gap = bool(has_incumbent) and (mip_gap is not None) and (not np.isnan(mip_gap)) and (float(mip_gap) <= float(MIPGAP) + 1e-12)
    is_acceptable = bool(is_optimal or is_within_target_gap)

    solver_rows.append({
        "run_id": RUN_ID,
        "model_tag": MODEL_TAG,
        "scenario_tag": SCENARIO_TAG,
        "demand_tag": DEMAND_TAG,
        "day_of_year": d,
        "date_utc": str(day_hours[0].date()),
        "solver_status": status,
        "solver_status_name": status_name(status),
        "runtime_seconds": runtime,
        "time_limit_s": TIME_LIMIT_S,
        "mipgap_target": float(MIPGAP),
        "mip_gap": pass1["pass_mipgap"],
        "has_incumbent": bool(has_incumbent),
        "is_optimal": bool(is_optimal),
        "hit_time_limit": bool(hit_time_limit),
        "is_within_target_gap": bool(is_within_target_gap),
        "is_acceptable": bool(is_acceptable),
        "obj_val": float(obj_val) if not np.isnan(obj_val) else np.nan,
        "obj_bound": float(obj_bnd) if not np.isnan(obj_bnd) else np.nan,

        "obj0_pass_status": pass0["pass_status"],
        "obj0_pass_status_name": pass0["pass_status_name"],
        "obj0_pass_runtime_s": pass0["pass_runtime"],
        "obj0_pass_objval": pass0["pass_objval"],
        "obj0_pass_objbound": pass0["pass_objbound"],
        "obj0_pass_mipgap": pass0["pass_mipgap"],

        "obj1_pass_status": pass1["pass_status"],
        "obj1_pass_status_name": pass1["pass_status_name"],
        "obj1_pass_runtime_s": pass1["pass_runtime"],
        "obj1_pass_objval": pass1["pass_objval"],
        "obj1_pass_objbound": pass1["pass_objbound"],
        "obj1_pass_mipgap": pass1["pass_mipgap"],

        "node_count": nodecnt,
        "sol_count": solcnt,
        "iter_count": float(itercnt) if not np.isnan(itercnt) else np.nan,
        "bar_iter_count": float(bariter) if not np.isnan(bariter) else np.nan,
        "work": float(work) if not np.isnan(work) else np.nan,
        "obj1_unmet_value": float(obj1_val) if not np.isnan(obj1_val) else np.nan,
        "obj2_emis_value": float(obj2_val) if not np.isnan(obj2_val) else np.nan,
        "metric_cost_value": float(obj3_val) if not np.isnan(obj3_val) else np.nan,
        "emis_ef_mode": EMIS_EF_MODE.lower(),
    })

    if status not in [GRB.OPTIMAL, GRB.TIME_LIMIT]:
        try:
            m.computeIIS()
            iis_path = day_dir / f"{MODEL_TAG}_day{d:03d}_iis.ilp"
            m.write(str(iis_path))
        except Exception:
            pass
        raise RuntimeError(f"Day {d}: solver status {status} (not optimal/time_limit).")

    # ---------------- Extract hourly results ----------------
    def val(v):
        return float(v.X)

    rows = []
    unmet_rows = []

    for t in T:
        ts = day_hours[t]
        pv_used = pv_av[t] - val(P_pv_cur[t])
        wd_used = wd_av[t] - val(P_wd_cur[t])
        grid_mw = val(P_grid[t])
        res_used_mw = pv_used + wd_used
        res_curt_mw = val(P_pv_cur[t]) + val(P_wd_cur[t])

        P_elz = val(P_elz_g[t]) + val(P_elz_ng[t])
        P_nh3 = val(P_nh3_g[t]) + val(P_nh3_ng[t])
        P_meo = val(P_meoh_g[t]) + val(P_meoh_ng[t])
        P_crk = val(P_crk_g[t]) + val(P_crk_ng[t])

        H_total = val(H_prod[t])
        H_g = val(H_prod_g[t])
        H_ng = val(H_prod_ng[t])

        nh3_total = val(Q_nh3[t])
        nh3_g = val(Q_nh3_g[t])
        nh3_ng = val(Q_nh3_ng[t])

        me_total = val(Q_meoh[t])
        me_g = val(Q_meoh_g[t])
        me_ng = val(Q_meoh_ng[t])

        h2_sell_g = val(H_sell_g[t])
        h2_sell_ng = val(H_sell_ng[t])
        nh3_sell_g = val(NH3_sell_g[t])
        nh3_sell_ng = val(NH3_sell_ng[t])
        meoh_sell_g = val(MEOH_sell_g[t])
        meoh_sell_ng = val(MEOH_sell_ng[t])

        u1 = val(u_H2[t]); u2 = val(u_NH3[t]); u3 = val(u_MEOH[t])

        def share(g, ng):
            tot = g + ng
            return (g / tot) if tot > 1e-9 else np.nan

        H2_min_to_NH3  = mH2_to_NH3 * f_NH3_min * Q_NH3_CAP * int(round(val(y_nh3[t])))
        H2_min_to_MeOH = mH2_to_MEOH * f_MEOH_min * Q_MEOH_CAP * int(round(val(y_meoh[t])))
        H2_min_to_CRK  = kH2_fuel * f_CRK_min * Q_CRK_CAP * int(round(val(y_crk[t])))
        H2_min_committed = H2_min_to_NH3 + H2_min_to_MeOH + H2_min_to_CRK
        H2_tank_start = val(S_H2_g[t]) + val(S_H2_ng[t])
        H2_free_margin = H2_tank_start - H2_min_committed

        elz_mod = {}
        for i in I_elz:
            elz_mod[f"elz_m{i}_y"]     = int(round(val(y_elz_m[i,t])))
            elz_mod[f"elz_m{i}_su"]    = int(round(val(su_elz_m[i,t])))
            elz_mod[f"elz_m{i}_b_g"]   = int(round(val(b_elz_g[i,t])))
            elz_mod[f"elz_m{i}_P_tot"] = float(val(P_elz_m_tot[i,t]))
            elz_mod[f"elz_m{i}_P_g"]   = float(val(P_elz_m_g[i,t]))
            elz_mod[f"elz_m{i}_P_ng"]  = float(val(P_elz_m_ng[i,t]))
            elz_mod[f"elz_m{i}_H_tot"] = float(val(H_elz_m[i,t]))
            elz_mod[f"elz_m{i}_H_g"]   = float(val(H_elz_m_g[i,t]))
            elz_mod[f"elz_m{i}_H_ng"]  = float(val(H_elz_m_ng[i,t]))

        rows.append({
            "run_id": RUN_ID,
            "model_tag": MODEL_TAG,
            "scenario_tag": SCENARIO_TAG,
            "demand_tag": DEMAND_TAG,
            "datetime_utc": ts.isoformat(),
            "day_of_year": d,
            "hour_in_day": t,
            "H2_tank_start_kg": H2_tank_start,
            "H2_min_committed_kgph": H2_min_committed,
            "H2_free_margin_kg": H2_free_margin,
            "pv_avail_mw": pv_av[t],
            "wind_avail_mw": wd_av[t],
            "pv_curt_mw": val(P_pv_cur[t]),
            "wind_curt_mw": val(P_wd_cur[t]),
            "res_used_mw": res_used_mw,
            "res_curtailed_mw": res_curt_mw,
            "grid_import_mw": grid_mw,
            "da_price_eur_per_mwh": pr_eur[t],
            "bat_ch_mw": val(P_ch[t]),
            "bat_dis_mw": val(P_dis[t]),
            "battery_soc": val(SOC[t]),
            "battery_soc_next": val(SOC[t+1]),
            "elz_grid_mw": val(P_elz_ng[t]),
            "nh3_grid_mw": val(P_nh3_ng[t]),
            "meoh_grid_mw": val(P_meoh_ng[t]),
            "crk_grid_mw": val(P_crk_ng[t]),
            "P_elz_mw": P_elz,
            "P_nh3_mw": P_nh3,
            "P_meoh_mw": P_meo,
            "P_crk_mw": P_crk,
            "elz_power_green_share": share(val(P_elz_g[t]), val(P_elz_ng[t])),
            "nh3_power_green_share": share(val(P_nh3_g[t]), val(P_nh3_ng[t])),
            "meoh_power_green_share": share(val(P_meoh_g[t]), val(P_meoh_ng[t])),
            "crk_power_green_share": share(val(P_crk_g[t]), val(P_crk_ng[t])),
            "H2_prod_kgph": H_total,
            "H2_prod_g_kgph": H_g,
            "H2_prod_ng_kgph": H_ng,
            "NH3_prod_kgph": nh3_total,
            "NH3_prod_g_kgph": nh3_g,
            "NH3_prod_ng_kgph": nh3_ng,
            "MeOH_prod_kgph": me_total,
            "MeOH_prod_g_kgph": me_g,
            "MeOH_prod_ng_kgph": me_ng,
            "crk_nh3_in_kgph": val(Q_crk_in[t]),
            "crk_h2_out_kgph": val(H_crk_out_g[t]) + val(H_crk_out_ng[t]),
            "crk_h2_fuel_kgph": val(H_crk_fuel_g[t]) + val(H_crk_fuel_ng[t]),
            "steam_kgph": val(Steam_g[t]) + val(Steam_ng[t]),
            "water_elz_m3ph": val(W_elz[t]),
            "water_steam_m3ph": val(W_steam[t]),
            "co2_buy_kgph": val(CO2_buy[t]),

            "H2_sell_g_kgph": h2_sell_g,
            "H2_sell_ng_kgph": h2_sell_ng,
            "H2_sell_kgph": h2_sell_g + h2_sell_ng,

            "NH3_sell_g_kgph": nh3_sell_g,
            "NH3_sell_ng_kgph": nh3_sell_ng,
            "NH3_sell_kgph": nh3_sell_g + nh3_sell_ng,

            "MeOH_sell_g_kgph": meoh_sell_g,
            "MeOH_sell_ng_kgph": meoh_sell_ng,
            "MeOH_sell_kgph": meoh_sell_g + meoh_sell_ng,

            "unmet_h2_kgph": u1,
            "unmet_nh3_kgph": u2,
            "unmet_meoh_kgph": u3,
            "tank_H2_g_kg": val(S_H2_g[t]),
            "tank_H2_ng_kg": val(S_H2_ng[t]),
            "tank_NH3_g_kg": val(S_NH3_g[t]),
            "tank_NH3_ng_kg": val(S_NH3_ng[t]),
            "tank_MeOH_g_kg": val(S_MEOH_g[t]),
            "tank_MeOH_ng_kg": val(S_MEOH_ng[t]),
            "b_ch": int(val(P_ch[t]) > 1e-6),

            "n_elz_on": int(round(val(n_elz_on[t]))),
            "y_elz": int(round(val(y_elz[t]))),
            "su_elz": int(round(val(su_elz[t]))),
            "y_nh3": int(round(val(y_nh3[t]))),
            "su_nh3": int(round(val(su_nh3[t]))),
            "y_meoh": int(round(val(y_meoh[t]))),
            "su_meoh": int(round(val(su_meoh[t]))),
            "y_crk": int(round(val(y_crk[t]))),
            "su_crk": int(round(val(su_crk[t]))),
            **elz_mod,
        })

        unmet_rows.append({
            "run_id": RUN_ID,
            "model_tag": MODEL_TAG,
            "scenario_tag": SCENARIO_TAG,
            "demand_tag": DEMAND_TAG,
            "datetime_utc": ts.isoformat(),
            "day_of_year": d,
            "hour_in_day": t,
            "unmet_h2_kgph": u1,
            "unmet_nh3_kgph": u2,
            "unmet_meoh_kgph": u3,
            "da_price_eur_per_mwh": pr_eur[t],
            "grid_import_mw": grid_mw,
            "res_curtailed_mw": res_curt_mw,
        })

    day_hourly = pd.DataFrame(rows)
    day_unmet  = pd.DataFrame(unmet_rows)

    day_hourly.to_csv(day_dir / "hourly_timeseries.csv", index=False)
    day_unmet.to_csv(day_dir / "hourly_unmet_demand.csv", index=False)

    day_hourly_all = day_hourly.copy()
    day_hourly_all["day_id"] = int(d)
    all_hourly_frames.append(day_hourly_all)

    # ---------------- Daily post-processing ----------------
    grid_cost_eur   = float(np.sum([pr_eur[t] * val(P_grid[t]) for t in T]))
    water_cost_eur  = float(c_H2O * np.sum([val(W_elz[t]) + val(W_steam[t]) for t in T]))
    co2_cost_eur    = float(c_CO2 * np.sum([val(CO2_buy[t]) for t in T]))
    startup_elz_eur = float(C_su_elz * np.sum([val(su_elz[t]) for t in T]))
    startup_nh3_eur = float(C_su_NH3 * np.sum([val(su_nh3[t]) for t in T]))
    startup_meoh_eur = float(C_su_MEOH * np.sum([val(su_meoh[t]) for t in T]))
    startup_crk_eur = float(C_su_CRK * np.sum([val(su_crk[t]) for t in T]))

    total_unmet_kg = float(np.sum(day_hourly["unmet_h2_kgph"].values + day_hourly["unmet_nh3_kgph"].values + day_hourly["unmet_meoh_kgph"].values))
    total_cost = float(obj3_val) if not np.isnan(obj3_val) else float(grid_cost_eur + water_cost_eur + co2_cost_eur + startup_elz_eur + startup_nh3_eur + startup_meoh_eur + startup_crk_eur)

    grid_emis_aef = float(np.sum([val(P_grid[t]) * 1000.0 * aef_k[t] for t in T]))
    grid_emis_mef_upramp = float(np.sum([val(P_grid[t]) * 1000.0 * mef_upramp_k[t] for t in T]))
    grid_emis_mef_soft = float(np.sum([val(P_grid[t]) * 1000.0 * mef_soft_k[t] for t in T]))

    direct_emis_nh3 = float(np.sum([de_NH3 * val(Q_nh3[t]) for t in T]))
    direct_emis_meoh = float(np.sum([de_MEOH * val(Q_meoh[t]) for t in T]))
    direct_emis_total = direct_emis_nh3 + direct_emis_meoh

    demand_H2_kg_day   = float(np.sum(dem_h2))
    demand_NH3_kg_day  = float(np.sum(dem_nh3))
    demand_MeOH_kg_day = float(np.sum(dem_meoh))

    elz_kwh_day  = float(np.sum(day_hourly["P_elz_mw"].values * 1000.0))
    nh3_kwh_day  = float(np.sum(day_hourly["P_nh3_mw"].values * 1000.0))
    meoh_kwh_day = float(np.sum(day_hourly["P_meoh_mw"].values * 1000.0))
    crk_kwh_day  = float(np.sum(day_hourly["P_crk_mw"].values * 1000.0))
    units_kwh_day = elz_kwh_day + nh3_kwh_day + meoh_kwh_day + crk_kwh_day

    res_gen_kwh = float(np.sum((pv_av + wd_av) * 1000.0))
    res_used_kwh = float(np.sum((pv_av - day_hourly["pv_curt_mw"].values + wd_av - day_hourly["wind_curt_mw"].values) * 1000.0))
    res_curt_kwh = float(np.sum((day_hourly["pv_curt_mw"].values + day_hourly["wind_curt_mw"].values) * 1000.0))
    grid_import_kwh = float(np.sum(day_hourly["grid_import_mw"].values * 1000.0))
    grid_share_percent = 100.0 * grid_import_kwh / max(grid_import_kwh + res_used_kwh, 1e-9)

    H2_g_sum = float(np.sum(day_hourly["H2_prod_g_kgph"].values))
    H2_ng_sum = float(np.sum(day_hourly["H2_prod_ng_kgph"].values))
    NH3_g_sum = float(np.sum(day_hourly["NH3_prod_g_kgph"].values))
    NH3_ng_sum = float(np.sum(day_hourly["NH3_prod_ng_kgph"].values))
    Me_g_sum = float(np.sum(day_hourly["MeOH_prod_g_kgph"].values))
    Me_ng_sum = float(np.sum(day_hourly["MeOH_prod_ng_kgph"].values))

    def green_share(sum_g, sum_ng):
        tot = sum_g + sum_ng
        return (sum_g / tot) if tot > 1e-9 else np.nan

    daily_rows.append({
        "run_id": RUN_ID,
        "model_tag": MODEL_TAG,
        "scenario_tag": SCENARIO_TAG,
        "demand_tag": DEMAND_TAG,
        "day_of_year": d,
        "date_utc": str(day_hours[0].date()),
        "has_incumbent": bool(has_incumbent),
        "is_optimal": bool(is_optimal),
        "hit_time_limit": bool(hit_time_limit),
        "mip_gap": float(mip_gap) if mip_gap is not None else np.nan,
        "is_within_target_gap": bool(is_within_target_gap),
        "is_acceptable": bool(is_acceptable),

        "total_cost_eur": total_cost,
        "obj1_total_unmet_kg": total_unmet_kg,
        "obj2_total_emissions_kg": float(obj2_val) if not np.isnan(obj2_val) else np.nan,
        "metric_operating_cost_eur": total_cost,
        "emis_ef_mode": EMIS_EF_MODE.lower(),

        "res_generation_kWh": res_gen_kwh,
        "res_used_kWh": res_used_kwh,
        "res_curtailed_kWh": res_curt_kwh,
        "grid_import_kWh": grid_import_kwh,
        "grid_share_percent": grid_share_percent,

        "battery_soc_avg": float(np.mean(day_hourly["battery_soc"].values)),
        "tank_H2_level_avg_kg": float(np.mean(day_hourly["tank_H2_g_kg"].values + day_hourly["tank_H2_ng_kg"].values)),
        "tank_NH3_level_avg_kg": float(np.mean(day_hourly["tank_NH3_g_kg"].values + day_hourly["tank_NH3_ng_kg"].values)),
        "tank_MeOH_level_avg_kg": float(np.mean(day_hourly["tank_MeOH_g_kg"].values + day_hourly["tank_MeOH_ng_kg"].values)),

        "H2_prod_green_share": green_share(H2_g_sum, H2_ng_sum),
        "NH3_prod_green_share": green_share(NH3_g_sum, NH3_ng_sum),
        "MeOH_prod_green_share": green_share(Me_g_sum, Me_ng_sum),

        # grid emissions only
        "emissions_grid_aef_kg": grid_emis_aef,
        "emissions_grid_mef_upramp_kg": grid_emis_mef_upramp,
        "emissions_grid_mef_soft_kg": grid_emis_mef_soft,

        # direct emissions separate
        "emissions_direct_nh3_kg": direct_emis_nh3,
        "emissions_direct_meoh_kg": direct_emis_meoh,
        "emissions_direct_total_kg": direct_emis_total,

        "unmet_H2_kg": float(np.sum(day_hourly["unmet_h2_kgph"].values)),
        "unmet_NH3_kg": float(np.sum(day_hourly["unmet_nh3_kgph"].values)),
        "unmet_MeOH_kg": float(np.sum(day_hourly["unmet_meoh_kgph"].values)),

        "demand_H2_kg_day": demand_H2_kg_day,
        "demand_NH3_kg_day": demand_NH3_kg_day,
        "demand_MeOH_kg_day": demand_MeOH_kg_day,

        "elz_consumption_kWh_day": elz_kwh_day,
        "nh3_consumption_kWh_day": nh3_kwh_day,
        "meoh_consumption_kWh_day": meoh_kwh_day,
        "crk_consumption_kWh_day": crk_kwh_day,
        "units_total_consumption_kWh_day": units_kwh_day,

        # sales accounting
        "H2_sell_g_kg_day": float(np.sum(day_hourly["H2_sell_g_kgph"].values)),
        "H2_sell_ng_kg_day": float(np.sum(day_hourly["H2_sell_ng_kgph"].values)),
        "H2_sell_total_kg_day": float(np.sum(day_hourly["H2_sell_kgph"].values)),

        "NH3_sell_g_kg_day": float(np.sum(day_hourly["NH3_sell_g_kgph"].values)),
        "NH3_sell_ng_kg_day": float(np.sum(day_hourly["NH3_sell_ng_kgph"].values)),
        "NH3_sell_total_kg_day": float(np.sum(day_hourly["NH3_sell_kgph"].values)),

        "MeOH_sell_g_kg_day": float(np.sum(day_hourly["MeOH_sell_g_kgph"].values)),
        "MeOH_sell_ng_kg_day": float(np.sum(day_hourly["MeOH_sell_ng_kgph"].values)),
        "MeOH_sell_total_kg_day": float(np.sum(day_hourly["MeOH_sell_kgph"].values)),

        "eod_H2_total_kg": float(val(S_H2_g[24]) + val(S_H2_ng[24])),
        "eod_NH3_total_kg": float(val(S_NH3_g[24]) + val(S_NH3_ng[24])),
        "eod_MeOH_total_kg": float(val(S_MEOH_g[24]) + val(S_MEOH_ng[24])),
        "eod_req_H2_kg": float(ALPHA_TANK_FILL * S_H2_CAP),
        "eod_req_NH3_kg": float(ALPHA_TANK_FILL * S_NH3_CAP),
        "eod_req_MeOH_kg": float(ALPHA_TANK_FILL * S_MEOH_CAP),
        "eod_margin_H2_kg": float((val(S_H2_g[24]) + val(S_H2_ng[24])) - ALPHA_TANK_FILL * S_H2_CAP),
        "eod_margin_NH3_kg": float((val(S_NH3_g[24]) + val(S_NH3_ng[24])) - ALPHA_TANK_FILL * S_NH3_CAP),
        "eod_margin_MeOH_kg": float((val(S_MEOH_g[24]) + val(S_MEOH_ng[24])) - ALPHA_TANK_FILL * S_MEOH_CAP),

        "cost_grid_eur": grid_cost_eur,
        "cost_water_eur": water_cost_eur,
        "cost_co2_eur": co2_cost_eur,
        "cost_startup_elz_eur": startup_elz_eur,
        "cost_startup_nh3_eur": startup_nh3_eur,
        "cost_startup_meoh_eur": startup_meoh_eur,
        "cost_startup_crk_eur": startup_crk_eur,
    })

    # ---------------- Carry-over update ----------------
    carry["soc"] = val(SOC[24])
    carry["S_H2_g"] = val(S_H2_g[24]); carry["S_H2_ng"] = val(S_H2_ng[24])
    carry["S_NH3_g"] = val(S_NH3_g[24]); carry["S_NH3_ng"] = val(S_NH3_ng[24])
    carry["S_MEOH_g"] = val(S_MEOH_g[24]); carry["S_MEOH_ng"] = val(S_MEOH_ng[24])
    carry["y_elz_prev_mod"] = [int(round(val(y_elz_m[i,23]))) for i in I_elz]
    carry["y_nh3_prev"]  = int(round(val(y_nh3[23])))
    carry["y_meoh_prev"] = int(round(val(y_meoh[23])))
    carry["y_crk_prev"]  = int(round(val(y_crk[23])))
    carry["Q_nh3_prev"]  = float(val(Q_nh3[23]))
    carry["Q_meoh_prev"] = float(val(Q_meoh[23]))
    carry["Q_crk_prev"]  = float(val(Q_crk_in[23]))

    state_rows.append({
        "run_id": RUN_ID,
        "day_of_year": d,
        "date_utc": str(day_hours[0].date()),
        "SOC_end": carry["soc"],
        "S_H2_g_end": carry["S_H2_g"],
        "S_H2_ng_end": carry["S_H2_ng"],
        "S_NH3_g_end": carry["S_NH3_g"],
        "S_NH3_ng_end": carry["S_NH3_ng"],
        "S_MEOH_g_end": carry["S_MEOH_g"],
        "S_MEOH_ng_end": carry["S_MEOH_ng"],
        "y_elz_prev_mod_end": carry["y_elz_prev_mod"],
        "y_nh3_end": carry["y_nh3_prev"],
        "y_meoh_end": carry["y_meoh_prev"],
        "y_crk_end": carry["y_crk_prev"],
        "Q_nh3_end": carry["Q_nh3_prev"],
        "Q_meoh_end": carry["Q_meoh_prev"],
        "Q_crk_end": carry["Q_crk_prev"],
    })

    eps = 1e-6
    sim_cd = float(np.sum((day_hourly["bat_ch_mw"].values > eps) & (day_hourly["bat_dis_mw"].values > eps)))
    soc_viol = float(np.sum((day_hourly["battery_soc"].values < SOC_MIN - 1e-6) | (day_hourly["battery_soc"].values > SOC_MAX + 1e-6)))
    unmet_any = float(np.sum((day_hourly["unmet_h2_kgph"].values > eps) | (day_hourly["unmet_nh3_kgph"].values > eps) | (day_hourly["unmet_meoh_kgph"].values > eps)))

    diag_rows.append({
        "run_id": RUN_ID,
        "day_of_year": d,
        "date_utc": str(day_hours[0].date()),
        "flag_simultaneous_charge_discharge_hours": sim_cd,
        "flag_soc_bound_violation_hours": soc_viol,
        "flag_unmet_demand_hours": unmet_any,
    })

# ---------------- End-of-run aggregation ----------------
daily_df = pd.DataFrame(daily_rows)
if daily_df.empty:
    raise RuntimeError("No days were solved (daily_df is empty).")

solver_df = pd.DataFrame(solver_rows)
state_df  = pd.DataFrame(state_rows)
diag_df   = pd.DataFrame(diag_rows)

if len(all_hourly_frames) > 0:
    hourly_all_df = pd.concat(all_hourly_frames, ignore_index=True)
    hourly_all_df.to_csv(OUT_POST / "hourly_timeseries_all_days.csv", index=False)
else:
    hourly_all_df = pd.DataFrame()

daily_df.to_csv(OUT_POST / "daily_summary.csv", index=False)
solver_df.to_csv(OUT_POST / "solver_log_daily.csv", index=False)
state_df.to_csv(OUT_POST / "state_trace.csv", index=False)
diag_df.to_csv(OUT_POST / "daily_diagnostics_flags.csv", index=False)

annual_summary = {
    "run_id": RUN_ID,
    "model_tag": MODEL_TAG,
    "scenario_tag": SCENARIO_TAG,
    "demand_tag": DEMAND_TAG,
    "start_day_of_year": START_DAY_OF_YEAR,
    "n_days_run": int(len(daily_df)),

    "total_cost_eur": float(daily_df["total_cost_eur"].sum()),

    "total_grid_emissions_aef_kg": float(daily_df["emissions_grid_aef_kg"].sum()),
    "total_grid_emissions_mef_upramp_kg": float(daily_df["emissions_grid_mef_upramp_kg"].sum()),
    "total_grid_emissions_mef_soft_kg": float(daily_df["emissions_grid_mef_soft_kg"].sum()),

    "total_direct_emissions_nh3_kg": float(daily_df["emissions_direct_nh3_kg"].sum()),
    "total_direct_emissions_meoh_kg": float(daily_df["emissions_direct_meoh_kg"].sum()),
    "total_direct_emissions_kg": float(daily_df["emissions_direct_total_kg"].sum()),

    "total_grid_import_kWh": float(daily_df["grid_import_kWh"].sum()),
    "total_res_used_kWh": float(daily_df["res_used_kWh"].sum()),
    "total_res_curtailed_kWh": float(daily_df["res_curtailed_kWh"].sum()),
    "avg_grid_share_percent": float(daily_df["grid_share_percent"].mean()),

    "total_unmet_H2_kg": float(daily_df["unmet_H2_kg"].sum()),
    "total_unmet_NH3_kg": float(daily_df["unmet_NH3_kg"].sum()),
    "total_unmet_MeOH_kg": float(daily_df["unmet_MeOH_kg"].sum()),

    "total_H2_sell_g_kg": float(daily_df["H2_sell_g_kg_day"].sum()),
    "total_H2_sell_ng_kg": float(daily_df["H2_sell_ng_kg_day"].sum()),
    "total_H2_sell_kg": float(daily_df["H2_sell_total_kg_day"].sum()),

    "total_NH3_sell_g_kg": float(daily_df["NH3_sell_g_kg_day"].sum()),
    "total_NH3_sell_ng_kg": float(daily_df["NH3_sell_ng_kg_day"].sum()),
    "total_NH3_sell_kg": float(daily_df["NH3_sell_total_kg_day"].sum()),

    "total_MeOH_sell_g_kg": float(daily_df["MeOH_sell_g_kg_day"].sum()),
    "total_MeOH_sell_ng_kg": float(daily_df["MeOH_sell_ng_kg_day"].sum()),
    "total_MeOH_sell_kg": float(daily_df["MeOH_sell_total_kg_day"].sum()),
}
pd.DataFrame([annual_summary]).to_csv(OUT_POST / "annual_summary.csv", index=False)

# ---------------- Plots ----------------
daily_df = daily_df.copy()
daily_df["date"] = pd.to_datetime(daily_df["date_utc"])

save_lineplot(daily_df, "date", ["total_cost_eur"], f"{RUN_ID} • Daily total cost", "EUR", OUT_PLOTS / "P01_daily_total_cost.png")

if "obj1_total_unmet_kg" in daily_df.columns:
    save_lineplot(
        daily_df, "date", ["obj1_total_unmet_kg"],
        f"{RUN_ID} • Objective-1 total unmet (daily)", "kg/day",
        OUT_PLOTS / "P07_obj1_total_unmet_daily.png"
    )

if "obj2_total_emissions_kg" in daily_df.columns:
    save_lineplot(
        daily_df, "date", ["obj2_total_emissions_kg"],
        f"{RUN_ID} • Objective-2 emissions (daily)", "kg CO2/day",
        OUT_PLOTS / "P02b_daily_objective2_emissions.png"
    )

tmp = daily_df[["date","emissions_grid_aef_kg","emissions_grid_mef_upramp_kg","emissions_grid_mef_soft_kg"]]
save_lineplot(
    tmp, "date", ["emissions_grid_aef_kg","emissions_grid_mef_upramp_kg","emissions_grid_mef_soft_kg"],
    f"{RUN_ID} • Daily grid emissions by signal", "kg CO2/day",
    OUT_PLOTS / "P02_daily_grid_emissions_three_signals.png"
)

tmp = daily_df[["date","emissions_direct_nh3_kg","emissions_direct_meoh_kg"]]
save_lineplot(
    tmp, "date", ["emissions_direct_nh3_kg","emissions_direct_meoh_kg"],
    f"{RUN_ID} • Daily direct emissions by unit", "kg CO2/day",
    OUT_PLOTS / "P17_direct_emissions_daily_total.png"
)

tmp = daily_df[["date","grid_import_kWh"]]
save_lineplot(tmp, "date", ["grid_import_kWh"], f"{RUN_ID} • Daily grid import", "kWh", OUT_PLOTS / "P03_daily_grid_import_kwh.png")

tmp = daily_df[["date","res_used_kWh","res_curtailed_kWh"]]
save_lineplot(
    tmp, "date", ["res_used_kWh","res_curtailed_kWh"],
    f"{RUN_ID} • Daily RES used vs curtailed", "kWh",
    OUT_PLOTS / "P04_daily_res_used_vs_curtailed.png"
)

solver_df = solver_df.copy()
solver_df["date"] = pd.to_datetime(solver_df["date_utc"])
tmp = solver_df[["date","runtime_seconds","mip_gap"]]
save_lineplot(tmp, "date", ["runtime_seconds"], f"{RUN_ID} • Daily solver runtime", "seconds", OUT_PLOTS / "P05_daily_solver_runtime.png")
save_lineplot(tmp, "date", ["mip_gap"], f"{RUN_ID} • Daily solver MIP gap", "-", OUT_PLOTS / "P06_daily_mip_gap.png")

if all(c in solver_df.columns for c in ["obj_val","obj_bound"]):
    tmp = solver_df[["date","obj_val","obj_bound"]].copy()
    save_lineplot(
        tmp, "date", ["obj_val","obj_bound"],
        f"{RUN_ID} • Daily solver objective value vs bound", "objective units",
        OUT_PLOTS / "P06b_daily_obj_val_vs_bound.png"
    )

if hourly_all_df is not None and len(hourly_all_df) > 0:
    hdf = hourly_all_df.copy()
    hdf["dt"] = pd.to_datetime(hdf["datetime_utc"], utc=True)
    hdf = hdf.sort_values("dt")
else:
    hdf = pd.DataFrame()

if len(hdf) > 0:
    hh = hdf.copy()

    hh["tank_H2_total_kg"]   = hh["tank_H2_g_kg"].values   + hh["tank_H2_ng_kg"].values
    hh["tank_NH3_total_kg"]  = hh["tank_NH3_g_kg"].values  + hh["tank_NH3_ng_kg"].values
    hh["tank_MeOH_total_kg"] = hh["tank_MeOH_g_kg"].values + hh["tank_MeOH_ng_kg"].values

    hh["demand_H2_kgph"]   = hh["H2_sell_kgph"].values   + hh["unmet_h2_kgph"].values
    hh["demand_NH3_kgph"]  = hh["NH3_sell_kgph"].values  + hh["unmet_nh3_kgph"].values
    hh["demand_MeOH_kgph"] = hh["MeOH_sell_kgph"].values + hh["unmet_meoh_kgph"].values

    hh["pct_demand_met_H2"] = np.where(hh["demand_H2_kgph"].values > 1e-9, 100.0 * (1.0 - hh["unmet_h2_kgph"].values / hh["demand_H2_kgph"].values), 100.0)
    hh["pct_demand_met_NH3"] = np.where(hh["demand_NH3_kgph"].values > 1e-9, 100.0 * (1.0 - hh["unmet_nh3_kgph"].values / hh["demand_NH3_kgph"].values), 100.0)
    hh["pct_demand_met_MeOH"] = np.where(hh["demand_MeOH_kgph"].values > 1e-9, 100.0 * (1.0 - hh["unmet_meoh_kgph"].values / hh["demand_MeOH_kgph"].values), 100.0)

    for c in ["pct_demand_met_H2","pct_demand_met_NH3","pct_demand_met_MeOH"]:
        hh[c] = hh[c].clip(lower=0.0, upper=100.0)

    hh["pct_tank_fill_H2"]   = 100.0 * hh["tank_H2_total_kg"].values / float(S_H2_CAP)
    hh["pct_tank_fill_NH3"]  = 100.0 * hh["tank_NH3_total_kg"].values / float(S_NH3_CAP)
    hh["pct_tank_fill_MeOH"] = 100.0 * hh["tank_MeOH_total_kg"].values / float(S_MEOH_CAP)

    for c in ["pct_tank_fill_H2","pct_tank_fill_NH3","pct_tank_fill_MeOH"]:
        hh[c] = hh[c].clip(lower=0.0, upper=100.0)

    binary_cols_present = [c for c in ["su_crk","y_crk","su_meoh","y_meoh","su_nh3","y_nh3","su_elz","y_elz","b_ch"] if c in hh.columns]

    ops_cols = (
        ["run_id","model_tag","scenario_tag","demand_tag","dt","day_id","day_of_year","hour_in_day"]
        + binary_cols_present
        + ["pct_demand_met_H2","pct_demand_met_NH3","pct_demand_met_MeOH",
           "pct_tank_fill_H2","pct_tank_fill_NH3","pct_tank_fill_MeOH"]
    )
    ops_cols = [c for c in ops_cols if c in hh.columns]
    hh[ops_cols].copy().to_csv(OUT_POST / "hourly_ops_signals.csv", index=False)

if len(hdf) > 0:
    tmp = hdf[["dt","grid_import_mw","res_curtailed_mw","res_used_mw"]].copy()
    save_lineplot(
        tmp, "dt", ["grid_import_mw","res_curtailed_mw","res_used_mw"],
        f"{RUN_ID} • Hourly grid import vs RES used/curtailed", "MW",
        OUT_PLOTS / "P03b_hourly_grid_vs_res_used_curtailed.png"
    )

    for prod_col, fname_stub, title_stub, ylab in [
        ("H2_prod_kgph",  "H2",  "H2 production",  "kg/h"),
        ("NH3_prod_kgph", "NH3", "NH3 production", "kg/h"),
        ("MeOH_prod_kgph","MeOH","MeOH production","kg/h"),
    ]:
        tmp = hdf[["dt", prod_col]]
        save_lineplot(tmp, "dt", [prod_col], f"{RUN_ID} • {title_stub} (hourly)", ylab,
                      OUT_PLOTS / f"P08_prod_{fname_stub}_hourly.png")

    for sell_cols, fname_stub, title_stub in [
        (["H2_sell_g_kgph","H2_sell_ng_kgph"], "H2", "H2 sales g/ng"),
        (["NH3_sell_g_kgph","NH3_sell_ng_kgph"], "NH3", "NH3 sales g/ng"),
        (["MeOH_sell_g_kgph","MeOH_sell_ng_kgph"], "MeOH", "MeOH sales g/ng"),
    ]:
        tmp = hdf[["dt"] + sell_cols].copy()
        save_lineplot(tmp, "dt", sell_cols, f"{RUN_ID} • {title_stub} (hourly)", "kg/h",
                      OUT_PLOTS / f"P08b_sales_{fname_stub}_hourly_gng.png")

    prod_daily = (
        hdf.groupby("day_id")[["H2_prod_kgph","NH3_prod_kgph","MeOH_prod_kgph"]]
        .sum()
        .rename(columns={
            "H2_prod_kgph":"H2_prod_kg_day",
            "NH3_prod_kgph":"NH3_prod_kg_day",
            "MeOH_prod_kgph":"MeOH_prod_kg_day",
        })
        .reset_index()
        .merge(daily_df[["day_of_year","date"]], left_on="day_id", right_on="day_of_year", how="left")
    )

    for col, fname_stub, title_stub in [
        ("H2_prod_kg_day",  "H2",  "H2 production"),
        ("NH3_prod_kg_day", "NH3", "NH3 production"),
        ("MeOH_prod_kg_day","MeOH","MeOH production"),
    ]:
        tmp = prod_daily[["date", col]].dropna()
        save_lineplot(tmp, "date", [col], f"{RUN_ID} • {title_stub} (daily total)", "kg/day",
                      OUT_PLOTS / f"P09_prod_{fname_stub}_daily_total.png")

    sell_daily = (
        hdf.groupby("day_id")[[
            "H2_sell_g_kgph","H2_sell_ng_kgph",
            "NH3_sell_g_kgph","NH3_sell_ng_kgph",
            "MeOH_sell_g_kgph","MeOH_sell_ng_kgph"
        ]]
        .sum()
        .rename(columns={
            "H2_sell_g_kgph":"H2_sell_g_kg_day",
            "H2_sell_ng_kgph":"H2_sell_ng_kg_day",
            "NH3_sell_g_kgph":"NH3_sell_g_kg_day",
            "NH3_sell_ng_kgph":"NH3_sell_ng_kg_day",
            "MeOH_sell_g_kgph":"MeOH_sell_g_kg_day",
            "MeOH_sell_ng_kgph":"MeOH_sell_ng_kg_day",
        })
        .reset_index()
        .merge(daily_df[["day_of_year","date"]], left_on="day_id", right_on="day_of_year", how="left")
    )

    for cols, fname_stub, title_stub in [
        (["H2_sell_g_kg_day","H2_sell_ng_kg_day"], "H2", "H2 sales g/ng"),
        (["NH3_sell_g_kg_day","NH3_sell_ng_kg_day"], "NH3", "NH3 sales g/ng"),
        (["MeOH_sell_g_kg_day","MeOH_sell_ng_kg_day"], "MeOH", "MeOH sales g/ng"),
    ]:
        tmp = sell_daily[["date"] + cols].dropna()
        save_lineplot(tmp, "date", cols, f"{RUN_ID} • {title_stub} (daily total)", "kg/day",
                      OUT_PLOTS / f"P09b_sales_{fname_stub}_daily_total_gng.png")

    hdf["tank_H2_total"] = hdf["tank_H2_g_kg"].values + hdf["tank_H2_ng_kg"].values
    hdf["tank_NH3_total"] = hdf["tank_NH3_g_kg"].values + hdf["tank_NH3_ng_kg"].values
    hdf["tank_MeOH_total"] = hdf["tank_MeOH_g_kg"].values + hdf["tank_MeOH_ng_kg"].values

    hdf["tank_H2_green_share"] = np.where(hdf["tank_H2_total"] > 1e-9, hdf["tank_H2_g_kg"] / hdf["tank_H2_total"], np.nan)
    hdf["tank_NH3_green_share"] = np.where(hdf["tank_NH3_total"] > 1e-9, hdf["tank_NH3_g_kg"] / hdf["tank_NH3_total"], np.nan)
    hdf["tank_MeOH_green_share"] = np.where(hdf["tank_MeOH_total"] > 1e-9, hdf["tank_MeOH_g_kg"] / hdf["tank_MeOH_total"], np.nan)

    for col, fname_stub, title_stub in [
        ("tank_H2_green_share", "H2", "Tank H2 green share"),
        ("tank_NH3_green_share", "NH3", "Tank NH3 green share"),
        ("tank_MeOH_green_share", "MeOH", "Tank MeOH green share"),
    ]:
        tmp = hdf[["dt", col]].copy()
        tmp[col] = tmp[col].fillna(0.0)
        save_lineplot(tmp, "dt", [col], f"{RUN_ID} • {title_stub} (hourly)", "share",
                      OUT_PLOTS / f"P10_tank_share_{fname_stub}_hourly.png")

for col, fname_stub, title_stub, ylab in [
    ("demand_H2_kg_day", "H2", "H2 demand", "kg/day"),
    ("demand_NH3_kg_day", "NH3", "NH3 demand", "kg/day"),
    ("demand_MeOH_kg_day", "MeOH", "MeOH demand", "kg/day"),
]:
    tmp = daily_df[["date", col]]
    save_lineplot(tmp, "date", [col], f"{RUN_ID} • {title_stub} (daily)", ylab,
                  OUT_PLOTS / f"P12_demand_{fname_stub}_daily.png")

for col, fname_stub, title_stub in [
    ("elz_consumption_kWh_day", "elz", "Electrolyzer electricity"),
    ("nh3_consumption_kWh_day", "nh3", "NH3 synthesis electricity"),
    ("meoh_consumption_kWh_day", "meoh", "MeOH synthesis electricity"),
    ("crk_consumption_kWh_day", "crk", "Cracker electricity"),
]:
    tmp = daily_df[["date", col]]
    save_lineplot(tmp, "date", [col], f"{RUN_ID} • {title_stub} (daily)", "kWh/day",
                  OUT_PLOTS / f"P14_power_{fname_stub}_daily.png")

tmp = daily_df[["date","elz_consumption_kWh_day","nh3_consumption_kWh_day","meoh_consumption_kWh_day","crk_consumption_kWh_day"]]
save_lineplot(
    tmp, "date", ["elz_consumption_kWh_day","nh3_consumption_kWh_day","meoh_consumption_kWh_day","crk_consumption_kWh_day"],
    f"{RUN_ID} • Unit electricity consumption (daily)", "kWh/day",
    OUT_PLOTS / "P15_power_units_combined_daily.png"
)

if len(hdf) > 0:
    hdf["direct_emis_nh3_kgph"]  = float(de_NH3) * hdf["NH3_prod_kgph"].values
    hdf["direct_emis_meoh_kgph"] = float(de_MEOH) * hdf["MeOH_prod_kgph"].values

    tmp = hdf[["dt","direct_emis_nh3_kgph","direct_emis_meoh_kgph"]]
    save_lineplot(
        tmp, "dt", ["direct_emis_nh3_kgph","direct_emis_meoh_kgph"],
        f"{RUN_ID} • Direct emissions by unit (hourly)", "kg CO2/h",
        OUT_PLOTS / "P16_direct_emissions_hourly.png"
    )

    aef_series = aef[[AEF_COL]].reset_index().rename(columns={"datetime_utc":"dt", AEF_COL:"aef_kg_per_kwh"})
    mef_upramp_series = mef_upramp[[MEF_UPRAMP_COL]].reset_index().rename(columns={"datetime_utc":"dt", MEF_UPRAMP_COL:"mef_upramp_kg_per_kwh"})
    mef_soft_series = mef_soft[[MEF_SOFT_COL]].reset_index().rename(columns={"datetime_utc":"dt", MEF_SOFT_COL:"mef_soft_kg_per_kwh"})

    aef_series["dt"] = pd.to_datetime(aef_series["dt"], utc=True)
    mef_upramp_series["dt"] = pd.to_datetime(mef_upramp_series["dt"], utc=True)
    mef_soft_series["dt"] = pd.to_datetime(mef_soft_series["dt"], utc=True)

    hdf["dt"] = pd.to_datetime(hdf["dt"], utc=True)
    hdf = hdf.merge(aef_series[["dt","aef_kg_per_kwh"]], on="dt", how="left")
    hdf = hdf.merge(mef_upramp_series[["dt","mef_upramp_kg_per_kwh"]], on="dt", how="left")
    hdf = hdf.merge(mef_soft_series[["dt","mef_soft_kg_per_kwh"]], on="dt", how="left")

    for tag, ef_col in [
        ("aef", "aef_kg_per_kwh"),
        ("mef_upramp", "mef_upramp_kg_per_kwh"),
        ("mef_soft", "mef_soft_kg_per_kwh"),
    ]:
        hdf[f"grid_emis_{tag}_elz_kgph"]  = 1000.0 * hdf["elz_grid_mw"].values  * hdf[ef_col].values
        hdf[f"grid_emis_{tag}_nh3_kgph"]  = 1000.0 * hdf["nh3_grid_mw"].values  * hdf[ef_col].values
        hdf[f"grid_emis_{tag}_meoh_kgph"] = 1000.0 * hdf["meoh_grid_mw"].values * hdf[ef_col].values
        hdf[f"grid_emis_{tag}_crk_kgph"]  = 1000.0 * hdf["crk_grid_mw"].values  * hdf[ef_col].values

        tmp = hdf[["dt",
                   f"grid_emis_{tag}_elz_kgph",
                   f"grid_emis_{tag}_nh3_kgph",
                   f"grid_emis_{tag}_meoh_kgph",
                   f"grid_emis_{tag}_crk_kgph"]].copy()

        save_lineplot(
            tmp, "dt",
            [f"grid_emis_{tag}_elz_kgph", f"grid_emis_{tag}_nh3_kgph", f"grid_emis_{tag}_meoh_kgph", f"grid_emis_{tag}_crk_kgph"],
            f"{RUN_ID} • Allocated grid emissions by unit (hourly, {tag})", "kg CO2/h",
            OUT_PLOTS / f"P17b_grid_emissions_by_unit_hourly_{tag}.png"
        )

        daily_alloc = (
            hdf.groupby("day_id")[
                [f"grid_emis_{tag}_elz_kgph", f"grid_emis_{tag}_nh3_kgph", f"grid_emis_{tag}_meoh_kgph", f"grid_emis_{tag}_crk_kgph"]
            ]
            .sum()
            .rename(columns={
                f"grid_emis_{tag}_elz_kgph": f"grid_emis_{tag}_elz_kg_day",
                f"grid_emis_{tag}_nh3_kgph": f"grid_emis_{tag}_nh3_kg_day",
                f"grid_emis_{tag}_meoh_kgph": f"grid_emis_{tag}_meoh_kg_day",
                f"grid_emis_{tag}_crk_kgph": f"grid_emis_{tag}_crk_kg_day",
            })
            .reset_index()
            .merge(daily_df[["day_of_year","date"]], left_on="day_id", right_on="day_of_year", how="left")
        )

        tmp = daily_alloc[["date",
                           f"grid_emis_{tag}_elz_kg_day",
                           f"grid_emis_{tag}_nh3_kg_day",
                           f"grid_emis_{tag}_meoh_kg_day",
                           f"grid_emis_{tag}_crk_kg_day"]].dropna()

        save_lineplot(
            tmp, "date",
            [f"grid_emis_{tag}_elz_kg_day", f"grid_emis_{tag}_nh3_kg_day", f"grid_emis_{tag}_meoh_kg_day", f"grid_emis_{tag}_crk_kg_day"],
            f"{RUN_ID} • Allocated grid emissions by unit (daily total, {tag})", "kg CO2/day",
            OUT_PLOTS / f"P17c_grid_emissions_by_unit_daily_{tag}.png"
        )

    hdf["grid_emis_aef_total_kgph"]        = 1000.0 * hdf["grid_import_mw"].values * hdf["aef_kg_per_kwh"].values
    hdf["grid_emis_mef_upramp_total_kgph"] = 1000.0 * hdf["grid_import_mw"].values * hdf["mef_upramp_kg_per_kwh"].values
    hdf["grid_emis_mef_soft_total_kgph"]   = 1000.0 * hdf["grid_import_mw"].values * hdf["mef_soft_kg_per_kwh"].values

    hdf["ef_delta_upramp_minus_aef"] = hdf["mef_upramp_kg_per_kwh"].values - hdf["aef_kg_per_kwh"].values
    hdf["ef_delta_soft_minus_aef"]   = hdf["mef_soft_kg_per_kwh"].values   - hdf["aef_kg_per_kwh"].values

    hdf["grid_emis_delta_upramp_minus_aef_kgph"] = hdf["grid_emis_mef_upramp_total_kgph"].values - hdf["grid_emis_aef_total_kgph"].values
    hdf["grid_emis_delta_soft_minus_aef_kgph"]   = hdf["grid_emis_mef_soft_total_kgph"].values   - hdf["grid_emis_aef_total_kgph"].values

    hdf["cum_delta_upramp_minus_aef_kg"] = np.cumsum(hdf["grid_emis_delta_upramp_minus_aef_kgph"].fillna(0.0).values)
    hdf["cum_delta_soft_minus_aef_kg"]   = np.cumsum(hdf["grid_emis_delta_soft_minus_aef_kgph"].fillna(0.0).values)

    tmp = hdf[["dt","cum_delta_upramp_minus_aef_kg","cum_delta_soft_minus_aef_kg"]]
    save_lineplot(
        tmp, "dt", ["cum_delta_upramp_minus_aef_kg","cum_delta_soft_minus_aef_kg"],
        f"{RUN_ID} • Cumulative Δ grid emissions vs AEF", "kg CO2",
        OUT_PLOTS / "P02e_cumulative_delta_grid_emissions_vs_aef.png"
    )

    hdf[[
        "run_id","model_tag","scenario_tag","demand_tag","dt","day_id","day_of_year","hour_in_day",
        "grid_import_mw",
        "aef_kg_per_kwh","mef_upramp_kg_per_kwh","mef_soft_kg_per_kwh",
        "grid_emis_aef_total_kgph","grid_emis_mef_upramp_total_kgph","grid_emis_mef_soft_total_kgph",
        "ef_delta_upramp_minus_aef","ef_delta_soft_minus_aef",
        "grid_emis_delta_upramp_minus_aef_kgph","grid_emis_delta_soft_minus_aef_kgph",
        "cum_delta_upramp_minus_aef_kg","cum_delta_soft_minus_aef_kg"
    ]].to_csv(OUT_POST / "hourly_emission_signal_diagnostics.csv", index=False)

if not hdf.empty:
    mod_ids = sorted({int(re.match(r"^elz_m(\d+)_y$", c).group(1)) for c in hdf.columns if re.match(r"^elz_m(\d+)_y$", c)})
    mod_ids = [i for i in mod_ids if (f"elz_m{i}_y" in hdf.columns) and (f"elz_m{i}_b_g" in hdf.columns)]

    if len(mod_ids) > 0:
        fig = plt.figure(figsize=(12, 6))
        ax1 = fig.add_subplot(2, 1, 1)
        for i in mod_ids:
            ax1.plot(hdf["dt"], hdf[f"elz_m{i}_y"].values, label=f"m{i}_y")
        ax1.set_title(f"{RUN_ID} • ELZ modules on/off (y)")
        ax1.set_ylabel("binary")
        ax1.legend(ncol=4, fontsize=8)

        ax2 = fig.add_subplot(2, 1, 2)
        for i in mod_ids:
            ax2.plot(hdf["dt"], hdf[f"elz_m{i}_b_g"].values, label=f"m{i}_b_g")
        ax2.set_title(f"{RUN_ID} • ELZ modules green-choice (b_g)")
        ax2.set_ylabel("binary")
        ax2.legend(ncol=4, fontsize=8)

        plt.tight_layout()
        plt.savefig(OUT_PLOTS / "P90_elz_modules_y_and_bg.png", dpi=150)
        plt.close()

        i0 = mod_ids[0]
        needed = [f"elz_m{i0}_P_tot", f"elz_m{i0}_P_g", f"elz_m{i0}_P_ng"]
        if all(c in hdf.columns for c in needed):
            tmp = hdf[["dt"] + needed].copy()
            save_lineplot(
                tmp, "dt", needed,
                f"{RUN_ID} • ELZ module m{i0} power split (P_tot, P_g, P_ng)",
                "MW",
                OUT_PLOTS / f"P91_elz_module_m{i0}_power_split.png"
            )

cost_cols = [
    ("cost_grid_eur", "Grid energy"),
    ("cost_water_eur", "Water"),
    ("cost_co2_eur", "CO2"),
    ("cost_startup_elz_eur", "Startup ELZ"),
    ("cost_startup_nh3_eur", "Startup NH3"),
    ("cost_startup_meoh_eur", "Startup MeOH"),
    ("cost_startup_crk_eur", "Startup CRK"),
]
save_stacked_bar_by_day(
    daily_df, "date", cost_cols,
    f"{RUN_ID} • Daily cost breakdown (stacked)", "EUR/day",
    OUT_PLOTS / "P18_daily_cost_breakdown_stacked.png"
)

res_cols = [
    ("res_used_kWh", "RES used"),
    ("res_curtailed_kWh", "RES curtailed"),
]
save_stacked_bar_by_day(
    daily_df, "date", res_cols,
    f"{RUN_ID} • Daily RES split (used vs curtailed)", "kWh/day",
    OUT_PLOTS / "P19_daily_res_used_vs_curtailed_stacked.png"
)

# ---------------- Packaging ----------------
ZIP_PATH = OUT_BASE / f"{RUN_ID}.zip"
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in OUT_RUN.rglob("*"):
        if p.is_file():
            zf.write(p, arcname=str(p.relative_to(OUT_RUN)))

print("Run completed.")
print("Run folder:", OUT_RUN)
print("ZIP:", ZIP_PATH)
print("Total wall time (s):", time.time() - t0_global)

## 5.3 M2 emission cap model


In [ ]:
# =====================  M2 =====================
# Assumes utility/precompute cell has already run and wrote /content/data/model_ready/*.csv and params_scenario_all.json
# min unmet demand >> min cost with daily emissions cap

# ---------------- Header / metadata ----------------
MODEL_TAG = "M2"
FORMULATION_TAG = "M2_final_v2"
SCENARIO_TAG = "S1"
DEMAND_TAG = "D1"
GREEN_TAG = "S1_green_split_with_rfnbo_hourly"

# Choose one emissions accounting basis for the cap constraint:
#   "aef"
#   "mef_upramp"
#   "mef_soft"
EMISSIONS_KEY = "aef"

#  pick one alpha manually for capping
ALPHA_OPTIONS = [0.2,0.5,0.8]
ALPHA_IDX = 0
ALPHA = float(ALPHA_OPTIONS[ALPHA_IDX])

# ---------------- Imports ----------------
import os, json, math, zipfile, shutil, time, re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gurobipy as gp
from gurobipy import GRB

# ---------------- Global configuration ----------------
DATA_DIR = Path("/content/data")
MR = DATA_DIR / "model_ready"

# Required model-ready files
RES_CSV         = MR / "res_hourly_2024.csv"
DEM_CSV         = MR / "demand_hourly_2024.csv"
PRICE_CSV       = MR / "grid_price_hourly_2024.csv"
AEF_CSV         = MR / "grid_aef_hourly_2024.csv"
MEF_UPRAMP_CSV  = MR / "grid_mef_upramp_hourly_2024.csv"
MEF_SOFT_CSV    = MR / "grid_mef_soft_hourly_2024.csv"
PARAMS_JSON     = MR / "params_scenario_all.json"

# Bounds file
M2_CAP_RANGES_CSV = MR / "M2_S1_emission_bounds_input_file.csv"

# Emission column names
AEF_COL        = "grid_aef_predicted_kg_per_kwh"
MEF_UPRAMP_COL = "grid_mef_upramp_pred_kg_per_kwh"
MEF_SOFT_COL   = "grid_mef_soft_kg_per_kwh"

# ---------------- Run controls ----------------
N_DAYS_TO_RUN     = 366
START_DAY_OF_YEAR = 1
MIPGAP            = 0.01
TIME_LIMIT_S      = 350
OUTPUTFLAG        = 1

# ---------------- License handling ----------------
for lic_path in [Path("/content/data/gurobi.lic")]:
    if lic_path.exists():
        os.environ["GRB_LICENSE_FILE"] = str(lic_path)
        break

# ---------------- Run ID builder ----------------
def build_run_id():
    stamp = pd.Timestamp.utcnow().strftime("%Y%m%dT%H%M%SZ")
    return f"{MODEL_TAG}__{SCENARIO_TAG}__{DEMAND_TAG}__start{START_DAY_OF_YEAR:03d}__ndays{N_DAYS_TO_RUN:03d}__{stamp}"

RUN_ID = build_run_id()

# ---------------- Output paths ----------------
OUT_BASE  = DATA_DIR / "stage2_runs"
OUT_RUN   = OUT_BASE / RUN_ID
OUT_DAILY = OUT_RUN / "daily"
OUT_POST  = OUT_RUN / "post"
OUT_PLOTS = OUT_RUN / "plots"
OUT_RUN.mkdir(parents=True, exist_ok=True)
OUT_DAILY.mkdir(parents=True, exist_ok=True)
OUT_POST.mkdir(parents=True, exist_ok=True)
OUT_PLOTS.mkdir(parents=True, exist_ok=True)

# ---------------- Input loading ----------------
def read_hourly_csv(path: Path, time_col="datetime_utc") -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    df = pd.read_csv(path)
    if time_col not in df.columns:
        raise ValueError(f"{path.name}: missing '{time_col}'. Found: {list(df.columns)}")
    df[time_col] = pd.to_datetime(df[time_col], utc=True)
    df = df.sort_values(time_col).set_index(time_col)
    df = df.tz_convert("UTC") if df.index.tz is not None else df.tz_localize("UTC")
    return df

res        = read_hourly_csv(RES_CSV)[["pv_mw","wind_mw","res_total_mw"]]
dem        = read_hourly_csv(DEM_CSV)[["h2_demand_kgph","nh3_demand_kgph","meoh_demand_kgph"]]
price      = read_hourly_csv(PRICE_CSV)[["da_elec_price_EURperMWh"]]
aef        = read_hourly_csv(AEF_CSV)[[AEF_COL]]
mef_upramp = read_hourly_csv(MEF_UPRAMP_CSV)[[MEF_UPRAMP_COL]]
mef_soft   = read_hourly_csv(MEF_SOFT_CSV)[[MEF_SOFT_COL]]

with open(PARAMS_JSON, "r") as f:
    params = json.load(f)

# ---------------- Bounds file ----------------
if not M2_CAP_RANGES_CSV.exists():
    raise FileNotFoundError(f"Missing M2 cap ranges file: {M2_CAP_RANGES_CSV}")

m2_bounds_raw = pd.read_csv(M2_CAP_RANGES_CSV)

required_cols = [
    "day_of_year", "scenario",
    "mef_upramp_lb_kg", "mef_upramp_up_kg",
    "mef_soft_lb_kg", "mef_soft_up_kg",
    "aef_lb_kg", "aef_up_kg"
]
missing = [c for c in required_cols if c not in m2_bounds_raw.columns]
if missing:
    raise ValueError(f"{M2_CAP_RANGES_CSV.name}: missing columns: {missing}. Found: {list(m2_bounds_raw.columns)}")

m2_bounds_raw["day_of_year"] = m2_bounds_raw["day_of_year"].astype(int)
m2_bounds_raw["scenario"] = m2_bounds_raw["scenario"].astype(str)

m2_bounds = (
    m2_bounds_raw.loc[m2_bounds_raw["scenario"].eq(str(SCENARIO_TAG))].copy()
    .sort_values("day_of_year")
    .set_index("day_of_year")
)

if m2_bounds.empty:
    raise ValueError(
        f"M2 bounds file has no rows for scenario='{SCENARIO_TAG}'. "
        f"Available scenarios: {sorted(m2_bounds_raw['scenario'].unique().tolist())}"
    )

def compute_daily_cap(lb_kg: float, ub_kg: float, alpha: float):
    lb = float(lb_kg)
    ub = float(ub_kg)

    is_negative_day = bool(ub < 0.0)
    is_cap_repair_day = bool((ub >= 0.0) and (ub < lb))

    if is_negative_day:
        return ub, np.nan, is_negative_day, is_cap_repair_day

    if is_cap_repair_day:
        return ub, np.nan, is_negative_day, is_cap_repair_day

    cap = lb + float(alpha) * (ub - lb)
    return cap, float(alpha), is_negative_day, is_cap_repair_day

def alpha_tag(a: float) -> str:
    s = f"{a:.3f}".rstrip("0").rstrip(".")
    return s.replace(".", "p")

# ---------------- Input validation ----------------
idx_2024 = pd.date_range("2024-01-01 00:00:00+00:00", "2024-12-31 23:00:00+00:00", freq="h", tz="UTC")

def assert_full_hourly(df: pd.DataFrame, name: str) -> pd.DataFrame:
    if df.index.duplicated().any():
        dup = df.index[df.index.duplicated()][0]
        raise ValueError(f"{name}: duplicated timestamp example: {dup}")
    missing = idx_2024.difference(df.index)
    extra   = df.index.difference(idx_2024)
    if len(missing) > 0:
        raise ValueError(f"{name}: missing {len(missing)} hours (first 5: {missing[:5].tolist()})")
    if len(extra) > 0:
        raise ValueError(f"{name}: extra {len(extra)} hours outside 2024 (first 5: {extra[:5].tolist()})")

    df2 = df.reindex(idx_2024)
    df2.index.name = "datetime_utc"

    if df2.isna().any().any():
        cols = df2.columns[df2.isna().any()].tolist()
        where = df2[df2[cols].isna().any(axis=1)].index[:5].tolist()
        raise ValueError(f"{name}: NaNs after reindex in cols {cols}; example {where}")
    return df2

res        = assert_full_hourly(res, "res")
dem        = assert_full_hourly(dem, "dem")
price      = assert_full_hourly(price, "price")
aef        = assert_full_hourly(aef, "aef")
mef_upramp = assert_full_hourly(mef_upramp, "mef_upramp")
mef_soft   = assert_full_hourly(mef_soft, "mef_soft")

# ---------------- Scalar parameters (aligned with M0-V2 / M1-V2) ----------------
P_GRID_MAX_MW = 295.58

# Battery
P_BAT_MAX_MW  = 50.0
E_BAT_MAX_MWH = 100.32
ETA_CH        = 0.964
ETA_DIS       = 0.964
SOC_MIN       = 0.15
SOC_MAX       = 0.95
SOC_INIT      = 0.5

# Electrolyzer
P_ELZ_MIN = 8
P_ELZ_MAX = 52.25
H_ELZ_MAX = 914.39161
A_elz = -0.075045052
B_elz = 21.73040904
C_elz = -16.14458074
N_stack = 4
omega_H2O = 0.015
c_H2O = 2.24
C_su_elz = 1392

P_ELZ_MAX_TOT = N_stack * P_ELZ_MAX
H_ELZ_MAX_TOT = N_stack * H_ELZ_MAX

H_min = A_elz*(P_ELZ_MIN**2) + B_elz*P_ELZ_MIN + C_elz
H_max = A_elz*(P_ELZ_MAX**2) + B_elz*P_ELZ_MAX + C_elz
Q1 = (H_max - H_min) / (P_ELZ_MAX - P_ELZ_MIN)
Q0 = H_min - Q1 * P_ELZ_MIN

# NH3
Q_NH3_CAP = 13472.04
f_NH3_min = 0.61
RR_NH3 = 0.2
mH2_to_NH3 = 0.18
mN2_to_NH3 = 0.87
SEC_HB = 0.36
SEC_ASU = 0.068
C_su_NH3 = 2397.46
de_NH3 = 0.00519

# MeOH
Q_MEOH_CAP = 5818.856
f_MEOH_min = 0.1
R_MEOH = 0.5 * Q_MEOH_CAP
mH2_to_MEOH = 0.22
SEC_MEOH = 161.75/1000.0
mCO2_to_MEOH = 1.59
c_CO2 = 0.2
C_su_MEOH = 1159.37
de_MEOH = 0.08

# Cracker
Q_CRK_CAP = 6062.416
f_CRK_min = 0.05
R_CRK = 0.5 * Q_CRK_CAP
kH2_out = 0.12024
kNH3_rec = 0.11848
kH2_fuel = 0.035294
SEC_CRK_E = 0.524667
kSteam = 0.941176
eSteam = 0.75
omega_steam = 0.001
C_su_CRK = 1000.0

# Tanks
H_TANK = 12.0
ALPHA_TANK_FILL = 0.15

S_H2_CAP   = H_TANK * H_ELZ_MAX_TOT
S_NH3_CAP  = H_TANK * Q_NH3_CAP
S_MEOH_CAP = H_TANK * Q_MEOH_CAP

S_H2_INIT   = ALPHA_TANK_FILL * S_H2_CAP
S_NH3_INIT  = ALPHA_TANK_FILL * S_NH3_CAP
S_MEOH_INIT = ALPHA_TANK_FILL * S_MEOH_CAP

# ---------------- Helpers ----------------
def day_to_time_slice(day_of_year: int):
    start_ts = idx_2024[0] + pd.Timedelta(days=day_of_year-1)
    end_ts = start_ts + pd.Timedelta(hours=24)
    return pd.date_range(start_ts, end_ts - pd.Timedelta(hours=1), freq="h", tz="UTC")

def save_lineplot(df, xcol, ycols, title, ylabel, outpath):
    plt.figure(figsize=(14, 4))
    x = df[xcol] if xcol in df.columns else df.index
    for c in ycols:
        plt.plot(x, df[c], label=c)
    plt.title(title)
    plt.ylabel(ylabel)
    plt.legend()
    plt.tight_layout()
    plt.savefig(outpath, dpi=150)
    plt.close()

def save_stacked_bar_by_day(df, date_col, components, title, ylabel, outpath):
    x = pd.to_datetime(df[date_col])
    plt.figure(figsize=(14,5))
    bottom = np.zeros(len(df), dtype=float)
    for col, label in components:
        vals = df[col].values.astype(float)
        plt.bar(x, vals, bottom=bottom, label=label)
        bottom += vals
    plt.title(title)
    plt.ylabel(ylabel)
    plt.legend()
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(outpath, dpi=150)
    plt.close()

def safe_get_attr(model, attr, default=np.nan):
    try:
        return getattr(model, attr)
    except Exception:
        return default

def status_name(code: int) -> str:
    mp = {
        GRB.LOADED: "LOADED",
        GRB.OPTIMAL: "OPTIMAL",
        GRB.INFEASIBLE: "INFEASIBLE",
        GRB.INF_OR_UNBD: "INF_OR_UNBD",
        GRB.UNBOUNDED: "UNBOUNDED",
        GRB.CUTOFF: "CUTOFF",
        GRB.ITERATION_LIMIT: "ITERATION_LIMIT",
        GRB.NODE_LIMIT: "NODE_LIMIT",
        GRB.TIME_LIMIT: "TIME_LIMIT",
        GRB.SOLUTION_LIMIT: "SOLUTION_LIMIT",
        GRB.INTERRUPTED: "INTERRUPTED",
        GRB.NUMERIC: "NUMERIC",
        GRB.SUBOPTIMAL: "SUBOPTIMAL",
        GRB.INPROGRESS: "INPROGRESS",
        GRB.USER_OBJ_LIMIT: "USER_OBJ_LIMIT",
    }
    return mp.get(int(code), f"STATUS_{int(code)}")

def get_objpass_metrics(model, obj_idx: int):
    out = {
        "obj_idx": int(obj_idx),
        "pass_status": np.nan,
        "pass_status_name": "NA",
        "pass_runtime": np.nan,
        "pass_objval": np.nan,
        "pass_objbound": np.nan,
        "pass_mipgap": np.nan,
    }
    try:
        prev_objnum  = getattr(model.Params, "ObjNumber", None)
        prev_passnum = getattr(model.Params, "ObjPassNumber", None)

        model.Params.ObjNumber = int(obj_idx)
        model.Params.ObjPassNumber = -1

        st = safe_get_attr(model, "ObjPassNStatus", np.nan)
        out["pass_status"] = float(st) if not np.isnan(st) else np.nan
        out["pass_status_name"] = status_name(int(st)) if not np.isnan(st) else "NA"
        out["pass_runtime"]  = float(safe_get_attr(model, "ObjPassNRuntime", np.nan))
        out["pass_objval"]   = float(safe_get_attr(model, "ObjPassNObjVal", np.nan))
        out["pass_objbound"] = float(safe_get_attr(model, "ObjPassNObjBound", np.nan))
        out["pass_mipgap"]   = float(safe_get_attr(model, "ObjPassNMipGap", np.nan))

        if prev_objnum is not None:
            model.Params.ObjNumber = prev_objnum
        if prev_passnum is not None:
            model.Params.ObjPassNumber = prev_passnum
    except Exception:
        pass
    return out

# ---------------- Carry-over initialization ----------------
carry = {
    "soc": SOC_INIT,
    "S_H2_g": 0.5*S_H2_INIT,
    "S_H2_ng": 0.5*S_H2_INIT,
    "S_NH3_g": 0.5*S_NH3_INIT,
    "S_NH3_ng": 0.5*S_NH3_INIT,
    "S_MEOH_g": 0.5*S_MEOH_INIT,
    "S_MEOH_ng": 0.5*S_MEOH_INIT,
    "y_elz_prev_mod": [0]*N_stack,
    "y_nh3_prev": 0,
    "y_meoh_prev": 0,
    "y_crk_prev": 0,
    "Q_nh3_prev": 0.0,
    "Q_meoh_prev": 0.0,
    "Q_crk_prev": 0.0,
}

daily_rows = []
solver_rows = []
state_rows = []
diag_rows = []
all_hourly_frames = []

# ---------------- Run loop ----------------
t0_global = time.time()

for d in range(START_DAY_OF_YEAR, START_DAY_OF_YEAR + N_DAYS_TO_RUN):
    day_hours = day_to_time_slice(d)

    if day_hours[0] < idx_2024[0] or day_hours[-1] > idx_2024[-1]:
        break

    day_dir = OUT_DAILY / f"day_{d:03d}"
    day_dir.mkdir(parents=True, exist_ok=True)

    pv_av    = res.loc[day_hours, "pv_mw"].values.astype(float)
    wd_av    = res.loc[day_hours, "wind_mw"].values.astype(float)
    pr_eur   = price.loc[day_hours, "da_elec_price_EURperMWh"].values.astype(float)
    dem_h2   = dem.loc[day_hours, "h2_demand_kgph"].values.astype(float)
    dem_nh3  = dem.loc[day_hours, "nh3_demand_kgph"].values.astype(float)
    dem_meoh = dem.loc[day_hours, "meoh_demand_kgph"].values.astype(float)

    aef_k        = aef.loc[day_hours, AEF_COL].values.astype(float)
    mef_upramp_k = mef_upramp.loc[day_hours, MEF_UPRAMP_COL].values.astype(float)
    mef_soft_k   = mef_soft.loc[day_hours, MEF_SOFT_COL].values.astype(float)

    # ---------------- Daily cap selection ----------------
    if d not in m2_bounds.index:
        raise ValueError(f"M2 bounds table missing day_of_year={d} for scenario='{SCENARIO_TAG}'")

    if EMISSIONS_KEY == "aef":
        lb_col, ub_col = "aef_lb_kg", "aef_up_kg"
    elif EMISSIONS_KEY == "mef_upramp":
        lb_col, ub_col = "mef_upramp_lb_kg", "mef_upramp_up_kg"
    elif EMISSIONS_KEY == "mef_soft":
        lb_col, ub_col = "mef_soft_lb_kg", "mef_soft_up_kg"
    else:
        raise ValueError("EMISSIONS_KEY must be one of: 'aef', 'mef_upramp', 'mef_soft'")

    lb_day = float(m2_bounds.loc[d, lb_col])
    ub_day = float(m2_bounds.loc[d, ub_col])
    EMIS_CAP_DAY_KG, ALPHA_USED_DAY, IS_NEGATIVE_DAY, IS_CAP_REPAIR_DAY = compute_daily_cap(lb_day, ub_day, ALPHA)

    # ---------------- Model ----------------
    m = gp.Model(f"{MODEL_TAG}_day{d:03d}")
    m.Params.MIPGap = MIPGAP
    m.Params.TimeLimit = TIME_LIMIT_S
    m.Params.OutputFlag = OUTPUTFLAG
    m.Params.NumericFocus = 1
    m.Params.MultiObjMethod = 2
    m.Params.MultiObjPre = 1

    T = range(24)
    T2 = range(25)

    S_sources = ["pv", "wd", "grid", "bat"]
    K_sinks = ["elz", "NH3", "MeOH", "crk", "bch"]
    S_res = ["pv", "wd"]

    # ---------------- Variables ----------------
    P = m.addVars(S_sources, K_sinks, T, lb=0.0, name="P_s2k")
    P_pv_cur = m.addVars(T, lb=0.0, name="P_pv_cur")
    P_wd_cur = m.addVars(T, lb=0.0, name="P_wd_cur")
    P_grid = m.addVars(T, lb=0.0, ub=P_GRID_MAX_MW, name="P_grid")

    P_ch  = m.addVars(T, lb=0.0, ub=P_BAT_MAX_MW, name="P_ch")
    P_dis = m.addVars(T, lb=0.0, ub=P_BAT_MAX_MW, name="P_dis")
    SOC   = m.addVars(T2, lb=SOC_MIN, ub=SOC_MAX, name="SOC")

    Pk_g  = m.addVars(K_sinks, T, lb=0.0, name="P_sink_g")
    Pk_ng = m.addVars(K_sinks, T, lb=0.0, name="P_sink_ng")

    # Electrolyzer
    I_elz = range(N_stack)
    y_elz_m  = m.addVars(I_elz, T, vtype=GRB.BINARY, name="y_elz_m")
    su_elz_m = m.addVars(I_elz, T, vtype=GRB.BINARY, name="su_elz_m")
    b_elz_g  = m.addVars(I_elz, T, vtype=GRB.BINARY, name="b_elz_g")

    P_elz_m_g   = m.addVars(I_elz, T, lb=0.0, ub=P_ELZ_MAX, name="P_elz_m_g")
    P_elz_m_ng  = m.addVars(I_elz, T, lb=0.0, ub=P_ELZ_MAX, name="P_elz_m_ng")
    P_elz_m_tot = m.addVars(I_elz, T, lb=0.0, ub=P_ELZ_MAX, name="P_elz_m_tot")

    H_elz_m    = m.addVars(I_elz, T, lb=0.0, ub=H_ELZ_MAX, name="H_elz_m")
    H_elz_m_g  = m.addVars(I_elz, T, lb=0.0, ub=H_ELZ_MAX, name="H_elz_m_g")
    H_elz_m_ng = m.addVars(I_elz, T, lb=0.0, ub=H_ELZ_MAX, name="H_elz_m_ng")

    y_elz    = m.addVars(T, vtype=GRB.BINARY, name="y_elz")
    n_elz_on = m.addVars(T, vtype=GRB.INTEGER, lb=0, ub=N_stack, name="n_elz_on")
    su_elz   = m.addVars(T, vtype=GRB.INTEGER, lb=0, ub=N_stack, name="su_elz")

    P_elz_g   = m.addVars(T, lb=0.0, ub=N_stack*P_ELZ_MAX, name="P_elz_g")
    P_elz_ng  = m.addVars(T, lb=0.0, ub=N_stack*P_ELZ_MAX, name="P_elz_ng")
    P_elz_tot = m.addVars(T, lb=0.0, ub=N_stack*P_ELZ_MAX, name="P_elz_tot")
    H_prod    = m.addVars(T, lb=0.0, ub=N_stack*H_ELZ_MAX, name="H_prod")
    H_prod_g  = m.addVars(T, lb=0.0, ub=N_stack*H_ELZ_MAX, name="H_prod_g")
    H_prod_ng = m.addVars(T, lb=0.0, ub=N_stack*H_ELZ_MAX, name="H_prod_ng")
    W_elz     = m.addVars(T, lb=0.0, name="W_elz_m3ph")

    # NH3
    y_nh3   = m.addVars(T, vtype=GRB.BINARY, name="y_nh3")
    su_nh3  = m.addVars(T, vtype=GRB.BINARY, name="su_nh3")
    Q_nh3_g = m.addVars(T, lb=0.0, name="Q_nh3_g")
    Q_nh3_ng = m.addVars(T, lb=0.0, name="Q_nh3_ng")
    Q_nh3   = m.addVars(T, lb=0.0, name="Q_nh3")
    P_nh3_g = m.addVars(T, lb=0.0, name="P_nh3_g")
    P_nh3_ng = m.addVars(T, lb=0.0, name="P_nh3_ng")

    # MeOH
    y_meoh   = m.addVars(T, vtype=GRB.BINARY, name="y_meoh")
    su_meoh  = m.addVars(T, vtype=GRB.BINARY, name="su_meoh")
    Q_meoh_g = m.addVars(T, lb=0.0, name="Q_meoh_g")
    Q_meoh_ng = m.addVars(T, lb=0.0, name="Q_meoh_ng")
    Q_meoh   = m.addVars(T, lb=0.0, name="Q_meoh")
    P_meoh_g = m.addVars(T, lb=0.0, name="P_meoh_g")
    P_meoh_ng = m.addVars(T, lb=0.0, name="P_meoh_ng")
    CO2_buy  = m.addVars(T, lb=0.0, name="CO2_buy_kgph")

    # Cracker
    y_crk    = m.addVars(T, vtype=GRB.BINARY, name="y_crk")
    su_crk   = m.addVars(T, vtype=GRB.BINARY, name="su_crk")
    Q_crk_in_g = m.addVars(T, lb=0.0, name="Q_crk_in_g")
    Q_crk_in_ng = m.addVars(T, lb=0.0, name="Q_crk_in_ng")
    Q_crk_in = m.addVars(T, lb=0.0, name="Q_crk_in")

    H_crk_out_g  = m.addVars(T, lb=0.0, name="H_crk_out_g")
    H_crk_out_ng = m.addVars(T, lb=0.0, name="H_crk_out_ng")
    NH3_rec_g    = m.addVars(T, lb=0.0, name="NH3_rec_g")
    NH3_rec_ng   = m.addVars(T, lb=0.0, name="NH3_rec_ng")
    H_crk_fuel_g  = m.addVars(T, lb=0.0, name="H_crk_fuel_g")
    H_crk_fuel_ng = m.addVars(T, lb=0.0, name="H_crk_fuel_ng")
    Steam_g      = m.addVars(T, lb=0.0, name="Steam_g_kgph")
    Steam_ng     = m.addVars(T, lb=0.0, name="Steam_ng_kgph")
    P_crk_g      = m.addVars(T, lb=0.0, name="P_crk_g")
    P_crk_ng     = m.addVars(T, lb=0.0, name="P_crk_ng")
    W_steam      = m.addVars(T, lb=0.0, name="W_steam_m3ph")

    # Tanks
    S_H2_g   = m.addVars(T2, lb=0.0, name="S_H2_g")
    S_H2_ng  = m.addVars(T2, lb=0.0, name="S_H2_ng")
    S_NH3_g  = m.addVars(T2, lb=0.0, name="S_NH3_g")
    S_NH3_ng = m.addVars(T2, lb=0.0, name="S_NH3_ng")
    S_MEOH_g = m.addVars(T2, lb=0.0, name="S_MEOH_g")
    S_MEOH_ng = m.addVars(T2, lb=0.0, name="S_MEOH_ng")

    # Sales
    H_sell_g = m.addVars(T, lb=0.0, name="H_sell_g")
    H_sell_ng = m.addVars(T, lb=0.0, name="H_sell_ng")
    NH3_sell_g = m.addVars(T, lb=0.0, name="NH3_sell_g")
    NH3_sell_ng = m.addVars(T, lb=0.0, name="NH3_sell_ng")
    MEOH_sell_g = m.addVars(T, lb=0.0, name="MEOH_sell_g")
    MEOH_sell_ng = m.addVars(T, lb=0.0, name="MEOH_sell_ng")

    # H2 to units
    H_to_NH3_g = m.addVars(T, lb=0.0, name="H_to_NH3_g")
    H_to_NH3_ng = m.addVars(T, lb=0.0, name="H_to_NH3_ng")
    H_to_MEOH_g = m.addVars(T, lb=0.0, name="H_to_MEOH_g")
    H_to_MEOH_ng = m.addVars(T, lb=0.0, name="H_to_MEOH_ng")

    # Unmet
    u_H2   = m.addVars(T, lb=0.0, name="u_H2")
    u_NH3  = m.addVars(T, lb=0.0, name="u_NH3")
    u_MEOH = m.addVars(T, lb=0.0, name="u_MEOH")

    # ---------------- Constraints ----------------
    for t in T:
        m.addConstr(P_pv_cur[t] <= pv_av[t], name=f"ub_pv_cur[{t}]")
        m.addConstr(P_wd_cur[t] <= wd_av[t], name=f"ub_wd_cur[{t}]")

    for t in T:
        for k in K_sinks:
            m.addConstr(P["pv",  k, t] <= pv_av[t],      name=f"ub_P_pv_{k}[{t}]")
            m.addConstr(P["wd",  k, t] <= wd_av[t],      name=f"ub_P_wd_{k}[{t}]")
            m.addConstr(P["grid",k, t] <= P_GRID_MAX_MW, name=f"ub_P_grid_{k}[{t}]")
            m.addConstr(P["bat", k, t] <= P_BAT_MAX_MW,  name=f"ub_P_bat_{k}[{t}]")

    for t in T:
        m.addConstr(gp.quicksum(P["pv",k,t] for k in K_sinks) + P_pv_cur[t] == pv_av[t], name=f"pv_balance[{t}]")
        m.addConstr(gp.quicksum(P["wd",k,t] for k in K_sinks) + P_wd_cur[t] == wd_av[t], name=f"wd_balance[{t}]")

        m.addConstr(gp.quicksum(P["grid",k,t] for k in K_sinks) == P_grid[t], name=f"grid_alloc[{t}]")
        m.addConstr(gp.quicksum(P["bat",k,t] for k in K_sinks) == P_dis[t], name=f"bat_alloc[{t}]")

        m.addConstr(gp.quicksum(P[s,"bch",t] for s in S_sources) == P_ch[t], name=f"bch_sink[{t}]")
        m.addConstr(P["grid","bch",t] == 0.0, name=f"no_grid_to_bch[{t}]")
        m.addConstr(P["bat","bch",t]  == 0.0, name=f"no_bat_to_bch[{t}]")
        m.addConstr(P["pv","bch",t] + P["wd","bch",t] == P_ch[t], name=f"bch_from_res[{t}]")

        m.addSOS(GRB.SOS_TYPE1, [P_ch[t], P_dis[t]])
        m.addConstr(P_dis[t] <= ETA_DIS * (SOC[t] - SOC_MIN) * E_BAT_MAX_MWH, name=f"ub_Pdis_soc[{t}]")
        m.addConstr(P_ch[t]  <= (SOC_MAX - SOC[t]) * E_BAT_MAX_MWH / ETA_CH, name=f"ub_Pch_soc[{t}]")

        for k in K_sinks:
            m.addConstr(Pk_ng[k,t] == P["grid",k,t] + P["bat",k,t], name=f"Pk_ng_def[{k},{t}]")
            m.addConstr(Pk_g[k,t]  == gp.quicksum(P[s,k,t] for s in S_res), name=f"Pk_g_def[{k},{t}]")

    m.addConstr(SOC[0] == float(carry["soc"]), name="SOC_init_carry")
    for t in T:
        m.addConstr(
            SOC[t+1] == SOC[t] + (ETA_CH*P_ch[t] - (1.0/ETA_DIS)*P_dis[t]) / E_BAT_MAX_MWH,
            name=f"SOC_dyn[{t}]"
        )

    m.addConstrs((y_elz_m[i,t] >= y_elz_m[i+1,t] for i in range(N_stack-1) for t in T), name="elz_module_order")

    # Electrolyzer
    for t in T:
        m.addConstr(P_elz_g[t]  == Pk_g["elz",t],  name=f"P_elz_g_link[{t}]")
        m.addConstr(P_elz_ng[t] == Pk_ng["elz",t], name=f"P_elz_ng_link[{t}]")
        m.addConstr(P_elz_g[t]  == gp.quicksum(P_elz_m_g[i,t] for i in I_elz), name=f"P_elz_g_sum_mod[{t}]")
        m.addConstr(P_elz_ng[t] == gp.quicksum(P_elz_m_ng[i,t] for i in I_elz), name=f"P_elz_ng_sum_mod[{t}]")
        m.addConstr(P_elz_tot[t] == P_elz_g[t] + P_elz_ng[t], name=f"P_elz_tot_sum[{t}]")

        m.addConstr(n_elz_on[t] == gp.quicksum(y_elz_m[i,t] for i in I_elz), name=f"n_elz_on_def[{t}]")
        m.addConstr(n_elz_on[t] >= y_elz[t], name=f"y_elz_lb[{t}]")
        m.addConstr(n_elz_on[t] <= N_stack * y_elz[t], name=f"y_elz_ub[{t}]")
        m.addConstr(su_elz[t] == gp.quicksum(su_elz_m[i,t] for i in I_elz), name=f"su_elz_sum[{t}]")

        m.addConstr(H_prod[t]    == gp.quicksum(H_elz_m[i,t] for i in I_elz), name=f"H_prod_sum_mod[{t}]")
        m.addConstr(H_prod_g[t]  == gp.quicksum(H_elz_m_g[i,t] for i in I_elz), name=f"H_prod_g_sum_mod[{t}]")
        m.addConstr(H_prod_ng[t] == gp.quicksum(H_elz_m_ng[i,t] for i in I_elz), name=f"H_prod_ng_sum_mod[{t}]")
        m.addConstr(H_prod[t] == H_prod_g[t] + H_prod_ng[t], name=f"H_prod_split[{t}]")
        m.addConstr(W_elz[t] == omega_H2O * H_prod[t], name=f"W_elz[{t}]")

        for i in range(N_stack - 1):
            m.addConstr(b_elz_g[i,t] >= b_elz_g[i+1,t], name=f"elz_bg_order[{i},{t}]")

        for i in I_elz:
            m.addConstr(P_elz_m_tot[i,t] == P_elz_m_g[i,t] + P_elz_m_ng[i,t], name=f"P_elz_m_tot_def[{i},{t}]")
            m.addConstr(P_elz_m_tot[i,t] <= P_ELZ_MAX * y_elz_m[i,t], name=f"elz_m_max[{i},{t}]")
            m.addConstr(P_elz_m_tot[i,t] >= P_ELZ_MIN * y_elz_m[i,t], name=f"elz_m_min[{i},{t}]")
            m.addConstr(b_elz_g[i,t] <= y_elz_m[i,t], name=f"b_elz_g_gate[{i},{t}]")
            m.addConstr(P_elz_m_g[i,t]  <= P_ELZ_MAX * b_elz_g[i,t], name=f"P_elz_m_g_cap[{i},{t}]")
            m.addConstr(P_elz_m_ng[i,t] <= P_ELZ_MAX * (y_elz_m[i,t] - b_elz_g[i,t]), name=f"P_elz_m_ng_cap[{i},{t}]")

            m.addConstr(H_elz_m[i,t] == H_elz_m_g[i,t] + H_elz_m_ng[i,t], name=f"H_elz_m_split[{i},{t}]")

            m.addQConstr(
                H_elz_m[i,t] <= A_elz*(P_elz_m_tot[i,t]*P_elz_m_tot[i,t]) + B_elz*P_elz_m_tot[i,t] + C_elz*y_elz_m[i,t],
                name=f"elz_m_quad[{i},{t}]"
            )
            m.addConstr(H_elz_m[i,t] >= Q1*P_elz_m_tot[i,t] + Q0*y_elz_m[i,t], name=f"elz_m_lin[{i},{t}]")

            m.addConstr(H_elz_m_g[i,t] <= H_ELZ_MAX * b_elz_g[i,t], name=f"H_elz_m_g_ub1[{i},{t}]")
            m.addConstr(H_elz_m_g[i,t] <= H_elz_m[i,t], name=f"H_elz_m_g_ub2[{i},{t}]")
            m.addConstr(H_elz_m_g[i,t] >= H_elz_m[i,t] - H_ELZ_MAX*(1 - b_elz_g[i,t]), name=f"H_elz_m_g_lb[{i},{t}]")

            m.addConstr(H_elz_m_ng[i,t] <= H_ELZ_MAX * (y_elz_m[i,t] - b_elz_g[i,t]), name=f"H_elz_m_ng_ub1[{i},{t}]")
            m.addConstr(H_elz_m_ng[i,t] <= H_elz_m[i,t], name=f"H_elz_m_ng_ub2[{i},{t}]")
            m.addConstr(H_elz_m_ng[i,t] >= H_elz_m[i,t] - H_ELZ_MAX*b_elz_g[i,t], name=f"H_elz_m_ng_lb[{i},{t}]")

            if t >= 1:
                m.addConstr(su_elz_m[i,t] <= y_elz_m[i,t], name=f"su_elz_m1[{i},{t}]")
                m.addConstr(su_elz_m[i,t] <= 1 - y_elz_m[i,t-1], name=f"su_elz_m2[{i},{t}]")
                m.addConstr(su_elz_m[i,t] >= y_elz_m[i,t] - y_elz_m[i,t-1], name=f"su_elz_m3[{i},{t}]")
            else:
                y_prev_i = int(carry["y_elz_prev_mod"][i])
                m.addConstr(su_elz_m[i,t] <= y_elz_m[i,t], name=f"su_elz_m1[{i},{t}]")
                m.addConstr(su_elz_m[i,t] <= 1 - y_prev_i, name=f"su_elz_m2[{i},{t}]")
                m.addConstr(su_elz_m[i,t] >= y_elz_m[i,t] - y_prev_i, name=f"su_elz_m3[{i},{t}]")

    # NH3
    for t in T:
        m.addConstr(Q_nh3[t] == Q_nh3_g[t] + Q_nh3_ng[t], name=f"Q_nh3_sum[{t}]")
        m.addConstr(Q_nh3[t] <= Q_NH3_CAP * y_nh3[t], name=f"nh3_max[{t}]")
        m.addConstr(Q_nh3[t] >= f_NH3_min * Q_NH3_CAP * y_nh3[t], name=f"nh3_min[{t}]")

        if t >= 1:
            m.addConstr(su_nh3[t] <= y_nh3[t])
            m.addConstr(su_nh3[t] <= 1 - y_nh3[t-1])
            m.addConstr(su_nh3[t] >= y_nh3[t] - y_nh3[t-1])
        else:
            y_prev = int(carry["y_nh3_prev"])
            m.addConstr(su_nh3[t] <= y_nh3[t])
            m.addConstr(su_nh3[t] <= 1 - y_prev)
            m.addConstr(su_nh3[t] >= y_nh3[t] - y_prev)

        m.addConstr(
            Q_nh3[t] <= f_NH3_min * Q_NH3_CAP + (Q_NH3_CAP - f_NH3_min * Q_NH3_CAP) * (1 - su_nh3[t]),
            name=f"nh3_start_minload[{t}]"
        )

        ramp_nh3 = RR_NH3 * Q_NH3_CAP
        if t == 0:
            Q_prev = float(carry["Q_nh3_prev"])
            y_prev = int(carry["y_nh3_prev"])
            m.addConstr(Q_nh3[t] - Q_prev <= ramp_nh3 + (Q_NH3_CAP - ramp_nh3) * (2 - y_nh3[t] - y_prev), name=f"nh3_rup0[{t}]")
            m.addConstr(Q_prev - Q_nh3[t] <= ramp_nh3 + (Q_NH3_CAP - ramp_nh3) * (2 - y_nh3[t] - y_prev), name=f"nh3_rdn0[{t}]")
        else:
            m.addConstr(Q_nh3[t] - Q_nh3[t-1] <= ramp_nh3 + (Q_NH3_CAP - ramp_nh3) * (2 - y_nh3[t] - y_nh3[t-1]), name=f"nh3_rup[{t}]")
            m.addConstr(Q_nh3[t-1] - Q_nh3[t] <= ramp_nh3 + (Q_NH3_CAP - ramp_nh3) * (2 - y_nh3[t] - y_nh3[t-1]), name=f"nh3_rdn[{t}]")

        m.addConstr(P_nh3_g[t]  == (SEC_HB*Q_nh3_g[t]  + SEC_ASU*(mN2_to_NH3*Q_nh3_g[t]))  / 1000.0, name=f"P_nh3_g_map[{t}]")
        m.addConstr(P_nh3_ng[t] == (SEC_HB*Q_nh3_ng[t] + SEC_ASU*(mN2_to_NH3*Q_nh3_ng[t])) / 1000.0, name=f"P_nh3_ng_map[{t}]")
        m.addConstr(P_nh3_g[t]  == Pk_g["NH3",t],  name=f"P_nh3_g_link[{t}]")
        m.addConstr(P_nh3_ng[t] == Pk_ng["NH3",t], name=f"P_nh3_ng_link[{t}]")
        m.addConstr(H_to_NH3_g[t]  == mH2_to_NH3 * Q_nh3_g[t],  name=f"H_to_NH3_g[{t}]")
        m.addConstr(H_to_NH3_ng[t] == mH2_to_NH3 * Q_nh3_ng[t], name=f"H_to_NH3_ng[{t}]")

    # MeOH
    for t in T:
        m.addConstr(Q_meoh[t] == Q_meoh_g[t] + Q_meoh_ng[t], name=f"Q_meoh_sum[{t}]")
        m.addConstr(Q_meoh[t] <= Q_MEOH_CAP * y_meoh[t], name=f"meoh_max[{t}]")
        m.addConstr(Q_meoh[t] >= f_MEOH_min * Q_MEOH_CAP * y_meoh[t], name=f"meoh_min[{t}]")

        if t >= 1:
            m.addConstr(su_meoh[t] <= y_meoh[t])
            m.addConstr(su_meoh[t] <= 1 - y_meoh[t-1])
            m.addConstr(su_meoh[t] >= y_meoh[t] - y_meoh[t-1])
        else:
            y_prev = int(carry["y_meoh_prev"])
            m.addConstr(su_meoh[t] <= y_meoh[t])
            m.addConstr(su_meoh[t] <= 1 - y_prev)
            m.addConstr(su_meoh[t] >= y_meoh[t] - y_prev)

        m.addConstr(
            Q_meoh[t] <= f_MEOH_min * Q_MEOH_CAP + (Q_MEOH_CAP - f_MEOH_min * Q_MEOH_CAP) * (1 - su_meoh[t]),
            name=f"meoh_start_minload[{t}]"
        )

        ramp_meoh = R_MEOH
        if t == 0:
            Q_prev = float(carry["Q_meoh_prev"])
            y_prev = int(carry["y_meoh_prev"])
            m.addConstr(Q_meoh[t] - Q_prev <= ramp_meoh + (Q_MEOH_CAP - ramp_meoh) * (2 - y_meoh[t] - y_prev), name=f"meoh_rup0[{t}]")
            m.addConstr(Q_prev - Q_meoh[t] <= ramp_meoh + (Q_MEOH_CAP - ramp_meoh) * (2 - y_meoh[t] - y_prev), name=f"meoh_rdn0[{t}]")
        else:
            m.addConstr(Q_meoh[t] - Q_meoh[t-1] <= ramp_meoh + (Q_MEOH_CAP - ramp_meoh) * (2 - y_meoh[t] - y_meoh[t-1]), name=f"meoh_rup[{t}]")
            m.addConstr(Q_meoh[t-1] - Q_meoh[t] <= ramp_meoh + (Q_MEOH_CAP - ramp_meoh) * (2 - y_meoh[t] - y_meoh[t-1]), name=f"meoh_rdn[{t}]")

        m.addConstr(P_meoh_g[t]  == (SEC_MEOH * Q_meoh_g[t])  / 1000.0, name=f"P_meoh_g_map[{t}]")
        m.addConstr(P_meoh_ng[t] == (SEC_MEOH * Q_meoh_ng[t]) / 1000.0, name=f"P_meoh_ng_map[{t}]")
        m.addConstr(P_meoh_g[t]  == Pk_g["MeOH",t],  name=f"P_meoh_g_link[{t}]")
        m.addConstr(P_meoh_ng[t] == Pk_ng["MeOH",t], name=f"P_meoh_ng_link[{t}]")
        m.addConstr(CO2_buy[t] == mCO2_to_MEOH * Q_meoh[t], name=f"CO2_buy[{t}]")
        m.addConstr(H_to_MEOH_g[t]  == mH2_to_MEOH * Q_meoh_g[t],  name=f"H_to_MEOH_g[{t}]")
        m.addConstr(H_to_MEOH_ng[t] == mH2_to_MEOH * Q_meoh_ng[t], name=f"H_to_MEOH_ng[{t}]")

    # Cracker
    for t in T:
        m.addConstr(Q_crk_in[t] == Q_crk_in_g[t] + Q_crk_in_ng[t], name=f"Q_crk_sum[{t}]")
        m.addConstr(Q_crk_in[t] <= Q_CRK_CAP * y_crk[t], name=f"crk_max[{t}]")
        m.addConstr(Q_crk_in[t] >= f_CRK_min * Q_CRK_CAP * y_crk[t], name=f"crk_min[{t}]")

        if t >= 1:
            m.addConstr(su_crk[t] <= y_crk[t])
            m.addConstr(su_crk[t] <= 1 - y_crk[t-1])
            m.addConstr(su_crk[t] >= y_crk[t] - y_crk[t-1])
        else:
            y_prev = int(carry["y_crk_prev"])
            m.addConstr(su_crk[t] <= y_crk[t])
            m.addConstr(su_crk[t] <= 1 - y_prev)
            m.addConstr(su_crk[t] >= y_crk[t] - y_prev)

        m.addConstr(
            Q_crk_in[t] <= f_CRK_min * Q_CRK_CAP + (Q_CRK_CAP - f_CRK_min * Q_CRK_CAP) * (1 - su_crk[t]),
            name=f"crk_start_minload[{t}]"
        )

        ramp_crk = R_CRK
        if t == 0:
            Q_prev = float(carry["Q_crk_prev"])
            y_prev = int(carry["y_crk_prev"])
            m.addConstr(Q_crk_in[t] - Q_prev <= ramp_crk + (Q_CRK_CAP - ramp_crk) * (2 - y_crk[t] - y_prev), name=f"crk_rup0[{t}]")
            m.addConstr(Q_prev - Q_crk_in[t] <= ramp_crk + (Q_CRK_CAP - ramp_crk) * (2 - y_crk[t] - y_prev), name=f"crk_rdn0[{t}]")
        else:
            m.addConstr(Q_crk_in[t] - Q_crk_in[t-1] <= ramp_crk + (Q_CRK_CAP - ramp_crk) * (2 - y_crk[t] - y_crk[t-1]), name=f"crk_rup[{t}]")
            m.addConstr(Q_crk_in[t-1] - Q_crk_in[t] <= ramp_crk + (Q_CRK_CAP - ramp_crk) * (2 - y_crk[t] - y_crk[t-1]), name=f"crk_rdn[{t}]")

        m.addConstr(H_crk_out_g[t]  == kH2_out * Q_crk_in_g[t],  name=f"Hcrk_out_g[{t}]")
        m.addConstr(H_crk_out_ng[t] == kH2_out * Q_crk_in_ng[t], name=f"Hcrk_out_ng[{t}]")
        m.addConstr(NH3_rec_g[t]    == kNH3_rec * Q_crk_in_g[t],  name=f"NH3rec_g[{t}]")
        m.addConstr(NH3_rec_ng[t]   == kNH3_rec * Q_crk_in_ng[t], name=f"NH3rec_ng[{t}]")
        m.addConstr(H_crk_fuel_g[t]  == kH2_fuel * Q_crk_in_g[t],  name=f"Hcrk_fuel_g[{t}]")
        m.addConstr(H_crk_fuel_ng[t] == kH2_fuel * Q_crk_in_ng[t], name=f"Hcrk_fuel_ng[{t}]")
        m.addConstr(Steam_g[t]  == kSteam * Q_crk_in_g[t],  name=f"Steam_g[{t}]")
        m.addConstr(Steam_ng[t] == kSteam * Q_crk_in_ng[t], name=f"Steam_ng[{t}]")
        m.addConstr(W_steam[t]  == omega_steam * (Steam_g[t] + Steam_ng[t]), name=f"W_steam[{t}]")
        m.addConstr(P_crk_g[t]  == (SEC_CRK_E*Q_crk_in_g[t]  + eSteam*Steam_g[t])  / 1000.0, name=f"P_crk_g_map[{t}]")
        m.addConstr(P_crk_ng[t] == (SEC_CRK_E*Q_crk_in_ng[t] + eSteam*Steam_ng[t]) / 1000.0, name=f"P_crk_ng_map[{t}]")
        m.addConstr(P_crk_g[t]  == Pk_g["crk",t],  name=f"P_crk_g_link[{t}]")
        m.addConstr(P_crk_ng[t] == Pk_ng["crk",t], name=f"P_crk_ng_link[{t}]")

    # Tanks / demand
    m.addConstr(S_H2_g[0]   == float(carry["S_H2_g"]), name="S_H2_g_init")
    m.addConstr(S_H2_ng[0]  == float(carry["S_H2_ng"]), name="S_H2_ng_init")
    m.addConstr(S_NH3_g[0]  == float(carry["S_NH3_g"]), name="S_NH3_g_init")
    m.addConstr(S_NH3_ng[0] == float(carry["S_NH3_ng"]), name="S_NH3_ng_init")
    m.addConstr(S_MEOH_g[0] == float(carry["S_MEOH_g"]), name="S_MEOH_g_init")
    m.addConstr(S_MEOH_ng[0] == float(carry["S_MEOH_ng"]), name="S_MEOH_ng_init")

    for tt in T2:
        m.addConstr(S_H2_g[tt] + S_H2_ng[tt] <= S_H2_CAP, name=f"S_H2_cap[{tt}]")
        m.addConstr(S_NH3_g[tt] + S_NH3_ng[tt] <= S_NH3_CAP, name=f"S_NH3_cap[{tt}]")
        m.addConstr(S_MEOH_g[tt] + S_MEOH_ng[tt] <= S_MEOH_CAP, name=f"S_MEOH_cap[{tt}]")

    m.addConstr(S_H2_g[24] + S_H2_ng[24] >= ALPHA_TANK_FILL * S_H2_CAP, name="eod_hard_H2_frac")
    m.addConstr(S_NH3_g[24] + S_NH3_ng[24] >= ALPHA_TANK_FILL * S_NH3_CAP, name="eod_hard_NH3_frac")
    m.addConstr(S_MEOH_g[24] + S_MEOH_ng[24] >= ALPHA_TANK_FILL * S_MEOH_CAP, name="eod_hard_MeOH_frac")

    for t in T:
        m.addConstr(H_sell_g[t] + H_sell_ng[t] + u_H2[t] == dem_h2[t], name=f"dem_H2[{t}]")
        m.addConstr(NH3_sell_g[t] + NH3_sell_ng[t] + u_NH3[t] == dem_nh3[t], name=f"dem_NH3[{t}]")
        m.addConstr(MEOH_sell_g[t] + MEOH_sell_ng[t] + u_MEOH[t] == dem_meoh[t], name=f"dem_MEOH[{t}]")

        m.addConstr(
            S_H2_g[t+1] == S_H2_g[t] + H_prod_g[t] + H_crk_out_g[t] - H_to_NH3_g[t] - H_to_MEOH_g[t] - H_sell_g[t] - H_crk_fuel_g[t],
            name=f"S_H2_g_dyn[{t}]"
        )
        m.addConstr(
            S_H2_ng[t+1] == S_H2_ng[t] + H_prod_ng[t] + H_crk_out_ng[t] - H_to_NH3_ng[t] - H_to_MEOH_ng[t] - H_sell_ng[t] - H_crk_fuel_ng[t],
            name=f"S_H2_ng_dyn[{t}]"
        )
        m.addConstr(
            S_NH3_g[t+1] == S_NH3_g[t] + Q_nh3_g[t] + NH3_rec_g[t] - NH3_sell_g[t] - Q_crk_in_g[t],
            name=f"S_NH3_g_dyn[{t}]"
        )
        m.addConstr(
            S_NH3_ng[t+1] == S_NH3_ng[t] + Q_nh3_ng[t] + NH3_rec_ng[t] - NH3_sell_ng[t] - Q_crk_in_ng[t],
            name=f"S_NH3_ng_dyn[{t}]"
        )
        m.addConstr(
            S_MEOH_g[t+1] == S_MEOH_g[t] + Q_meoh_g[t] - MEOH_sell_g[t],
            name=f"S_MEOH_g_dyn[{t}]"
        )
        m.addConstr(
            S_MEOH_ng[t+1] == S_MEOH_ng[t] + Q_meoh_ng[t] - MEOH_sell_ng[t],
            name=f"S_MEOH_ng_dyn[{t}]"
        )

        m.addConstr(H_to_NH3_g[t] + H_to_MEOH_g[t] + H_sell_g[t] + H_crk_fuel_g[t] <= S_H2_g[t], name=f"dir_H2_g[{t}]")
        m.addConstr(H_to_NH3_ng[t] + H_to_MEOH_ng[t] + H_sell_ng[t] + H_crk_fuel_ng[t] <= S_H2_ng[t], name=f"dir_H2_ng[{t}]")
        m.addConstr(NH3_sell_g[t] + Q_crk_in_g[t] <= S_NH3_g[t], name=f"dir_NH3_g[{t}]")
        m.addConstr(NH3_sell_ng[t] + Q_crk_in_ng[t] <= S_NH3_ng[t], name=f"dir_NH3_ng[{t}]")
        m.addConstr(MEOH_sell_g[t] <= S_MEOH_g[t], name=f"dir_MEOH_g[{t}]")
        m.addConstr(MEOH_sell_ng[t] <= S_MEOH_ng[t], name=f"dir_MEOH_ng[{t}]")

    # ---------------- Emissions cap expression ----------------
    if EMISSIONS_KEY == "aef":
        grid_emis_cap_expr = gp.quicksum(1000.0 * P_grid[t] * float(aef_k[t]) for t in T)
    elif EMISSIONS_KEY == "mef_upramp":
        grid_emis_cap_expr = gp.quicksum(1000.0 * P_grid[t] * float(mef_upramp_k[t]) for t in T)
    elif EMISSIONS_KEY == "mef_soft":
        grid_emis_cap_expr = gp.quicksum(1000.0 * P_grid[t] * float(mef_soft_k[t]) for t in T)
    else:
        raise ValueError("EMISSIONS_KEY must be one of: 'aef', 'mef_upramp', 'mef_soft'")

    direct_emis_cap_expr = gp.quicksum(float(de_NH3) * Q_nh3[t] + float(de_MEOH) * Q_meoh[t] for t in T)
    total_emis_cap_expr = grid_emis_cap_expr + direct_emis_cap_expr
    m.addConstr(total_emis_cap_expr <= float(EMIS_CAP_DAY_KG), name="M2_emissions_cap_total_kg")

    # ---------------- Objectives ----------------
    unmet_total = gp.quicksum(u_H2[t] + u_NH3[t] + u_MEOH[t] for t in T)

    cost_total = gp.LinExpr()
    for t in T:
        cost_total += pr_eur[t] * P_grid[t]
        cost_total += c_H2O * (W_elz[t] + W_steam[t])
        cost_total += c_CO2 * CO2_buy[t]
        cost_total += C_su_elz * gp.quicksum(su_elz_m[i,t] for i in I_elz)
        cost_total += C_su_NH3 * su_nh3[t]
        cost_total += C_su_MEOH * su_meoh[t]
        cost_total += C_su_CRK * su_crk[t]

    m.setObjectiveN(unmet_total, index=0, priority=2, reltol=1e-4, abstol=1e-6, name="OBJ1_min_unmet")
    m.setObjectiveN(cost_total,  index=1, priority=1, reltol=1e-4, abstol=1e-6, name="OBJ2_min_cost")

    # ---------------- Solve ----------------
    m.optimize()

    status = int(safe_get_attr(m, "Status", -1))
    runtime = float(safe_get_attr(m, "Runtime", np.nan))
    nodecnt = float(safe_get_attr(m, "NodeCount", np.nan))
    solcnt  = int(safe_get_attr(m, "SolCount", -1))

    pass0 = get_objpass_metrics(m, 0)
    pass1 = get_objpass_metrics(m, 1)

    mip_gap = pass1["pass_mipgap"]
    obj_val = pass1["pass_objval"]
    obj_bnd = pass1["pass_objbound"]

    itercnt = safe_get_attr(m, "IterCount", np.nan)
    bariter = safe_get_attr(m, "BarIterCount", np.nan)
    work = safe_get_attr(m, "Work", np.nan)

    obj1_val = np.nan
    obj2_val = np.nan
    try:
        obj1_val = float(unmet_total.getValue())
    except Exception:
        pass
    try:
        obj2_val = float(cost_total.getValue())
    except Exception:
        pass

    is_optimal = (status == GRB.OPTIMAL)
    hit_time_limit = (status == GRB.TIME_LIMIT)
    has_incumbent = (solcnt is not None) and (solcnt >= 1)
    is_within_target_gap = bool(has_incumbent) and (mip_gap is not None) and (not np.isnan(mip_gap)) and (float(mip_gap) <= float(MIPGAP) + 1e-12)
    is_acceptable = bool(is_optimal or is_within_target_gap)

    solver_rows.append({
        "run_id": RUN_ID,
        "model_tag": MODEL_TAG,
        "scenario_tag": SCENARIO_TAG,
        "demand_tag": DEMAND_TAG,
        "day_of_year": d,
        "date_utc": str(day_hours[0].date()),
        "solver_status": status,
        "solver_status_name": status_name(status),

        "m2_emissions_key": str(EMISSIONS_KEY),
        "m2_alpha_used_day": (float(ALPHA_USED_DAY) if (ALPHA_USED_DAY is not None and not np.isnan(ALPHA_USED_DAY)) else np.nan),
        "m2_emis_cap_day_kg": float(EMIS_CAP_DAY_KG),

        "runtime_seconds": runtime,
        "time_limit_s": TIME_LIMIT_S,
        "mipgap_target": float(MIPGAP),
        "mip_gap": pass1["pass_mipgap"],
        "has_incumbent": bool(has_incumbent),
        "is_optimal": bool(is_optimal),
        "hit_time_limit": bool(hit_time_limit),
        "is_within_target_gap": bool(is_within_target_gap),
        "is_acceptable": bool(is_acceptable),
        "obj_val": float(obj_val) if not np.isnan(obj_val) else np.nan,
        "obj_bound": float(obj_bnd) if not np.isnan(obj_bnd) else np.nan,

        "obj0_pass_status": pass0["pass_status"],
        "obj0_pass_status_name": pass0["pass_status_name"],
        "obj0_pass_runtime_s": pass0["pass_runtime"],
        "obj0_pass_objval": pass0["pass_objval"],
        "obj0_pass_objbound": pass0["pass_objbound"],
        "obj0_pass_mipgap": pass0["pass_mipgap"],

        "obj1_pass_status": pass1["pass_status"],
        "obj1_pass_status_name": pass1["pass_status_name"],
        "obj1_pass_runtime_s": pass1["pass_runtime"],
        "obj1_pass_objval": pass1["pass_objval"],
        "obj1_pass_objbound": pass1["pass_objbound"],
        "obj1_pass_mipgap": pass1["pass_mipgap"],

        "node_count": nodecnt,
        "sol_count": solcnt,
        "iter_count": float(itercnt) if not np.isnan(itercnt) else np.nan,
        "bar_iter_count": float(bariter) if not np.isnan(bariter) else np.nan,
        "work": float(work) if not np.isnan(work) else np.nan,
        "obj1_unmet_value": float(obj1_val) if not np.isnan(obj1_val) else np.nan,
        "obj2_cost_value": float(obj2_val) if not np.isnan(obj2_val) else np.nan,
    })

    if status not in [GRB.OPTIMAL, GRB.TIME_LIMIT]:
        try:
            m.computeIIS()
            iis_path = day_dir / f"{MODEL_TAG}_day{d:03d}_iis.ilp"
            m.write(str(iis_path))
        except Exception:
            pass
        raise RuntimeError(f"Day {d}: solver status {status} (not optimal/time_limit).")

    # ---------------- Extract hourly results ----------------
    def val(v):
        return float(v.X)

    rows = []
    unmet_rows = []

    for t in T:
        ts = day_hours[t]
        pv_used = pv_av[t] - val(P_pv_cur[t])
        wd_used = wd_av[t] - val(P_wd_cur[t])
        grid_mw = val(P_grid[t])
        res_used_mw = pv_used + wd_used
        res_curt_mw = val(P_pv_cur[t]) + val(P_wd_cur[t])

        P_elz = val(P_elz_g[t]) + val(P_elz_ng[t])
        P_nh3 = val(P_nh3_g[t]) + val(P_nh3_ng[t])
        P_meo = val(P_meoh_g[t]) + val(P_meoh_ng[t])
        P_crk = val(P_crk_g[t]) + val(P_crk_ng[t])

        H_total = val(H_prod[t])
        H_g = val(H_prod_g[t])
        H_ng = val(H_prod_ng[t])

        nh3_total = val(Q_nh3[t])
        nh3_g = val(Q_nh3_g[t])
        nh3_ng = val(Q_nh3_ng[t])

        me_total = val(Q_meoh[t])
        me_g = val(Q_meoh_g[t])
        me_ng = val(Q_meoh_ng[t])

        h2_sell_g = val(H_sell_g[t])
        h2_sell_ng = val(H_sell_ng[t])
        nh3_sell_g = val(NH3_sell_g[t])
        nh3_sell_ng = val(NH3_sell_ng[t])
        meoh_sell_g = val(MEOH_sell_g[t])
        meoh_sell_ng = val(MEOH_sell_ng[t])

        u1 = val(u_H2[t]); u2 = val(u_NH3[t]); u3 = val(u_MEOH[t])

        def share(g, ng):
            tot = g + ng
            return (g / tot) if tot > 1e-9 else np.nan

        H2_min_to_NH3  = mH2_to_NH3 * f_NH3_min * Q_NH3_CAP * int(round(val(y_nh3[t])))
        H2_min_to_MeOH = mH2_to_MEOH * f_MEOH_min * Q_MEOH_CAP * int(round(val(y_meoh[t])))
        H2_min_to_CRK  = kH2_fuel * f_CRK_min * Q_CRK_CAP * int(round(val(y_crk[t])))
        H2_min_committed = H2_min_to_NH3 + H2_min_to_MeOH + H2_min_to_CRK
        H2_tank_start = val(S_H2_g[t]) + val(S_H2_ng[t])
        H2_free_margin = H2_tank_start - H2_min_committed

        elz_mod = {}
        for i in I_elz:
            elz_mod[f"elz_m{i}_y"]     = int(round(val(y_elz_m[i,t])))
            elz_mod[f"elz_m{i}_su"]    = int(round(val(su_elz_m[i,t])))
            elz_mod[f"elz_m{i}_b_g"]   = int(round(val(b_elz_g[i,t])))
            elz_mod[f"elz_m{i}_P_tot"] = float(val(P_elz_m_tot[i,t]))
            elz_mod[f"elz_m{i}_P_g"]   = float(val(P_elz_m_g[i,t]))
            elz_mod[f"elz_m{i}_P_ng"]  = float(val(P_elz_m_ng[i,t]))
            elz_mod[f"elz_m{i}_H_tot"] = float(val(H_elz_m[i,t]))
            elz_mod[f"elz_m{i}_H_g"]   = float(val(H_elz_m_g[i,t]))
            elz_mod[f"elz_m{i}_H_ng"]  = float(val(H_elz_m_ng[i,t]))

        rows.append({
            "run_id": RUN_ID,
            "model_tag": MODEL_TAG,
            "scenario_tag": SCENARIO_TAG,
            "demand_tag": DEMAND_TAG,
            "datetime_utc": ts.isoformat(),
            "day_of_year": d,
            "hour_in_day": t,
            "H2_tank_start_kg": H2_tank_start,
            "H2_min_committed_kgph": H2_min_committed,
            "H2_free_margin_kg": H2_free_margin,
            "pv_avail_mw": pv_av[t],
            "wind_avail_mw": wd_av[t],
            "pv_curt_mw": val(P_pv_cur[t]),
            "wind_curt_mw": val(P_wd_cur[t]),
            "res_used_mw": res_used_mw,
            "res_curtailed_mw": res_curt_mw,
            "grid_import_mw": grid_mw,
            "da_price_eur_per_mwh": pr_eur[t],
            "bat_ch_mw": val(P_ch[t]),
            "bat_dis_mw": val(P_dis[t]),
            "battery_soc": val(SOC[t]),
            "battery_soc_next": val(SOC[t+1]),
            "elz_grid_mw": val(P_elz_ng[t]),
            "nh3_grid_mw": val(P_nh3_ng[t]),
            "meoh_grid_mw": val(P_meoh_ng[t]),
            "crk_grid_mw": val(P_crk_ng[t]),
            "P_elz_mw": P_elz,
            "P_nh3_mw": P_nh3,
            "P_meoh_mw": P_meo,
            "P_crk_mw": P_crk,
            "elz_power_green_share": share(val(P_elz_g[t]), val(P_elz_ng[t])),
            "nh3_power_green_share": share(val(P_nh3_g[t]), val(P_nh3_ng[t])),
            "meoh_power_green_share": share(val(P_meoh_g[t]), val(P_meoh_ng[t])),
            "crk_power_green_share": share(val(P_crk_g[t]), val(P_crk_ng[t])),
            "H2_prod_kgph": H_total,
            "H2_prod_g_kgph": H_g,
            "H2_prod_ng_kgph": H_ng,
            "NH3_prod_kgph": nh3_total,
            "NH3_prod_g_kgph": nh3_g,
            "NH3_prod_ng_kgph": nh3_ng,
            "MeOH_prod_kgph": me_total,
            "MeOH_prod_g_kgph": me_g,
            "MeOH_prod_ng_kgph": me_ng,
            "crk_nh3_in_kgph": val(Q_crk_in[t]),
            "crk_h2_out_kgph": val(H_crk_out_g[t]) + val(H_crk_out_ng[t]),
            "crk_h2_fuel_kgph": val(H_crk_fuel_g[t]) + val(H_crk_fuel_ng[t]),
            "steam_kgph": val(Steam_g[t]) + val(Steam_ng[t]),
            "water_elz_m3ph": val(W_elz[t]),
            "water_steam_m3ph": val(W_steam[t]),
            "co2_buy_kgph": val(CO2_buy[t]),

            "H2_sell_g_kgph": h2_sell_g,
            "H2_sell_ng_kgph": h2_sell_ng,
            "H2_sell_kgph": h2_sell_g + h2_sell_ng,

            "NH3_sell_g_kgph": nh3_sell_g,
            "NH3_sell_ng_kgph": nh3_sell_ng,
            "NH3_sell_kgph": nh3_sell_g + nh3_sell_ng,

            "MeOH_sell_g_kgph": meoh_sell_g,
            "MeOH_sell_ng_kgph": meoh_sell_ng,
            "MeOH_sell_kgph": meoh_sell_g + meoh_sell_ng,

            "unmet_h2_kgph": u1,
            "unmet_nh3_kgph": u2,
            "unmet_meoh_kgph": u3,
            "tank_H2_g_kg": val(S_H2_g[t]),
            "tank_H2_ng_kg": val(S_H2_ng[t]),
            "tank_NH3_g_kg": val(S_NH3_g[t]),
            "tank_NH3_ng_kg": val(S_NH3_ng[t]),
            "tank_MeOH_g_kg": val(S_MEOH_g[t]),
            "tank_MeOH_ng_kg": val(S_MEOH_ng[t]),
            "b_ch": int(val(P_ch[t]) > 1e-6),

            "n_elz_on": int(round(val(n_elz_on[t]))),
            "y_elz": int(round(val(y_elz[t]))),
            "su_elz": int(round(val(su_elz[t]))),
            "y_nh3": int(round(val(y_nh3[t]))),
            "su_nh3": int(round(val(su_nh3[t]))),
            "y_meoh": int(round(val(y_meoh[t]))),
            "su_meoh": int(round(val(su_meoh[t]))),
            "y_crk": int(round(val(y_crk[t]))),
            "su_crk": int(round(val(su_crk[t]))),
            **elz_mod,
        })

        unmet_rows.append({
            "run_id": RUN_ID,
            "model_tag": MODEL_TAG,
            "scenario_tag": SCENARIO_TAG,
            "demand_tag": DEMAND_TAG,
            "datetime_utc": ts.isoformat(),
            "day_of_year": d,
            "hour_in_day": t,
            "unmet_h2_kgph": u1,
            "unmet_nh3_kgph": u2,
            "unmet_meoh_kgph": u3,
            "da_price_eur_per_mwh": pr_eur[t],
            "grid_import_mw": grid_mw,
            "res_curtailed_mw": res_curt_mw,
        })

    day_hourly = pd.DataFrame(rows)
    day_unmet  = pd.DataFrame(unmet_rows)

    day_hourly.to_csv(day_dir / "hourly_timeseries.csv", index=False)
    day_unmet.to_csv(day_dir / "hourly_unmet_demand.csv", index=False)

    day_hourly_all = day_hourly.copy()
    day_hourly_all["day_id"] = int(d)
    all_hourly_frames.append(day_hourly_all)

    # ---------------- Daily accounting ----------------
    grid_cost_eur   = float(np.sum([pr_eur[t] * val(P_grid[t]) for t in T]))
    water_cost_eur  = float(c_H2O * np.sum([val(W_elz[t]) + val(W_steam[t]) for t in T]))
    co2_cost_eur    = float(c_CO2 * np.sum([val(CO2_buy[t]) for t in T]))
    startup_elz_eur = float(C_su_elz * np.sum([val(su_elz[t]) for t in T]))
    startup_nh3_eur = float(C_su_NH3 * np.sum([val(su_nh3[t]) for t in T]))
    startup_meoh_eur = float(C_su_MEOH * np.sum([val(su_meoh[t]) for t in T]))
    startup_crk_eur = float(C_su_CRK * np.sum([val(su_crk[t]) for t in T]))

    total_unmet_kg = float(np.sum(day_hourly["unmet_h2_kgph"].values + day_hourly["unmet_nh3_kgph"].values + day_hourly["unmet_meoh_kgph"].values))
    total_cost = float(obj2_val) if not np.isnan(obj2_val) else float(grid_cost_eur + water_cost_eur + co2_cost_eur + startup_elz_eur + startup_nh3_eur + startup_meoh_eur + startup_crk_eur)

    grid_emis_aef = float(np.sum([val(P_grid[t]) * 1000.0 * aef_k[t] for t in T]))
    grid_emis_mef_upramp = float(np.sum([val(P_grid[t]) * 1000.0 * mef_upramp_k[t] for t in T]))
    grid_emis_mef_soft = float(np.sum([val(P_grid[t]) * 1000.0 * mef_soft_k[t] for t in T]))

    direct_emis_nh3 = float(np.sum([de_NH3 * val(Q_nh3[t]) for t in T]))
    direct_emis_meoh = float(np.sum([de_MEOH * val(Q_meoh[t]) for t in T]))
    direct_emis_total = direct_emis_nh3 + direct_emis_meoh

    if EMISSIONS_KEY == "aef":
        realized_emis_cap_basis = grid_emis_aef + direct_emis_total
    elif EMISSIONS_KEY == "mef_upramp":
        realized_emis_cap_basis = grid_emis_mef_upramp + direct_emis_total
    elif EMISSIONS_KEY == "mef_soft":
        realized_emis_cap_basis = grid_emis_mef_soft + direct_emis_total
    else:
        raise ValueError("EMISSIONS_KEY must be one of: 'aef', 'mef_upramp', 'mef_soft'")

    demand_H2_kg_day   = float(np.sum(dem_h2))
    demand_NH3_kg_day  = float(np.sum(dem_nh3))
    demand_MeOH_kg_day = float(np.sum(dem_meoh))

    elz_kwh_day  = float(np.sum(day_hourly["P_elz_mw"].values * 1000.0))
    nh3_kwh_day  = float(np.sum(day_hourly["P_nh3_mw"].values * 1000.0))
    meoh_kwh_day = float(np.sum(day_hourly["P_meoh_mw"].values * 1000.0))
    crk_kwh_day  = float(np.sum(day_hourly["P_crk_mw"].values * 1000.0))
    units_kwh_day = elz_kwh_day + nh3_kwh_day + meoh_kwh_day + crk_kwh_day

    res_gen_kwh = float(np.sum((pv_av + wd_av) * 1000.0))
    res_used_kwh = float(np.sum((pv_av - day_hourly["pv_curt_mw"].values + wd_av - day_hourly["wind_curt_mw"].values) * 1000.0))
    res_curt_kwh = float(np.sum((day_hourly["pv_curt_mw"].values + day_hourly["wind_curt_mw"].values) * 1000.0))
    grid_import_kwh = float(np.sum(day_hourly["grid_import_mw"].values * 1000.0))
    grid_share_percent = 100.0 * grid_import_kwh / max(grid_import_kwh + res_used_kwh, 1e-9)

    H2_g_sum = float(np.sum(day_hourly["H2_prod_g_kgph"].values))
    H2_ng_sum = float(np.sum(day_hourly["H2_prod_ng_kgph"].values))
    NH3_g_sum = float(np.sum(day_hourly["NH3_prod_g_kgph"].values))
    NH3_ng_sum = float(np.sum(day_hourly["NH3_prod_ng_kgph"].values))
    Me_g_sum = float(np.sum(day_hourly["MeOH_prod_g_kgph"].values))
    Me_ng_sum = float(np.sum(day_hourly["MeOH_prod_ng_kgph"].values))

    def green_share(sum_g, sum_ng):
        tot = sum_g + sum_ng
        return (sum_g / tot) if tot > 1e-9 else np.nan

    daily_rows.append({
        "run_id": RUN_ID,
        "model_tag": MODEL_TAG,
        "scenario_tag": SCENARIO_TAG,
        "demand_tag": DEMAND_TAG,
        "day_of_year": d,
        "date_utc": str(day_hours[0].date()),
        "has_incumbent": bool(has_incumbent),
        "is_optimal": bool(is_optimal),
        "hit_time_limit": bool(hit_time_limit),
        "mip_gap": float(mip_gap) if mip_gap is not None else np.nan,
        "is_within_target_gap": bool(is_within_target_gap),
        "is_acceptable": bool(is_acceptable),

        "total_cost_eur": total_cost,
        "obj1_total_unmet_kg": total_unmet_kg,
        "obj2_operating_cost_eur": total_cost,

        "res_generation_kWh": res_gen_kwh,
        "res_used_kWh": res_used_kwh,
        "res_curtailed_kWh": res_curt_kwh,
        "grid_import_kWh": grid_import_kwh,
        "grid_share_percent": grid_share_percent,

        "battery_soc_avg": float(np.mean(day_hourly["battery_soc"].values)),
        "tank_H2_level_avg_kg": float(np.mean(day_hourly["tank_H2_g_kg"].values + day_hourly["tank_H2_ng_kg"].values)),
        "tank_NH3_level_avg_kg": float(np.mean(day_hourly["tank_NH3_g_kg"].values + day_hourly["tank_NH3_ng_kg"].values)),
        "tank_MeOH_level_avg_kg": float(np.mean(day_hourly["tank_MeOH_g_kg"].values + day_hourly["tank_MeOH_ng_kg"].values)),

        "H2_prod_green_share": green_share(H2_g_sum, H2_ng_sum),
        "NH3_prod_green_share": green_share(NH3_g_sum, NH3_ng_sum),
        "MeOH_prod_green_share": green_share(Me_g_sum, Me_ng_sum),

        # grid emissions only
        "emissions_grid_aef_kg": grid_emis_aef,
        "emissions_grid_mef_upramp_kg": grid_emis_mef_upramp,
        "emissions_grid_mef_soft_kg": grid_emis_mef_soft,

        # direct emissions separate
        "emissions_direct_nh3_kg": direct_emis_nh3,
        "emissions_direct_meoh_kg": direct_emis_meoh,
        "emissions_direct_total_kg": direct_emis_total,

        "unmet_H2_kg": float(np.sum(day_hourly["unmet_h2_kgph"].values)),
        "unmet_NH3_kg": float(np.sum(day_hourly["unmet_nh3_kgph"].values)),
        "unmet_MeOH_kg": float(np.sum(day_hourly["unmet_meoh_kgph"].values)),

        "demand_H2_kg_day": demand_H2_kg_day,
        "demand_NH3_kg_day": demand_NH3_kg_day,
        "demand_MeOH_kg_day": demand_MeOH_kg_day,

        "elz_consumption_kWh_day": elz_kwh_day,
        "nh3_consumption_kWh_day": nh3_kwh_day,
        "meoh_consumption_kWh_day": meoh_kwh_day,
        "crk_consumption_kWh_day": crk_kwh_day,
        "units_total_consumption_kWh_day": units_kwh_day,

        # sales accounting
        "H2_sell_g_kg_day": float(np.sum(day_hourly["H2_sell_g_kgph"].values)),
        "H2_sell_ng_kg_day": float(np.sum(day_hourly["H2_sell_ng_kgph"].values)),
        "H2_sell_total_kg_day": float(np.sum(day_hourly["H2_sell_kgph"].values)),

        "NH3_sell_g_kg_day": float(np.sum(day_hourly["NH3_sell_g_kgph"].values)),
        "NH3_sell_ng_kg_day": float(np.sum(day_hourly["NH3_sell_ng_kgph"].values)),
        "NH3_sell_total_kg_day": float(np.sum(day_hourly["NH3_sell_kgph"].values)),

        "MeOH_sell_g_kg_day": float(np.sum(day_hourly["MeOH_sell_g_kgph"].values)),
        "MeOH_sell_ng_kg_day": float(np.sum(day_hourly["MeOH_sell_ng_kgph"].values)),
        "MeOH_sell_total_kg_day": float(np.sum(day_hourly["MeOH_sell_kgph"].values)),

        "eod_H2_total_kg": float(val(S_H2_g[24]) + val(S_H2_ng[24])),
        "eod_NH3_total_kg": float(val(S_NH3_g[24]) + val(S_NH3_ng[24])),
        "eod_MeOH_total_kg": float(val(S_MEOH_g[24]) + val(S_MEOH_ng[24])),
        "eod_req_H2_kg": float(ALPHA_TANK_FILL * S_H2_CAP),
        "eod_req_NH3_kg": float(ALPHA_TANK_FILL * S_NH3_CAP),
        "eod_req_MeOH_kg": float(ALPHA_TANK_FILL * S_MEOH_CAP),
        "eod_margin_H2_kg": float((val(S_H2_g[24]) + val(S_H2_ng[24])) - ALPHA_TANK_FILL * S_H2_CAP),
        "eod_margin_NH3_kg": float((val(S_NH3_g[24]) + val(S_NH3_ng[24])) - ALPHA_TANK_FILL * S_NH3_CAP),
        "eod_margin_MeOH_kg": float((val(S_MEOH_g[24]) + val(S_MEOH_ng[24])) - ALPHA_TANK_FILL * S_MEOH_CAP),

        "cost_grid_eur": grid_cost_eur,
        "cost_water_eur": water_cost_eur,
        "cost_co2_eur": co2_cost_eur,
        "cost_startup_elz_eur": startup_elz_eur,
        "cost_startup_nh3_eur": startup_nh3_eur,
        "cost_startup_meoh_eur": startup_meoh_eur,
        "cost_startup_crk_eur": startup_crk_eur,
        "m2_cap_repair_flag": int(IS_CAP_REPAIR_DAY),
        "m2_emissions_key": str(EMISSIONS_KEY),
        "m2_alpha_used_day": (float(ALPHA_USED_DAY) if (ALPHA_USED_DAY is not None and not np.isnan(ALPHA_USED_DAY)) else np.nan),
        "m2_emis_cap_day_kg": float(EMIS_CAP_DAY_KG),
        "m2_realized_total_emis_cap_basis_kg": float(realized_emis_cap_basis),
        "m2_cap_slack_kg": float(EMIS_CAP_DAY_KG - realized_emis_cap_basis),
    })

    # ---------------- Carry-over update ----------------
    carry["soc"] = val(SOC[24])
    carry["S_H2_g"] = val(S_H2_g[24]); carry["S_H2_ng"] = val(S_H2_ng[24])
    carry["S_NH3_g"] = val(S_NH3_g[24]); carry["S_NH3_ng"] = val(S_NH3_ng[24])
    carry["S_MEOH_g"] = val(S_MEOH_g[24]); carry["S_MEOH_ng"] = val(S_MEOH_ng[24])
    carry["y_elz_prev_mod"] = [int(round(val(y_elz_m[i,23]))) for i in I_elz]
    carry["y_nh3_prev"]  = int(round(val(y_nh3[23])))
    carry["y_meoh_prev"] = int(round(val(y_meoh[23])))
    carry["y_crk_prev"]  = int(round(val(y_crk[23])))
    carry["Q_nh3_prev"]  = float(val(Q_nh3[23]))
    carry["Q_meoh_prev"] = float(val(Q_meoh[23]))
    carry["Q_crk_prev"]  = float(val(Q_crk_in[23]))

    state_rows.append({
        "run_id": RUN_ID,
        "day_of_year": d,
        "date_utc": str(day_hours[0].date()),
        "SOC_end": carry["soc"],
        "S_H2_g_end": carry["S_H2_g"],
        "S_H2_ng_end": carry["S_H2_ng"],
        "S_NH3_g_end": carry["S_NH3_g"],
        "S_NH3_ng_end": carry["S_NH3_ng"],
        "S_MEOH_g_end": carry["S_MEOH_g"],
        "S_MEOH_ng_end": carry["S_MEOH_ng"],
        "y_elz_prev_mod_end": carry["y_elz_prev_mod"],
        "y_nh3_end": carry["y_nh3_prev"],
        "y_meoh_end": carry["y_meoh_prev"],
        "y_crk_end": carry["y_crk_prev"],
        "Q_nh3_end": carry["Q_nh3_prev"],
        "Q_meoh_end": carry["Q_meoh_prev"],
        "Q_crk_end": carry["Q_crk_prev"],
    })

    eps = 1e-6
    sim_cd = float(np.sum((day_hourly["bat_ch_mw"].values > eps) & (day_hourly["bat_dis_mw"].values > eps)))
    soc_viol = float(np.sum((day_hourly["battery_soc"].values < SOC_MIN - 1e-6) | (day_hourly["battery_soc"].values > SOC_MAX + 1e-6)))
    unmet_any = float(np.sum((day_hourly["unmet_h2_kgph"].values > eps) | (day_hourly["unmet_nh3_kgph"].values > eps) | (day_hourly["unmet_meoh_kgph"].values > eps)))

    diag_rows.append({
        "run_id": RUN_ID,
        "day_of_year": d,
        "date_utc": str(day_hours[0].date()),
        "flag_simultaneous_charge_discharge_hours": sim_cd,
        "flag_soc_bound_violation_hours": soc_viol,
        "flag_unmet_demand_hours": unmet_any,
    })

# ---------------- End-of-run aggregation ----------------
daily_df = pd.DataFrame(daily_rows)
if daily_df.empty:
    raise RuntimeError("No days were solved (daily_df is empty).")

solver_df = pd.DataFrame(solver_rows)
state_df  = pd.DataFrame(state_rows)
diag_df   = pd.DataFrame(diag_rows)

if len(all_hourly_frames) > 0:
    hourly_all_df = pd.concat(all_hourly_frames, ignore_index=True)
    hourly_all_df.to_csv(OUT_POST / "hourly_timeseries_all_days.csv", index=False)
else:
    hourly_all_df = pd.DataFrame()

daily_df.to_csv(OUT_POST / "daily_summary.csv", index=False)
solver_df.to_csv(OUT_POST / "solver_log_daily.csv", index=False)
state_df.to_csv(OUT_POST / "state_trace.csv", index=False)
diag_df.to_csv(OUT_POST / "daily_diagnostics_flags.csv", index=False)

annual_summary = {
    "run_id": RUN_ID,
    "model_tag": MODEL_TAG,
    "scenario_tag": SCENARIO_TAG,
    "demand_tag": DEMAND_TAG,
    "start_day_of_year": START_DAY_OF_YEAR,
    "n_days_run": int(len(daily_df)),
    "m2_emissions_key": str(EMISSIONS_KEY),
    "m2_alpha_configured": float(ALPHA),
    "m2_emis_cap_day_kg_sum": float(daily_df["m2_emis_cap_day_kg"].sum()),
    "m2_alpha_used_day_mean": float(daily_df["m2_alpha_used_day"].mean()),
    "m2_alpha_used_day_count": int(daily_df["m2_alpha_used_day"].notna().sum()),
    "m2_cap_repair_days_count": int(daily_df["m2_cap_repair_flag"].sum()),
    "total_cost_eur": float(daily_df["total_cost_eur"].sum()),

    "total_grid_emissions_aef_kg": float(daily_df["emissions_grid_aef_kg"].sum()),
    "total_grid_emissions_mef_upramp_kg": float(daily_df["emissions_grid_mef_upramp_kg"].sum()),
    "total_grid_emissions_mef_soft_kg": float(daily_df["emissions_grid_mef_soft_kg"].sum()),

    "total_direct_emissions_nh3_kg": float(daily_df["emissions_direct_nh3_kg"].sum()),
    "total_direct_emissions_meoh_kg": float(daily_df["emissions_direct_meoh_kg"].sum()),
    "total_direct_emissions_kg": float(daily_df["emissions_direct_total_kg"].sum()),

    "total_grid_import_kWh": float(daily_df["grid_import_kWh"].sum()),
    "total_res_used_kWh": float(daily_df["res_used_kWh"].sum()),
    "total_res_curtailed_kWh": float(daily_df["res_curtailed_kWh"].sum()),
    "avg_grid_share_percent": float(daily_df["grid_share_percent"].mean()),

    "total_unmet_H2_kg": float(daily_df["unmet_H2_kg"].sum()),
    "total_unmet_NH3_kg": float(daily_df["unmet_NH3_kg"].sum()),
    "total_unmet_MeOH_kg": float(daily_df["unmet_MeOH_kg"].sum()),

    "total_H2_sell_g_kg": float(daily_df["H2_sell_g_kg_day"].sum()),
    "total_H2_sell_ng_kg": float(daily_df["H2_sell_ng_kg_day"].sum()),
    "total_H2_sell_kg": float(daily_df["H2_sell_total_kg_day"].sum()),

    "total_NH3_sell_g_kg": float(daily_df["NH3_sell_g_kg_day"].sum()),
    "total_NH3_sell_ng_kg": float(daily_df["NH3_sell_ng_kg_day"].sum()),
    "total_NH3_sell_kg": float(daily_df["NH3_sell_total_kg_day"].sum()),

    "total_MeOH_sell_g_kg": float(daily_df["MeOH_sell_g_kg_day"].sum()),
    "total_MeOH_sell_ng_kg": float(daily_df["MeOH_sell_ng_kg_day"].sum()),
    "total_MeOH_sell_kg": float(daily_df["MeOH_sell_total_kg_day"].sum()),

}
pd.DataFrame([annual_summary]).to_csv(OUT_POST / "annual_summary.csv", index=False)

# ---------------- Plots ----------------
daily_df = daily_df.copy()
daily_df["date"] = pd.to_datetime(daily_df["date_utc"])

if all(c in daily_df.columns for c in ["eod_margin_H2_kg","eod_margin_NH3_kg","eod_margin_MeOH_kg"]):
    tmp = daily_df[["date","eod_margin_H2_kg","eod_margin_NH3_kg","eod_margin_MeOH_kg"]]
    save_lineplot(
        tmp, "date",
        ["eod_margin_H2_kg","eod_margin_NH3_kg","eod_margin_MeOH_kg"],
        f"{RUN_ID} • EOD inventory constraint margins", "kg",
        OUT_PLOTS / "P20_eod_margins.png"
    )

save_lineplot(daily_df, "date", ["total_cost_eur"], f"{RUN_ID} • Daily total cost", "EUR", OUT_PLOTS / "P01_daily_total_cost.png")

if "obj1_total_unmet_kg" in daily_df.columns:
    save_lineplot(
        daily_df, "date", ["obj1_total_unmet_kg"],
        f"{RUN_ID} • Objective-1 total unmet (daily)", "kg/day",
        OUT_PLOTS / "P07_obj1_total_unmet_daily.png"
    )

tmp = daily_df[["date","emissions_grid_aef_kg","emissions_grid_mef_upramp_kg","emissions_grid_mef_soft_kg"]]
save_lineplot(
    tmp, "date", ["emissions_grid_aef_kg","emissions_grid_mef_upramp_kg","emissions_grid_mef_soft_kg"],
    f"{RUN_ID} • Daily grid emissions by signal", "kg CO2/day",
    OUT_PLOTS / "P02_daily_grid_emissions_three_signals.png"
)

tmp = daily_df[["date","emissions_direct_nh3_kg","emissions_direct_meoh_kg"]]
save_lineplot(
    tmp, "date", ["emissions_direct_nh3_kg","emissions_direct_meoh_kg"],
    f"{RUN_ID} • Daily direct emissions by unit", "kg CO2/day",
    OUT_PLOTS / "P17_direct_emissions_daily_total.png"
)

tmp = daily_df[["date","grid_import_kWh"]]
save_lineplot(tmp, "date", ["grid_import_kWh"], f"{RUN_ID} • Daily grid import", "kWh", OUT_PLOTS / "P03_daily_grid_import_kwh.png")

tmp = daily_df[["date","res_used_kWh","res_curtailed_kWh"]]
save_lineplot(
    tmp, "date", ["res_used_kWh","res_curtailed_kWh"],
    f"{RUN_ID} • Daily RES used vs curtailed", "kWh",
    OUT_PLOTS / "P04_daily_res_used_vs_curtailed.png"
)

solver_df = solver_df.copy()
solver_df["date"] = pd.to_datetime(solver_df["date_utc"])
tmp = solver_df[["date","runtime_seconds","mip_gap"]]
save_lineplot(tmp, "date", ["runtime_seconds"], f"{RUN_ID} • Daily solver runtime", "seconds", OUT_PLOTS / "P05_daily_solver_runtime.png")
save_lineplot(tmp, "date", ["mip_gap"], f"{RUN_ID} • Daily solver MIP gap", "-", OUT_PLOTS / "P06_daily_mip_gap.png")

if all(c in solver_df.columns for c in ["obj_val","obj_bound"]):
    tmp = solver_df[["date","obj_val","obj_bound"]].copy()
    save_lineplot(
        tmp, "date", ["obj_val","obj_bound"],
        f"{RUN_ID} • Daily solver objective value vs bound", "objective units",
        OUT_PLOTS / "P06b_daily_obj_val_vs_bound.png"
    )

if hourly_all_df is not None and len(hourly_all_df) > 0:
    hdf = hourly_all_df.copy()
    hdf["dt"] = pd.to_datetime(hdf["datetime_utc"], utc=True)
    hdf = hdf.sort_values("dt")
else:
    hdf = pd.DataFrame()

if len(hdf) > 0:
    hh = hdf.copy()
    hh["tank_H2_total_kg"]   = hh["tank_H2_g_kg"].values + hh["tank_H2_ng_kg"].values
    hh["tank_NH3_total_kg"]  = hh["tank_NH3_g_kg"].values + hh["tank_NH3_ng_kg"].values
    hh["tank_MeOH_total_kg"] = hh["tank_MeOH_g_kg"].values + hh["tank_MeOH_ng_kg"].values

    hh["demand_H2_kgph"]   = hh["H2_sell_kgph"].values + hh["unmet_h2_kgph"].values
    hh["demand_NH3_kgph"]  = hh["NH3_sell_kgph"].values + hh["unmet_nh3_kgph"].values
    hh["demand_MeOH_kgph"] = hh["MeOH_sell_kgph"].values + hh["unmet_meoh_kgph"].values

    hh["pct_demand_met_H2"] = np.where(hh["demand_H2_kgph"].values > 1e-9, 100.0 * (1.0 - hh["unmet_h2_kgph"].values / hh["demand_H2_kgph"].values), 100.0)
    hh["pct_demand_met_NH3"] = np.where(hh["demand_NH3_kgph"].values > 1e-9, 100.0 * (1.0 - hh["unmet_nh3_kgph"].values / hh["demand_NH3_kgph"].values), 100.0)
    hh["pct_demand_met_MeOH"] = np.where(hh["demand_MeOH_kgph"].values > 1e-9, 100.0 * (1.0 - hh["unmet_meoh_kgph"].values / hh["demand_MeOH_kgph"].values), 100.0)

    for c in ["pct_demand_met_H2","pct_demand_met_NH3","pct_demand_met_MeOH"]:
        hh[c] = hh[c].clip(lower=0.0, upper=100.0)

    hh["pct_tank_fill_H2"] = 100.0 * hh["tank_H2_total_kg"].values / float(S_H2_CAP)
    hh["pct_tank_fill_NH3"] = 100.0 * hh["tank_NH3_total_kg"].values / float(S_NH3_CAP)
    hh["pct_tank_fill_MeOH"] = 100.0 * hh["tank_MeOH_total_kg"].values / float(S_MEOH_CAP)

    for c in ["pct_tank_fill_H2","pct_tank_fill_NH3","pct_tank_fill_MeOH"]:
        hh[c] = hh[c].clip(lower=0.0, upper=100.0)

    binary_cols_present = [c for c in ["su_crk","y_crk","su_meoh","y_meoh","su_nh3","y_nh3","su_elz","y_elz","b_ch"] if c in hh.columns]

    ops_cols = (
        ["run_id","model_tag","scenario_tag","demand_tag","dt","day_id","day_of_year","hour_in_day"]
        + binary_cols_present
        + ["pct_demand_met_H2","pct_demand_met_NH3","pct_demand_met_MeOH",
           "pct_tank_fill_H2","pct_tank_fill_NH3","pct_tank_fill_MeOH"]
    )
    ops_cols = [c for c in ops_cols if c in hh.columns]
    hh[ops_cols].copy().to_csv(OUT_POST / "hourly_ops_signals.csv", index=False)

    tmp = hdf[["dt","grid_import_mw","res_curtailed_mw","res_used_mw"]].copy()
    save_lineplot(
        tmp, "dt", ["grid_import_mw","res_curtailed_mw","res_used_mw"],
        f"{RUN_ID} • Hourly grid import vs RES used/curtailed", "MW",
        OUT_PLOTS / "P03b_hourly_grid_vs_res_used_curtailed.png"
    )

    for prod_col, fname_stub, title_stub, ylab in [
        ("H2_prod_kgph",  "H2",  "H2 production",  "kg/h"),
        ("NH3_prod_kgph", "NH3", "NH3 production", "kg/h"),
        ("MeOH_prod_kgph","MeOH","MeOH production","kg/h"),
    ]:
        tmp = hdf[["dt", prod_col]]
        save_lineplot(tmp, "dt", [prod_col], f"{RUN_ID} • {title_stub} (hourly)", ylab,
                      OUT_PLOTS / f"P08_prod_{fname_stub}_hourly.png")

    for sell_cols, fname_stub, title_stub in [
        (["H2_sell_g_kgph","H2_sell_ng_kgph"], "H2", "H2 sales g/ng"),
        (["NH3_sell_g_kgph","NH3_sell_ng_kgph"], "NH3", "NH3 sales g/ng"),
        (["MeOH_sell_g_kgph","MeOH_sell_ng_kgph"], "MeOH", "MeOH sales g/ng"),
    ]:
        tmp = hdf[["dt"] + sell_cols].copy()
        save_lineplot(tmp, "dt", sell_cols, f"{RUN_ID} • {title_stub} (hourly)", "kg/h",
                      OUT_PLOTS / f"P08b_sales_{fname_stub}_hourly_gng.png")

    prod_daily = (
        hdf.groupby("day_id")[["H2_prod_kgph","NH3_prod_kgph","MeOH_prod_kgph"]]
        .sum()
        .rename(columns={
            "H2_prod_kgph":"H2_prod_kg_day",
            "NH3_prod_kgph":"NH3_prod_kg_day",
            "MeOH_prod_kgph":"MeOH_prod_kg_day",
        })
        .reset_index()
        .merge(daily_df[["day_of_year","date"]], left_on="day_id", right_on="day_of_year", how="left")
    )

    for col, fname_stub, title_stub in [
        ("H2_prod_kg_day",  "H2",  "H2 production"),
        ("NH3_prod_kg_day", "NH3", "NH3 production"),
        ("MeOH_prod_kg_day","MeOH","MeOH production"),
    ]:
        tmp = prod_daily[["date", col]].dropna()
        save_lineplot(tmp, "date", [col], f"{RUN_ID} • {title_stub} (daily total)", "kg/day",
                      OUT_PLOTS / f"P09_prod_{fname_stub}_daily_total.png")

    sell_daily = (
        hdf.groupby("day_id")[[
            "H2_sell_g_kgph","H2_sell_ng_kgph",
            "NH3_sell_g_kgph","NH3_sell_ng_kgph",
            "MeOH_sell_g_kgph","MeOH_sell_ng_kgph"
        ]]
        .sum()
        .rename(columns={
            "H2_sell_g_kgph":"H2_sell_g_kg_day",
            "H2_sell_ng_kgph":"H2_sell_ng_kg_day",
            "NH3_sell_g_kgph":"NH3_sell_g_kg_day",
            "NH3_sell_ng_kgph":"NH3_sell_ng_kg_day",
            "MeOH_sell_g_kgph":"MeOH_sell_g_kg_day",
            "MeOH_sell_ng_kgph":"MeOH_sell_ng_kg_day",
        })
        .reset_index()
        .merge(daily_df[["day_of_year","date"]], left_on="day_id", right_on="day_of_year", how="left")
    )

    for cols, fname_stub, title_stub in [
        (["H2_sell_g_kg_day","H2_sell_ng_kg_day"], "H2", "H2 sales g/ng"),
        (["NH3_sell_g_kg_day","NH3_sell_ng_kg_day"], "NH3", "NH3 sales g/ng"),
        (["MeOH_sell_g_kg_day","MeOH_sell_ng_kg_day"], "MeOH", "MeOH sales g/ng"),
    ]:
        tmp = sell_daily[["date"] + cols].dropna()
        save_lineplot(tmp, "date", cols, f"{RUN_ID} • {title_stub} (daily total)", "kg/day",
                      OUT_PLOTS / f"P09b_sales_{fname_stub}_daily_total_gng.png")

    hdf["tank_H2_total"] = hdf["tank_H2_g_kg"].values + hdf["tank_H2_ng_kg"].values
    hdf["tank_NH3_total"] = hdf["tank_NH3_g_kg"].values + hdf["tank_NH3_ng_kg"].values
    hdf["tank_MeOH_total"] = hdf["tank_MeOH_g_kg"].values + hdf["tank_MeOH_ng_kg"].values

    hdf["tank_H2_green_share"] = np.where(hdf["tank_H2_total"] > 1e-9, hdf["tank_H2_g_kg"] / hdf["tank_H2_total"], np.nan)
    hdf["tank_NH3_green_share"] = np.where(hdf["tank_NH3_total"] > 1e-9, hdf["tank_NH3_g_kg"] / hdf["tank_NH3_total"], np.nan)
    hdf["tank_MeOH_green_share"] = np.where(hdf["tank_MeOH_total"] > 1e-9, hdf["tank_MeOH_g_kg"] / hdf["tank_MeOH_total"], np.nan)

    for col, fname_stub, title_stub in [
        ("tank_H2_green_share", "H2", "Tank H2 green share"),
        ("tank_NH3_green_share", "NH3", "Tank NH3 green share"),
        ("tank_MeOH_green_share", "MeOH", "Tank MeOH green share"),
    ]:
        tmp = hdf[["dt", col]].copy()
        tmp[col] = tmp[col].fillna(0.0)
        save_lineplot(tmp, "dt", [col], f"{RUN_ID} • {title_stub} (hourly)", "share",
                      OUT_PLOTS / f"P10_tank_share_{fname_stub}_hourly.png")

for col, fname_stub, title_stub, ylab in [
    ("demand_H2_kg_day", "H2", "H2 demand", "kg/day"),
    ("demand_NH3_kg_day", "NH3", "NH3 demand", "kg/day"),
    ("demand_MeOH_kg_day", "MeOH", "MeOH demand", "kg/day"),
]:
    tmp = daily_df[["date", col]]
    save_lineplot(tmp, "date", [col], f"{RUN_ID} • {title_stub} (daily)", ylab,
                  OUT_PLOTS / f"P12_demand_{fname_stub}_daily.png")

for col, fname_stub, title_stub in [
    ("elz_consumption_kWh_day", "elz", "Electrolyzer electricity"),
    ("nh3_consumption_kWh_day", "nh3", "NH3 synthesis electricity"),
    ("meoh_consumption_kWh_day", "meoh", "MeOH synthesis electricity"),
    ("crk_consumption_kWh_day", "crk", "Cracker electricity"),
]:
    tmp = daily_df[["date", col]]
    save_lineplot(tmp, "date", [col], f"{RUN_ID} • {title_stub} (daily)", "kWh/day",
                  OUT_PLOTS / f"P14_power_{fname_stub}_daily.png")

tmp = daily_df[["date","elz_consumption_kWh_day","nh3_consumption_kWh_day","meoh_consumption_kWh_day","crk_consumption_kWh_day"]]
save_lineplot(
    tmp, "date", ["elz_consumption_kWh_day","nh3_consumption_kWh_day","meoh_consumption_kWh_day","crk_consumption_kWh_day"],
    f"{RUN_ID} • Unit electricity consumption (daily)", "kWh/day",
    OUT_PLOTS / "P15_power_units_combined_daily.png"
)

if len(hdf) > 0:
    hdf["direct_emis_nh3_kgph"]  = float(de_NH3) * hdf["NH3_prod_kgph"].values
    hdf["direct_emis_meoh_kgph"] = float(de_MEOH) * hdf["MeOH_prod_kgph"].values

    tmp = hdf[["dt","direct_emis_nh3_kgph","direct_emis_meoh_kgph"]]
    save_lineplot(
        tmp, "dt", ["direct_emis_nh3_kgph","direct_emis_meoh_kgph"],
        f"{RUN_ID} • Direct emissions by unit (hourly)", "kg CO2/h",
        OUT_PLOTS / "P16_direct_emissions_hourly.png"
    )

    aef_series = aef[[AEF_COL]].reset_index().rename(columns={"datetime_utc":"dt", AEF_COL:"aef_kg_per_kwh"})
    mef_upramp_series = mef_upramp[[MEF_UPRAMP_COL]].reset_index().rename(columns={"datetime_utc":"dt", MEF_UPRAMP_COL:"mef_upramp_kg_per_kwh"})
    mef_soft_series = mef_soft[[MEF_SOFT_COL]].reset_index().rename(columns={"datetime_utc":"dt", MEF_SOFT_COL:"mef_soft_kg_per_kwh"})

    aef_series["dt"] = pd.to_datetime(aef_series["dt"], utc=True)
    mef_upramp_series["dt"] = pd.to_datetime(mef_upramp_series["dt"], utc=True)
    mef_soft_series["dt"] = pd.to_datetime(mef_soft_series["dt"], utc=True)

    hdf["dt"] = pd.to_datetime(hdf["dt"], utc=True)
    hdf = hdf.merge(aef_series[["dt","aef_kg_per_kwh"]], on="dt", how="left")
    hdf = hdf.merge(mef_upramp_series[["dt","mef_upramp_kg_per_kwh"]], on="dt", how="left")
    hdf = hdf.merge(mef_soft_series[["dt","mef_soft_kg_per_kwh"]], on="dt", how="left")

    for tag, ef_col in [
        ("aef", "aef_kg_per_kwh"),
        ("mef_upramp", "mef_upramp_kg_per_kwh"),
        ("mef_soft", "mef_soft_kg_per_kwh"),
    ]:
        hdf[f"grid_emis_{tag}_elz_kgph"]  = 1000.0 * hdf["elz_grid_mw"].values  * hdf[ef_col].values
        hdf[f"grid_emis_{tag}_nh3_kgph"]  = 1000.0 * hdf["nh3_grid_mw"].values  * hdf[ef_col].values
        hdf[f"grid_emis_{tag}_meoh_kgph"] = 1000.0 * hdf["meoh_grid_mw"].values * hdf[ef_col].values
        hdf[f"grid_emis_{tag}_crk_kgph"]  = 1000.0 * hdf["crk_grid_mw"].values  * hdf[ef_col].values

        tmp = hdf[["dt",
                   f"grid_emis_{tag}_elz_kgph",
                   f"grid_emis_{tag}_nh3_kgph",
                   f"grid_emis_{tag}_meoh_kgph",
                   f"grid_emis_{tag}_crk_kgph"]].copy()

        save_lineplot(
            tmp, "dt",
            [f"grid_emis_{tag}_elz_kgph", f"grid_emis_{tag}_nh3_kgph", f"grid_emis_{tag}_meoh_kgph", f"grid_emis_{tag}_crk_kgph"],
            f"{RUN_ID} • Allocated grid emissions by unit (hourly, {tag})",
            "kg CO2/h",
            OUT_PLOTS / f"P17b_grid_emissions_by_unit_hourly_{tag}.png"
        )

        daily_alloc = (
            hdf.groupby("day_id")[
                [f"grid_emis_{tag}_elz_kgph", f"grid_emis_{tag}_nh3_kgph", f"grid_emis_{tag}_meoh_kgph", f"grid_emis_{tag}_crk_kgph"]
            ]
            .sum()
            .rename(columns={
                f"grid_emis_{tag}_elz_kgph": f"grid_emis_{tag}_elz_kg_day",
                f"grid_emis_{tag}_nh3_kgph": f"grid_emis_{tag}_nh3_kg_day",
                f"grid_emis_{tag}_meoh_kgph": f"grid_emis_{tag}_meoh_kg_day",
                f"grid_emis_{tag}_crk_kgph": f"grid_emis_{tag}_crk_kg_day",
            })
            .reset_index()
            .merge(daily_df[["day_of_year","date"]], left_on="day_id", right_on="day_of_year", how="left")
        )

        tmp = daily_alloc[["date",
                           f"grid_emis_{tag}_elz_kg_day",
                           f"grid_emis_{tag}_nh3_kg_day",
                           f"grid_emis_{tag}_meoh_kg_day",
                           f"grid_emis_{tag}_crk_kg_day"]].dropna()

        save_lineplot(
            tmp, "date",
            [f"grid_emis_{tag}_elz_kg_day", f"grid_emis_{tag}_nh3_kg_day", f"grid_emis_{tag}_meoh_kg_day", f"grid_emis_{tag}_crk_kg_day"],
            f"{RUN_ID} • Allocated grid emissions by unit (daily total, {tag})",
            "kg CO2/day",
            OUT_PLOTS / f"P17c_grid_emissions_by_unit_daily_{tag}.png"
        )

    hdf["grid_emis_aef_total_kgph"] = 1000.0 * hdf["grid_import_mw"].values * hdf["aef_kg_per_kwh"].values
    hdf["grid_emis_mef_upramp_total_kgph"] = 1000.0 * hdf["grid_import_mw"].values * hdf["mef_upramp_kg_per_kwh"].values
    hdf["grid_emis_mef_soft_total_kgph"] = 1000.0 * hdf["grid_import_mw"].values * hdf["mef_soft_kg_per_kwh"].values

    hdf["ef_delta_upramp_minus_aef"] = hdf["mef_upramp_kg_per_kwh"].values - hdf["aef_kg_per_kwh"].values
    hdf["ef_delta_soft_minus_aef"]   = hdf["mef_soft_kg_per_kwh"].values   - hdf["aef_kg_per_kwh"].values

    hdf["grid_emis_delta_upramp_minus_aef_kgph"] = hdf["grid_emis_mef_upramp_total_kgph"].values - hdf["grid_emis_aef_total_kgph"].values
    hdf["grid_emis_delta_soft_minus_aef_kgph"]   = hdf["grid_emis_mef_soft_total_kgph"].values   - hdf["grid_emis_aef_total_kgph"].values

    hdf["cum_delta_upramp_minus_aef_kg"] = np.cumsum(hdf["grid_emis_delta_upramp_minus_aef_kgph"].fillna(0.0).values)
    hdf["cum_delta_soft_minus_aef_kg"]   = np.cumsum(hdf["grid_emis_delta_soft_minus_aef_kgph"].fillna(0.0).values)

    tmp = hdf[["dt","cum_delta_upramp_minus_aef_kg","cum_delta_soft_minus_aef_kg"]]
    save_lineplot(
        tmp, "dt", ["cum_delta_upramp_minus_aef_kg","cum_delta_soft_minus_aef_kg"],
        f"{RUN_ID} • Cumulative Δ grid emissions vs AEF", "kg CO2",
        OUT_PLOTS / "P02e_cumulative_delta_grid_emissions_vs_aef.png"
    )

    hdf[[
        "run_id","model_tag","scenario_tag","demand_tag","dt","day_id","day_of_year","hour_in_day",
        "grid_import_mw",
        "aef_kg_per_kwh","mef_upramp_kg_per_kwh","mef_soft_kg_per_kwh",
        "grid_emis_aef_total_kgph","grid_emis_mef_upramp_total_kgph","grid_emis_mef_soft_total_kgph",
        "ef_delta_upramp_minus_aef","ef_delta_soft_minus_aef",
        "grid_emis_delta_upramp_minus_aef_kgph","grid_emis_delta_soft_minus_aef_kgph",
        "cum_delta_upramp_minus_aef_kg","cum_delta_soft_minus_aef_kg"
    ]].to_csv(OUT_POST / "hourly_emission_signal_diagnostics.csv", index=False)

if not hdf.empty:
    mod_ids = sorted({int(re.match(r"^elz_m(\d+)_y$", c).group(1)) for c in hdf.columns if re.match(r"^elz_m(\d+)_y$", c)})
    mod_ids = [i for i in mod_ids if (f"elz_m{i}_y" in hdf.columns) and (f"elz_m{i}_b_g" in hdf.columns)]

    if len(mod_ids) > 0:
        fig = plt.figure(figsize=(12, 6))
        ax1 = fig.add_subplot(2, 1, 1)
        for i in mod_ids:
            ax1.plot(hdf["dt"], hdf[f"elz_m{i}_y"].values, label=f"m{i}_y")
        ax1.set_title(f"{RUN_ID} • ELZ modules on/off (y)")
        ax1.set_ylabel("binary")
        ax1.legend(ncol=4, fontsize=8)

        ax2 = fig.add_subplot(2, 1, 2)
        for i in mod_ids:
            ax2.plot(hdf["dt"], hdf[f"elz_m{i}_b_g"].values, label=f"m{i}_b_g")
        ax2.set_title(f"{RUN_ID} • ELZ modules green-choice (b_g)")
        ax2.set_ylabel("binary")
        ax2.legend(ncol=4, fontsize=8)

        plt.tight_layout()
        plt.savefig(OUT_PLOTS / "P90_elz_modules_y_and_bg.png", dpi=150)
        plt.close()

        i0 = mod_ids[0]
        needed = [f"elz_m{i0}_P_tot", f"elz_m{i0}_P_g", f"elz_m{i0}_P_ng"]
        if all(c in hdf.columns for c in needed):
            tmp = hdf[["dt"] + needed].copy()
            save_lineplot(
                tmp, "dt", needed,
                f"{RUN_ID} • ELZ module m{i0} power split (P_tot, P_g, P_ng)",
                "MW",
                OUT_PLOTS / f"P91_elz_module_m{i0}_power_split.png"
            )

cost_cols = [
    ("cost_grid_eur", "Grid energy"),
    ("cost_water_eur", "Water"),
    ("cost_co2_eur", "CO2"),
    ("cost_startup_elz_eur", "Startup ELZ"),
    ("cost_startup_nh3_eur", "Startup NH3"),
    ("cost_startup_meoh_eur", "Startup MeOH"),
    ("cost_startup_crk_eur", "Startup CRK"),
]
save_stacked_bar_by_day(
    daily_df, "date", cost_cols,
    f"{RUN_ID} • Daily cost breakdown (stacked)", "EUR/day",
    OUT_PLOTS / "P18_daily_cost_breakdown_stacked.png"
)

res_cols = [
    ("res_used_kWh", "RES used"),
    ("res_curtailed_kWh", "RES curtailed"),
]
save_stacked_bar_by_day(
    daily_df, "date", res_cols,
    f"{RUN_ID} • Daily RES split (used vs curtailed)", "kWh/day",
    OUT_PLOTS / "P19_daily_res_used_vs_curtailed_stacked.png"
)

ZIP_PATH = OUT_BASE / f"{RUN_ID}.zip"
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in OUT_RUN.rglob("*"):
        if p.is_file():
            zf.write(p, arcname=str(p.relative_to(OUT_RUN)))

print("Run completed.")
print("Run folder:", OUT_RUN)
print("ZIP:", ZIP_PATH)
print("Total wall time (s):", time.time() - t0_global)

# 6. Representative and stress week selection

This block keeps the week selection logic


## 6.1 Representative center week and UpRamp stress week


In [ ]:
# ============================================================
# REPRESENTATIVE WEEK (CENTER) + STRESS WEEK SELECTION
#
# - Center week: multivariate robust central week across:
#     emissions signals + RES + demand + grid price
# - Stress week: upramp MEF stress-based week
# - Uses only full Monday start 168 h weeks
# ============================================================

import os
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

from google.colab import files

# -----------------------------
# Config
# -----------------------------
BASE_DIR = "/content"
OUT_DIR = os.path.join(BASE_DIR, "week_selection_outputs")
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

AEF_PATH = os.path.join(BASE_DIR, "grid_aef_hourly_2024.csv")
SOFT_PATH = os.path.join(BASE_DIR, "grid_mef_soft_hourly_2024.csv")
UPRAMP_PATH = os.path.join(BASE_DIR, "grid_mef_upramp_hourly_2024.csv")
RES_PATH = os.path.join(BASE_DIR, "res_hourly_2024.csv")
DEM_PATH = os.path.join(BASE_DIR, "demand_hourly_2024.csv")
PRICE_PATH = os.path.join(BASE_DIR, "grid_price_hourly_2024.csv")

# expected columns
COL_AEF = "grid_aef_predicted_kg_per_kwh"
COL_SOFT = "grid_mef_soft_kg_per_kwh"
COL_UPRAMP = "grid_mef_upramp_pred_kg_per_kwh"
COL_RES = "res_total_mw"
COL_H2 = "h2_demand_kgph"
COL_NH3 = "nh3_demand_kgph"
COL_MEOH = "meoh_demand_kgph"
COL_PRICE = "da_elec_price_EURperMWh"

# -----------------------------
# Helpers
# -----------------------------
def read_hourly_csv(path, keep_cols):
    df = pd.read_csv(path).copy()
    if "datetime_utc" not in df.columns:
        raise ValueError(f"{os.path.basename(path)} is missing 'datetime_utc'")
    df["datetime_utc"] = pd.to_datetime(df["datetime_utc"], utc=True, errors="coerce")
    df = df.dropna(subset=["datetime_utc"]).sort_values("datetime_utc")

    missing = [c for c in keep_cols if c not in df.columns]
    if missing:
        raise ValueError(f"{os.path.basename(path)} is missing columns: {missing}")

    return df[["datetime_utc"] + keep_cols].copy()

def robust_scale(series):
    med = series.median()
    q25 = series.quantile(0.25)
    q75 = series.quantile(0.75)
    iqr = q75 - q25
    if pd.isna(iqr) or iqr == 0:
        raise ValueError(f"IQR is zero for feature '{series.name}', cannot robust-scale.")
    z = (series - med) / iqr
    return z, med, q25, q75, iqr

# -----------------------------
# Read data
# -----------------------------
df_aef = read_hourly_csv(AEF_PATH, [COL_AEF]).rename(columns={COL_AEF: "AEF"})
df_soft = read_hourly_csv(SOFT_PATH, [COL_SOFT]).rename(columns={COL_SOFT: "SOFT"})
df_upr = read_hourly_csv(UPRAMP_PATH, [COL_UPRAMP]).rename(columns={COL_UPRAMP: "UPRAMP"})
df_res = read_hourly_csv(RES_PATH, [COL_RES]).rename(columns={COL_RES: "RES_MW"})
df_dem = read_hourly_csv(DEM_PATH, [COL_H2, COL_NH3, COL_MEOH]).rename(columns={
    COL_H2: "D_H2",
    COL_NH3: "D_NH3",
    COL_MEOH: "D_MEOH"
})
df_price = read_hourly_csv(PRICE_PATH, [COL_PRICE]).rename(columns={COL_PRICE: "PRICE"})

df = (
    df_aef.merge(df_soft, on="datetime_utc", how="inner")
          .merge(df_upr, on="datetime_utc", how="inner")
          .merge(df_res, on="datetime_utc", how="inner")
          .merge(df_dem, on="datetime_utc", how="inner")
          .merge(df_price, on="datetime_utc", how="inner")
          .sort_values("datetime_utc")
          .reset_index(drop=True)
)

print("Merged hourly rows:", len(df))
print("Time range:", df["datetime_utc"].min(), "to", df["datetime_utc"].max())

# -----------------------------
# Build Monday-start weeks
# -----------------------------
df["week_start"] = df["datetime_utc"].dt.floor("D") - pd.to_timedelta(df["datetime_utc"].dt.dayofweek, unit="D")
df["week_start"] = pd.to_datetime(df["week_start"].dt.date, utc=True)

# annual upramp p90 threshold for stress logic
annual_upramp_p90 = df["UPRAMP"].quantile(0.90)

weekly = df.groupby("week_start").agg(
    AEF_mean=("AEF", "mean"),
    SOFT_mean=("SOFT", "mean"),
    UPRAMP_mean=("UPRAMP", "mean"),
    SOFT_std=("SOFT", "std"),
    UPRAMP_std=("UPRAMP", "std"),
    UPRAMP_p90=("UPRAMP", lambda x: x.quantile(0.90)),
    PRICE_mean=("PRICE", "mean"),
    PRICE_std=("PRICE", "std"),
    RES_energy_MWh=("RES_MW", "sum"),
    D_H2_week=("D_H2", "sum"),
    D_NH3_week=("D_NH3", "sum"),
    D_MeOH_week=("D_MEOH", "sum"),
    n_hours=("UPRAMP", "count"),
    hours_above_annual_upramp_p90=("UPRAMP", lambda x: (x > annual_upramp_p90).sum())
).reset_index()

weekly["share_above_annual_upramp_p90"] = weekly["hours_above_annual_upramp_p90"] / weekly["n_hours"]

# only full weeks
weekly_full = weekly[weekly["n_hours"] == 168].copy().reset_index(drop=True)

if weekly_full.empty:
    raise RuntimeError("No full 168 h Monday-start weeks found.")

# -----------------------------
# Center week score
# -----------------------------
center_features = [
    "AEF_mean",
    "SOFT_mean",
    "UPRAMP_mean",
    "SOFT_std",
    "UPRAMP_std",
    "PRICE_mean",
    "PRICE_std",
    "RES_energy_MWh",
    "D_H2_week",
    "D_NH3_week",
    "D_MeOH_week",
]

# -----------------------------
# Stress week score (upramp-based only)
# -----------------------------
stress_features = [
    "UPRAMP_mean",
    "UPRAMP_p90",
    "share_above_annual_upramp_p90",
]

scaler_rows = []

for col in center_features:
    z, med, q25, q75, iqr = robust_scale(weekly_full[col])
    weekly_full[f"z_center_{col}"] = z
    scaler_rows.append({
        "selection_type": "center",
        "feature": col,
        "median": med,
        "q25": q25,
        "q75": q75,
        "iqr": iqr
    })

for col in stress_features:
    z, med, q25, q75, iqr = robust_scale(weekly_full[col])
    weekly_full[f"z_stress_{col}"] = z
    scaler_rows.append({
        "selection_type": "stress",
        "feature": col,
        "median": med,
        "q25": q25,
        "q75": q75,
        "iqr": iqr
    })

weekly_full["center_distance"] = np.sqrt(
    np.sum([weekly_full[f"z_center_{c}"]**2 for c in center_features], axis=0)
)

weekly_full["stress_score_upramp"] = np.sqrt(
    np.sum([weekly_full[f"z_stress_{c}"]**2 for c in stress_features], axis=0)
)

# rank
weekly_center_ranked = weekly_full.sort_values("center_distance", ascending=True).reset_index(drop=True)
weekly_stress_ranked = weekly_full.sort_values("stress_score_upramp", ascending=False).reset_index(drop=True)

center_week = weekly_center_ranked.iloc[0].copy()
stress_week = weekly_stress_ranked.iloc[0].copy()

center_start = center_week["week_start"]
center_end = center_start + pd.Timedelta(days=7)
stress_start = stress_week["week_start"]
stress_end = stress_start + pd.Timedelta(days=7)

print("\nChosen center week:")
print("  start:", center_start)
print("  end  :", center_end)

print("\nChosen stress week (upramp-based):")
print("  start:", stress_start)
print("  end  :", stress_end)

# -----------------------------
# Save supporting CSVs
# -----------------------------
weekly_out = weekly_full.copy()
weekly_out["week_end"] = weekly_out["week_start"] + pd.Timedelta(days=7)

all_scores_path = os.path.join(OUT_DIR, "all_full_weeks_selection_scores.csv")
weekly_out.to_csv(all_scores_path, index=False)

top10_center_path = os.path.join(OUT_DIR, "top10_center_weeks.csv")
weekly_center_ranked.head(10).to_csv(top10_center_path, index=False)

top10_stress_path = os.path.join(OUT_DIR, "top10_stress_weeks.csv")
weekly_stress_ranked.head(10).to_csv(top10_stress_path, index=False)

scalers_path = os.path.join(OUT_DIR, "week_selection_scalers.csv")
pd.DataFrame(scaler_rows).to_csv(scalers_path, index=False)

# -----------------------------
# center week summary
# -----------------------------
table5_cols = [
    "week_start",
    "AEF_mean",
    "SOFT_mean",
    "UPRAMP_mean",
    "SOFT_std",
    "UPRAMP_std",
    "UPRAMP_p90",
    "PRICE_mean",
    "PRICE_std",
    "share_above_annual_upramp_p90",
    "RES_energy_MWh",
    "D_H2_week",
    "D_NH3_week",
    "D_MeOH_week",
    "center_distance",
]

table5_df = pd.DataFrame([center_week[table5_cols]])
table5_df["week_end"] = table5_df["week_start"] + pd.Timedelta(days=7)
table5_df["selection_note"] = (
    "Representative week selected as the minimum robust multivariate distance "
    "across emissions signals, grid price, RES, and weekly demand features."
)

table5_path = os.path.join(OUT_DIR, "table5_center_week_summary.csv")
table5_df.to_csv(table5_path, index=False)

# -----------------------------
# stress week summary
# -----------------------------
tableS14_cols = [
    "week_start",
    "AEF_mean",
    "SOFT_mean",
    "UPRAMP_mean",
    "SOFT_std",
    "UPRAMP_std",
    "UPRAMP_p90",
    "PRICE_mean",
    "PRICE_std",
    "share_above_annual_upramp_p90",
    "RES_energy_MWh",
    "D_H2_week",
    "D_NH3_week",
    "D_MeOH_week",
    "stress_score_upramp",
]

tableS14_df = pd.DataFrame([stress_week[tableS14_cols]])
tableS14_df["week_end"] = tableS14_df["week_start"] + pd.Timedelta(days=7)
tableS14_df["selection_note"] = (
    "Stress week selected as the maximum upramp-MEF stress score using weekly "
    "upramp mean, upramp p90, and share of hours above the annual upramp p90 threshold. "
    "Grid price is reported for context but not used in the stress score."
)

tableS14_path = os.path.join(OUT_DIR, "tableS14_stress_week_summary.csv")
tableS14_df.to_csv(tableS14_path, index=False)


# -----------------------------
summary_txt_path = os.path.join(OUT_DIR, "chosen_weeks_summary.txt")
with open(summary_txt_path, "w", encoding="utf-8") as f:
    f.write("Chosen center week:\n")
    f.write(f"start = {center_start}\n")
    f.write(f"end   = {center_end}\n\n")
    f.write("Chosen stress week (upramp-based):\n")
    f.write(f"start = {stress_start}\n")
    f.write(f"end   = {stress_end}\n")

# -----------------------------
# Zip outputs
# -----------------------------
saved = [
    all_scores_path,
    top10_center_path,
    top10_stress_path,
    scalers_path,
    table5_path,
    tableS14_path,
    summary_txt_path,
]

zip_path = os.path.join(OUT_DIR, "week_selection_outputs.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for fp in saved:
        zf.write(fp, arcname=os.path.basename(fp))

print("\nSaved files:")
for fp in saved:
    print(" ", fp)

print("\nZIP created:", zip_path)

files.download(zip_path)

# 7. Ex-post sold product carbon intensity accounting

This is the productc level carbon propagation block used to calculate sold product CI and green-attribution outputs after optimization.


## 7.1 Sold product CI and green attribution


In [ ]:
# SOLD PRODUCT CARBON ACCOUNTING + GREEN ATTRIBUTION
#
# INPUT: put all files in /content
#   m0.csv
#   m2-aef0.5.csv
#   m2-soft mef0.5.csv
#   m2-upramp soft0.5.csv
#   grid_aef_hourly_2024.csv
#   grid_mef_soft_hourly_2024.csv
#   grid_mef_upramp_hourly_2024.csv
#
# OUTPUT: /content/product_carbon_accounting_outputs
#

# 1) Initial tank inventories 0
# 2) Electricity carbon is based on the exact reconstructed sink-level
#    grid-vs-battery split:
#       total unit power + saved green share + proportional battery discharge allocation
#    Battery-fed power carries zero electricity carbon.
# ============================================================

import json
import zipfile
from pathlib import Path
from matplotlib import colors
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -----------------------------
# USER PATHS
# -----------------------------
BASE_DIR = Path("/content")
OUT_ROOT = BASE_DIR / "product_carbon_accounting_outputs"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

DIR_TABLES   = OUT_ROOT / "tables"
DIR_FIG_MAIN = OUT_ROOT / "figures_main"
DIR_FIG_SI   = OUT_ROOT / "figures_SI"
DIR_INTER    = OUT_ROOT / "intermediate_csv"
DIR_MODEL    = OUT_ROOT / "model_level_outputs"

for d in [DIR_TABLES, DIR_FIG_MAIN, DIR_FIG_SI, DIR_INTER, DIR_MODEL]:
    d.mkdir(parents=True, exist_ok=True)

# -----------------------------
# INPUT FILES
# -----------------------------
MODEL_FILES = {
    "M0": BASE_DIR / "m0.csv",
    "M2-AEF0.5": BASE_DIR / "m2-aef0.5.csv",
    "M2-Soft MEF0.5": BASE_DIR / "m2-soft mef0.5.csv",
    "M2-UpRamp MEF0.5": BASE_DIR / "m2-upramp soft0.5.csv",
}

SIGNAL_FILES = {
    "AEF": (BASE_DIR / "grid_aef_hourly_2024.csv", "grid_aef_predicted_kg_per_kwh"),
    "Soft MEF": (BASE_DIR / "grid_mef_soft_hourly_2024.csv", "grid_mef_soft_kg_per_kwh"),
    "UpRamp MEF": (BASE_DIR / "grid_mef_upramp_hourly_2024.csv", "grid_mef_upramp_pred_kg_per_kwh"),
}

MODEL_ORDER  = ["M0", "M2-AEF0.5", "M2-Soft MEF0.5", "M2-UpRamp MEF0.5"]
SIGNAL_ORDER = ["AEF", "Soft MEF", "UpRamp MEF"]
PRODUCTS     = ["H2", "NH3", "MeOH"]

MATCHED_SIGNAL = {
    "M2-AEF0.5": "AEF",
    "M2-Soft MEF0.5": "Soft MEF",
    "M2-UpRamp MEF0.5": "UpRamp MEF",
}

SIGNAL_COLOR = {
    "AEF": "#0457ac",
    "Soft MEF": "#7fc6d4",
    "UpRamp MEF": "#a2596f",
}

MODEL_COLOR = {
    "M0": "#0457ac",
    "M2-AEF0.5": "#7fc6d4",
    "M2-Soft MEF0.5": "#a2596f",
    "M2-UpRamp MEF0.5": "#6b7280",
}

# -----------------------------
# STYLE
# -----------------------------
def apply_plot_settings(ax, title, xlabel, ylabel):
    ax.set_title(title, fontsize=14, fontweight="bold", pad=15)
    ax.set_xlabel(xlabel, fontsize=12, fontweight="bold", labelpad=10)
    ax.set_ylabel(ylabel, fontsize=12, fontweight="bold", labelpad=10)
    ax.tick_params(axis="both", which="major", labelsize=10)
    for spine in ax.spines.values():
        spine.set_color("#dadedf")
        spine.set_linewidth(2)
    ax.grid(False)

def savefig_png_svg(fig, out_noext, dpi=150):
    fig.savefig(str(out_noext) + ".png", dpi=dpi, bbox_inches="tight")
    fig.savefig(str(out_noext) + ".svg", bbox_inches="tight")
    plt.close(fig)


# -----------------------------
M_H2_TO_NH3  = 0.18
M_H2_TO_MEOH = 0.22
K_H2_OUT     = 0.12024
K_NH3_REC    = 0.11848
K_H2_FUEL    = 0.035294

DE_NH3  = 0.00519   # kgCO2 / kg NH3
DE_MEOH = 0.08      # kgCO2 / kg MeOH

TOL = 1e-6

# -----------------------------
# REQUIRED MODEL COLUMNS
# -----------------------------
REQUIRED_MODEL_COLS = [
    "model_tag", "datetime_utc", "day_of_year", "hour_in_day",

    "P_elz_mw", "P_nh3_mw", "P_meoh_mw", "P_crk_mw",

    "elz_power_green_share",
    "nh3_power_green_share",
    "meoh_power_green_share",
    "crk_power_green_share",

    "grid_import_mw",
    "res_used_mw",
    "bat_ch_mw",
    "bat_dis_mw",

    "H2_prod_g_kgph", "H2_prod_ng_kgph",
    "NH3_prod_g_kgph", "NH3_prod_ng_kgph",
    "MeOH_prod_g_kgph", "MeOH_prod_ng_kgph",

    "crk_nh3_in_kgph", "crk_h2_out_kgph", "crk_h2_fuel_kgph",

    "H2_sell_g_kgph", "H2_sell_ng_kgph",
    "NH3_sell_g_kgph", "NH3_sell_ng_kgph",
    "MeOH_sell_g_kgph", "MeOH_sell_ng_kgph",

    "tank_H2_g_kg", "tank_H2_ng_kg",
    "tank_NH3_g_kg", "tank_NH3_ng_kg",
    "tank_MeOH_g_kg", "tank_MeOH_ng_kg",
]

# -----------------------------
# HELPERS
# -----------------------------
def clip_small_negative(series, tol=TOL):
    x = series.copy()
    x[(x < 0) & (x > -tol)] = 0.0
    return x

def load_model_df(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing model file: {path}")
    df = pd.read_csv(path)
    missing = [c for c in REQUIRED_MODEL_COLS if c not in df.columns]
    if missing:
        raise KeyError(f"{path.name} missing columns: {missing}")
    df = df.copy()
    df["datetime_utc"] = pd.to_datetime(df["datetime_utc"], utc=True)
    df = df.sort_values("datetime_utc").reset_index(drop=True)
    return df

def load_signal_df(path: Path, ef_col: str) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing signal file: {path}")
    df = pd.read_csv(path)
    if "datetime_utc" not in df.columns or ef_col not in df.columns:
        raise KeyError(f"{path.name} must contain datetime_utc and {ef_col}")
    df = df[["datetime_utc", ef_col]].copy()
    df["datetime_utc"] = pd.to_datetime(df["datetime_utc"], utc=True)
    df = df.sort_values("datetime_utc").drop_duplicates(subset=["datetime_utc"])
    df = df.rename(columns={ef_col: "ef_kg_per_kwh"})
    return df

def reconstruct_unit_grid_battery_split(df: pd.DataFrame, tol=TOL):
    d = df.copy()

    d["P_elz_total_mw"] = d["P_elz_mw"]
    d["P_nh3_total_mw"] = d["P_nh3_mw"]
    d["P_meoh_total_mw"] = d["P_meoh_mw"]
    d["P_crk_total_mw"] = d["P_crk_mw"]

    for col in ["elz_power_green_share", "nh3_power_green_share", "meoh_power_green_share", "crk_power_green_share"]:
        d[col] = pd.to_numeric(d[col], errors="coerce").clip(lower=0.0, upper=1.0).fillna(0.0)

    d["P_elz_green_mw"] = d["P_elz_total_mw"] * d["elz_power_green_share"]
    d["P_nh3_green_mw"] = d["P_nh3_total_mw"] * d["nh3_power_green_share"]
    d["P_meoh_green_mw"] = d["P_meoh_total_mw"] * d["meoh_power_green_share"]
    d["P_crk_green_mw"] = d["P_crk_total_mw"] * d["crk_power_green_share"]

    d["P_elz_ng_mw"] = d["P_elz_total_mw"] - d["P_elz_green_mw"]
    d["P_nh3_ng_mw"] = d["P_nh3_total_mw"] - d["P_nh3_green_mw"]
    d["P_meoh_ng_mw"] = d["P_meoh_total_mw"] - d["P_meoh_green_mw"]
    d["P_crk_ng_mw"] = d["P_crk_total_mw"] - d["P_crk_green_mw"]

    for col in ["P_elz_ng_mw", "P_nh3_ng_mw", "P_meoh_ng_mw", "P_crk_ng_mw"]:
        d[col] = clip_small_negative(d[col], tol=tol)

    d["P_ng_sum_units_mw"] = (
        d["P_elz_ng_mw"] +
        d["P_nh3_ng_mw"] +
        d["P_meoh_ng_mw"] +
        d["P_crk_ng_mw"]
    )

    def alloc_bat(unit_ng, ng_sum, bat_dis):
        return np.where(
            (ng_sum > tol) & (bat_dis > tol),
            bat_dis * unit_ng / ng_sum,
            0.0
        )

    d["P_elz_bat_mw"] = alloc_bat(d["P_elz_ng_mw"], d["P_ng_sum_units_mw"], d["bat_dis_mw"])
    d["P_nh3_bat_mw"] = alloc_bat(d["P_nh3_ng_mw"], d["P_ng_sum_units_mw"], d["bat_dis_mw"])
    d["P_meoh_bat_mw"] = alloc_bat(d["P_meoh_ng_mw"], d["P_ng_sum_units_mw"], d["bat_dis_mw"])
    d["P_crk_bat_mw"] = alloc_bat(d["P_crk_ng_mw"], d["P_ng_sum_units_mw"], d["bat_dis_mw"])

    d["P_elz_grid_exact_mw"] = d["P_elz_ng_mw"] - d["P_elz_bat_mw"]
    d["P_nh3_grid_exact_mw"] = d["P_nh3_ng_mw"] - d["P_nh3_bat_mw"]
    d["P_meoh_grid_exact_mw"] = d["P_meoh_ng_mw"] - d["P_meoh_bat_mw"]
    d["P_crk_grid_exact_mw"] = d["P_crk_ng_mw"] - d["P_crk_bat_mw"]

    for col in ["P_elz_grid_exact_mw", "P_nh3_grid_exact_mw", "P_meoh_grid_exact_mw", "P_crk_grid_exact_mw"]:
        d[col] = clip_small_negative(d[col], tol=tol)

    d["grid_unit_sum_exact_mw"] = (
        d["P_elz_grid_exact_mw"] +
        d["P_nh3_grid_exact_mw"] +
        d["P_meoh_grid_exact_mw"] +
        d["P_crk_grid_exact_mw"]
    )
    d["bat_unit_sum_exact_mw"] = (
        d["P_elz_bat_mw"] +
        d["P_nh3_bat_mw"] +
        d["P_meoh_bat_mw"] +
        d["P_crk_bat_mw"]
    )

    d["grid_balance_gap_mw"] = d["grid_unit_sum_exact_mw"] - d["grid_import_mw"]
    d["bat_balance_gap_mw"] = d["bat_unit_sum_exact_mw"] - d["bat_dis_mw"]

    diag = pd.DataFrame({
        "metric": [
            "max_abs_grid_balance_gap_mw",
            "max_abs_battery_balance_gap_mw",
            "mean_abs_grid_balance_gap_mw",
            "mean_abs_battery_balance_gap_mw",
        ],
        "value": [
            float(d["grid_balance_gap_mw"].abs().max()),
            float(d["bat_balance_gap_mw"].abs().max()),
            float(d["grid_balance_gap_mw"].abs().mean()),
            float(d["bat_balance_gap_mw"].abs().mean()),
        ]
    })

    return d, diag

# -----------------------------
# STEP 1 — RECONSTRUCT INTERMEDIATES
#since I forgot to extract the cracker flows in optimization loops, i reconstrcuted them from other flows. you can extract directly and skip this part
# -----------------------------
def reconstruct_intermediates(df: pd.DataFrame):
    d = df.copy()

    for p in ["H2", "NH3", "MeOH"]:
        for tag in ["g", "ng"]:
            d[f"tank_{p}_{tag}_next_kg"] = d[f"tank_{p}_{tag}_kg"].shift(-1)
            d[f"delta_tank_{p}_{tag}_kg"] = d[f"tank_{p}_{tag}_next_kg"] - d[f"tank_{p}_{tag}_kg"]

    d["has_next_state"] = d["tank_H2_g_next_kg"].notna()

    d["H_to_NH3_g_kgph"]   = M_H2_TO_NH3  * d["NH3_prod_g_kgph"]
    d["H_to_NH3_ng_kgph"]  = M_H2_TO_NH3  * d["NH3_prod_ng_kgph"]
    d["H_to_MeOH_g_kgph"]  = M_H2_TO_MEOH * d["MeOH_prod_g_kgph"]
    d["H_to_MeOH_ng_kgph"] = M_H2_TO_MEOH * d["MeOH_prod_ng_kgph"]

    denom = 1.0 - K_NH3_REC
    d["Q_crk_in_g_recon_kgph"] = (
        d["NH3_prod_g_kgph"] - d["NH3_sell_g_kgph"] - d["delta_tank_NH3_g_kg"]
    ) / denom
    d["Q_crk_in_ng_recon_kgph"] = (
        d["NH3_prod_ng_kgph"] - d["NH3_sell_ng_kgph"] - d["delta_tank_NH3_ng_kg"]
    ) / denom

    d["Q_crk_in_g_recon_kgph"] = clip_small_negative(d["Q_crk_in_g_recon_kgph"])
    d["Q_crk_in_ng_recon_kgph"] = clip_small_negative(d["Q_crk_in_ng_recon_kgph"])

    d["Q_crk_in_recon_sum_kgph"] = d["Q_crk_in_g_recon_kgph"] + d["Q_crk_in_ng_recon_kgph"]
    d["Q_crk_total_gap_kgph"] = d["Q_crk_in_recon_sum_kgph"] - d["crk_nh3_in_kgph"]

    d["Q_crk_split_valid"] = (
        d["has_next_state"] &
        d["Q_crk_in_g_recon_kgph"].notna() &
        d["Q_crk_in_ng_recon_kgph"].notna() &
        (d["Q_crk_total_gap_kgph"].abs() <= 5e-4)
    )

    valid = d["Q_crk_split_valid"] & (d["Q_crk_in_recon_sum_kgph"] > TOL)
    scale = np.where(valid, d["crk_nh3_in_kgph"] / d["Q_crk_in_recon_sum_kgph"], 1.0)
    d.loc[valid, "Q_crk_in_g_recon_kgph"] *= scale[valid]
    d.loc[valid, "Q_crk_in_ng_recon_kgph"] *= scale[valid]

    d["NH3_rec_g_recon_kgph"] = K_NH3_REC * d["Q_crk_in_g_recon_kgph"]
    d["NH3_rec_ng_recon_kgph"] = K_NH3_REC * d["Q_crk_in_ng_recon_kgph"]

    d["H_crk_out_g_recon_kgph"] = K_H2_OUT * d["Q_crk_in_g_recon_kgph"]
    d["H_crk_out_ng_recon_kgph"] = K_H2_OUT * d["Q_crk_in_ng_recon_kgph"]

    d["H_crk_fuel_g_recon_kgph"] = K_H2_FUEL * d["Q_crk_in_g_recon_kgph"]
    d["H_crk_fuel_ng_recon_kgph"] = K_H2_FUEL * d["Q_crk_in_ng_recon_kgph"]

    d["H_crk_out_total_gap_kgph"] = (
        d["H_crk_out_g_recon_kgph"] + d["H_crk_out_ng_recon_kgph"] - d["crk_h2_out_kgph"]
    )
    d["H_crk_fuel_total_gap_kgph"] = (
        d["H_crk_fuel_g_recon_kgph"] + d["H_crk_fuel_ng_recon_kgph"] - d["crk_h2_fuel_kgph"]
    )

    diag = pd.DataFrame({
        "metric": [
            "n_rows_total",
            "n_rows_with_next_state",
            "n_rows_valid_cracker_split",
            "n_rows_dropped_for_split_reconstruction",
            "max_abs_cracker_nh3in_gap_kgph",
            "max_abs_cracker_h2out_gap_kgph",
            "max_abs_cracker_h2fuel_gap_kgph",
        ],
        "value": [
            int(len(d)),
            int(d["has_next_state"].sum()),
            int(d["Q_crk_split_valid"].sum()),
            int((~d["Q_crk_split_valid"]).sum()),
            float(d["Q_crk_total_gap_kgph"].abs().max()),
            float(d["H_crk_out_total_gap_kgph"].abs().max()),
            float(d["H_crk_fuel_total_gap_kgph"].abs().max()),
        ]
    })

    return d, diag

# -----------------------------
# INITIAL INVENTORY RULE
# -----------------------------
def derive_zero_initial_cis(recon_df: pd.DataFrame, signal_df: pd.DataFrame, signal_name: str):
    """
    Simplified assumption:
    all initial tank inventories at the start of the horizon are assigned zero carbon intensity.
    """
    return {
        "initial_inventory_rule_note": (
            "All initial tank inventories at the start of the study horizon are assigned zero carbon intensity."
        ),
        "initial_inventory_basis_signal": signal_name,
        "initial_inventory_basis_datetime_utc": "assumed_zero_initial_inventory",
        "initial_inventory_basis_day_of_year": 1,
        "initial_inventory_basis_hour_in_day": 0,
        "initial_inventory_basis_ef_kg_per_kwh": 0.0,
        "CI_init_H2_g_kgco2_perkg": 0.0,
        "CI_init_H2_ng_kgco2_perkg": 0.0,
        "CI_init_NH3_g_kgco2_perkg": 0.0,
        "CI_init_NH3_ng_kgco2_perkg": 0.0,
        "CI_init_MeOH_g_kgco2_perkg": 0.0,
        "CI_init_MeOH_ng_kgco2_perkg": 0.0,
    }
# -----------------------------
# STEP 2 — DYNAMIC CARBON LEDGER
# -----------------------------
def run_carbon_ledger(recon_df: pd.DataFrame, signal_name: str, signal_df: pd.DataFrame, init_ci_dict: dict):
    d = recon_df.merge(signal_df, on="datetime_utc", how="left").copy()
    if d["ef_kg_per_kwh"].isna().any():
        raise ValueError(f"Missing EF values after merge for {signal_name}")

    d["elec_carbon_elz_kgph"]  = 1000.0 * d["P_elz_grid_exact_mw"]  * d["ef_kg_per_kwh"]
    d["elec_carbon_nh3_kgph"]  = 1000.0 * d["P_nh3_grid_exact_mw"]  * d["ef_kg_per_kwh"]
    d["elec_carbon_meoh_kgph"] = 1000.0 * d["P_meoh_grid_exact_mw"] * d["ef_kg_per_kwh"]
    d["elec_carbon_crk_kgph"]  = 1000.0 * d["P_crk_grid_exact_mw"]  * d["ef_kg_per_kwh"]



    d["C_new_H2_g_kgco2"] = 0.0
    d["C_new_H2_ng_kgco2"] = d["elec_carbon_elz_kgph"]

    d["CI_new_H2_g_batch"] = np.where(d["H2_prod_g_kgph"] > TOL, d["C_new_H2_g_kgco2"] / d["H2_prod_g_kgph"], np.nan)
    d["CI_new_H2_ng_batch"] = np.where(d["H2_prod_ng_kgph"] > TOL, d["C_new_H2_ng_kgco2"] / d["H2_prod_ng_kgph"], np.nan)

    CH2g = float(d.loc[0, "tank_H2_g_kg"]   * init_ci_dict["CI_init_H2_g_kgco2_perkg"])
    CH2ng = float(d.loc[0, "tank_H2_ng_kg"] * init_ci_dict["CI_init_H2_ng_kgco2_perkg"])
    CNH3g = float(d.loc[0, "tank_NH3_g_kg"]   * init_ci_dict["CI_init_NH3_g_kgco2_perkg"])
    CNH3ng = float(d.loc[0, "tank_NH3_ng_kg"] * init_ci_dict["CI_init_NH3_ng_kgco2_perkg"])
    CMeOHg = float(d.loc[0, "tank_MeOH_g_kg"]   * init_ci_dict["CI_init_MeOH_g_kgco2_perkg"])
    CMeOHng = float(d.loc[0, "tank_MeOH_ng_kg"] * init_ci_dict["CI_init_MeOH_ng_kgco2_perkg"])

    rows = []

    for _, r in d.iterrows():
        ci_H2_g = CH2g / r["tank_H2_g_kg"] if r["tank_H2_g_kg"] > TOL else 0.0
        ci_H2_ng = CH2ng / r["tank_H2_ng_kg"] if r["tank_H2_ng_kg"] > TOL else 0.0

        ci_NH3_g = CNH3g / r["tank_NH3_g_kg"] if r["tank_NH3_g_kg"] > TOL else 0.0
        ci_NH3_ng = CNH3ng / r["tank_NH3_ng_kg"] if r["tank_NH3_ng_kg"] > TOL else 0.0

        ci_MeOH_g = CMeOHg / r["tank_MeOH_g_kg"] if r["tank_MeOH_g_kg"] > TOL else 0.0
        ci_MeOH_ng = CMeOHng / r["tank_MeOH_ng_kg"] if r["tank_MeOH_ng_kg"] > TOL else 0.0

        C_H2_to_NH3_g = ci_H2_g * r["H_to_NH3_g_kgph"]
        C_H2_to_NH3_ng = ci_H2_ng * r["H_to_NH3_ng_kgph"]
        C_H2_to_MeOH_g = ci_H2_g * r["H_to_MeOH_g_kgph"]
        C_H2_to_MeOH_ng = ci_H2_ng * r["H_to_MeOH_ng_kgph"]

        C_direct_NH3_g = DE_NH3 * r["NH3_prod_g_kgph"]
        C_direct_NH3_ng = DE_NH3 * r["NH3_prod_ng_kgph"]
        C_direct_MeOH_g = DE_MEOH * r["MeOH_prod_g_kgph"]
        C_direct_MeOH_ng = DE_MEOH * r["MeOH_prod_ng_kgph"]

        # Green downstream products inherit carbon from green H2 and direct process emissions only.
        # Grid-electricity carbon is assigned only to the non-green production stream.
        # Battery-fed non-green production remains non-green for attribution but receives zero grid CO2,
        # because elec_carbon_* is based only on reconstructed grid-fed unit electricity.

        C_new_NH3_g = (
            C_H2_to_NH3_g
            + C_direct_NH3_g
        )

        C_new_NH3_ng = (
            C_H2_to_NH3_ng
            + r["elec_carbon_nh3_kgph"]
            + C_direct_NH3_ng
        )

        C_new_MeOH_g = (
            C_H2_to_MeOH_g
            + C_direct_MeOH_g
        )

        C_new_MeOH_ng = (
            C_H2_to_MeOH_ng
            + r["elec_carbon_meoh_kgph"]
            + C_direct_MeOH_ng
        )

        if bool(r["Q_crk_split_valid"]):
            Qg = r["Q_crk_in_g_recon_kgph"]
            Qng = r["Q_crk_in_ng_recon_kgph"]

            NH3_rec_g = r["NH3_rec_g_recon_kgph"]
            NH3_rec_ng = r["NH3_rec_ng_recon_kgph"]

            H_out_g = r["H_crk_out_g_recon_kgph"]
            H_out_ng = r["H_crk_out_ng_recon_kgph"]

            H_fuel_g = r["H_crk_fuel_g_recon_kgph"]
            H_fuel_ng = r["H_crk_fuel_ng_recon_kgph"]

            # Cracker carbon propagation:
            # Incoming NH3 carries the current CI of the corresponding labeled NH3 tank.
            # Recovered NH3 returns with inherited NH3 carbon.
            # The remaining inherited carbon is assigned to recovered H2.
            # Cracker grid-electricity carbon is assigned only to the non-green recovered H2 stream.

            C_in_crk_NH3_g = ci_NH3_g * Qg
            C_in_crk_NH3_ng = ci_NH3_ng * Qng

            C_rec_NH3_g = ci_NH3_g * NH3_rec_g
            C_rec_NH3_ng = ci_NH3_ng * NH3_rec_ng

            C_H2_from_crk_g = C_in_crk_NH3_g - C_rec_NH3_g
            C_H2_from_crk_ng = C_in_crk_NH3_ng - C_rec_NH3_ng + r["elec_carbon_crk_kgph"]

            # Numerical safety for reconstructed cracker carbon flows.
            # Small negative values can occur from floating-point noise and reconstructed cracker splits.
            # Values below -1 kgCO2/h should be inspected because they may indicate a real reconstruction issue.

            CRK_CARBON_TOL = 1.0  # kgCO2/h

            if C_H2_from_crk_g < -CRK_CARBON_TOL:
                raise ValueError(
                    f"Large negative green cracker H2 carbon at {r['datetime_utc']}: {C_H2_from_crk_g}"
                )

            if C_H2_from_crk_ng < -CRK_CARBON_TOL:
                raise ValueError(
                    f"Large negative non-green cracker H2 carbon at {r['datetime_utc']}: {C_H2_from_crk_ng}"
                )

            C_H2_from_crk_g = max(C_H2_from_crk_g, 0.0)
            C_H2_from_crk_ng = max(C_H2_from_crk_ng, 0.0)
        else:
            Qg = Qng = 0.0
            H_fuel_g = H_fuel_ng = 0.0
            C_rec_NH3_g = C_rec_NH3_ng = 0.0
            C_H2_from_crk_g = C_H2_from_crk_ng = 0.0

        C_sell_H2_g = ci_H2_g * r["H2_sell_g_kgph"]
        C_sell_H2_ng = ci_H2_ng * r["H2_sell_ng_kgph"]

        C_sell_NH3_g = ci_NH3_g * r["NH3_sell_g_kgph"]
        C_sell_NH3_ng = ci_NH3_ng * r["NH3_sell_ng_kgph"]

        C_sell_MeOH_g = ci_MeOH_g * r["MeOH_sell_g_kgph"]
        C_sell_MeOH_ng = ci_MeOH_ng * r["MeOH_sell_ng_kgph"]

        def checked_nonnegative(x, name, timestamp, tol=1.0):
            """
            Clip small negative carbon stocks caused by numerical noise or reconstructed flows.
            Raise an error only for large negative values.
            tol is in kgCO2.
            """
            if x < -tol:
                raise ValueError(f"Large negative carbon stock in {name} at {timestamp}: {x}")
            return max(x, 0.0)

        CH2g_new = (
            CH2g
            - C_sell_H2_g
            - C_H2_to_NH3_g
            - C_H2_to_MeOH_g
            + r["C_new_H2_g_kgco2"]
            + C_H2_from_crk_g
            - ci_H2_g * H_fuel_g
        )

        CH2ng_new = (
            CH2ng
            - C_sell_H2_ng
            - C_H2_to_NH3_ng
            - C_H2_to_MeOH_ng
            + r["C_new_H2_ng_kgco2"]
            + C_H2_from_crk_ng
            - ci_H2_ng * H_fuel_ng
        )

        CNH3g_new = (
            CNH3g
            - C_sell_NH3_g
            - ci_NH3_g * Qg
            + C_new_NH3_g
            + C_rec_NH3_g
        )

        CNH3ng_new = (
            CNH3ng
            - C_sell_NH3_ng
            - ci_NH3_ng * Qng
            + C_new_NH3_ng
            + C_rec_NH3_ng
        )

        CMeOHg_new = (
            CMeOHg
            - C_sell_MeOH_g
            + C_new_MeOH_g
        )

        CMeOHng_new = (
            CMeOHng
            - C_sell_MeOH_ng
            + C_new_MeOH_ng
        )

        CH2g = checked_nonnegative(CH2g_new, "H2_g", r["datetime_utc"])
        CH2ng = checked_nonnegative(CH2ng_new, "H2_ng", r["datetime_utc"])
        CNH3g = checked_nonnegative(CNH3g_new, "NH3_g", r["datetime_utc"])
        CNH3ng = checked_nonnegative(CNH3ng_new, "NH3_ng", r["datetime_utc"])
        CMeOHg = checked_nonnegative(CMeOHg_new, "MeOH_g", r["datetime_utc"])
        CMeOHng = checked_nonnegative(CMeOHng_new, "MeOH_ng", r["datetime_utc"])

        rows.append({
            "datetime_utc": r["datetime_utc"],
            "day_of_year": int(r["day_of_year"]),
            "hour_in_day": int(r["hour_in_day"]),
            "CI_tank_H2_g": ci_H2_g,
            "CI_tank_H2_ng": ci_H2_ng,
            "CI_tank_NH3_g": ci_NH3_g,
            "CI_tank_NH3_ng": ci_NH3_ng,
            "CI_tank_MeOH_g": ci_MeOH_g,
            "CI_tank_MeOH_ng": ci_MeOH_ng,
            "C_sell_H2_g_kgco2": C_sell_H2_g,
            "C_sell_H2_ng_kgco2": C_sell_H2_ng,
            "C_sell_NH3_g_kgco2": C_sell_NH3_g,
            "C_sell_NH3_ng_kgco2": C_sell_NH3_ng,
            "C_sell_MeOH_g_kgco2": C_sell_MeOH_g,
            "C_sell_MeOH_ng_kgco2": C_sell_MeOH_ng,
            "H2_sold_total_kgph": float(r["H2_sell_g_kgph"] + r["H2_sell_ng_kgph"]),
            "NH3_sold_total_kgph": float(r["NH3_sell_g_kgph"] + r["NH3_sell_ng_kgph"]),
            "MeOH_sold_total_kgph": float(r["MeOH_sell_g_kgph"] + r["MeOH_sell_ng_kgph"]),
            "H2_sold_CI_kgco2_perkg_hourly": float((C_sell_H2_g + C_sell_H2_ng) / max(r["H2_sell_g_kgph"] + r["H2_sell_ng_kgph"], TOL)) if (r["H2_sell_g_kgph"] + r["H2_sell_ng_kgph"]) > TOL else np.nan,
            "NH3_sold_CI_kgco2_perkg_hourly": float((C_sell_NH3_g + C_sell_NH3_ng) / max(r["NH3_sell_g_kgph"] + r["NH3_sell_ng_kgph"], TOL)) if (r["NH3_sell_g_kgph"] + r["NH3_sell_ng_kgph"]) > TOL else np.nan,
            "MeOH_sold_CI_kgco2_perkg_hourly": float((C_sell_MeOH_g + C_sell_MeOH_ng) / max(r["MeOH_sell_g_kgph"] + r["MeOH_sell_ng_kgph"], TOL)) if (r["MeOH_sell_g_kgph"] + r["MeOH_sell_ng_kgph"]) > TOL else np.nan,
            "elec_carbon_elz_kgph": r["elec_carbon_elz_kgph"],
            "elec_carbon_nh3_kgph": r["elec_carbon_nh3_kgph"],
            "elec_carbon_meoh_kgph": r["elec_carbon_meoh_kgph"],
            "elec_carbon_crk_kgph": r["elec_carbon_crk_kgph"],
            "Q_crk_split_valid": bool(r["Q_crk_split_valid"]),
        })

    hourly = pd.DataFrame(rows)


    if len(hourly) >= 2:
        last_idx = hourly.index[-1]
        prev_idx = hourly.index[-2]
        fill_cols = [
            "CI_tank_H2_g", "CI_tank_H2_ng",
            "CI_tank_NH3_g", "CI_tank_NH3_ng",
            "CI_tank_MeOH_g", "CI_tank_MeOH_ng",
            "C_sell_H2_g_kgco2", "C_sell_H2_ng_kgco2",
            "C_sell_NH3_g_kgco2", "C_sell_NH3_ng_kgco2",
            "C_sell_MeOH_g_kgco2", "C_sell_MeOH_ng_kgco2",
        ]
        hourly.loc[last_idx, fill_cols] = hourly.loc[last_idx, fill_cols].where(hourly.loc[last_idx, fill_cols].notna(), hourly.loc[prev_idx, fill_cols])

    hourly["date"] = pd.to_datetime(hourly["datetime_utc"]).dt.date

    d2 = d.merge(
        hourly[[
            "datetime_utc",
            "CI_tank_H2_g", "CI_tank_H2_ng",
            "CI_tank_NH3_g", "CI_tank_NH3_ng",
            "CI_tank_MeOH_g", "CI_tank_MeOH_ng",
            "C_sell_H2_g_kgco2", "C_sell_H2_ng_kgco2",
            "C_sell_NH3_g_kgco2", "C_sell_NH3_ng_kgco2",
            "C_sell_MeOH_g_kgco2", "C_sell_MeOH_ng_kgco2"
        ]],
        on="datetime_utc",
        how="left"
    )

    daily = d2.groupby("day_of_year", as_index=False).agg(
        date=("datetime_utc", lambda x: str(pd.to_datetime(x.iloc[0]).date())),

        H2_sold_g_kg=("H2_sell_g_kgph", "sum"),
        H2_sold_ng_kg=("H2_sell_ng_kgph", "sum"),
        H2_carbon_g_kg=("C_sell_H2_g_kgco2", "sum"),
        H2_carbon_ng_kg=("C_sell_H2_ng_kgco2", "sum"),

        NH3_sold_g_kg=("NH3_sell_g_kgph", "sum"),
        NH3_sold_ng_kg=("NH3_sell_ng_kgph", "sum"),
        NH3_carbon_g_kg=("C_sell_NH3_g_kgco2", "sum"),
        NH3_carbon_ng_kg=("C_sell_NH3_ng_kgco2", "sum"),

        MeOH_sold_g_kg=("MeOH_sell_g_kgph", "sum"),
        MeOH_sold_ng_kg=("MeOH_sell_ng_kgph", "sum"),
        MeOH_carbon_g_kg=("C_sell_MeOH_g_kgco2", "sum"),
        MeOH_carbon_ng_kg=("C_sell_MeOH_ng_kgco2", "sum"),
    )

    for p in PRODUCTS:
        daily[f"{p}_sold_total_kg"] = daily[f"{p}_sold_g_kg"] + daily[f"{p}_sold_ng_kg"]
        daily[f"{p}_sold_green_share_pct"] = np.where(
            daily[f"{p}_sold_total_kg"] > TOL,
            100.0 * daily[f"{p}_sold_g_kg"] / daily[f"{p}_sold_total_kg"],
            np.nan
        )
        daily[f"{p}_sold_CI_kgco2_perkg"] = np.where(
            daily[f"{p}_sold_total_kg"] > TOL,
            (daily[f"{p}_carbon_g_kg"] + daily[f"{p}_carbon_ng_kg"]) / daily[f"{p}_sold_total_kg"],
            np.nan
        )

    summary = pd.DataFrame([{
        "initial_inventory_method_note": init_ci_dict["initial_inventory_rule_note"],
        "initial_inventory_basis_signal": init_ci_dict["initial_inventory_basis_signal"],
        "initial_inventory_basis_ef_kg_per_kwh": init_ci_dict["initial_inventory_basis_ef_kg_per_kwh"],
        "initial_inventory_basis_datetime_utc": init_ci_dict["initial_inventory_basis_datetime_utc"],
        "initial_inventory_basis_day_of_year": init_ci_dict["initial_inventory_basis_day_of_year"],
        "initial_inventory_basis_hour_in_day": init_ci_dict["initial_inventory_basis_hour_in_day"],

        "CI_init_H2_g_kgco2_perkg": init_ci_dict["CI_init_H2_g_kgco2_perkg"],
        "CI_init_H2_ng_kgco2_perkg": init_ci_dict["CI_init_H2_ng_kgco2_perkg"],
        "CI_init_NH3_g_kgco2_perkg": init_ci_dict["CI_init_NH3_g_kgco2_perkg"],
        "CI_init_NH3_ng_kgco2_perkg": init_ci_dict["CI_init_NH3_ng_kgco2_perkg"],
        "CI_init_MeOH_g_kgco2_perkg": init_ci_dict["CI_init_MeOH_g_kgco2_perkg"],
        "CI_init_MeOH_ng_kgco2_perkg": init_ci_dict["CI_init_MeOH_ng_kgco2_perkg"],

        "H2_sold_green_share_pct": float(100.0 * daily["H2_sold_g_kg"].sum() / max(daily["H2_sold_total_kg"].sum(), TOL)),
        "NH3_sold_green_share_pct": float(100.0 * daily["NH3_sold_g_kg"].sum() / max(daily["NH3_sold_total_kg"].sum(), TOL)),
        "MeOH_sold_green_share_pct": float(100.0 * daily["MeOH_sold_g_kg"].sum() / max(daily["MeOH_sold_total_kg"].sum(), TOL)),

        "H2_sold_CI_kgco2_perkg": float((daily["H2_carbon_g_kg"].sum() + daily["H2_carbon_ng_kg"].sum()) / max(daily["H2_sold_total_kg"].sum(), TOL)),
        "NH3_sold_CI_kgco2_perkg": float((daily["NH3_carbon_g_kg"].sum() + daily["NH3_carbon_ng_kg"].sum()) / max(daily["NH3_sold_total_kg"].sum(), TOL)),
        "MeOH_sold_CI_kgco2_perkg": float((daily["MeOH_carbon_g_kg"].sum() + daily["MeOH_carbon_ng_kg"].sum()) / max(daily["MeOH_sold_total_kg"].sum(), TOL)),

        "hours_excluded_from_propagation_due_to_missing_next_state": int((~d["has_next_state"]).sum()),
        "hours_excluded_from_cracker_split_due_to_invalid_reconstruction": int((~d["Q_crk_split_valid"]).sum()),
        "electricity_split_method_note": (
            "Exact sink-level grid power is reconstructed from total unit power, saved green share, "
            "and proportional allocation of battery discharge across non-green unit demand. "
            "Battery-fed power carries zero electricity carbon."
        ),
    }])

    return d2, hourly, daily, summary

# -----------------------------
# LOAD SIGNALS
# -----------------------------
signal_dfs = {name: load_signal_df(path, col) for name, (path, col) in SIGNAL_FILES.items()}

# -----------------------------
# RUN ALL MODELS
# -----------------------------
annual_green_rows = []
annual_ci_rows = []
all_daily_ci = []
all_hourly_ci = []
all_recon_diag = []

for model_name in MODEL_ORDER:
    model_df = load_model_df(MODEL_FILES[model_name])

    recon_power_df, diag_power = reconstruct_unit_grid_battery_split(model_df)
    recon_df, diag_crk = reconstruct_intermediates(recon_power_df)


    model_dir = DIR_MODEL / model_name.replace(" ", "_")
    model_dir.mkdir(parents=True, exist_ok=True)

    diag_power2 = diag_power.copy()
    diag_power2.insert(0, "Model", model_name)
    diag_power2.insert(1, "Diagnostic block", "power_split")
    all_recon_diag.append(diag_power2)

    diag_crk2 = diag_crk.copy()
    diag_crk2.insert(0, "Model", model_name)
    diag_crk2.insert(1, "Diagnostic block", "cracker_split")
    all_recon_diag.append(diag_crk2)

    recon_df.to_csv(model_dir / "01_reconstructed_intermediates.csv", index=False)

    for signal_name in SIGNAL_ORDER:
        signal_df = signal_dfs[signal_name]
        init_ci_dict = derive_zero_initial_cis(recon_df, signal_df, signal_name)
        merged, hourly_ci, daily_ci, summary = run_carbon_ledger(recon_df, signal_name, signal_df, init_ci_dict)

        tag = signal_name.lower().replace(" ", "_")

        merged.to_csv(model_dir / f"02_merged_with_signal_{tag}.csv", index=False)
        hourly_ci.to_csv(model_dir / f"03_hourly_carbon_ledger_{tag}.csv", index=False)
        daily_ci.to_csv(model_dir / f"04_daily_product_CI_{tag}.csv", index=False)
        summary.to_csv(model_dir / f"05_annual_product_CI_summary_{tag}.csv", index=False)

        annual_green_rows.append({
            "Model": model_name,
            "Evaluation signal": signal_name,
            "H2 sold green share (%)": float(summary["H2_sold_green_share_pct"].iloc[0]),
            "NH3 sold green share (%)": float(summary["NH3_sold_green_share_pct"].iloc[0]),
            "MeOH sold green share (%)": float(summary["MeOH_sold_green_share_pct"].iloc[0]),
        })

        annual_ci_rows.append({
            "Model": model_name,
            "Evaluation signal": signal_name,
            "H2 sold CI (kgCO2/kg H2)": float(summary["H2_sold_CI_kgco2_perkg"].iloc[0]),
            "NH3 sold CI (kgCO2/kg NH3)": float(summary["NH3_sold_CI_kgco2_perkg"].iloc[0]),
            "MeOH sold CI (kgCO2/kg MeOH)": float(summary["MeOH_sold_CI_kgco2_perkg"].iloc[0]),
            "Hours excluded (missing next-state)": int(summary["hours_excluded_from_propagation_due_to_missing_next_state"].iloc[0]),
            "Hours excluded (invalid cracker split)": int(summary["hours_excluded_from_cracker_split_due_to_invalid_reconstruction"].iloc[0]),
            "Initialization note": summary["initial_inventory_method_note"].iloc[0],
        })

        daily_ci2 = daily_ci.copy()
        daily_ci2.insert(0, "Model", model_name)
        daily_ci2.insert(1, "Evaluation signal", signal_name)
        all_daily_ci.append(daily_ci2)

        hourly_ci2 = hourly_ci.copy()
        hourly_ci2.insert(0, "Model", model_name)
        hourly_ci2.insert(1, "Evaluation signal", signal_name)
        all_hourly_ci.append(hourly_ci2)

# -----------------------------
# AGGREGATE TABLE FRAMES
# -----------------------------
annual_green_df = pd.DataFrame(annual_green_rows)
annual_green_df["Model"] = pd.Categorical(annual_green_df["Model"], categories=MODEL_ORDER, ordered=True)
annual_green_df["Evaluation signal"] = pd.Categorical(annual_green_df["Evaluation signal"], categories=SIGNAL_ORDER, ordered=True)
annual_green_df = annual_green_df.sort_values(["Model", "Evaluation signal"]).reset_index(drop=True)

annual_ci_df = pd.DataFrame(annual_ci_rows)
annual_ci_df["Model"] = pd.Categorical(annual_ci_df["Model"], categories=MODEL_ORDER, ordered=True)
annual_ci_df["Evaluation signal"] = pd.Categorical(annual_ci_df["Evaluation signal"], categories=SIGNAL_ORDER, ordered=True)
annual_ci_df = annual_ci_df.sort_values(["Model", "Evaluation signal"]).reset_index(drop=True)

daily_all = pd.concat(all_daily_ci, ignore_index=True)
daily_all["Model"] = pd.Categorical(daily_all["Model"], categories=MODEL_ORDER, ordered=True)
daily_all["Evaluation signal"] = pd.Categorical(daily_all["Evaluation signal"], categories=SIGNAL_ORDER, ordered=True)

hourly_all = pd.concat(all_hourly_ci, ignore_index=True)
hourly_all["Model"] = pd.Categorical(hourly_all["Model"], categories=MODEL_ORDER, ordered=True)
hourly_all["Evaluation signal"] = pd.Categorical(hourly_all["Evaluation signal"], categories=SIGNAL_ORDER, ordered=True)

recon_diag_all = pd.concat(all_recon_diag, ignore_index=True)

# -----------------------------
# MAIN-TEXT TABLES
# -----------------------------
# -----------------------------
# MATCHED-SIGNAL SUBSETS
# -----------------------------
matched_m2_ci = annual_ci_df[
    annual_ci_df.apply(
        lambda r: (str(r["Model"]) in MATCHED_SIGNAL) and
                  (MATCHED_SIGNAL[str(r["Model"])] == str(r["Evaluation signal"])),
        axis=1
    )
].copy()

# sold green share is operational and does not depend on evaluation signal in the same way as CI;

matched_green = annual_green_df[
    annual_green_df.apply(
        lambda r: (str(r["Model"]) == "M0" and str(r["Evaluation signal"]) == "AEF") or
                  ((str(r["Model"]) in MATCHED_SIGNAL) and
                   (MATCHED_SIGNAL[str(r["Model"])] == str(r["Evaluation signal"]))),
        axis=1
    )
].copy()

# Use all M0 rows across all evaluation signals as the baseline map
m0_signal_map = annual_ci_df[annual_ci_df["Model"].eq("M0")].set_index("Evaluation signal")


# Use CI reduction relative to matched M0
table_6_rows = []

m0_signal_map = annual_ci_df[annual_ci_df["Model"].eq("M0")].set_index("Evaluation signal")

for _, r in matched_m2_ci.iterrows():
    signal_name = str(r["Evaluation signal"])

    if signal_name not in m0_signal_map.index:
        raise KeyError(f"M0 baseline missing for evaluation signal: {signal_name}")

    base_row = m0_signal_map.loc[signal_name]

    row = {
        "Suggested table title": "Annual sold-product carbon intensity for matched-signal cases, with CI reduction relative to the matched M0 baseline.",
        "Model": str(r["Model"]),
        "Matched evaluation signal": signal_name,
        "H2 sold CI (kgCO2/kg H2)": float(r["H2 sold CI (kgCO2/kg H2)"]),
        "NH3 sold CI (kgCO2/kg NH3)": float(r["NH3 sold CI (kgCO2/kg NH3)"]),
        "MeOH sold CI (kgCO2/kg MeOH)": float(r["MeOH sold CI (kgCO2/kg MeOH)"]),
    }

    for prod, col in [
        ("H2", "H2 sold CI (kgCO2/kg H2)"),
        ("NH3", "NH3 sold CI (kgCO2/kg NH3)"),
        ("MeOH", "MeOH sold CI (kgCO2/kg MeOH)")
    ]:
        baseline = float(base_row[col])
        row[f"{prod} CI reduction vs matched M0 (%)"] = (
            100.0 * (baseline - float(r[col])) / baseline if baseline > TOL else np.nan
        )

    table_6_rows.append(row)

table_6 = pd.DataFrame(table_6_rows)
table_6.to_csv(DIR_TABLES / "Table_6_annual_sold_product_CI_matched_signal_with_reduction_vs_M0.csv", index=False)

# -----------------------------
# SI TABLES
# the table/figure numbers are not same as the paper.
# -----------------------------
# Table S18: sold green share
table_s18 = matched_green.copy()
table_s18.insert(0, "Suggested table title", "Annual sold green-attributed share for the matched-signal cases.")
table_s18.to_csv(DIR_TABLES / "Table_S18_annual_sold_green_share_matched_signal.csv", index=False)

# Table S19: all model / all signal CI normalized by the M0 baseline within each evaluation signal.
table_s19 = annual_ci_df.copy()
m0_all_signal_map = annual_ci_df[annual_ci_df["Model"].eq("M0")].set_index("Evaluation signal")
for prod, col in [("H2", "H2 sold CI (kgCO2/kg H2)"), ("NH3", "NH3 sold CI (kgCO2/kg NH3)"), ("MeOH", "MeOH sold CI (kgCO2/kg MeOH)")]:
    base_map = m0_all_signal_map[col].to_dict()
    table_s19[f"{prod} relative CI vs M0 within signal"] = table_s19.apply(
        lambda r: float(r[col]) / float(base_map[str(r["Evaluation signal"])]) if str(r["Evaluation signal"]) in base_map and float(base_map[str(r["Evaluation signal"])]) > TOL else np.nan,
        axis=1
    )
table_s19.insert(0, "Suggested table title", "Annual sold-product carbon intensity normalized by the M0 baseline within each evaluation signal.")
table_s19.to_csv(DIR_TABLES / "Table_S19_relative_annual_sold_product_CI_vs_M0_within_signal.csv", index=False)



# -----------------------------
# SAVE INTERMEDIATE CSVs
# -----------------------------
daily_all.to_csv(DIR_INTER / "all_models_all_signals_daily_product_CI.csv", index=False)
hourly_all.to_csv(DIR_INTER / "all_models_all_signals_hourly_carbon_ledger.csv", index=False)
recon_diag_all.to_csv(DIR_INTER / "all_models_reconstruction_diagnostics.csv", index=False)

# -----------------------------
# FIGURE 11 — Annual sold product CI reduction relative to matched M0
# signal-centered main figure with positive improvement bars
# -----------------------------
fig, axes = plt.subplots(1, 3, figsize=(15.5, 4.9), dpi=150, sharey=True)

signal_to_model = {
    "AEF": "M2-AEF0.5",
    "Soft MEF": "M2-Soft MEF0.5",
    "UpRamp MEF": "M2-UpRamp MEF0.5",
}

fig_11_rows = []
for ax, signal_name in zip(axes, SIGNAL_ORDER):
    model_name = signal_to_model[signal_name]
    sub = table_6[(table_6["Model"].eq(model_name)) & (table_6["Matched evaluation signal"].eq(signal_name))].iloc[0]

    yvals = [
        float(sub["H2 CI reduction vs matched M0 (%)"]),
        float(sub["NH3 CI reduction vs matched M0 (%)"]),
        float(sub["MeOH CI reduction vs matched M0 (%)"]),
    ]
    x = np.arange(len(PRODUCTS))
    ax.bar(x, yvals, color=SIGNAL_COLOR[signal_name])
    ax.axhline(0.0, linewidth=1.5, color="#444444")
    ax.set_xticks(x)
    ax.set_xticklabels(["H2", "NH3", "MeOH"])
    apply_plot_settings(
        ax,
        f"{signal_name}: matched M2 vs M0",
        "Product",
        "Annual sold-product CI reduction vs matched M0 (%)"
    )
    ymin = min(0.0, min(yvals) * 1.18)
    ymax = max(5.0, max(yvals) * 1.18)
    ax.set_ylim(ymin, ymax)

    for i, y in enumerate(yvals):
        offset = 0.02 * (ymax - ymin)
        if np.isfinite(y):
            va = "bottom" if y >= 0 else "top"
            y_text = y + offset if y >= 0 else y - offset
            ax.text(i, y_text, f"{y:.1f}", ha="center", va=va, fontsize=9)

    fig_11_rows.append({
        "Suggested figure title": "Annual sold-product carbon-intensity reduction for matched-signal M2 cases relative to the matched M0 baseline.",
        "Signal": signal_name,
        "Model": model_name,
        "H2 CI reduction vs matched M0 (%)": yvals[0],
        "NH3 CI reduction vs matched M0 (%)": yvals[1],
        "MeOH CI reduction vs matched M0 (%)": yvals[2],
    })

plt.tight_layout()
savefig_png_svg(fig, DIR_FIG_MAIN / "Figure_11_annual_sold_product_CI_reduction_matched_signal_vs_M0")
pd.DataFrame(fig_11_rows).to_csv(DIR_FIG_MAIN / "Figure_11_source_data.csv", index=False)

# -----------------------------
# -----------------------------
# FIGURE 12 — Day x hour normalized sold-product CI heatmap
# rows = products, cols = matched-signal M2 cases
# each cell = hourly sold-product CI normalized by annual M0 CI under the same evaluation signal
# shared color scale across all panels + shared nonlinear normalization
# -----------------------------
fig, axes = plt.subplots(
    3, 3,
    figsize=(16.2, 11.6),
    dpi=150,
    sharex=True,
    sharey=True,
    constrained_layout=True
)

heatmap_models = {
    "AEF": "M2-AEF0.5",
    "Soft MEF": "M2-Soft MEF0.5",
    "UpRamp MEF": "M2-UpRamp MEF0.5",
}
heatmap_hourly_col = {
    "H2": "H2_sold_CI_kgco2_perkg_hourly",
    "NH3": "NH3_sold_CI_kgco2_perkg_hourly",
    "MeOH": "MeOH_sold_CI_kgco2_perkg_hourly",
}
heatmap_annual_col = {
    "H2": "H2 sold CI (kgCO2/kg H2)",
    "NH3": "NH3 sold CI (kgCO2/kg NH3)",
    "MeOH": "MeOH sold CI (kgCO2/kg MeOH)",
}

# ---- first pass: build all matrices and get one shared robust color scale
fig12_panel_data = {}
fig12_all_vals = []

for prod in PRODUCTS:
    for signal_name in SIGNAL_ORDER:
        model_name = heatmap_models[signal_name]
        base_row = annual_ci_df[
            (annual_ci_df["Model"].eq("M0")) &
            (annual_ci_df["Evaluation signal"].eq(signal_name))
        ].iloc[0]
        baseline = float(base_row[heatmap_annual_col[prod]])

        sub = hourly_all[
            (hourly_all["Model"].eq(model_name)) &
            (hourly_all["Evaluation signal"].eq(signal_name))
        ].copy()

        sub["norm_hourly_ci"] = sub[heatmap_hourly_col[prod]] / baseline if baseline > TOL else np.nan
        mat = sub.pivot(index="day_of_year", columns="hour_in_day", values="norm_hourly_ci")

        day_index = np.arange(1, int(sub["day_of_year"].max()) + 1)
        hour_index = np.arange(0, 24)
        mat = mat.reindex(index=day_index, columns=hour_index)

        arr = mat.values.astype(float)
        fig12_panel_data[(prod, signal_name)] = (arr, baseline, model_name)

        finite = arr[np.isfinite(arr)]
        if finite.size > 0:
            fig12_all_vals.append(finite)

if len(fig12_all_vals) > 0:
    fig12_all_vals = np.concatenate(fig12_all_vals)
    fig12_vmax = float(np.nanquantile(fig12_all_vals, 0.98))
    fig12_vmax = max(fig12_vmax, 1.0)
else:
    fig12_vmax = 1.0

# shared nonlinear normalization
# gamma < 1 expands low-mid range visually while preserving shared comparability
fig12_norm = colors.PowerNorm(gamma=0.65, vmin=0.0, vmax=fig12_vmax)

fig_g3_source_rows = []
im = None

# ---- second pass: plot all panels with the same scale and same normalization
for i, prod in enumerate(PRODUCTS):
    for j, signal_name in enumerate(SIGNAL_ORDER):
        ax = axes[i, j]
        arr, baseline, model_name = fig12_panel_data[(prod, signal_name)]

        im = ax.imshow(
            arr,
            aspect="auto",
            origin="lower",
            norm=fig12_norm,
            cmap="viridis"
        )

        ax.set_xticks([0, 5, 11, 17, 23])
        ax.set_xticklabels([0, 5, 11, 17, 23])
        ax.set_yticks([0, 90, 181, 273, 365])
        ax.set_yticklabels([1, 91, 182, 274, 366])

        apply_plot_settings(
            ax,
            f"{prod} | {signal_name}",
            "Hour of day",
            "Day of year"
        )

        fig_g3_source_rows.append({
            "Suggested figure title": "Day-by-hour heatmaps of hourly sold-product carbon intensity normalized by the annual matched M0 carbon intensity under the same evaluation signal.",
            "Product": prod,
            "Signal": signal_name,
            "Model": model_name,
            "Normalization baseline model": "M0",
            "Normalization baseline signal": signal_name,
            "Normalization baseline annual CI": baseline,
            "Shared colorbar vmax": fig12_vmax,
            "Shared normalization": "PowerNorm(gamma=0.65)",
        })

# one shared colorbar for the whole figure
cbar = fig.colorbar(im, ax=axes.ravel().tolist(), fraction=0.02, pad=0.02)
cbar.ax.tick_params(labelsize=9)
cbar.set_label("Hourly sold CI / annual M0 CI", fontsize=10, fontweight="bold")


savefig_png_svg(fig, DIR_FIG_MAIN / "Figure_12_day_hour_normalized_sold_product_CI_heatmap")
pd.DataFrame(fig_g3_source_rows).to_csv(DIR_FIG_MAIN / "Figure_12_source_data.csv", index=False)

# -----------------------------
# -----------------------------
# FIGURE 13 — Hour-of-day median CI reduction heatmap
# rows = products, cols = matched-signal M2 cases
# each cell = median across days of hourly CI reduction relative to matched M0 at the same hour
# shared symmetric color scale across all panels
# -----------------------------
fig, axes = plt.subplots(
    3, 3,
    figsize=(16.2, 11.6),
    dpi=150,
    sharex=True,
    sharey=True,
    constrained_layout=True
)

fig_13_rows = []

# ---- first pass: build all panel matrices and get one shared symmetric range
fig13_panel_data = {}
fig13_all_abs_vals = []

for prod in PRODUCTS:
    m2_col = {
        "H2": "H2_sold_CI_kgco2_perkg_hourly",
        "NH3": "NH3_sold_CI_kgco2_perkg_hourly",
        "MeOH": "MeOH_sold_CI_kgco2_perkg_hourly",
    }[prod]

    for signal_name in SIGNAL_ORDER:
        model_name = signal_to_model[signal_name]

        m2_sub = hourly_all[
            (hourly_all["Model"].eq(model_name)) &
            (hourly_all["Evaluation signal"].eq(signal_name))
        ][["datetime_utc", "hour_in_day", m2_col]].copy()

        m0_sub = hourly_all[
            (hourly_all["Model"].eq("M0")) &
            (hourly_all["Evaluation signal"].eq(signal_name))
        ][["datetime_utc", "hour_in_day", m2_col]].copy()

        merged = m2_sub.merge(m0_sub, on=["datetime_utc", "hour_in_day"], suffixes=("_m2", "_m0"))
        merged["ci_reduction_pct"] = np.where(
            merged[f"{m2_col}_m0"] > TOL,
            100.0 * (merged[f"{m2_col}_m0"] - merged[f"{m2_col}_m2"]) / merged[f"{m2_col}_m0"],
            np.nan
        )

        median_by_hour = merged.groupby("hour_in_day", observed=False)["ci_reduction_pct"].median().reindex(range(24))
        mat = np.array([median_by_hour.values.astype(float)])

        fig13_panel_data[(prod, signal_name)] = (mat, median_by_hour, model_name)

        finite = mat[np.isfinite(mat)]
        if finite.size > 0:
            fig13_all_abs_vals.append(np.abs(finite))

if len(fig13_all_abs_vals) > 0:
    fig13_all_abs_vals = np.concatenate(fig13_all_abs_vals)
    fig13_vabs = float(np.nanquantile(fig13_all_abs_vals, 0.98))   # robust shared symmetric bound
    fig13_vabs = max(fig13_vabs, 1.0)
else:
    fig13_vabs = 1.0

im = None

# ---- second pass: plot all panels with the same symmetric scale
for i, prod in enumerate(PRODUCTS):
    for j, signal_name in enumerate(SIGNAL_ORDER):
        ax = axes[i, j]
        mat, median_by_hour, model_name = fig13_panel_data[(prod, signal_name)]

        im = ax.imshow(
            mat,
            aspect="auto",
            origin="lower",
            vmin=-fig13_vabs,
            vmax=fig13_vabs,
            cmap="RdBu_r"
        )

        ax.set_xticks([0, 5, 11, 17, 23])
        ax.set_xticklabels([0, 5, 11, 17, 23])
        ax.set_yticks([0])
        ax.set_yticklabels(["Median\nover days"])

        apply_plot_settings(ax, f"{prod} | {signal_name}", "Hour of day", "Aggregation")

        for h, val in enumerate(median_by_hour.values):
            fig_13_rows.append({
                "Suggested figure title": "Hour-of-day median sold-product carbon-intensity reduction for matched-signal M2 cases relative to matched M0.",
                "Product": prod,
                "Signal": signal_name,
                "Model": model_name,
                "Hour of day": h,
                "Median CI reduction vs matched M0 (%)": val,
                "Shared symmetric colorbar absmax": fig13_vabs,
            })

# one shared colorbar for the whole figure
cbar = fig.colorbar(im, ax=axes.ravel().tolist(), fraction=0.02, pad=0.02)
cbar.ax.tick_params(labelsize=9)
cbar.set_label("Median hourly CI reduction vs matched M0 (%)", fontsize=10, fontweight="bold")


savefig_png_svg(fig, DIR_FIG_MAIN / "Figure_13_hour_of_day_median_CI_reduction_heatmap")
pd.DataFrame(fig_13_rows).to_csv(DIR_FIG_MAIN / "Figure_13_source_data.csv", index=False)


# -----------------------------
# FIGURE S16 — Daily sold-product CI distributions grouped by evaluation signal
# matched presentation only: M0 + matched M2 for each signal
# -----------------------------
fig, axes = plt.subplots(3, 3, figsize=(15.8, 11.8), dpi=150, sharex=False)

signal_to_model = {
    "AEF": "M2-AEF0.5",
    "Soft MEF": "M2-Soft MEF0.5",
    "UpRamp MEF": "M2-UpRamp MEF0.5",
}

fig_s16_source_rows = []

for i, prod in enumerate(PRODUCTS):
    ci_col = f"{prod}_sold_CI_kgco2_perkg"
    for j, signal_name in enumerate(SIGNAL_ORDER):
        ax = axes[i, j]

        models_to_plot = ["M0", signal_to_model[signal_name]]

        positions = []
        boxdata = []
        labels = []

        for pos, model in enumerate(models_to_plot, start=1):
            vals = daily_all.loc[
                (daily_all["Model"] == model) &
                (daily_all["Evaluation signal"] == signal_name),
                ci_col
            ].dropna().values

            if len(vals) > 0:
                boxdata.append(vals)
                positions.append(pos)
                labels.append(model)

                for v in vals:
                    fig_s16_source_rows.append({
                        "Suggested figure title": "Daily sold-product carbon-intensity distributions grouped by evaluation signal, shown only for M0 and the matched M2 case of that signal.",
                        "Product": prod,
                        "Evaluation signal": signal_name,
                        "Model": model,
                        "Daily sold CI (kgCO2/kg)": float(v),
                    })

        if boxdata:
            bp = ax.boxplot(
                boxdata,
                positions=positions,
                widths=0.65,
                patch_artist=True,
                showfliers=False
            )
            for patch, lbl in zip(bp["boxes"], labels):
                patch.set_facecolor(MODEL_COLOR[lbl])
                patch.set_alpha(0.85)

        ax.set_xticks(positions)
        ax.set_xticklabels(labels, rotation=25, ha="right", fontsize=8)

        apply_plot_settings(
            ax,
            f"{prod} | {signal_name}",
            "Model",
            f"{prod} sold CI (kgCO2/kg)"
        )

plt.tight_layout()
savefig_png_svg(fig, DIR_FIG_SI / "Figure_S16_daily_sold_product_CI_distributions_grouped_by_signal")

fig_s16_source = pd.DataFrame(fig_s16_source_rows)
fig_s16_source.to_csv(DIR_FIG_SI / "Figure_S16_source_data.csv", index=False)

fig_s16_source.to_csv(DIR_FIG_SI / "Figure_S16_source_data.csv", index=False)

# -----------------------------
# FIGURE S17 — Relative annual sold-product CI vs M0 within signal
# all model/signal combinations retained
# -----------------------------
fig, axes = plt.subplots(1, 3, figsize=(12.8, 4.8), dpi=150)

for ax, prod in zip(axes, PRODUCTS):
    col = f"{prod} relative CI vs M0 within signal"
    sub = table_s19.copy()
    mat = pd.DataFrame(index=MODEL_ORDER, columns=SIGNAL_ORDER, dtype=float)
    for _, r in sub.iterrows():
        m = str(r["Model"])
        s = str(r["Evaluation signal"])
        if m in MODEL_ORDER and s in SIGNAL_ORDER:
            mat.loc[m, s] = float(r[col]) if pd.notna(r[col]) else np.nan

    arr = mat.to_numpy(dtype=float)
    finite = arr[np.isfinite(arr)]
    vmin = float(np.nanmin(finite)) if finite.size else 0.0
    vmax = float(np.nanmax(finite)) if finite.size else 1.0
    im = ax.imshow(arr, aspect="auto", vmin=vmin, vmax=vmax)
    ax.set_xticks(np.arange(len(SIGNAL_ORDER)))
    ax.set_yticks(np.arange(len(MODEL_ORDER)))
    ax.set_xticklabels(SIGNAL_ORDER)
    ax.set_yticklabels(MODEL_ORDER)

    for r in range(mat.shape[0]):
        for c in range(mat.shape[1]):
            val = mat.iloc[r, c]
            if pd.notna(val):
                ax.text(c, r, f"{float(val):.2f}", ha="center", va="center", fontsize=9)

    apply_plot_settings(ax, f"{prod} annual sold CI / M0", "Evaluation signal", "Model")

plt.tight_layout()
savefig_png_svg(fig, DIR_FIG_SI / "Figure_S17_relative_annual_sold_product_CI_heatmap_vs_M0_within_signal")
fig_s17_source = table_s19.copy()
fig_s17_source.insert(0, "Suggested figure title", "Annual sold-product carbon intensity normalized by M0 within each evaluation signal.")
fig_s17_source.to_csv(DIR_FIG_SI / "Figure_S17_source_data.csv", index=False)

# -----------------------------
# FIGURE S18 — Annual sold green-attributed share for matched cases
# moved from main text to SI
# -----------------------------
fig, axes = plt.subplots(1, 3, figsize=(14.5, 4.8), dpi=150)

for ax, prod in zip(axes, PRODUCTS):
    col = f"{prod} sold green share (%)"
    plot_df = matched_green.set_index("Model").reindex(MODEL_ORDER).reset_index()
    x = np.arange(len(plot_df))
    y = plot_df[col].astype(float).values
    bars = ax.bar(x, y, width=0.68, color=[MODEL_COLOR.get(m, "#7fc6d4") for m in plot_df["Model"]], alpha=0.9)
    ax.set_xticks(x)
    ax.set_xticklabels(plot_df["Model"], rotation=25, ha="right")
    apply_plot_settings(ax, prod, "Model", "Sold green share (%)")
    for b, val in zip(bars, y):
        if np.isfinite(val):
            ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.4, f"{val:.1f}", ha="center", va="bottom", fontsize=8)

plt.tight_layout()
savefig_png_svg(fig, DIR_FIG_SI / "Figure_S18_annual_sold_green_share_matched_signal")
fig_s18_source = matched_green.copy()
fig_s18_source.insert(0, "Suggested figure title", "Annual sold green-attributed share for the matched-signal cases.")
fig_s18_source.to_csv(DIR_FIG_SI / "Figure_S18_source_data.csv", index=False)

# -----------------------------
# ZIP OUTPUTS
# -----------------------------
ZIP_PATH = OUT_ROOT / "product_carbon_accounting_outputs.zip"
if ZIP_PATH.exists():
    ZIP_PATH.unlink()

with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in OUT_ROOT.rglob("*"):
        if p.is_file() and p != ZIP_PATH:
            zf.write(p, arcname=str(p.relative_to(OUT_ROOT)))

print("Done.")
print("Output root:", OUT_ROOT)
print("ZIP:", ZIP_PATH)
